In [ ]:
# -*- coding: utf-8 -*-
"""
年次（10/1）割安高質バックテスト（100株単位・税引後）に
FF5+MOMのリスク制御（C：レジーム減速＋β制約、riskoff=0.5）を統合。

【最小改修で高速化】
- prices_df を Code ごとの辞書 prices_by_code に前処理
- get_near_price() は巨大DFを毎回フィルタせず、prices_by_code[code]のみを探索

税金:
- 簡易：年次（10/1→翌10/1）の最終損益にのみ課税

出力（統合前後＋診断）:
- annual_returns_raw.csv / annual_returns_riskcontrol.csv
- cumulative_curve_raw.csv / cumulative_curve_riskcontrol.csv
- performance_summary_raw.csv / performance_summary_riskcontrol.csv
- regression_ff5mom_raw.csv / regression_ff5mom_riskcontrol.csv
- diag_invest_ratio_monthly.csv

【追加出力（既存は変更しない）】
- daily_equity_curve_raw.csv / daily_equity_curve_riskcontrol.csv
- daily_mdd_summary.csv
- daily_drawdown_events_raw.csv / daily_drawdown_events_riskcontrol.csv
- daily_drawdown_events_max_summary.csv

依存:
  pip install pandas numpy pyarrow tqdm
"""

import warnings
warnings.filterwarnings("ignore")

import os
import logging
from pathlib import Path
from typing import Dict, Tuple, List, Optional

import numpy as np
import pandas as pd
from tqdm import tqdm


# ===================================
# ロギング設定
# ===================================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler("backtest_october_unit_with_ff5mom_riskcontrol.log", encoding="utf-8"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)


# ===================================
# パス（あなたの環境）
# ===================================
FACTORS_DIR = Path(r"C:\Users\yongr\Project\merged_data_all_stocks\factors")
FF5MOM_FACTOR_PATH = FACTORS_DIR / "ff5_mom_factors_monthly.parquet"

CACHE_FILE = "topix_quarterly_statements.csv"
OHLCV_DIR = "./OHLCV_Adjusted"

OUT_DIR = FACTORS_DIR / "bt_october_unit_with_ff5mom_riskcontrol"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_ANNUAL_RAW = OUT_DIR / "annual_returns_raw.csv"
OUT_ANNUAL_RC  = OUT_DIR / "annual_returns_riskcontrol.csv"

OUT_CURVE_RAW  = OUT_DIR / "cumulative_curve_raw.csv"
OUT_CURVE_RC   = OUT_DIR / "cumulative_curve_riskcontrol.csv"

OUT_PERF_RAW   = OUT_DIR / "performance_summary_raw.csv"
OUT_PERF_RC    = OUT_DIR / "performance_summary_riskcontrol.csv"

OUT_REG_RAW    = OUT_DIR / "regression_ff5mom_raw.csv"
OUT_REG_RC     = OUT_DIR / "regression_ff5mom_riskcontrol.csv"

OUT_IR_DIAG    = OUT_DIR / "diag_invest_ratio_monthly.csv"

# --- 追加出力（日次）
OUT_DAILY_CURVE_RAW = OUT_DIR / "daily_equity_curve_raw.csv"
OUT_DAILY_CURVE_RC  = OUT_DIR / "daily_equity_curve_riskcontrol.csv"
OUT_DAILY_MDD_SUMMARY = OUT_DIR / "daily_mdd_summary.csv"

OUT_DD_EVENTS_RAW = OUT_DIR / "daily_drawdown_events_raw.csv"
OUT_DD_EVENTS_RC  = OUT_DIR / "daily_drawdown_events_riskcontrol.csv"
OUT_DD_EVENTS_MAX = OUT_DIR / "daily_drawdown_events_max_summary.csv"


# ===================================
# 税・単位株
# ===================================
TAX_RATE = 0.20315
UNIT_SHARES = 100
INITIAL_CAPITAL = 10_000_000


# ===================================
# リスク制御パラメータ（指定）
# ===================================
RISKOFF_RATIO = 0.5

REGIME_LOOKBACK_M = 3
REGIME_OFF_IF_SUM_MKT_LT = 0.0
REGIME_OFF_IF_SUM_WML_LT = 0.0

BETA_CAP_CMA_LT = -0.8
BETA_CAP_ABS_MKT_GT = 0.9

BETA_EST_WINDOW_M = 12
BETA_EST_MIN_OBS = 10

FACTOR_COLS = ["MKT", "SMB", "HML", "RMW", "CMA", "WML"]

BAD_CODE_STRINGS = {"None", "nan", "", "NaN", "NULL", "null"}


# ===================================
# ユーティリティ
# ===================================
def normalize_code(code) -> str:
    if code is None:
        return ""
    s = str(code).strip()
    if s in BAD_CODE_STRINGS:
        return ""
    return s


def ols_alpha_beta(y: np.ndarray, X: np.ndarray) -> Tuple[float, np.ndarray, float]:
    n = len(y)
    if n < 3:
        return np.nan, np.full(X.shape[1], np.nan), np.nan

    X1 = np.column_stack([np.ones(n), X])
    XtX = X1.T @ X1
    try:
        inv = np.linalg.inv(XtX)
    except np.linalg.LinAlgError:
        inv = np.linalg.pinv(XtX)
    b = inv @ (X1.T @ y)

    yhat = X1 @ b
    resid = y - yhat
    sse = float(np.sum(resid**2))
    sst = float(np.sum((y - y.mean())**2))
    r2 = np.nan if sst <= 0 else (1.0 - sse / sst)

    alpha = float(b[0])
    betas = b[1:].astype(float)
    return alpha, betas, float(r2)


def compute_drawdown(cum: pd.Series) -> pd.Series:
    peak = cum.cummax()
    return cum / peak - 1.0


def perf_stats(annual_ret: pd.Series) -> Dict:
    r = annual_ret.dropna().astype(float)
    if r.empty:
        return {"n_years": 0, "CAGR": np.nan, "ann_mean": np.nan, "ann_vol": np.nan, "sharpe0": np.nan, "maxDD": np.nan, "cum_end": np.nan}

    n = len(r)
    cum = (1.0 + r).cumprod()
    years = n
    cagr = float(cum.iloc[-1] ** (1/years) - 1.0) if years > 0 else np.nan
    ann_mean = float(r.mean())
    ann_vol = float(r.std(ddof=1)) if n >= 2 else np.nan
    sharpe0 = float(ann_mean / ann_vol) if ann_vol and ann_vol > 0 else np.nan
    dd = compute_drawdown(cum)
    maxdd = float(dd.min())

    return {
        "n_years": int(n),
        "CAGR": cagr,
        "ann_mean": ann_mean,
        "ann_vol": ann_vol,
        "sharpe0": sharpe0,
        "maxDD": maxdd,
        "cum_end": float(cum.iloc[-1]),
    }


def extract_max_drawdown_event_from_daily_curve(df_daily: pd.DataFrame, label: str = "") -> pd.DataFrame:
    if df_daily is None or df_daily.empty:
        return pd.DataFrame([{
            "label": label,
            "peak_date": pd.NaT,
            "trough_date": pd.NaT,
            "recovery_date": pd.NaT,
            "dd_min": np.nan,
            "peak_equity": np.nan,
            "trough_equity": np.nan,
            "recovery_equity": np.nan,
            "days_to_trough": np.nan,
            "days_to_recovery": np.nan,
            "dd_duration_days": np.nan,
        }])

    df = df_daily.copy()
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df = df.dropna(subset=["Date"]).sort_values("Date").reset_index(drop=True)

    for col in ["equity_total", "dd_total"]:
        if col not in df.columns:
            raise KeyError(f"daily curve missing required column: {col}")

    df["equity_total"] = pd.to_numeric(df["equity_total"], errors="coerce")
    df["dd_total"] = pd.to_numeric(df["dd_total"], errors="coerce")
    df = df.dropna(subset=["equity_total", "dd_total"]).copy()
    if df.empty:
        return pd.DataFrame([{
            "label": label,
            "peak_date": pd.NaT,
            "trough_date": pd.NaT,
            "recovery_date": pd.NaT,
            "dd_min": np.nan,
            "peak_equity": np.nan,
            "trough_equity": np.nan,
            "recovery_equity": np.nan,
            "days_to_trough": np.nan,
            "days_to_recovery": np.nan,
            "dd_duration_days": np.nan,
        }])

    trough_idx = int(df["dd_total"].idxmin())
    dd_min = float(df.loc[trough_idx, "dd_total"])
    trough_date = pd.Timestamp(df.loc[trough_idx, "Date"])
    trough_equity = float(df.loc[trough_idx, "equity_total"])

    df_pre = df.loc[:trough_idx].copy()
    peak_equity = float(df_pre["equity_total"].max())
    peak_idx = int(df_pre["equity_total"].idxmax())
    peak_date = pd.Timestamp(df.loc[peak_idx, "Date"])

    df_post = df.loc[trough_idx:].copy()
    rec = df_post[df_post["equity_total"] >= peak_equity]
    if len(rec) == 0:
        recovery_date = pd.NaT
        recovery_equity = np.nan
        days_to_recovery = np.nan
        dd_duration_days = np.nan
    else:
        rec_idx = int(rec.index[0])
        recovery_date = pd.Timestamp(df.loc[rec_idx, "Date"])
        recovery_equity = float(df.loc[rec_idx, "equity_total"])
        days_to_recovery = int((recovery_date - peak_date).days)
        dd_duration_days = int((recovery_date - peak_date).days)

    days_to_trough = int((trough_date - peak_date).days)

    return pd.DataFrame([{
        "label": label,
        "peak_date": peak_date,
        "trough_date": trough_date,
        "recovery_date": recovery_date,
        "dd_min": dd_min,
        "peak_equity": peak_equity,
        "trough_equity": trough_equity,
        "recovery_equity": recovery_equity,
        "days_to_trough": days_to_trough,
        "days_to_recovery": days_to_recovery,
        "dd_duration_days": dd_duration_days,
    }])


def daily_mdd(df_daily: pd.DataFrame) -> float:
    if df_daily is None or df_daily.empty or "dd_total" not in df_daily.columns:
        return np.nan
    s = pd.to_numeric(df_daily["dd_total"], errors="coerce").dropna()
    return float(s.min()) if len(s) else np.nan


# ===================================
# A. 財務データ読み込み（列名自動判定）
# ===================================
def load_financial_data(cache_filename: str = CACHE_FILE) -> pd.DataFrame:
    if not os.path.exists(cache_filename):
        logger.error(f"財務データファイル '{cache_filename}' が見つかりません")
        return pd.DataFrame()

    try:
        df = pd.read_csv(cache_filename, encoding="utf-8-sig", parse_dates=["DisclosedDate"], low_memory=False)
        logger.info(f"財務データ読み込み成功: {len(df):,}件")

        column_mapping = {
            "IssuedShareTotal": ["IssuedShareTotal", "NumberOfIssuedAndOutstandingSharesAtTheEndOfFiscalYearIncludingTreasuryStock"],
            "Equity": ["Equity", "NetAssets", "TotalEquity"],
            "Profit": ["Profit", "NetIncome", "ProfitAttributableToOwnersOfParent"],
        }

        available_cols = df.columns.tolist()
        required_columns = {}
        for target_col, possible_names in column_mapping.items():
            found = False
            for possible_name in possible_names:
                if possible_name in available_cols:
                    required_columns[target_col] = possible_name
                    found = True
                    break
            if not found:
                required_columns[target_col] = None

        rename_dict = {v: k for k, v in required_columns.items() if v is not None}
        df = df.rename(columns=rename_dict)

        for col in ["IssuedShareTotal", "Equity", "Profit"]:
            if col not in df.columns:
                df[col] = 0

        base_cols = ["Code", "DisclosedDate"]
        if "CompanyName" in df.columns:
            base_cols.append("CompanyName")
        final_cols = base_cols + ["Profit", "Equity", "IssuedShareTotal"]
        df = df[final_cols].copy()

        logger.info(f"使用列: {df.columns.tolist()}")
        return df

    except Exception as e:
        logger.error(f"財務データ読み込みエラー: {e}", exc_info=True)
        return pd.DataFrame()


# ===================================
# B. 株価データ読み込み
# ===================================
def load_existing_price_data(ohlcv_dir: str = OHLCV_DIR) -> pd.DataFrame:
    if not os.path.exists(ohlcv_dir):
        logger.error(f"株価データディレクトリ '{ohlcv_dir}' が見つかりません。")
        return pd.DataFrame()

    csv_files = sorted([f for f in os.listdir(ohlcv_dir)
                        if f.startswith("OHLCV_Adjusted_") and f.endswith(".csv") and f != "OHLCV_Adjusted_TOPIX.csv"])

    if not csv_files:
        logger.error(f"ディレクトリ '{ohlcv_dir}' 内にCSVファイルが見つかりません。")
        return pd.DataFrame()

    logger.info(f"株価ファイル数: {len(csv_files)}個")

    all_dataframes = []
    usecols = ["Date", "Ticker", "AdjustmentClose"]

    for csv_file in tqdm(csv_files, desc="株価ファイル読み込み中"):
        file_path = os.path.join(ohlcv_dir, csv_file)
        try:
            df = pd.read_csv(
                file_path,
                usecols=usecols,
                parse_dates=["Date"],
                dtype={"Ticker": "Int64", "AdjustmentClose": "float32"},
            )
            df = df.drop_duplicates(subset=["Ticker", "Date"], keep="first")
            all_dataframes.append(df)
        except Exception as e:
            logger.warning(f"ファイル読み込みエラー ({csv_file}): {e}")
            continue

    if not all_dataframes:
        return pd.DataFrame()

    logger.info("全ファイルを結合中...")
    df_all = pd.concat(all_dataframes, ignore_index=True)
    logger.info(f"結合完了: {len(df_all):,}件")

    df_all = df_all.rename(columns={"Ticker": "Code", "AdjustmentClose": "Close"})
    df_all["Code"] = df_all["Code"].astype("str").str.replace("<NA>", "0").str.zfill(4)
    df_all = df_all.dropna(subset=["Close"])

    logger.info("重複除去 & ソート中...")
    df_all = df_all.sort_values(["Code", "Date"])
    df_all = df_all.drop_duplicates(subset=["Code", "Date"], keep="first")
    logger.info(f"重複除去後: {len(df_all):,}件")

    return df_all


def build_prices_by_code(prices_df: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    d = {}
    tmp = prices_df[["Code", "Date", "Close"]].copy()
    tmp["Code"] = tmp["Code"].astype(str).map(normalize_code)
    tmp = tmp[tmp["Code"] != ""].copy()
    tmp = tmp.sort_values(["Code", "Date"])

    for code, g in tmp.groupby("Code", sort=False):
        gg = g[["Date", "Close"]].drop_duplicates(subset=["Date"], keep="last").sort_values("Date").copy()
        gg = gg.set_index("Date", drop=False)
        d[code] = gg

    logger.info(f"prices_by_code built: {len(d):,} codes")
    return d


def get_near_price_fast(prices_by_code: Dict[str, pd.DataFrame],
                        code: str,
                        ref_date: pd.Timestamp,
                        kind: str = "last") -> Optional[float]:
    code = normalize_code(code)
    if not code or code not in prices_by_code:
        return None
    g = prices_by_code[code]
    lo = ref_date - pd.Timedelta(days=5)
    hi = ref_date + pd.Timedelta(days=5)
    w = g.loc[(g["Date"] >= lo) & (g["Date"] <= hi)]
    if w.empty:
        return None
    return float(w.iloc[0]["Close"]) if kind == "first" else float(w.iloc[-1]["Close"])


def safe_code_to_int(code_series: pd.Series) -> pd.Series:
    cleaned = code_series.astype(str).str.replace(r"\D", "", regex=True)
    cleaned = cleaned.replace("", "0")
    return pd.to_numeric(cleaned, errors="coerce").fillna(0).astype("int64")


def calculate_market_metrics_fast_chunked(statements_df: pd.DataFrame,
                                          prices_df: pd.DataFrame,
                                          chunk_size: int = 200) -> pd.DataFrame:
    logger.info("時価総額・PBR・ROE計算中（チャンク処理版）...")

    if prices_df.empty:
        logger.error("株価データが空です。")
        return pd.DataFrame()

    req_cols = {"Code", "Date", "Close"}
    if not req_cols.issubset(set(prices_df.columns)):
        logger.error(f"株価データに必要な列が存在しません。存在する列: {prices_df.columns.tolist()}")
        return pd.DataFrame()

    statements_df = statements_df.copy()
    statements_df["Profit"] = pd.to_numeric(statements_df["Profit"], errors="coerce").fillna(0)
    statements_df["Equity"] = pd.to_numeric(statements_df["Equity"], errors="coerce").fillna(0)
    statements_df["IssuedShareTotal"] = pd.to_numeric(statements_df["IssuedShareTotal"], errors="coerce").fillna(1)

    statements_df = statements_df[(statements_df["Equity"] > 0) & (statements_df["IssuedShareTotal"] > 0)]
    logger.info(f"有効な財務データ: {len(statements_df):,}件")

    statements_df["Code_int"] = safe_code_to_int(statements_df["Code"])
    prices_df = prices_df.copy()
    prices_df["Code_int"] = safe_code_to_int(prices_df["Code"])

    statements_df = statements_df[statements_df["Code_int"] > 0]
    prices_df = prices_df[prices_df["Code_int"] > 0]

    statements_df = statements_df.sort_values(["Code_int", "DisclosedDate"]).reset_index(drop=True)
    prices_df = prices_df.sort_values(["Code_int", "Date"]).reset_index(drop=True)

    statements_df = statements_df.drop_duplicates(subset=["Code_int", "DisclosedDate"], keep="first")
    prices_df = prices_df.drop_duplicates(subset=["Code_int", "Date"], keep="first")

    logger.info(f"ソート・重複除去後: 財務 {len(statements_df):,}件, 株価 {len(prices_df):,}件")

    statements_groups = list(statements_df.groupby("Code_int"))
    prices_dict = {code: group for code, group in prices_df.groupby("Code_int")}

    merged_list = []
    num_chunks = (len(statements_groups) + chunk_size - 1) // chunk_size

    for chunk_idx in tqdm(range(num_chunks), desc="マージ処理"):
        start_idx = chunk_idx * chunk_size
        end_idx = min((chunk_idx + 1) * chunk_size, len(statements_groups))
        chunk_groups = statements_groups[start_idx:end_idx]

        for code, stmt_code in chunk_groups:
            if code not in prices_dict:
                continue
            price_code = prices_dict[code]
            if len(price_code) == 0:
                continue

            stmt_code = stmt_code.sort_values("DisclosedDate").reset_index(drop=True)
            price_code = price_code.sort_values("Date").reset_index(drop=True)

            try:
                merged = pd.merge_asof(
                    stmt_code,
                    price_code[["Date", "Close"]],
                    left_on="DisclosedDate",
                    right_on="Date",
                    direction="backward",
                    tolerance=pd.Timedelta(days=10),
                )
                if not merged.empty:
                    merged_list.append(merged)
            except Exception:
                continue

    if not merged_list:
        logger.error("マージ結果が空です")
        return pd.DataFrame()

    df_merged = pd.concat(merged_list, ignore_index=True)
    df_merged = df_merged.dropna(subset=["Close"])
    logger.info(f"マージ完了: {len(df_merged):,}件")

    df_merged["MarketCap"] = df_merged["Close"] * df_merged["IssuedShareTotal"]
    df_merged["PBR"] = df_merged["MarketCap"] / df_merged["Equity"]
    df_merged["ROE"] = (df_merged["Profit"] / df_merged["Equity"]) * 100

    result_cols = ["Code", "DisclosedDate", "Close", "MarketCap", "PBR", "ROE", "Date"]
    if "CompanyName" in df_merged.columns:
        result_cols.insert(1, "CompanyName")

    result_df = df_merged[result_cols].copy()
    result_df = result_df.rename(columns={"Close": "StockPrice", "Date": "PriceDate"})

    mask = (
        (result_df["PBR"] > 0) &
        (result_df["PBR"] < 50) &
        (result_df["ROE"] > -100) &
        (result_df["ROE"] < 100) &
        (result_df["MarketCap"] > 1_000_000_000)
    )
    result_df = result_df[mask].copy()

    logger.info(f"計算完了: {len(result_df):,}件")
    return result_df


def build_unit_share_portfolio(stock_candidates: pd.DataFrame,
                               target_positions: int = 20,
                               initial_capital: float = 10_000_000) -> dict:
    if len(stock_candidates) == 0:
        return {"stocks": [], "shares": [], "prices": [], "amounts": []}

    selected = stock_candidates.head(target_positions).copy()
    capital_per_stock = initial_capital / len(selected)

    stocks, shares_list, prices_list, amounts_list = [], [], [], []

    for _, row in selected.iterrows():
        code = row["Code"]
        price = row["StockPrice"]
        required_amount = price * UNIT_SHARES

        if required_amount <= capital_per_stock:
            shares = int(capital_per_stock // required_amount) * UNIT_SHARES
            if shares > 0:
                stocks.append(code)
                shares_list.append(shares)
                prices_list.append(price)
                amounts_list.append(shares * price)

    return {"stocks": stocks, "shares": shares_list, "prices": prices_list, "amounts": amounts_list}


def make_month_ends(start: pd.Timestamp, end: pd.Timestamp) -> List[pd.Timestamp]:
    m0 = pd.Timestamp(start.year, start.month, 1) + pd.offsets.MonthEnd(0)
    m1 = pd.Timestamp(end.year, end.month, 1) + pd.offsets.MonthEnd(0)
    months = pd.date_range(m0, m1, freq="M")
    return [pd.Timestamp(x).normalize() for x in months]


def build_daily_equity_curve_for_period(
    portfolio: dict,
    start_date: pd.Timestamp,
    end_date: pd.Timestamp,
    prices_by_code: Dict[str, pd.DataFrame],
    invest_ratio_by_monthend: Optional[Dict[pd.Timestamp, float]] = None,
) -> pd.DataFrame:
    stocks = portfolio.get("stocks", [])
    shares = portfolio.get("shares", [])
    if not stocks:
        return pd.DataFrame()

    start_date = pd.to_datetime(start_date).normalize()
    end_date = pd.to_datetime(end_date).normalize()

    date_sets = []
    for code in stocks:
        code = normalize_code(code)
        if code in prices_by_code:
            g = prices_by_code[code]
            d = g[(g["Date"] >= start_date - pd.Timedelta(days=10)) & (g["Date"] <= end_date + pd.Timedelta(days=10))]["Date"]
            if len(d):
                date_sets.append(d)

    if not date_sets:
        return pd.DataFrame()

    dates = pd.Index(sorted(pd.unique(pd.concat(date_sets)))).astype("datetime64[ns]")
    dates = dates[(dates >= start_date) & (dates <= end_date)]
    if len(dates) == 0:
        return pd.DataFrame()

    values = []
    for dt in dates:
        v = 0.0
        ok = False
        for i, code in enumerate(stocks):
            code = normalize_code(code)
            if code not in prices_by_code:
                continue
            p = get_near_price_fast(prices_by_code, code, pd.Timestamp(dt), kind="last")
            if p is None:
                continue
            v += float(shares[i]) * float(p)
            ok = True
        values.append(v if ok else np.nan)

    df = pd.DataFrame({"Date": pd.to_datetime(dates), "equity_stock": values})
    df = df.dropna(subset=["equity_stock"]).copy()
    if df.empty:
        return df

    df = df.sort_values("Date").reset_index(drop=True)
    df["ret_stock"] = df["equity_stock"].pct_change().fillna(0.0)

    df["MonthEnd"] = (df["Date"] + pd.offsets.MonthEnd(0)).dt.normalize()

    if invest_ratio_by_monthend is None:
        df["invest_ratio"] = 1.0
        df["ret_total"] = df["ret_stock"]
    else:
        df["invest_ratio"] = df["MonthEnd"].map(invest_ratio_by_monthend).fillna(1.0).astype(float)
        df["ret_total"] = df["invest_ratio"] * df["ret_stock"]

    df["equity_total"] = (1.0 + df["ret_total"]).cumprod()
    peak = df["equity_total"].cummax()
    df["dd_total"] = df["equity_total"] / peak - 1.0

    return df[["Date", "MonthEnd", "equity_stock", "ret_stock", "invest_ratio", "ret_total", "equity_total", "dd_total"]]


def load_ff5mom_factors_monthly() -> pd.DataFrame:
    fac = pd.read_parquet(FF5MOM_FACTOR_PATH).copy()
    fac["MonthEnd"] = pd.to_datetime(fac["MonthEnd"], errors="coerce").dt.normalize()
    need = ["MonthEnd"] + FACTOR_COLS
    missing = [c for c in need if c not in fac.columns]
    if missing:
        raise KeyError(f"FF5MOM factors missing columns: {missing} in {FF5MOM_FACTOR_PATH}")
    fac = fac[need].sort_values("MonthEnd").reset_index(drop=True)
    return fac


def compute_regime_off(fac: pd.DataFrame, month_end: pd.Timestamp) -> Tuple[bool, Dict]:
    fac2 = fac.set_index("MonthEnd")
    idx = fac2.index[fac2.index < month_end]
    if len(idx) < REGIME_LOOKBACK_M:
        return False, {"regime_ready": False}

    win = idx[-REGIME_LOOKBACK_M:]
    s_mkt = float(pd.to_numeric(fac2.loc[win, "MKT"], errors="coerce").sum())
    s_wml = float(pd.to_numeric(fac2.loc[win, "WML"], errors="coerce").sum())

    # ===== PATCH START (Option 1): OR -> AND =====
    off = (s_mkt < REGIME_OFF_IF_SUM_MKT_LT) and (s_wml < REGIME_OFF_IF_SUM_WML_LT)
    # ===== PATCH END =====

    return bool(off), {"regime_ready": True, "sum_MKT_3m": s_mkt, "sum_WML_3m": s_wml}


def estimate_port_beta(monthly_port_rets: pd.DataFrame, fac: pd.DataFrame, month_end: pd.Timestamp) -> Tuple[bool, Dict]:
    fac2 = fac.set_index("MonthEnd")
    idx = fac2.index[fac2.index < month_end]
    if len(idx) < BETA_EST_WINDOW_M:
        return False, {"beta_ready": False}

    win = idx[-BETA_EST_WINDOW_M:]

    m = monthly_port_rets.copy()
    m["MonthEnd"] = pd.to_datetime(m["MonthEnd"], errors="coerce").dt.normalize()
    m = m.dropna(subset=["MonthEnd", "port_ret"]).copy()

    # MonthEnd重複を1行化（merge validate="one_to_one"維持）
    m = (
        m.groupby("MonthEnd", as_index=False)["port_ret"]
         .apply(lambda s: (1.0 + s.astype(float)).prod() - 1.0)
    )

    df = m[m["MonthEnd"].isin(win)].merge(
        fac, on="MonthEnd", how="left", validate="one_to_one"
    ).dropna(subset=["port_ret"] + FACTOR_COLS)

    if len(df) < BETA_EST_MIN_OBS:
        return False, {"beta_ready": False, "n_obs": int(len(df))}

    y = df["port_ret"].to_numpy(dtype=float)
    X = df[FACTOR_COLS].to_numpy(dtype=float)
    alpha, betas, r2 = ols_alpha_beta(y, X)

    out = {"beta_ready": True, "n_obs": int(len(df)), "r2": float(r2), "alpha": float(alpha)}
    for c, b in zip(FACTOR_COLS, betas):
        out[f"beta_{c}"] = float(b)
    return True, out


def decide_invest_ratio(month_end: pd.Timestamp,
                        fac: pd.DataFrame,
                        port_hist: pd.DataFrame) -> Tuple[float, Dict]:
    invest_ratio = 1.0
    diag = {
        "MonthEnd": month_end,
        "invest_ratio": 1.0,
        "regime_off": False,
        "beta_off": False,
        "regime_ready": False,
        "beta_ready": False,
        "sum_MKT_3m": np.nan,
        "sum_WML_3m": np.nan,
        "beta_MKT": np.nan,
        "beta_CMA": np.nan,
        "beta_n_obs": np.nan,
        "beta_r2": np.nan,
    }

    off_reg, info = compute_regime_off(fac, month_end)
    diag["regime_off"] = bool(off_reg)
    diag["regime_ready"] = bool(info.get("regime_ready", False))
    diag["sum_MKT_3m"] = info.get("sum_MKT_3m", np.nan)
    diag["sum_WML_3m"] = info.get("sum_WML_3m", np.nan)
    if off_reg:
        invest_ratio = min(invest_ratio, RISKOFF_RATIO)

    ok_beta, binfo = estimate_port_beta(port_hist, fac, month_end)
    diag["beta_ready"] = bool(binfo.get("beta_ready", False))
    if binfo.get("beta_ready", False):
        b_mkt = float(binfo.get("beta_MKT", np.nan))
        b_cma = float(binfo.get("beta_CMA", np.nan))
        diag["beta_MKT"] = b_mkt
        diag["beta_CMA"] = b_cma
        diag["beta_n_obs"] = float(binfo.get("n_obs", np.nan))
        diag["beta_r2"] = float(binfo.get("r2", np.nan))

        off_beta = (b_cma < BETA_CAP_CMA_LT) or (abs(b_mkt) > BETA_CAP_ABS_MKT_GT)
        diag["beta_off"] = bool(off_beta)
        if off_beta:
            invest_ratio = min(invest_ratio, RISKOFF_RATIO)

    diag["invest_ratio"] = float(invest_ratio)
    return float(invest_ratio), diag


def compute_monthly_portfolio_returns_with_riskcontrol_fast(
    portfolio: dict,
    start_date: pd.Timestamp,
    end_date: pd.Timestamp,
    prices_by_code: Dict[str, pd.DataFrame],
    fac: pd.DataFrame
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    stocks = portfolio["stocks"]
    shares = portfolio["shares"]
    if not stocks:
        return pd.DataFrame(), pd.DataFrame()

    month_ends = make_month_ends(start_date, end_date)
    boundaries = [start_date] + [me for me in month_ends if (me > start_date) and (me < end_date)] + [end_date]

    rows = []
    ir_rows = []

    port_hist = pd.DataFrame(columns=["MonthEnd", "port_ret"])

    for j in range(len(boundaries) - 1):
        d0 = boundaries[j]
        d1 = boundaries[j + 1]

        me = pd.Timestamp(d0.year, d0.month, 1) + pd.offsets.MonthEnd(0)
        me = pd.Timestamp(me).normalize()

        start_vals = []
        end_vals = []
        for i, code in enumerate(stocks):
            p0 = get_near_price_fast(prices_by_code, code, d0, kind="first")
            p1 = get_near_price_fast(prices_by_code, code, d1, kind="last")
            if p0 is None or p1 is None:
                continue
            sh = shares[i]
            start_vals.append(sh * p0)
            end_vals.append(sh * p1)

        if len(start_vals) == 0:
            continue

        start_v = float(np.sum(start_vals))
        end_v = float(np.sum(end_vals))
        stock_ret = (end_v / start_v) - 1.0

        port_hist = pd.concat([port_hist, pd.DataFrame([{"MonthEnd": me, "port_ret": stock_ret}])], ignore_index=True)

        invest_ratio, diag = decide_invest_ratio(me, fac, port_hist)
        ir_rows.append(diag)

        total_ret = invest_ratio * stock_ret

        rows.append({
            "MonthEnd": me,
            "period_start": d0,
            "period_end": d1,
            "port_ret_stock": stock_ret,
            "invest_ratio": invest_ratio,
            "port_ret_total": total_ret,
        })

    monthly_df = pd.DataFrame(rows)
    ir_diag_df = pd.DataFrame(ir_rows)
    return monthly_df, ir_diag_df


def build_long_candidates(enhanced_financial_data: pd.DataFrame, rebalance_date: pd.Timestamp) -> pd.DataFrame:
    current_data = enhanced_financial_data[enhanced_financial_data["DisclosedDate"] <= rebalance_date].copy()
    current_data = current_data.sort_values("DisclosedDate").groupby("Code").tail(1)

    if len(current_data) < 100:
        return pd.DataFrame()

    current_data["PBR_Rank"] = current_data["PBR"].rank(method="first", ascending=True)
    current_data["ROE_Rank"] = current_data["ROE"].rank(method="first", ascending=False)

    current_data["PBR_Quartile"] = pd.qcut(current_data["PBR_Rank"], q=4, labels=[1, 2, 3, 4])
    current_data["ROE_Quartile"] = pd.qcut(current_data["ROE_Rank"], q=4, labels=[1, 2, 3, 4])

    long_candidates = current_data[
        (current_data["PBR_Quartile"] == 1) &
        (current_data["ROE_Quartile"] == 4)
    ].nsmallest(50, "PBR")

    return long_candidates


def annual_return_from_monthly(monthly_rets: pd.Series) -> float:
    if monthly_rets.empty:
        return 0.0
    return float((1.0 + monthly_rets).prod() - 1.0)


def apply_tax_annual(gross_return: float, initial_capital: float) -> Tuple[float, float]:
    profit = gross_return * initial_capital
    taxable = max(profit, 0.0)
    tax = taxable * TAX_RATE
    net_profit = profit - tax
    net_return = net_profit / initial_capital
    tax_rate_total = tax / initial_capital
    return float(net_return), float(tax_rate_total)


def run_annual_backtest_with_and_without_riskcontrol_fast(
    enhanced_financial_data: pd.DataFrame,
    prices_df: pd.DataFrame,
    prices_by_code: Dict[str, pd.DataFrame],
    fac: pd.DataFrame,
    initial_capital: float = INITIAL_CAPITAL
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    rebalance_dates = [pd.Timestamp(f"{year}-10-01") for year in range(2016, 2026)]

    results_raw = []
    results_rc = []
    diag_ir_all = []

    daily_curves_raw = []
    daily_curves_rc  = []

    for i, rebalance_date in enumerate(tqdm(rebalance_dates[:-1], desc="10月1日リバランス（統合・高速）")):
        next_rebalance = rebalance_dates[i + 1]

        long_candidates = build_long_candidates(enhanced_financial_data, rebalance_date)
        if long_candidates.empty:
            logger.warning(f"{rebalance_date}: データ不足でスキップ")
            continue

        long_portfolio = build_unit_share_portfolio(
            long_candidates,
            target_positions=20,
            initial_capital=initial_capital
        )

        n_long = len(long_portfolio["stocks"])
        inv_amount = float(np.sum(long_portfolio["amounts"])) if n_long > 0 else 0.0
        inv_ratio_raw = inv_amount / initial_capital if initial_capital > 0 else 0.0

        logger.info(f"{rebalance_date.strftime('%Y-%m')}: long={n_long} invest={inv_amount:,.0f} ratio={inv_ratio_raw:.3f}")

        # --- RAW（年次 start->end の価格で計算）
        total_profit = 0.0
        total_start_value = 0.0

        for j, code in enumerate(long_portfolio["stocks"]):
            sh = long_portfolio["shares"][j]
            p0 = long_portfolio["prices"][j]
            p1 = get_near_price_fast(prices_by_code, code, next_rebalance, kind="last")
            if p1 is None:
                continue
            start_value = sh * p0
            end_value = sh * p1
            total_start_value += start_value
            total_profit += (end_value - start_value)

        gross_return_raw = (total_profit / initial_capital) if initial_capital > 0 else 0.0
        net_return_raw, tax_rate_raw = apply_tax_annual(gross_return_raw, initial_capital)

        results_raw.append({
            "date": next_rebalance,
            "strategy_return_gross": gross_return_raw,
            "strategy_return_net": net_return_raw,
            "tax": tax_rate_raw,
            "long_count": n_long,
            "investment_ratio": inv_ratio_raw,
        })

        # --- RiskControl（月次で縮尺）
        monthly_df, ir_df = compute_monthly_portfolio_returns_with_riskcontrol_fast(
            long_portfolio, rebalance_date, next_rebalance, prices_by_code, fac
        )

        if monthly_df.empty:
            gross_return_rc = 0.0
            inv_ratio_rc_avg = 0.0
        else:
            gross_return_rc = annual_return_from_monthly(monthly_df["port_ret_total"])
            inv_ratio_rc_avg = float(monthly_df["invest_ratio"].mean())

        net_return_rc, tax_rate_rc = apply_tax_annual(gross_return_rc, initial_capital)

        results_rc.append({
            "date": next_rebalance,
            "strategy_return_gross": gross_return_rc,
            "strategy_return_net": net_return_rc,
            "tax": tax_rate_rc,
            "long_count": n_long,
            "investment_ratio": inv_ratio_rc_avg,
        })

        if not ir_df.empty:
            ir_df = ir_df.copy()
            ir_df["rebalance_start"] = rebalance_date
            ir_df["rebalance_end"] = next_rebalance
            diag_ir_all.append(ir_df)

        # --- 日次曲線（RAW / RC）を追加生成
        invest_ratio_map = None
        if not ir_df.empty:
            tmp_ir = ir_df.copy()
            tmp_ir["MonthEnd"] = pd.to_datetime(tmp_ir["MonthEnd"], errors="coerce").dt.normalize()
            tmp_ir = tmp_ir.dropna(subset=["MonthEnd"])
            tmp_ir = tmp_ir.sort_values("MonthEnd").drop_duplicates(subset=["MonthEnd"], keep="last")
            invest_ratio_map = dict(zip(tmp_ir["MonthEnd"], tmp_ir["invest_ratio"].astype(float)))

        daily_raw = build_daily_equity_curve_for_period(
            long_portfolio, rebalance_date, next_rebalance, prices_by_code, invest_ratio_by_monthend=None
        )
        if not daily_raw.empty:
            daily_raw["rebalance_start"] = rebalance_date
            daily_raw["rebalance_end"] = next_rebalance
            daily_curves_raw.append(daily_raw)

        daily_rc = build_daily_equity_curve_for_period(
            long_portfolio, rebalance_date, next_rebalance, prices_by_code, invest_ratio_by_monthend=invest_ratio_map
        )
        if not daily_rc.empty:
            daily_rc["rebalance_start"] = rebalance_date
            daily_rc["rebalance_end"] = next_rebalance
            daily_curves_rc.append(daily_rc)

    df_raw = pd.DataFrame(results_raw)
    df_rc = pd.DataFrame(results_rc)
    diag_ir = pd.concat(diag_ir_all, ignore_index=True) if len(diag_ir_all) else pd.DataFrame()

    daily_raw_all = pd.concat(daily_curves_raw, ignore_index=True) if len(daily_curves_raw) else pd.DataFrame()
    daily_rc_all  = pd.concat(daily_curves_rc,  ignore_index=True) if len(daily_curves_rc)  else pd.DataFrame()

    run_annual_backtest_with_and_without_riskcontrol_fast._daily_raw_all = daily_raw_all
    run_annual_backtest_with_and_without_riskcontrol_fast._daily_rc_all = daily_rc_all

    return df_raw, df_rc, diag_ir


def build_annual_factor_from_monthly(fac: pd.DataFrame, start_date: pd.Timestamp, end_date: pd.Timestamp) -> Dict:
    fac2 = fac.copy()
    fac2 = fac2[(fac2["MonthEnd"] >= (start_date + pd.offsets.MonthEnd(0))) &
                (fac2["MonthEnd"] <= (end_date + pd.offsets.MonthEnd(-1)))].copy()
    out = {}
    for c in FACTOR_COLS:
        s = pd.to_numeric(fac2[c], errors="coerce").dropna()
        out[c] = float((1.0 + s).prod() - 1.0) if len(s) else np.nan
    return out


def run_factor_regression(results_df: pd.DataFrame, fac: pd.DataFrame) -> pd.DataFrame:
    if results_df.empty:
        return pd.DataFrame()

    rows = []
    for _, row in results_df.iterrows():
        end_date = pd.Timestamp(row["date"])
        start_date = end_date - pd.DateOffset(years=1)
        ann_fac = build_annual_factor_from_monthly(fac, start_date, end_date)
        rec = {"date": end_date}
        rec.update(ann_fac)
        rec["y"] = float(row["strategy_return_net"])
        rows.append(rec)

    df = pd.DataFrame(rows).dropna(subset=["y"] + FACTOR_COLS).copy()
    if len(df) < 3:
        return pd.DataFrame([{
            "n_years": int(len(df)),
            "R2": np.nan,
            "alpha": np.nan,
            **{f"beta_{c}": np.nan for c in FACTOR_COLS}
        }])

    y = df["y"].to_numpy(dtype=float)
    X = df[FACTOR_COLS].to_numpy(dtype=float)
    alpha, betas, r2 = ols_alpha_beta(y, X)

    out = {"n_years": int(len(df)), "R2": float(r2), "alpha": float(alpha)}
    for c, b in zip(FACTOR_COLS, betas):
        out[f"beta_{c}"] = float(b)
    return pd.DataFrame([out])


def save_annual_bundle(df: pd.DataFrame, out_annual_csv: Path, out_curve_csv: Path, out_perf_csv: Path):
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").reset_index(drop=True)

    df.to_csv(out_annual_csv, index=False, encoding="utf-8-sig")

    r = df["strategy_return_net"].astype(float)
    cum = (1.0 + r).cumprod()
    dd = compute_drawdown(cum)
    curve = pd.DataFrame({"date": df["date"], "ret": r, "cum": cum, "dd": dd})
    curve.to_csv(out_curve_csv, index=False, encoding="utf-8-sig")

    stats = perf_stats(r)
    perf = pd.DataFrame([stats])
    perf.to_csv(out_perf_csv, index=False, encoding="utf-8-sig")


def main():
    logger.info("=" * 110)
    logger.info("10/1 年次「割安高質」バックテスト + FF5+MOM RiskControl(C) [FAST get_near_price]")
    logger.info("=" * 110)
    logger.info(f"FF5MOM_FACTOR_PATH: {FF5MOM_FACTOR_PATH}")
    logger.info(f"OUT_DIR: {OUT_DIR}")
    logger.info(f"Tax mode: simple annual tax only (TAX_RATE={TAX_RATE})")
    logger.info(f"RiskControl: regime(3m MKT<0 AND WML<0) + beta(CMA<-0.8 or |MKT|>0.9) => riskoff={RISKOFF_RATIO}  [Option1]")
    logger.info(f"Beta estimation window: {BETA_EST_WINDOW_M} months (min_obs={BETA_EST_MIN_OBS})")

    statements_df = load_financial_data()
    if statements_df.empty:
        logger.error("財務データ読み込み失敗")
        return

    prices_df = load_existing_price_data()
    if prices_df.empty:
        logger.error("株価データ読み込み失敗")
        return

    fac = load_ff5mom_factors_monthly()
    prices_by_code = build_prices_by_code(prices_df)

    enhanced_financial_data = calculate_market_metrics_fast_chunked(statements_df, prices_df, chunk_size=200)
    if enhanced_financial_data.empty:
        logger.error("財務指標計算失敗")
        return

    df_raw, df_rc, diag_ir = run_annual_backtest_with_and_without_riskcontrol_fast(
        enhanced_financial_data, prices_df, prices_by_code, fac, initial_capital=INITIAL_CAPITAL
    )

    if df_raw.empty or df_rc.empty:
        logger.error("年次バックテスト結果が空です")
        return

    save_annual_bundle(df_raw, OUT_ANNUAL_RAW, OUT_CURVE_RAW, OUT_PERF_RAW)
    save_annual_bundle(df_rc,  OUT_ANNUAL_RC,  OUT_CURVE_RC,  OUT_PERF_RC)

    reg_raw = run_factor_regression(df_raw, fac)
    reg_rc = run_factor_regression(df_rc, fac)
    reg_raw.to_csv(OUT_REG_RAW, index=False, encoding="utf-8-sig")
    reg_rc.to_csv(OUT_REG_RC, index=False, encoding="utf-8-sig")

    if not diag_ir.empty:
        diag_ir["MonthEnd"] = pd.to_datetime(diag_ir["MonthEnd"])
        diag_ir.sort_values(["rebalance_start", "MonthEnd"], inplace=True)
        diag_ir.to_csv(OUT_IR_DIAG, index=False, encoding="utf-8-sig")

    # ==============================
    # 追加：日次曲線保存 + 日次MDD + 最大DDイベント抽出（ログ出力まで）
    # ==============================
    daily_raw_all = getattr(run_annual_backtest_with_and_without_riskcontrol_fast, "_daily_raw_all", pd.DataFrame())
    daily_rc_all  = getattr(run_annual_backtest_with_and_without_riskcontrol_fast, "_daily_rc_all", pd.DataFrame())

    if not daily_raw_all.empty:
        daily_raw_all.to_csv(OUT_DAILY_CURVE_RAW, index=False, encoding="utf-8-sig")
    if not daily_rc_all.empty:
        daily_rc_all.to_csv(OUT_DAILY_CURVE_RC, index=False, encoding="utf-8-sig")

    mdd_raw_d = daily_mdd(daily_raw_all)
    mdd_rc_d  = daily_mdd(daily_rc_all)

    pd.DataFrame([{
        "daily_maxDD_raw": mdd_raw_d,
        "daily_maxDD_riskcontrol": mdd_rc_d,
        "n_days_raw": int(len(daily_raw_all)) if not daily_raw_all.empty else 0,
        "n_days_riskcontrol": int(len(daily_rc_all)) if not daily_rc_all.empty else 0,
    }]).to_csv(OUT_DAILY_MDD_SUMMARY, index=False, encoding="utf-8-sig")

    ev_raw = extract_max_drawdown_event_from_daily_curve(daily_raw_all, label="RAW")
    ev_rc  = extract_max_drawdown_event_from_daily_curve(daily_rc_all,  label="RISKCONTROL")

    ev_raw.to_csv(OUT_DD_EVENTS_RAW, index=False, encoding="utf-8-sig")
    ev_rc.to_csv(OUT_DD_EVENTS_RC, index=False, encoding="utf-8-sig")
    pd.concat([ev_raw, ev_rc], ignore_index=True).to_csv(OUT_DD_EVENTS_MAX, index=False, encoding="utf-8-sig")

    logger.info("-" * 110)
    logger.info("✅ SAVED (RAW)")
    logger.info(f"  - {OUT_ANNUAL_RAW}")
    logger.info(f"  - {OUT_CURVE_RAW}")
    logger.info(f"  - {OUT_PERF_RAW}")
    logger.info(f"  - {OUT_REG_RAW}")
    logger.info("✅ SAVED (RISKCONTROL)")
    logger.info(f"  - {OUT_ANNUAL_RC}")
    logger.info(f"  - {OUT_CURVE_RC}")
    logger.info(f"  - {OUT_PERF_RC}")
    logger.info(f"  - {OUT_REG_RC}")
    logger.info("✅ DIAGNOSTIC")
    logger.info(f"  - {OUT_IR_DIAG}")
    logger.info("✅ DAILY (ADDED)")
    logger.info(f"  - {OUT_DAILY_CURVE_RAW}")
    logger.info(f"  - {OUT_DAILY_CURVE_RC}")
    logger.info(f"  - {OUT_DAILY_MDD_SUMMARY}")
    logger.info("✅ DAILY DD EVENTS (ADDED)")
    logger.info(f"  - {OUT_DD_EVENTS_RAW}")
    logger.info(f"  - {OUT_DD_EVENTS_RC}")
    logger.info(f"  - {OUT_DD_EVENTS_MAX}")
    logger.info("-" * 110)

    raw_perf = pd.read_csv(OUT_PERF_RAW).iloc[0].to_dict()
    rc_perf  = pd.read_csv(OUT_PERF_RC).iloc[0].to_dict()
    logger.info("[PERF RAW] " + " | ".join([f"{k}={raw_perf.get(k)}" for k in ["n_years","CAGR","ann_mean","ann_vol","sharpe0","maxDD","cum_end"]]))
    logger.info("[PERF RC ] " + " | ".join([f"{k}={rc_perf.get(k)}"  for k in ["n_years","CAGR","ann_mean","ann_vol","sharpe0","maxDD","cum_end"]]))

    logger.info(f"[DAILY MDD] RAW={mdd_raw_d:.6f} | RC={mdd_rc_d:.6f}")
    logger.info("[DAILY MAX DD EVENT RAW] " + " | ".join([f"{k}={ev_raw.iloc[0][k]}" for k in ["peak_date","trough_date","recovery_date","dd_min"]]))
    logger.info("[DAILY MAX DD EVENT  RC] " + " | ".join([f"{k}={ev_rc.iloc[0][k]}"  for k in ["peak_date","trough_date","recovery_date","dd_min"]]))

if __name__ == "__main__":
    main()


In [1]:
# -*- coding: utf-8 -*-
"""
年次（10/1）割安高質バックテスト（100株単位・税引後）に
FF5+MOMのリスク制御（C：レジーム減速＋β制約）を統合。

【追加：日次DDストップ・グリッド実験】
- RISKOFF_RATIO = 0.7 固定
- 月次RiskControl（regime/beta）で出るinvest_ratio（月次）をベースに日次カーブを作成
- その日次カーブ上の dd_total を見て、日次で invest_ratio を強制的に下げる（0までOK）
- DD閾値をグリッドで比較し、日次MDD / CAGR / days_to_recovery を横並び出力

税金:
- 年次のみ簡易課税（10/1→翌10/1の最終損益にのみ課税）
- 日次は税無視（実験目的）

出力:
- ケース別サブフォルダ（従来CSV一式 + ddstop版のCSVも追加）
- grid_summary.csv（横並び）

依存:
  pip install pandas numpy pyarrow tqdm
"""

import warnings
warnings.filterwarnings("ignore")

import os
import logging
from pathlib import Path
from typing import Dict, Tuple, List, Optional

import numpy as np
import pandas as pd
from tqdm import tqdm


# ===================================
# ロギング設定
# ===================================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler("backtest_october_unit_with_ff5mom_ddstop_grid.log", encoding="utf-8"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)


# ===================================
# パス（あなたの環境）
# ===================================
FACTORS_DIR = Path(r"C:\Users\yongr\Project\merged_data_all_stocks\factors")
FF5MOM_FACTOR_PATH = FACTORS_DIR / "ff5_mom_factors_monthly.parquet"

CACHE_FILE = "topix_quarterly_statements.csv"
OHLCV_DIR = "./OHLCV_Adjusted"


# ===================================
# 出力（グリッド用ベース）
# ===================================
BASE_OUT_DIR = FACTORS_DIR / "bt_october_unit_with_ff5mom_ddstop_grid"
BASE_OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_GRID_SUMMARY = BASE_OUT_DIR / "grid_summary.csv"


# ===================================
# 税・単位株
# ===================================
TAX_RATE = 0.20315
UNIT_SHARES = 100
INITIAL_CAPITAL = 10_000_000


# ===================================
# リスク制御パラメータ（固定/既存）
# ===================================
RISKOFF_RATIO = 0.7  # 固定

REGIME_LOOKBACK_M = 3
REGIME_OFF_IF_SUM_MKT_LT = 0.0  # 必要なら固定で変えてもOK（今回は据え置き）
REGIME_OFF_IF_SUM_WML_LT = 0.0

BETA_CAP_CMA_LT = -0.8
BETA_CAP_ABS_MKT_GT = 0.9

BETA_EST_WINDOW_M = 12
BETA_EST_MIN_OBS = 10

FACTOR_COLS = ["MKT", "SMB", "HML", "RMW", "CMA", "WML"]

BAD_CODE_STRINGS = {"None", "nan", "", "NaN", "NULL", "null"}


# ===================================
# 出力パス（ケースごとに set_output_dir で差し替える）
# ===================================
OUT_DIR = BASE_OUT_DIR / "case_tmp"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_ANNUAL_RAW = OUT_DIR / "annual_returns_raw.csv"
OUT_ANNUAL_RC  = OUT_DIR / "annual_returns_riskcontrol.csv"
OUT_ANNUAL_DDSTOP = OUT_DIR / "annual_returns_ddstop.csv"

OUT_CURVE_RAW  = OUT_DIR / "cumulative_curve_raw.csv"
OUT_CURVE_RC   = OUT_DIR / "cumulative_curve_riskcontrol.csv"
OUT_CURVE_DDSTOP = OUT_DIR / "cumulative_curve_ddstop.csv"

OUT_PERF_RAW   = OUT_DIR / "performance_summary_raw.csv"
OUT_PERF_RC    = OUT_DIR / "performance_summary_riskcontrol.csv"
OUT_PERF_DDSTOP = OUT_DIR / "performance_summary_ddstop.csv"

OUT_REG_RAW    = OUT_DIR / "regression_ff5mom_raw.csv"
OUT_REG_RC     = OUT_DIR / "regression_ff5mom_riskcontrol.csv"
OUT_REG_DDSTOP = OUT_DIR / "regression_ff5mom_ddstop.csv"

OUT_IR_DIAG    = OUT_DIR / "diag_invest_ratio_monthly.csv"

# 日次
OUT_DAILY_CURVE_RAW = OUT_DIR / "daily_equity_curve_raw.csv"
OUT_DAILY_CURVE_RC  = OUT_DIR / "daily_equity_curve_riskcontrol.csv"
OUT_DAILY_CURVE_DDSTOP = OUT_DIR / "daily_equity_curve_ddstop.csv"

OUT_DAILY_MDD_SUMMARY = OUT_DIR / "daily_mdd_summary.csv"

OUT_DD_EVENTS_RAW = OUT_DIR / "daily_drawdown_events_raw.csv"
OUT_DD_EVENTS_RC  = OUT_DIR / "daily_drawdown_events_riskcontrol.csv"
OUT_DD_EVENTS_DDSTOP = OUT_DIR / "daily_drawdown_events_ddstop.csv"
OUT_DD_EVENTS_MAX = OUT_DIR / "daily_drawdown_events_max_summary.csv"


def set_output_dir(base_dir: Path, suffix: str):
    global OUT_DIR
    global OUT_ANNUAL_RAW, OUT_ANNUAL_RC, OUT_ANNUAL_DDSTOP
    global OUT_CURVE_RAW, OUT_CURVE_RC, OUT_CURVE_DDSTOP
    global OUT_PERF_RAW, OUT_PERF_RC, OUT_PERF_DDSTOP
    global OUT_REG_RAW, OUT_REG_RC, OUT_REG_DDSTOP
    global OUT_IR_DIAG
    global OUT_DAILY_CURVE_RAW, OUT_DAILY_CURVE_RC, OUT_DAILY_CURVE_DDSTOP
    global OUT_DAILY_MDD_SUMMARY
    global OUT_DD_EVENTS_RAW, OUT_DD_EVENTS_RC, OUT_DD_EVENTS_DDSTOP, OUT_DD_EVENTS_MAX

    OUT_DIR = base_dir / suffix
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    OUT_ANNUAL_RAW = OUT_DIR / "annual_returns_raw.csv"
    OUT_ANNUAL_RC  = OUT_DIR / "annual_returns_riskcontrol.csv"
    OUT_ANNUAL_DDSTOP = OUT_DIR / "annual_returns_ddstop.csv"

    OUT_CURVE_RAW  = OUT_DIR / "cumulative_curve_raw.csv"
    OUT_CURVE_RC   = OUT_DIR / "cumulative_curve_riskcontrol.csv"
    OUT_CURVE_DDSTOP = OUT_DIR / "cumulative_curve_ddstop.csv"

    OUT_PERF_RAW   = OUT_DIR / "performance_summary_raw.csv"
    OUT_PERF_RC    = OUT_DIR / "performance_summary_riskcontrol.csv"
    OUT_PERF_DDSTOP = OUT_DIR / "performance_summary_ddstop.csv"

    OUT_REG_RAW    = OUT_DIR / "regression_ff5mom_raw.csv"
    OUT_REG_RC     = OUT_DIR / "regression_ff5mom_riskcontrol.csv"
    OUT_REG_DDSTOP = OUT_DIR / "regression_ff5mom_ddstop.csv"

    OUT_IR_DIAG    = OUT_DIR / "diag_invest_ratio_monthly.csv"

    OUT_DAILY_CURVE_RAW = OUT_DIR / "daily_equity_curve_raw.csv"
    OUT_DAILY_CURVE_RC  = OUT_DIR / "daily_equity_curve_riskcontrol.csv"
    OUT_DAILY_CURVE_DDSTOP = OUT_DIR / "daily_equity_curve_ddstop.csv"

    OUT_DAILY_MDD_SUMMARY = OUT_DIR / "daily_mdd_summary.csv"

    OUT_DD_EVENTS_RAW = OUT_DIR / "daily_drawdown_events_raw.csv"
    OUT_DD_EVENTS_RC  = OUT_DIR / "daily_drawdown_events_riskcontrol.csv"
    OUT_DD_EVENTS_DDSTOP = OUT_DIR / "daily_drawdown_events_ddstop.csv"
    OUT_DD_EVENTS_MAX = OUT_DIR / "daily_drawdown_events_max_summary.csv"


def fmt_cut(x: float) -> str:
    # -0.10 -> m0p10, -0.08 -> m0p08
    x = float(x)
    s = f"{abs(x):.2f}".replace(".", "p")
    return f"m{s}" if x < 0 else s


# ===================================
# ユーティリティ
# ===================================
def normalize_code(code) -> str:
    if code is None:
        return ""
    s = str(code).strip()
    if s in BAD_CODE_STRINGS:
        return ""
    return s


def ols_alpha_beta(y: np.ndarray, X: np.ndarray) -> Tuple[float, np.ndarray, float]:
    n = len(y)
    if n < 3:
        return np.nan, np.full(X.shape[1], np.nan), np.nan

    X1 = np.column_stack([np.ones(n), X])
    XtX = X1.T @ X1
    try:
        inv = np.linalg.inv(XtX)
    except np.linalg.LinAlgError:
        inv = np.linalg.pinv(XtX)
    b = inv @ (X1.T @ y)

    yhat = X1 @ b
    resid = y - yhat
    sse = float(np.sum(resid**2))
    sst = float(np.sum((y - y.mean())**2))
    r2 = np.nan if sst <= 0 else (1.0 - sse / sst)

    alpha = float(b[0])
    betas = b[1:].astype(float)
    return alpha, betas, float(r2)


def compute_drawdown(cum: pd.Series) -> pd.Series:
    peak = cum.cummax()
    return cum / peak - 1.0


def perf_stats(annual_ret: pd.Series) -> Dict:
    r = annual_ret.dropna().astype(float)
    if r.empty:
        return {"n_years": 0, "CAGR": np.nan, "ann_mean": np.nan, "ann_vol": np.nan, "sharpe0": np.nan, "maxDD": np.nan, "cum_end": np.nan}

    n = len(r)
    cum = (1.0 + r).cumprod()
    years = n
    cagr = float(cum.iloc[-1] ** (1/years) - 1.0) if years > 0 else np.nan
    ann_mean = float(r.mean())
    ann_vol = float(r.std(ddof=1)) if n >= 2 else np.nan
    sharpe0 = float(ann_mean / ann_vol) if ann_vol and ann_vol > 0 else np.nan
    dd = compute_drawdown(cum)
    maxdd = float(dd.min())

    return {
        "n_years": int(n),
        "CAGR": cagr,
        "ann_mean": ann_mean,
        "ann_vol": ann_vol,
        "sharpe0": sharpe0,
        "maxDD": maxdd,
        "cum_end": float(cum.iloc[-1]),
    }


def extract_max_drawdown_event_from_daily_curve(df_daily: pd.DataFrame, label: str = "") -> pd.DataFrame:
    if df_daily is None or df_daily.empty:
        return pd.DataFrame([{
            "label": label,
            "peak_date": pd.NaT,
            "trough_date": pd.NaT,
            "recovery_date": pd.NaT,
            "dd_min": np.nan,
            "peak_equity": np.nan,
            "trough_equity": np.nan,
            "recovery_equity": np.nan,
            "days_to_trough": np.nan,
            "days_to_recovery": np.nan,
            "dd_duration_days": np.nan,
        }])

    df = df_daily.copy()
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df = df.dropna(subset=["Date"]).sort_values("Date").reset_index(drop=True)

    for col in ["equity_total", "dd_total"]:
        if col not in df.columns:
            raise KeyError(f"daily curve missing required column: {col}")

    df["equity_total"] = pd.to_numeric(df["equity_total"], errors="coerce")
    df["dd_total"] = pd.to_numeric(df["dd_total"], errors="coerce")
    df = df.dropna(subset=["equity_total", "dd_total"]).copy()
    if df.empty:
        return pd.DataFrame([{
            "label": label,
            "peak_date": pd.NaT,
            "trough_date": pd.NaT,
            "recovery_date": pd.NaT,
            "dd_min": np.nan,
            "peak_equity": np.nan,
            "trough_equity": np.nan,
            "recovery_equity": np.nan,
            "days_to_trough": np.nan,
            "days_to_recovery": np.nan,
            "dd_duration_days": np.nan,
        }])

    trough_idx = int(df["dd_total"].idxmin())
    dd_min = float(df.loc[trough_idx, "dd_total"])
    trough_date = pd.Timestamp(df.loc[trough_idx, "Date"])
    trough_equity = float(df.loc[trough_idx, "equity_total"])

    df_pre = df.loc[:trough_idx].copy()
    peak_equity = float(df_pre["equity_total"].max())
    peak_idx = int(df_pre["equity_total"].idxmax())
    peak_date = pd.Timestamp(df.loc[peak_idx, "Date"])

    df_post = df.loc[trough_idx:].copy()
    rec = df_post[df_post["equity_total"] >= peak_equity]
    if len(rec) == 0:
        recovery_date = pd.NaT
        recovery_equity = np.nan
        days_to_recovery = np.nan
        dd_duration_days = np.nan
    else:
        rec_idx = int(rec.index[0])
        recovery_date = pd.Timestamp(df.loc[rec_idx, "Date"])
        recovery_equity = float(df.loc[rec_idx, "equity_total"])
        days_to_recovery = int((recovery_date - peak_date).days)
        dd_duration_days = int((recovery_date - peak_date).days)

    days_to_trough = int((trough_date - peak_date).days)

    return pd.DataFrame([{
        "label": label,
        "peak_date": peak_date,
        "trough_date": trough_date,
        "recovery_date": recovery_date,
        "dd_min": dd_min,
        "peak_equity": peak_equity,
        "trough_equity": trough_equity,
        "recovery_equity": recovery_equity,
        "days_to_trough": days_to_trough,
        "days_to_recovery": days_to_recovery,
        "dd_duration_days": dd_duration_days,
    }])


def daily_mdd(df_daily: pd.DataFrame) -> float:
    if df_daily is None or df_daily.empty or "dd_total" not in df_daily.columns:
        return np.nan
    s = pd.to_numeric(df_daily["dd_total"], errors="coerce").dropna()
    return float(s.min()) if len(s) else np.nan


# ===================================
# A. 財務データ読み込み（列名自動判定）
# ===================================
def load_financial_data(cache_filename: str = CACHE_FILE) -> pd.DataFrame:
    if not os.path.exists(cache_filename):
        logger.error(f"財務データファイル '{cache_filename}' が見つかりません")
        return pd.DataFrame()

    try:
        df = pd.read_csv(cache_filename, encoding="utf-8-sig", parse_dates=["DisclosedDate"], low_memory=False)
        logger.info(f"財務データ読み込み成功: {len(df):,}件")

        column_mapping = {
            "IssuedShareTotal": ["IssuedShareTotal", "NumberOfIssuedAndOutstandingSharesAtTheEndOfFiscalYearIncludingTreasuryStock"],
            "Equity": ["Equity", "NetAssets", "TotalEquity"],
            "Profit": ["Profit", "NetIncome", "ProfitAttributableToOwnersOfParent"],
        }

        available_cols = df.columns.tolist()
        required_columns = {}
        for target_col, possible_names in column_mapping.items():
            found = False
            for possible_name in possible_names:
                if possible_name in available_cols:
                    required_columns[target_col] = possible_name
                    found = True
                    break
            if not found:
                required_columns[target_col] = None

        rename_dict = {v: k for k, v in required_columns.items() if v is not None}
        df = df.rename(columns=rename_dict)

        for col in ["IssuedShareTotal", "Equity", "Profit"]:
            if col not in df.columns:
                df[col] = 0

        base_cols = ["Code", "DisclosedDate"]
        if "CompanyName" in df.columns:
            base_cols.append("CompanyName")
        final_cols = base_cols + ["Profit", "Equity", "IssuedShareTotal"]
        df = df[final_cols].copy()

        logger.info(f"使用列: {df.columns.tolist()}")
        return df

    except Exception as e:
        logger.error(f"財務データ読み込みエラー: {e}", exc_info=True)
        return pd.DataFrame()


# ===================================
# B. 株価データ読み込み
# ===================================
def load_existing_price_data(ohlcv_dir: str = OHLCV_DIR) -> pd.DataFrame:
    if not os.path.exists(ohlcv_dir):
        logger.error(f"株価データディレクトリ '{ohlcv_dir}' が見つかりません。")
        return pd.DataFrame()

    csv_files = sorted([f for f in os.listdir(ohlcv_dir)
                        if f.startswith("OHLCV_Adjusted_") and f.endswith(".csv") and f != "OHLCV_Adjusted_TOPIX.csv"])

    if not csv_files:
        logger.error(f"ディレクトリ '{ohlcv_dir}' 内にCSVファイルが見つかりません。")
        return pd.DataFrame()

    logger.info(f"株価ファイル数: {len(csv_files)}個")

    all_dataframes = []
    usecols = ["Date", "Ticker", "AdjustmentClose"]

    for csv_file in tqdm(csv_files, desc="株価ファイル読み込み中"):
        file_path = os.path.join(ohlcv_dir, csv_file)
        try:
            df = pd.read_csv(
                file_path,
                usecols=usecols,
                parse_dates=["Date"],
                dtype={"Ticker": "Int64", "AdjustmentClose": "float32"},
            )
            df = df.drop_duplicates(subset=["Ticker", "Date"], keep="first")
            all_dataframes.append(df)
        except Exception as e:
            logger.warning(f"ファイル読み込みエラー ({csv_file}): {e}")
            continue

    if not all_dataframes:
        return pd.DataFrame()

    logger.info("全ファイルを結合中...")
    df_all = pd.concat(all_dataframes, ignore_index=True)
    logger.info(f"結合完了: {len(df_all):,}件")

    df_all = df_all.rename(columns={"Ticker": "Code", "AdjustmentClose": "Close"})
    df_all["Code"] = df_all["Code"].astype("str").str.replace("<NA>", "0").str.zfill(4)
    df_all = df_all.dropna(subset=["Close"])

    logger.info("重複除去 & ソート中...")
    df_all = df_all.sort_values(["Code", "Date"])
    df_all = df_all.drop_duplicates(subset=["Code", "Date"], keep="first")
    logger.info(f"重複除去後: {len(df_all):,}件")

    return df_all


def build_prices_by_code(prices_df: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    d = {}
    tmp = prices_df[["Code", "Date", "Close"]].copy()
    tmp["Code"] = tmp["Code"].astype(str).map(normalize_code)
    tmp = tmp[tmp["Code"] != ""].copy()
    tmp = tmp.sort_values(["Code", "Date"])

    for code, g in tmp.groupby("Code", sort=False):
        gg = g[["Date", "Close"]].drop_duplicates(subset=["Date"], keep="last").sort_values("Date").copy()
        gg = gg.set_index("Date", drop=False)
        d[code] = gg

    logger.info(f"prices_by_code built: {len(d):,} codes")
    return d


def get_near_price_fast(prices_by_code: Dict[str, pd.DataFrame],
                        code: str,
                        ref_date: pd.Timestamp,
                        kind: str = "last") -> Optional[float]:
    code = normalize_code(code)
    if not code or code not in prices_by_code:
        return None
    g = prices_by_code[code]
    lo = ref_date - pd.Timedelta(days=5)
    hi = ref_date + pd.Timedelta(days=5)
    w = g.loc[(g["Date"] >= lo) & (g["Date"] <= hi)]
    if w.empty:
        return None
    return float(w.iloc[0]["Close"]) if kind == "first" else float(w.iloc[-1]["Close"])


def safe_code_to_int(code_series: pd.Series) -> pd.Series:
    cleaned = code_series.astype(str).str.replace(r"\D", "", regex=True)
    cleaned = cleaned.replace("", "0")
    return pd.to_numeric(cleaned, errors="coerce").fillna(0).astype("int64")


def calculate_market_metrics_fast_chunked(statements_df: pd.DataFrame,
                                          prices_df: pd.DataFrame,
                                          chunk_size: int = 200) -> pd.DataFrame:
    logger.info("時価総額・PBR・ROE計算中（チャンク処理版）...")

    if prices_df.empty:
        logger.error("株価データが空です。")
        return pd.DataFrame()

    req_cols = {"Code", "Date", "Close"}
    if not req_cols.issubset(set(prices_df.columns)):
        logger.error(f"株価データに必要な列が存在しません。存在する列: {prices_df.columns.tolist()}")
        return pd.DataFrame()

    statements_df = statements_df.copy()
    statements_df["Profit"] = pd.to_numeric(statements_df["Profit"], errors="coerce").fillna(0)
    statements_df["Equity"] = pd.to_numeric(statements_df["Equity"], errors="coerce").fillna(0)
    statements_df["IssuedShareTotal"] = pd.to_numeric(statements_df["IssuedShareTotal"], errors="coerce").fillna(1)

    statements_df = statements_df[(statements_df["Equity"] > 0) & (statements_df["IssuedShareTotal"] > 0)]
    logger.info(f"有効な財務データ: {len(statements_df):,}件")

    statements_df["Code_int"] = safe_code_to_int(statements_df["Code"])
    prices_df = prices_df.copy()
    prices_df["Code_int"] = safe_code_to_int(prices_df["Code"])

    statements_df = statements_df[statements_df["Code_int"] > 0]
    prices_df = prices_df[prices_df["Code_int"] > 0]

    statements_df = statements_df.sort_values(["Code_int", "DisclosedDate"]).reset_index(drop=True)
    prices_df = prices_df.sort_values(["Code_int", "Date"]).reset_index(drop=True)

    statements_df = statements_df.drop_duplicates(subset=["Code_int", "DisclosedDate"], keep="first")
    prices_df = prices_df.drop_duplicates(subset=["Code_int", "Date"], keep="first")

    logger.info(f"ソート・重複除去後: 財務 {len(statements_df):,}件, 株価 {len(prices_df):,}件")

    statements_groups = list(statements_df.groupby("Code_int"))
    prices_dict = {code: group for code, group in prices_df.groupby("Code_int")}

    merged_list = []
    num_chunks = (len(statements_groups) + chunk_size - 1) // chunk_size

    for chunk_idx in tqdm(range(num_chunks), desc="マージ処理"):
        start_idx = chunk_idx * chunk_size
        end_idx = min((chunk_idx + 1) * chunk_size, len(statements_groups))
        chunk_groups = statements_groups[start_idx:end_idx]

        for code, stmt_code in chunk_groups:
            if code not in prices_dict:
                continue
            price_code = prices_dict[code]
            if len(price_code) == 0:
                continue

            stmt_code = stmt_code.sort_values("DisclosedDate").reset_index(drop=True)
            price_code = price_code.sort_values("Date").reset_index(drop=True)

            try:
                merged = pd.merge_asof(
                    stmt_code,
                    price_code[["Date", "Close"]],
                    left_on="DisclosedDate",
                    right_on="Date",
                    direction="backward",
                    tolerance=pd.Timedelta(days=10),
                )
                if not merged.empty:
                    merged_list.append(merged)
            except Exception:
                continue

    if not merged_list:
        logger.error("マージ結果が空です")
        return pd.DataFrame()

    df_merged = pd.concat(merged_list, ignore_index=True)
    df_merged = df_merged.dropna(subset=["Close"])
    logger.info(f"マージ完了: {len(df_merged):,}件")

    df_merged["MarketCap"] = df_merged["Close"] * df_merged["IssuedShareTotal"]
    df_merged["PBR"] = df_merged["MarketCap"] / df_merged["Equity"]
    df_merged["ROE"] = (df_merged["Profit"] / df_merged["Equity"]) * 100

    result_cols = ["Code", "DisclosedDate", "Close", "MarketCap", "PBR", "ROE", "Date"]
    if "CompanyName" in df_merged.columns:
        result_cols.insert(1, "CompanyName")

    result_df = df_merged[result_cols].copy()
    result_df = result_df.rename(columns={"Close": "StockPrice", "Date": "PriceDate"})

    mask = (
        (result_df["PBR"] > 0) &
        (result_df["PBR"] < 50) &
        (result_df["ROE"] > -100) &
        (result_df["ROE"] < 100) &
        (result_df["MarketCap"] > 1_000_000_000)
    )
    result_df = result_df[mask].copy()

    logger.info(f"計算完了: {len(result_df):,}件")
    return result_df


def build_unit_share_portfolio(stock_candidates: pd.DataFrame,
                               target_positions: int = 20,
                               initial_capital: float = 10_000_000) -> dict:
    if len(stock_candidates) == 0:
        return {"stocks": [], "shares": [], "prices": [], "amounts": []}

    selected = stock_candidates.head(target_positions).copy()
    capital_per_stock = initial_capital / len(selected)

    stocks, shares_list, prices_list, amounts_list = [], [], [], []

    for _, row in selected.iterrows():
        code = row["Code"]
        price = row["StockPrice"]
        required_amount = price * UNIT_SHARES

        if required_amount <= capital_per_stock:
            shares = int(capital_per_stock // required_amount) * UNIT_SHARES
            if shares > 0:
                stocks.append(code)
                shares_list.append(shares)
                prices_list.append(price)
                amounts_list.append(shares * price)

    return {"stocks": stocks, "shares": shares_list, "prices": prices_list, "amounts": amounts_list}


def make_month_ends(start: pd.Timestamp, end: pd.Timestamp) -> List[pd.Timestamp]:
    m0 = pd.Timestamp(start.year, start.month, 1) + pd.offsets.MonthEnd(0)
    m1 = pd.Timestamp(end.year, end.month, 1) + pd.offsets.MonthEnd(0)
    months = pd.date_range(m0, m1, freq="M")
    return [pd.Timestamp(x).normalize() for x in months]


def build_daily_equity_curve_for_period(
    portfolio: dict,
    start_date: pd.Timestamp,
    end_date: pd.Timestamp,
    prices_by_code: Dict[str, pd.DataFrame],
    invest_ratio_by_monthend: Optional[Dict[pd.Timestamp, float]] = None,
) -> pd.DataFrame:
    stocks = portfolio.get("stocks", [])
    shares = portfolio.get("shares", [])
    if not stocks:
        return pd.DataFrame()

    start_date = pd.to_datetime(start_date).normalize()
    end_date = pd.to_datetime(end_date).normalize()

    date_sets = []
    for code in stocks:
        code = normalize_code(code)
        if code in prices_by_code:
            g = prices_by_code[code]
            d = g[(g["Date"] >= start_date - pd.Timedelta(days=10)) & (g["Date"] <= end_date + pd.Timedelta(days=10))]["Date"]
            if len(d):
                date_sets.append(d)

    if not date_sets:
        return pd.DataFrame()

    dates = pd.Index(sorted(pd.unique(pd.concat(date_sets)))).astype("datetime64[ns]")
    dates = dates[(dates >= start_date) & (dates <= end_date)]
    if len(dates) == 0:
        return pd.DataFrame()

    values = []
    for dt in dates:
        v = 0.0
        ok = False
        for i, code in enumerate(stocks):
            code = normalize_code(code)
            if code not in prices_by_code:
                continue
            p = get_near_price_fast(prices_by_code, code, pd.Timestamp(dt), kind="last")
            if p is None:
                continue
            v += float(shares[i]) * float(p)
            ok = True
        values.append(v if ok else np.nan)

    df = pd.DataFrame({"Date": pd.to_datetime(dates), "equity_stock": values})
    df = df.dropna(subset=["equity_stock"]).copy()
    if df.empty:
        return df

    df = df.sort_values("Date").reset_index(drop=True)
    df["ret_stock"] = df["equity_stock"].pct_change().fillna(0.0)
    df["MonthEnd"] = (df["Date"] + pd.offsets.MonthEnd(0)).dt.normalize()

    if invest_ratio_by_monthend is None:
        df["invest_ratio"] = 1.0
        df["ret_total"] = df["ret_stock"]
    else:
        df["invest_ratio"] = df["MonthEnd"].map(invest_ratio_by_monthend).fillna(1.0).astype(float)
        df["ret_total"] = df["invest_ratio"] * df["ret_stock"]

    df["equity_total"] = (1.0 + df["ret_total"]).cumprod()
    peak = df["equity_total"].cummax()
    df["dd_total"] = df["equity_total"] / peak - 1.0

    return df[["Date", "MonthEnd", "equity_stock", "ret_stock", "invest_ratio", "ret_total", "equity_total", "dd_total"]]


def load_ff5mom_factors_monthly() -> pd.DataFrame:
    fac = pd.read_parquet(FF5MOM_FACTOR_PATH).copy()
    fac["MonthEnd"] = pd.to_datetime(fac["MonthEnd"], errors="coerce").dt.normalize()
    need = ["MonthEnd"] + FACTOR_COLS
    missing = [c for c in need if c not in fac.columns]
    if missing:
        raise KeyError(f"FF5MOM factors missing columns: {missing} in {FF5MOM_FACTOR_PATH}")
    fac = fac[need].sort_values("MonthEnd").reset_index(drop=True)
    return fac


def compute_regime_off(fac: pd.DataFrame, month_end: pd.Timestamp) -> Tuple[bool, Dict]:
    fac2 = fac.set_index("MonthEnd")
    idx = fac2.index[fac2.index < month_end]
    if len(idx) < REGIME_LOOKBACK_M:
        return False, {"regime_ready": False}

    win = idx[-REGIME_LOOKBACK_M:]
    s_mkt = float(pd.to_numeric(fac2.loc[win, "MKT"], errors="coerce").sum())
    s_wml = float(pd.to_numeric(fac2.loc[win, "WML"], errors="coerce").sum())

    off = (s_mkt < REGIME_OFF_IF_SUM_MKT_LT) or (s_wml < REGIME_OFF_IF_SUM_WML_LT)
    return bool(off), {"regime_ready": True, "sum_MKT_3m": s_mkt, "sum_WML_3m": s_wml}


def estimate_port_beta(monthly_port_rets: pd.DataFrame, fac: pd.DataFrame, month_end: pd.Timestamp) -> Tuple[bool, Dict]:
    fac2 = fac.set_index("MonthEnd")
    idx = fac2.index[fac2.index < month_end]
    if len(idx) < BETA_EST_WINDOW_M:
        return False, {"beta_ready": False}

    win = idx[-BETA_EST_WINDOW_M:]

    m = monthly_port_rets.copy()
    m["MonthEnd"] = pd.to_datetime(m["MonthEnd"], errors="coerce").dt.normalize()
    m = m.dropna(subset=["MonthEnd", "port_ret"]).copy()

    m = (
        m.groupby("MonthEnd", as_index=False)["port_ret"]
         .apply(lambda s: (1.0 + s.astype(float)).prod() - 1.0)
    )

    df = m[m["MonthEnd"].isin(win)].merge(
        fac, on="MonthEnd", how="left", validate="one_to_one"
    ).dropna(subset=["port_ret"] + FACTOR_COLS)

    if len(df) < BETA_EST_MIN_OBS:
        return False, {"beta_ready": False, "n_obs": int(len(df))}

    y = df["port_ret"].to_numpy(dtype=float)
    X = df[FACTOR_COLS].to_numpy(dtype=float)
    alpha, betas, r2 = ols_alpha_beta(y, X)

    out = {"beta_ready": True, "n_obs": int(len(df)), "r2": float(r2), "alpha": float(alpha)}
    for c, b in zip(FACTOR_COLS, betas):
        out[f"beta_{c}"] = float(b)
    return True, out


def decide_invest_ratio(month_end: pd.Timestamp,
                        fac: pd.DataFrame,
                        port_hist: pd.DataFrame) -> Tuple[float, Dict]:
    invest_ratio = 1.0
    diag = {
        "MonthEnd": month_end,
        "invest_ratio": 1.0,
        "regime_off": False,
        "beta_off": False,
        "regime_ready": False,
        "beta_ready": False,
        "sum_MKT_3m": np.nan,
        "sum_WML_3m": np.nan,
        "beta_MKT": np.nan,
        "beta_CMA": np.nan,
        "beta_n_obs": np.nan,
        "beta_r2": np.nan,
    }

    off_reg, info = compute_regime_off(fac, month_end)
    diag["regime_off"] = bool(off_reg)
    diag["regime_ready"] = bool(info.get("regime_ready", False))
    diag["sum_MKT_3m"] = info.get("sum_MKT_3m", np.nan)
    diag["sum_WML_3m"] = info.get("sum_WML_3m", np.nan)
    if off_reg:
        invest_ratio = min(invest_ratio, RISKOFF_RATIO)

    ok_beta, binfo = estimate_port_beta(port_hist, fac, month_end)
    diag["beta_ready"] = bool(binfo.get("beta_ready", False))
    if binfo.get("beta_ready", False):
        b_mkt = float(binfo.get("beta_MKT", np.nan))
        b_cma = float(binfo.get("beta_CMA", np.nan))
        diag["beta_MKT"] = b_mkt
        diag["beta_CMA"] = b_cma
        diag["beta_n_obs"] = float(binfo.get("n_obs", np.nan))
        diag["beta_r2"] = float(binfo.get("r2", np.nan))

        off_beta = (b_cma < BETA_CAP_CMA_LT) or (abs(b_mkt) > BETA_CAP_ABS_MKT_GT)
        diag["beta_off"] = bool(off_beta)
        if off_beta:
            invest_ratio = min(invest_ratio, RISKOFF_RATIO)

    diag["invest_ratio"] = float(invest_ratio)
    return float(invest_ratio), diag


def compute_monthly_portfolio_returns_with_riskcontrol_fast(
    portfolio: dict,
    start_date: pd.Timestamp,
    end_date: pd.Timestamp,
    prices_by_code: Dict[str, pd.DataFrame],
    fac: pd.DataFrame
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    stocks = portfolio["stocks"]
    shares = portfolio["shares"]
    if not stocks:
        return pd.DataFrame(), pd.DataFrame()

    month_ends = make_month_ends(start_date, end_date)
    boundaries = [start_date] + [me for me in month_ends if (me > start_date) and (me < end_date)] + [end_date]

    rows = []
    ir_rows = []
    port_hist = pd.DataFrame(columns=["MonthEnd", "port_ret"])

    for j in range(len(boundaries) - 1):
        d0 = boundaries[j]
        d1 = boundaries[j + 1]

        me = pd.Timestamp(d0.year, d0.month, 1) + pd.offsets.MonthEnd(0)
        me = pd.Timestamp(me).normalize()

        start_vals = []
        end_vals = []
        for i, code in enumerate(stocks):
            p0 = get_near_price_fast(prices_by_code, code, d0, kind="first")
            p1 = get_near_price_fast(prices_by_code, code, d1, kind="last")
            if p0 is None or p1 is None:
                continue
            sh = shares[i]
            start_vals.append(sh * p0)
            end_vals.append(sh * p1)

        if len(start_vals) == 0:
            continue

        start_v = float(np.sum(start_vals))
        end_v = float(np.sum(end_vals))
        stock_ret = (end_v / start_v) - 1.0

        port_hist = pd.concat([port_hist, pd.DataFrame([{"MonthEnd": me, "port_ret": stock_ret}])], ignore_index=True)

        invest_ratio, diag = decide_invest_ratio(me, fac, port_hist)
        ir_rows.append(diag)

        total_ret = invest_ratio * stock_ret

        rows.append({
            "MonthEnd": me,
            "period_start": d0,
            "period_end": d1,
            "port_ret_stock": stock_ret,
            "invest_ratio": invest_ratio,
            "port_ret_total": total_ret,
        })

    monthly_df = pd.DataFrame(rows)
    ir_diag_df = pd.DataFrame(ir_rows)
    return monthly_df, ir_diag_df


def build_long_candidates(enhanced_financial_data: pd.DataFrame, rebalance_date: pd.Timestamp) -> pd.DataFrame:
    current_data = enhanced_financial_data[enhanced_financial_data["DisclosedDate"] <= rebalance_date].copy()
    current_data = current_data.sort_values("DisclosedDate").groupby("Code").tail(1)

    if len(current_data) < 100:
        return pd.DataFrame()

    current_data["PBR_Rank"] = current_data["PBR"].rank(method="first", ascending=True)
    current_data["ROE_Rank"] = current_data["ROE"].rank(method="first", ascending=False)

    current_data["PBR_Quartile"] = pd.qcut(current_data["PBR_Rank"], q=4, labels=[1, 2, 3, 4])
    current_data["ROE_Quartile"] = pd.qcut(current_data["ROE_Rank"], q=4, labels=[1, 2, 3, 4])

    long_candidates = current_data[
        (current_data["PBR_Quartile"] == 1) &
        (current_data["ROE_Quartile"] == 4)
    ].nsmallest(50, "PBR")

    return long_candidates


def annual_return_from_monthly(monthly_rets: pd.Series) -> float:
    if monthly_rets.empty:
        return 0.0
    return float((1.0 + monthly_rets).prod() - 1.0)


def apply_tax_annual(gross_return: float, initial_capital: float) -> Tuple[float, float]:
    profit = gross_return * initial_capital
    taxable = max(profit, 0.0)
    tax = taxable * TAX_RATE
    net_profit = profit - tax
    net_return = net_profit / initial_capital
    tax_rate_total = tax / initial_capital
    return float(net_return), float(tax_rate_total)


# ===================================
# ★ 追加：日次DDストップ適用（軽量）
# ===================================
def apply_dd_stop_to_daily_curve(
    df_daily_rc: pd.DataFrame,
    dd_cutoff_1: float,
    dd_ratio_1: float,
    dd_cutoff_2: float,
    dd_ratio_2: float,
) -> pd.DataFrame:
    """
    入力 df_daily_rc は既に
      Date, ret_stock, invest_ratio, ret_total, equity_total, dd_total ...
    を持っている想定。

    ここでは dd_total を見ながら日次で invest_ratio を上書きし、
    ret_total / equity_total / dd_total を作り直す。

    ※ dd_total は「新しいequity_total」から再計算されるので、
       ストップの効果が反映される。
    """
    if df_daily_rc is None or df_daily_rc.empty:
        return pd.DataFrame()

    df = df_daily_rc.copy()
    df = df.sort_values("Date").reset_index(drop=True)

    # 入力整形
    df["ret_stock"] = pd.to_numeric(df["ret_stock"], errors="coerce").fillna(0.0).astype(float)
    base_ir = pd.to_numeric(df["invest_ratio"], errors="coerce").fillna(1.0).astype(float).to_numpy()

    # 日次で上書きする投資比率（最初は月次ベース）
    ir = base_ir.copy()

    # equityを再構築しながらDDを監視
    eq = np.empty(len(df), dtype=float)
    peak = np.empty(len(df), dtype=float)
    dd = np.empty(len(df), dtype=float)

    eq_val = 1.0
    peak_val = 1.0

    for i in range(len(df)):
        # 直前までのDDでストップを発動させる（当日反映でもOKだが、ここでは当日リターンに反映）
        # ルール：当日計算に使う ir[i] を、現在の(直前の)ddで抑える
        # 初日はdd=0扱い
        cur_dd = dd[i-1] if i > 0 else 0.0

        if cur_dd <= dd_cutoff_2:
            ir[i] = min(ir[i], dd_ratio_2)
        elif cur_dd <= dd_cutoff_1:
            ir[i] = min(ir[i], dd_ratio_1)

        r_total = ir[i] * df.loc[i, "ret_stock"]
        eq_val *= (1.0 + r_total)

        peak_val = max(peak_val, eq_val)
        dd_val = (eq_val / peak_val) - 1.0

        eq[i] = eq_val
        peak[i] = peak_val
        dd[i] = dd_val

    df["invest_ratio_ddstop"] = ir
    df["ret_total_ddstop"] = df["ret_stock"] * df["invest_ratio_ddstop"]
    df["equity_total_ddstop"] = eq
    df["dd_total_ddstop"] = dd

    return df


def build_annual_from_daily_equity(df_daily: pd.DataFrame, equity_col: str) -> pd.DataFrame:
    """
    df_daily（全期間連結）の equity_col を使って、年次（10/1→翌10/1）リターンを作る。
    返り値: columns = ['date', 'strategy_return_gross', 'strategy_return_net', 'tax', ...] の形に合わせる。
    ※ 日次税は無視、年次で apply_tax_annual する。
    """
    if df_daily is None or df_daily.empty:
        return pd.DataFrame()

    df = df_daily.copy()
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df = df.dropna(subset=["Date"]).sort_values("Date").reset_index(drop=True)

    # 10/1区切りの年次区間を作成
    rebalance_dates = [pd.Timestamp(f"{year}-10-01") for year in range(2016, 2026)]
    rows = []
    for i in range(len(rebalance_dates) - 1):
        start = rebalance_dates[i]
        end = rebalance_dates[i + 1]

        w = df[(df["Date"] >= start) & (df["Date"] <= end)].copy()
        if w.empty:
            continue

        # start近傍の最初 / end近傍の最後
        eq0 = float(w.iloc[0][equity_col])
        eq1 = float(w.iloc[-1][equity_col])
        if eq0 <= 0:
            continue

        gross = (eq1 / eq0) - 1.0
        net, tax = apply_tax_annual(gross, INITIAL_CAPITAL)

        rows.append({
            "date": end,
            "strategy_return_gross": float(gross),
            "strategy_return_net": float(net),
            "tax": float(tax),
            "long_count": np.nan,
            "investment_ratio": np.nan,
        })

    return pd.DataFrame(rows)


def run_factor_regression(results_df: pd.DataFrame, fac: pd.DataFrame) -> pd.DataFrame:
    if results_df.empty:
        return pd.DataFrame()

    rows = []
    for _, row in results_df.iterrows():
        end_date = pd.Timestamp(row["date"])
        start_date = end_date - pd.DateOffset(years=1)

        fac2 = fac.copy()
        fac2 = fac2[(fac2["MonthEnd"] >= (start_date + pd.offsets.MonthEnd(0))) &
                    (fac2["MonthEnd"] <= (end_date + pd.offsets.MonthEnd(-1)))].copy()

        ann_fac = {}
        for c in FACTOR_COLS:
            s = pd.to_numeric(fac2[c], errors="coerce").dropna()
            ann_fac[c] = float((1.0 + s).prod() - 1.0) if len(s) else np.nan

        rec = {"date": end_date}
        rec.update(ann_fac)
        rec["y"] = float(row["strategy_return_net"])
        rows.append(rec)

    df = pd.DataFrame(rows).dropna(subset=["y"] + FACTOR_COLS).copy()
    if len(df) < 3:
        return pd.DataFrame([{
            "n_years": int(len(df)),
            "R2": np.nan,
            "alpha": np.nan,
            **{f"beta_{c}": np.nan for c in FACTOR_COLS}
        }])

    y = df["y"].to_numpy(dtype=float)
    X = df[FACTOR_COLS].to_numpy(dtype=float)
    alpha, betas, r2 = ols_alpha_beta(y, X)

    out = {"n_years": int(len(df)), "R2": float(r2), "alpha": float(alpha)}
    for c, b in zip(FACTOR_COLS, betas):
        out[f"beta_{c}"] = float(b)
    return pd.DataFrame([out])


def save_annual_bundle(df: pd.DataFrame, out_annual_csv: Path, out_curve_csv: Path, out_perf_csv: Path):
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").reset_index(drop=True)

    df.to_csv(out_annual_csv, index=False, encoding="utf-8-sig")

    r = df["strategy_return_net"].astype(float)
    cum = (1.0 + r).cumprod()
    dd = compute_drawdown(cum)
    curve = pd.DataFrame({"date": df["date"], "ret": r, "cum": cum, "dd": dd})
    curve.to_csv(out_curve_csv, index=False, encoding="utf-8-sig")

    stats = perf_stats(r)
    perf = pd.DataFrame([stats])
    perf.to_csv(out_perf_csv, index=False, encoding="utf-8-sig")


def run_annual_backtest_with_and_without_riskcontrol_fast(
    enhanced_financial_data: pd.DataFrame,
    prices_df: pd.DataFrame,
    prices_by_code: Dict[str, pd.DataFrame],
    fac: pd.DataFrame,
    initial_capital: float = INITIAL_CAPITAL
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    rebalance_dates = [pd.Timestamp(f"{year}-10-01") for year in range(2016, 2026)]

    results_raw = []
    results_rc = []
    diag_ir_all = []

    daily_curves_raw = []
    daily_curves_rc  = []

    for i, rebalance_date in enumerate(tqdm(rebalance_dates[:-1], desc="10月1日リバランス（統合・高速）")):
        next_rebalance = rebalance_dates[i + 1]

        long_candidates = build_long_candidates(enhanced_financial_data, rebalance_date)
        if long_candidates.empty:
            logger.warning(f"{rebalance_date}: データ不足でスキップ")
            continue

        long_portfolio = build_unit_share_portfolio(
            long_candidates,
            target_positions=20,
            initial_capital=initial_capital
        )

        n_long = len(long_portfolio["stocks"])
        inv_amount = float(np.sum(long_portfolio["amounts"])) if n_long > 0 else 0.0
        inv_ratio_raw = inv_amount / initial_capital if initial_capital > 0 else 0.0

        logger.info(f"{rebalance_date.strftime('%Y-%m')}: long={n_long} invest={inv_amount:,.0f} ratio={inv_ratio_raw:.3f}")

        # RAW（年次）
        total_profit = 0.0
        for j, code in enumerate(long_portfolio["stocks"]):
            sh = long_portfolio["shares"][j]
            p0 = long_portfolio["prices"][j]
            p1 = get_near_price_fast(prices_by_code, code, next_rebalance, kind="last")
            if p1 is None:
                continue
            total_profit += (sh * p1 - sh * p0)

        gross_return_raw = (total_profit / initial_capital) if initial_capital > 0 else 0.0
        net_return_raw, tax_rate_raw = apply_tax_annual(gross_return_raw, initial_capital)

        results_raw.append({
            "date": next_rebalance,
            "strategy_return_gross": gross_return_raw,
            "strategy_return_net": net_return_raw,
            "tax": tax_rate_raw,
            "long_count": n_long,
            "investment_ratio": inv_ratio_raw,
        })

        # RiskControl（月次縮尺）
        monthly_df, ir_df = compute_monthly_portfolio_returns_with_riskcontrol_fast(
            long_portfolio, rebalance_date, next_rebalance, prices_by_code, fac
        )

        if monthly_df.empty:
            gross_return_rc = 0.0
            inv_ratio_rc_avg = 0.0
        else:
            gross_return_rc = annual_return_from_monthly(monthly_df["port_ret_total"])
            inv_ratio_rc_avg = float(monthly_df["invest_ratio"].mean())

        net_return_rc, tax_rate_rc = apply_tax_annual(gross_return_rc, initial_capital)

        results_rc.append({
            "date": next_rebalance,
            "strategy_return_gross": gross_return_rc,
            "strategy_return_net": net_return_rc,
            "tax": tax_rate_rc,
            "long_count": n_long,
            "investment_ratio": inv_ratio_rc_avg,
        })

        if not ir_df.empty:
            ir_df = ir_df.copy()
            ir_df["rebalance_start"] = rebalance_date
            ir_df["rebalance_end"] = next_rebalance
            diag_ir_all.append(ir_df)

        invest_ratio_map = None
        if not ir_df.empty:
            tmp_ir = ir_df.copy()
            tmp_ir["MonthEnd"] = pd.to_datetime(tmp_ir["MonthEnd"], errors="coerce").dt.normalize()
            tmp_ir = tmp_ir.dropna(subset=["MonthEnd"])
            tmp_ir = tmp_ir.sort_values("MonthEnd").drop_duplicates(subset=["MonthEnd"], keep="last")
            invest_ratio_map = dict(zip(tmp_ir["MonthEnd"], tmp_ir["invest_ratio"].astype(float)))

        daily_raw = build_daily_equity_curve_for_period(
            long_portfolio, rebalance_date, next_rebalance, prices_by_code, invest_ratio_by_monthend=None
        )
        if not daily_raw.empty:
            daily_raw["rebalance_start"] = rebalance_date
            daily_raw["rebalance_end"] = next_rebalance
            daily_curves_raw.append(daily_raw)

        daily_rc = build_daily_equity_curve_for_period(
            long_portfolio, rebalance_date, next_rebalance, prices_by_code, invest_ratio_by_monthend=invest_ratio_map
        )
        if not daily_rc.empty:
            daily_rc["rebalance_start"] = rebalance_date
            daily_rc["rebalance_end"] = next_rebalance
            daily_curves_rc.append(daily_rc)

    df_raw = pd.DataFrame(results_raw)
    df_rc = pd.DataFrame(results_rc)
    diag_ir = pd.concat(diag_ir_all, ignore_index=True) if len(diag_ir_all) else pd.DataFrame()

    daily_raw_all = pd.concat(daily_curves_raw, ignore_index=True) if len(daily_curves_raw) else pd.DataFrame()
    daily_rc_all  = pd.concat(daily_curves_rc,  ignore_index=True) if len(daily_curves_rc)  else pd.DataFrame()

    run_annual_backtest_with_and_without_riskcontrol_fast._daily_raw_all = daily_raw_all
    run_annual_backtest_with_and_without_riskcontrol_fast._daily_rc_all = daily_rc_all

    return df_raw, df_rc, diag_ir


# ===================================
# 1ケース実行（DDストップ）
# ===================================
def run_one_case_ddstop(
    dd_cutoff_1: float,
    dd_cutoff_2: float,
    dd_ratio_1: float,
    dd_ratio_2: float,
    enhanced_financial_data: pd.DataFrame,
    prices_df: pd.DataFrame,
    prices_by_code: Dict[str, pd.DataFrame],
    fac: pd.DataFrame,
) -> Dict:

    # 不整合はスキップ（例：cut2が浅い）
    if not (dd_cutoff_2 < dd_cutoff_1):
        return {
            "dd_cutoff_1": float(dd_cutoff_1),
            "dd_cutoff_2": float(dd_cutoff_2),
            "dd_ratio_1": float(dd_ratio_1),
            "dd_ratio_2": float(dd_ratio_2),
            "daily_mdd": np.nan,
            "cagr": np.nan,
            "days_to_recovery": np.nan,
            "out_dir": "",
            "note": "skipped: cutoff_2 must be < cutoff_1"
        }

    suffix = f"dd1_{fmt_cut(dd_cutoff_1)}_r{int(dd_ratio_1*10)}__dd2_{fmt_cut(dd_cutoff_2)}_r{int(dd_ratio_2*10)}"
    set_output_dir(BASE_OUT_DIR, suffix)

    logger.info("=" * 110)
    logger.info(f"[GRID] START DDSTOP case | RISKOFF_RATIO={RISKOFF_RATIO} | dd1={dd_cutoff_1}→{dd_ratio_1} | dd2={dd_cutoff_2}→{dd_ratio_2}")
    logger.info(f"[GRID] OUT_DIR={OUT_DIR}")
    logger.info("=" * 110)

    # まず既存の RAW/RC を作る（データ取得部分は既存のまま）
    df_raw, df_rc, diag_ir = run_annual_backtest_with_and_without_riskcontrol_fast(
        enhanced_financial_data, prices_df, prices_by_code, fac, initial_capital=INITIAL_CAPITAL
    )
    if df_raw.empty or df_rc.empty:
        logger.error("年次バックテスト結果が空です")
        return {
            "dd_cutoff_1": float(dd_cutoff_1),
            "dd_cutoff_2": float(dd_cutoff_2),
            "dd_ratio_1": float(dd_ratio_1),
            "dd_ratio_2": float(dd_ratio_2),
            "daily_mdd": np.nan,
            "cagr": np.nan,
            "days_to_recovery": np.nan,
            "out_dir": str(OUT_DIR),
            "note": "empty_result"
        }

    # 既存保存（RAW/RC）
    save_annual_bundle(df_raw, OUT_ANNUAL_RAW, OUT_CURVE_RAW, OUT_PERF_RAW)
    save_annual_bundle(df_rc,  OUT_ANNUAL_RC,  OUT_CURVE_RC,  OUT_PERF_RC)

    reg_raw = run_factor_regression(df_raw, fac)
    reg_rc  = run_factor_regression(df_rc, fac)
    reg_raw.to_csv(OUT_REG_RAW, index=False, encoding="utf-8-sig")
    reg_rc.to_csv(OUT_REG_RC,  index=False, encoding="utf-8-sig")

    if not diag_ir.empty:
        diag_ir = diag_ir.copy()
        diag_ir["MonthEnd"] = pd.to_datetime(diag_ir["MonthEnd"])
        diag_ir.sort_values(["rebalance_start", "MonthEnd"], inplace=True)
        diag_ir.to_csv(OUT_IR_DIAG, index=False, encoding="utf-8-sig")

    # 日次（RAW/RC）
    daily_raw_all = getattr(run_annual_backtest_with_and_without_riskcontrol_fast, "_daily_raw_all", pd.DataFrame())
    daily_rc_all  = getattr(run_annual_backtest_with_and_without_riskcontrol_fast, "_daily_rc_all",  pd.DataFrame())

    if not daily_raw_all.empty:
        daily_raw_all.to_csv(OUT_DAILY_CURVE_RAW, index=False, encoding="utf-8-sig")
    if not daily_rc_all.empty:
        daily_rc_all.to_csv(OUT_DAILY_CURVE_RC, index=False, encoding="utf-8-sig")

    # ★DDストップ適用（日次）
    ddstop_daily = apply_dd_stop_to_daily_curve(
        daily_rc_all,
        dd_cutoff_1=dd_cutoff_1,
        dd_ratio_1=dd_ratio_1,
        dd_cutoff_2=dd_cutoff_2,
        dd_ratio_2=dd_ratio_2,
    )
    if ddstop_daily.empty:
        return {
            "dd_cutoff_1": float(dd_cutoff_1),
            "dd_cutoff_2": float(dd_cutoff_2),
            "dd_ratio_1": float(dd_ratio_1),
            "dd_ratio_2": float(dd_ratio_2),
            "daily_mdd": np.nan,
            "cagr": np.nan,
            "days_to_recovery": np.nan,
            "out_dir": str(OUT_DIR),
            "note": "ddstop_daily_empty"
        }

    # ddstop版 日次CSV
    ddstop_out = ddstop_daily.copy()
    # 互換的に列名を揃える（equity_total, dd_total等を ddstop のものに差し替えたビューも作る）
    ddstop_view = ddstop_out.copy()
    ddstop_view["invest_ratio"] = ddstop_view["invest_ratio_ddstop"]
    ddstop_view["ret_total"] = ddstop_view["ret_total_ddstop"]
    ddstop_view["equity_total"] = ddstop_view["equity_total_ddstop"]
    ddstop_view["dd_total"] = ddstop_view["dd_total_ddstop"]
    ddstop_view.to_csv(OUT_DAILY_CURVE_DDSTOP, index=False, encoding="utf-8-sig")

    # 日次MDD（ddstop）
    mdd_ddstop = float(pd.to_numeric(ddstop_view["dd_total"], errors="coerce").dropna().min())

    # DDイベント（raw/rc/ddstop）
    ev_raw = extract_max_drawdown_event_from_daily_curve(daily_raw_all, label="RAW")
    ev_rc  = extract_max_drawdown_event_from_daily_curve(daily_rc_all,  label="RISKCONTROL")
    ev_dd  = extract_max_drawdown_event_from_daily_curve(ddstop_view,    label="DDSTOP")

    ev_raw.to_csv(OUT_DD_EVENTS_RAW, index=False, encoding="utf-8-sig")
    ev_rc.to_csv(OUT_DD_EVENTS_RC, index=False, encoding="utf-8-sig")
    ev_dd.to_csv(OUT_DD_EVENTS_DDSTOP, index=False, encoding="utf-8-sig")
    pd.concat([ev_raw, ev_rc, ev_dd], ignore_index=True).to_csv(OUT_DD_EVENTS_MAX, index=False, encoding="utf-8-sig")

    # ddstop版 年次リターン（日次equityから再構築→年次課税）
    df_ddstop = build_annual_from_daily_equity(ddstop_view, equity_col="equity_total")
    if df_ddstop.empty:
        cagr_ddstop = np.nan
        days_to_recovery = np.nan
    else:
        save_annual_bundle(df_ddstop, OUT_ANNUAL_DDSTOP, OUT_CURVE_DDSTOP, OUT_PERF_DDSTOP)
        reg_dd = run_factor_regression(df_ddstop, fac)
        reg_dd.to_csv(OUT_REG_DDSTOP, index=False, encoding="utf-8-sig")

        perf_dd = pd.read_csv(OUT_PERF_DDSTOP).iloc[0].to_dict()
        cagr_ddstop = float(perf_dd.get("CAGR", np.nan))
        days_to_recovery = float(ev_dd.iloc[0].get("days_to_recovery", np.nan)) if not ev_dd.empty else np.nan

    # サマリ（ケース内）
    pd.DataFrame([{
        "daily_maxDD_raw": daily_mdd(daily_raw_all),
        "daily_maxDD_riskcontrol": daily_mdd(daily_rc_all),
        "daily_maxDD_ddstop": mdd_ddstop,
        "dd_cutoff_1": dd_cutoff_1,
        "dd_ratio_1": dd_ratio_1,
        "dd_cutoff_2": dd_cutoff_2,
        "dd_ratio_2": dd_ratio_2,
    }]).to_csv(OUT_DAILY_MDD_SUMMARY, index=False, encoding="utf-8-sig")

    logger.info(f"[GRID] DONE DDSTOP case | dailyMDD={mdd_ddstop:.6f} | CAGR={cagr_ddstop} | days_to_recovery={days_to_recovery}")

    return {
        "riskoff_ratio": float(RISKOFF_RATIO),
        "dd_cutoff_1": float(dd_cutoff_1),
        "dd_ratio_1": float(dd_ratio_1),
        "dd_cutoff_2": float(dd_cutoff_2),
        "dd_ratio_2": float(dd_ratio_2),
        "daily_mdd": float(mdd_ddstop) if np.isfinite(mdd_ddstop) else np.nan,
        "cagr": float(cagr_ddstop) if np.isfinite(cagr_ddstop) else np.nan,
        "days_to_recovery": float(days_to_recovery) if np.isfinite(days_to_recovery) else np.nan,
        "out_dir": str(OUT_DIR),
        "note": ""
    }


def main():
    logger.info("=" * 110)
    logger.info("10/1 年次「割安高質」バックテスト + FF5+MOM RiskControl(C) [DAILY DD-STOP GRID]")
    logger.info("=" * 110)
    logger.info(f"FF5MOM_FACTOR_PATH: {FF5MOM_FACTOR_PATH}")
    logger.info(f"BASE_OUT_DIR: {BASE_OUT_DIR}")
    logger.info(f"Tax mode: annual simple tax only (TAX_RATE={TAX_RATE}) | daily tax ignored")
    logger.info(f"Fixed: RISKOFF_RATIO={RISKOFF_RATIO}")
    logger.info("=" * 110)

    # 入力データ（1回だけ）
    statements_df = load_financial_data()
    if statements_df.empty:
        logger.error("財務データ読み込み失敗")
        return

    prices_df = load_existing_price_data()
    if prices_df.empty:
        logger.error("株価データ読み込み失敗")
        return

    fac = load_ff5mom_factors_monthly()
    prices_by_code = build_prices_by_code(prices_df)

    enhanced_financial_data = calculate_market_metrics_fast_chunked(statements_df, prices_df, chunk_size=200)
    if enhanced_financial_data.empty:
        logger.error("財務指標計算失敗")
        return

    # グリッド設定（必要ならここを調整）
    dd1_list = [-0.08, -0.10, -0.12]
    dd2_list = [-0.12, -0.15, -0.18]
    dd_ratio_1 = 0.3
    dd_ratio_2 = 0.0

    rows = []
    for dd1 in dd1_list:
        for dd2 in dd2_list:
            try:
                rec = run_one_case_ddstop(
                    dd_cutoff_1=dd1,
                    dd_cutoff_2=dd2,
                    dd_ratio_1=dd_ratio_1,
                    dd_ratio_2=dd_ratio_2,
                    enhanced_financial_data=enhanced_financial_data,
                    prices_df=prices_df,
                    prices_by_code=prices_by_code,
                    fac=fac,
                )
                rows.append(rec)
            except Exception as e:
                logger.error(f"[GRID] case failed: dd1={dd1} dd2={dd2} | err={e}", exc_info=True)
                rows.append({
                    "riskoff_ratio": float(RISKOFF_RATIO),
                    "dd_cutoff_1": float(dd1),
                    "dd_ratio_1": float(dd_ratio_1),
                    "dd_cutoff_2": float(dd2),
                    "dd_ratio_2": float(dd_ratio_2),
                    "daily_mdd": np.nan,
                    "cagr": np.nan,
                    "days_to_recovery": np.nan,
                    "out_dir": "",
                    "note": f"error: {e}"
                })

    summary = pd.DataFrame(rows)
    summary = summary.sort_values(["dd_cutoff_1", "dd_cutoff_2"]).reset_index(drop=True)
    summary.to_csv(OUT_GRID_SUMMARY, index=False, encoding="utf-8-sig")

    logger.info("-" * 110)
    logger.info("✅ GRID SUMMARY SAVED")
    logger.info(f"  - {OUT_GRID_SUMMARY}")
    logger.info("-" * 110)
    logger.info("[GRID TABLE]")
    logger.info(summary.to_string(index=False))


if __name__ == "__main__":
    main()


2026-01-27 04:27:38,771 - INFO - ==============================================================================================================
2026-01-27 04:27:38,772 - INFO - 10/1 年次「割安高質」バックテスト + FF5+MOM RiskControl(C) [DAILY DD-STOP GRID]
2026-01-27 04:27:38,772 - INFO - ==============================================================================================================
2026-01-27 04:27:38,772 - INFO - FF5MOM_FACTOR_PATH: C:\Users\yongr\Project\merged_data_all_stocks\factors\ff5_mom_factors_monthly.parquet
2026-01-27 04:27:38,773 - INFO - BASE_OUT_DIR: C:\Users\yongr\Project\merged_data_all_stocks\factors\bt_october_unit_with_ff5mom_ddstop_grid
2026-01-27 04:27:38,774 - INFO - Tax mode: annual simple tax only (TAX_RATE=0.20315) | daily tax ignored
2026-01-27 04:27:38,775 - INFO - Fixed: RISKOFF_RATIO=0.7
2026-01-27 04:27:38,775 - INFO - ==============================================================================================================
2026-01-27 04:27:39,382 - 

In [ ]:
# -*- coding: utf-8 -*-
"""
年次（10/1）割安高質バックテスト（100株単位・税引後）に
FF5+MOMのリスク制御（C：レジーム減速＋β制約）を統合。

【追加：グリッド実験（Regime閾値）】
- RISKOFF_RATIO は 0.7 固定
- REGIME_OFF_IF_SUM_MKT_LT と REGIME_OFF_IF_SUM_WML_LT を
  (0.00, -0.02, -0.05) のグリッド比較（9ケース）
- 指標: 日次MDD・CAGR（年次税引後）・days_to_recovery（最大DDイベントの回復日数）
- 出力: ケース別サブフォルダ（従来CSV一式） + grid_summary.csv

税金:
- 簡易：年次（10/1→翌10/1）の最終損益にのみ課税
- 日次は税考慮しない（実験目的のため）

依存:
  pip install pandas numpy pyarrow tqdm
"""

import warnings
warnings.filterwarnings("ignore")

import os
import logging
from pathlib import Path
from typing import Dict, Tuple, List, Optional

import numpy as np
import pandas as pd
from tqdm import tqdm


# ===================================
# ロギング設定
# ===================================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler("backtest_october_unit_with_ff5mom_regime_grid.log", encoding="utf-8"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)


# ===================================
# パス（あなたの環境）
# ===================================
FACTORS_DIR = Path(r"C:\Users\yongr\Project\merged_data_all_stocks\factors")
FF5MOM_FACTOR_PATH = FACTORS_DIR / "ff5_mom_factors_monthly.parquet"

CACHE_FILE = "topix_quarterly_statements.csv"
OHLCV_DIR = "./OHLCV_Adjusted"


# ===================================
# 出力（グリッド用ベース）
# ===================================
BASE_OUT_DIR = FACTORS_DIR / "bt_october_unit_with_ff5mom_regime_grid"
BASE_OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_GRID_SUMMARY = BASE_OUT_DIR / "grid_summary.csv"


# ===================================
# 税・単位株
# ===================================
TAX_RATE = 0.20315
UNIT_SHARES = 100
INITIAL_CAPITAL = 10_000_000


# ===================================
# リスク制御パラメータ
# ===================================
# ★固定
RISKOFF_RATIO = 0.7

REGIME_LOOKBACK_M = 3
# ★グリッド実験で動的に差し替える
REGIME_OFF_IF_SUM_MKT_LT = 0.0
REGIME_OFF_IF_SUM_WML_LT = 0.0

BETA_CAP_CMA_LT = -0.8
BETA_CAP_ABS_MKT_GT = 0.9

BETA_EST_WINDOW_M = 12
BETA_EST_MIN_OBS = 10

FACTOR_COLS = ["MKT", "SMB", "HML", "RMW", "CMA", "WML"]

BAD_CODE_STRINGS = {"None", "nan", "", "NaN", "NULL", "null"}


# ===================================
# 出力パス（ケースごとに set_output_dir で差し替える）
# ===================================
OUT_DIR = BASE_OUT_DIR / "case_tmp"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_ANNUAL_RAW = OUT_DIR / "annual_returns_raw.csv"
OUT_ANNUAL_RC  = OUT_DIR / "annual_returns_riskcontrol.csv"

OUT_CURVE_RAW  = OUT_DIR / "cumulative_curve_raw.csv"
OUT_CURVE_RC   = OUT_DIR / "cumulative_curve_riskcontrol.csv"

OUT_PERF_RAW   = OUT_DIR / "performance_summary_raw.csv"
OUT_PERF_RC    = OUT_DIR / "performance_summary_riskcontrol.csv"

OUT_REG_RAW    = OUT_DIR / "regression_ff5mom_raw.csv"
OUT_REG_RC     = OUT_DIR / "regression_ff5mom_riskcontrol.csv"

OUT_IR_DIAG    = OUT_DIR / "diag_invest_ratio_monthly.csv"

OUT_DAILY_CURVE_RAW = OUT_DIR / "daily_equity_curve_raw.csv"
OUT_DAILY_CURVE_RC  = OUT_DIR / "daily_equity_curve_riskcontrol.csv"
OUT_DAILY_MDD_SUMMARY = OUT_DIR / "daily_mdd_summary.csv"

OUT_DD_EVENTS_RAW = OUT_DIR / "daily_drawdown_events_raw.csv"
OUT_DD_EVENTS_RC  = OUT_DIR / "daily_drawdown_events_riskcontrol.csv"
OUT_DD_EVENTS_MAX = OUT_DIR / "daily_drawdown_events_max_summary.csv"


def set_output_dir(base_dir: Path, suffix: str):
    """
    ループ実験で出力先を切り替えるための関数。
    suffix例: 'mkt0p00_wmlm0p02'
    """
    global OUT_DIR
    global OUT_ANNUAL_RAW, OUT_ANNUAL_RC
    global OUT_CURVE_RAW, OUT_CURVE_RC
    global OUT_PERF_RAW, OUT_PERF_RC
    global OUT_REG_RAW, OUT_REG_RC
    global OUT_IR_DIAG
    global OUT_DAILY_CURVE_RAW, OUT_DAILY_CURVE_RC, OUT_DAILY_MDD_SUMMARY
    global OUT_DD_EVENTS_RAW, OUT_DD_EVENTS_RC, OUT_DD_EVENTS_MAX

    OUT_DIR = base_dir / suffix
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    OUT_ANNUAL_RAW = OUT_DIR / "annual_returns_raw.csv"
    OUT_ANNUAL_RC  = OUT_DIR / "annual_returns_riskcontrol.csv"

    OUT_CURVE_RAW  = OUT_DIR / "cumulative_curve_raw.csv"
    OUT_CURVE_RC   = OUT_DIR / "cumulative_curve_riskcontrol.csv"

    OUT_PERF_RAW   = OUT_DIR / "performance_summary_raw.csv"
    OUT_PERF_RC    = OUT_DIR / "performance_summary_riskcontrol.csv"

    OUT_REG_RAW    = OUT_DIR / "regression_ff5mom_raw.csv"
    OUT_REG_RC     = OUT_DIR / "regression_ff5mom_riskcontrol.csv"

    OUT_IR_DIAG    = OUT_DIR / "diag_invest_ratio_monthly.csv"

    OUT_DAILY_CURVE_RAW = OUT_DIR / "daily_equity_curve_raw.csv"
    OUT_DAILY_CURVE_RC  = OUT_DIR / "daily_equity_curve_riskcontrol.csv"
    OUT_DAILY_MDD_SUMMARY = OUT_DIR / "daily_mdd_summary.csv"

    OUT_DD_EVENTS_RAW = OUT_DIR / "daily_drawdown_events_raw.csv"
    OUT_DD_EVENTS_RC  = OUT_DIR / "daily_drawdown_events_riskcontrol.csv"
    OUT_DD_EVENTS_MAX = OUT_DIR / "daily_drawdown_events_max_summary.csv"


def fmt_thr(x: float) -> str:
    """
    フォルダ名用の閾値表記
    0.00 -> 0p00, -0.02 -> m0p02
    """
    x = float(x)
    s = f"{abs(x):.2f}".replace(".", "p")
    return f"m{s}" if x < 0 else s


# ===================================
# ユーティリティ
# ===================================
def normalize_code(code) -> str:
    if code is None:
        return ""
    s = str(code).strip()
    if s in BAD_CODE_STRINGS:
        return ""
    return s


def ols_alpha_beta(y: np.ndarray, X: np.ndarray) -> Tuple[float, np.ndarray, float]:
    n = len(y)
    if n < 3:
        return np.nan, np.full(X.shape[1], np.nan), np.nan

    X1 = np.column_stack([np.ones(n), X])
    XtX = X1.T @ X1
    try:
        inv = np.linalg.inv(XtX)
    except np.linalg.LinAlgError:
        inv = np.linalg.pinv(XtX)
    b = inv @ (X1.T @ y)

    yhat = X1 @ b
    resid = y - yhat
    sse = float(np.sum(resid**2))
    sst = float(np.sum((y - y.mean())**2))
    r2 = np.nan if sst <= 0 else (1.0 - sse / sst)

    alpha = float(b[0])
    betas = b[1:].astype(float)
    return alpha, betas, float(r2)


def compute_drawdown(cum: pd.Series) -> pd.Series:
    peak = cum.cummax()
    return cum / peak - 1.0


def perf_stats(annual_ret: pd.Series) -> Dict:
    r = annual_ret.dropna().astype(float)
    if r.empty:
        return {"n_years": 0, "CAGR": np.nan, "ann_mean": np.nan, "ann_vol": np.nan, "sharpe0": np.nan, "maxDD": np.nan, "cum_end": np.nan}

    n = len(r)
    cum = (1.0 + r).cumprod()
    years = n
    cagr = float(cum.iloc[-1] ** (1/years) - 1.0) if years > 0 else np.nan
    ann_mean = float(r.mean())
    ann_vol = float(r.std(ddof=1)) if n >= 2 else np.nan
    sharpe0 = float(ann_mean / ann_vol) if ann_vol and ann_vol > 0 else np.nan
    dd = compute_drawdown(cum)
    maxdd = float(dd.min())

    return {
        "n_years": int(n),
        "CAGR": cagr,
        "ann_mean": ann_mean,
        "ann_vol": ann_vol,
        "sharpe0": sharpe0,
        "maxDD": maxdd,
        "cum_end": float(cum.iloc[-1]),
    }


def extract_max_drawdown_event_from_daily_curve(df_daily: pd.DataFrame, label: str = "") -> pd.DataFrame:
    if df_daily is None or df_daily.empty:
        return pd.DataFrame([{
            "label": label,
            "peak_date": pd.NaT,
            "trough_date": pd.NaT,
            "recovery_date": pd.NaT,
            "dd_min": np.nan,
            "peak_equity": np.nan,
            "trough_equity": np.nan,
            "recovery_equity": np.nan,
            "days_to_trough": np.nan,
            "days_to_recovery": np.nan,
            "dd_duration_days": np.nan,
        }])

    df = df_daily.copy()
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df = df.dropna(subset=["Date"]).sort_values("Date").reset_index(drop=True)

    for col in ["equity_total", "dd_total"]:
        if col not in df.columns:
            raise KeyError(f"daily curve missing required column: {col}")

    df["equity_total"] = pd.to_numeric(df["equity_total"], errors="coerce")
    df["dd_total"] = pd.to_numeric(df["dd_total"], errors="coerce")
    df = df.dropna(subset=["equity_total", "dd_total"]).copy()
    if df.empty:
        return pd.DataFrame([{
            "label": label,
            "peak_date": pd.NaT,
            "trough_date": pd.NaT,
            "recovery_date": pd.NaT,
            "dd_min": np.nan,
            "peak_equity": np.nan,
            "trough_equity": np.nan,
            "recovery_equity": np.nan,
            "days_to_trough": np.nan,
            "days_to_recovery": np.nan,
            "dd_duration_days": np.nan,
        }])

    trough_idx = int(df["dd_total"].idxmin())
    dd_min = float(df.loc[trough_idx, "dd_total"])
    trough_date = pd.Timestamp(df.loc[trough_idx, "Date"])
    trough_equity = float(df.loc[trough_idx, "equity_total"])

    df_pre = df.loc[:trough_idx].copy()
    peak_equity = float(df_pre["equity_total"].max())
    peak_idx = int(df_pre["equity_total"].idxmax())
    peak_date = pd.Timestamp(df.loc[peak_idx, "Date"])

    df_post = df.loc[trough_idx:].copy()
    rec = df_post[df_post["equity_total"] >= peak_equity]
    if len(rec) == 0:
        recovery_date = pd.NaT
        recovery_equity = np.nan
        days_to_recovery = np.nan
        dd_duration_days = np.nan
    else:
        rec_idx = int(rec.index[0])
        recovery_date = pd.Timestamp(df.loc[rec_idx, "Date"])
        recovery_equity = float(df.loc[rec_idx, "equity_total"])
        days_to_recovery = int((recovery_date - peak_date).days)
        dd_duration_days = int((recovery_date - peak_date).days)

    days_to_trough = int((trough_date - peak_date).days)

    return pd.DataFrame([{
        "label": label,
        "peak_date": peak_date,
        "trough_date": trough_date,
        "recovery_date": recovery_date,
        "dd_min": dd_min,
        "peak_equity": peak_equity,
        "trough_equity": trough_equity,
        "recovery_equity": recovery_equity,
        "days_to_trough": days_to_trough,
        "days_to_recovery": days_to_recovery,
        "dd_duration_days": dd_duration_days,
    }])


def daily_mdd(df_daily: pd.DataFrame) -> float:
    if df_daily is None or df_daily.empty or "dd_total" not in df_daily.columns:
        return np.nan
    s = pd.to_numeric(df_daily["dd_total"], errors="coerce").dropna()
    return float(s.min()) if len(s) else np.nan


# ===================================
# A. 財務データ読み込み（列名自動判定）
# ===================================
def load_financial_data(cache_filename: str = CACHE_FILE) -> pd.DataFrame:
    if not os.path.exists(cache_filename):
        logger.error(f"財務データファイル '{cache_filename}' が見つかりません")
        return pd.DataFrame()

    try:
        df = pd.read_csv(cache_filename, encoding="utf-8-sig", parse_dates=["DisclosedDate"], low_memory=False)
        logger.info(f"財務データ読み込み成功: {len(df):,}件")

        column_mapping = {
            "IssuedShareTotal": ["IssuedShareTotal", "NumberOfIssuedAndOutstandingSharesAtTheEndOfFiscalYearIncludingTreasuryStock"],
            "Equity": ["Equity", "NetAssets", "TotalEquity"],
            "Profit": ["Profit", "NetIncome", "ProfitAttributableToOwnersOfParent"],
        }

        available_cols = df.columns.tolist()
        required_columns = {}
        for target_col, possible_names in column_mapping.items():
            found = False
            for possible_name in possible_names:
                if possible_name in available_cols:
                    required_columns[target_col] = possible_name
                    found = True
                    break
            if not found:
                required_columns[target_col] = None

        rename_dict = {v: k for k, v in required_columns.items() if v is not None}
        df = df.rename(columns=rename_dict)

        for col in ["IssuedShareTotal", "Equity", "Profit"]:
            if col not in df.columns:
                df[col] = 0

        base_cols = ["Code", "DisclosedDate"]
        if "CompanyName" in df.columns:
            base_cols.append("CompanyName")
        final_cols = base_cols + ["Profit", "Equity", "IssuedShareTotal"]
        df = df[final_cols].copy()

        logger.info(f"使用列: {df.columns.tolist()}")
        return df

    except Exception as e:
        logger.error(f"財務データ読み込みエラー: {e}", exc_info=True)
        return pd.DataFrame()


# ===================================
# B. 株価データ読み込み
# ===================================
def load_existing_price_data(ohlcv_dir: str = OHLCV_DIR) -> pd.DataFrame:
    if not os.path.exists(ohlcv_dir):
        logger.error(f"株価データディレクトリ '{ohlcv_dir}' が見つかりません。")
        return pd.DataFrame()

    csv_files = sorted([f for f in os.listdir(ohlcv_dir)
                        if f.startswith("OHLCV_Adjusted_") and f.endswith(".csv") and f != "OHLCV_Adjusted_TOPIX.csv"])

    if not csv_files:
        logger.error(f"ディレクトリ '{ohlcv_dir}' 内にCSVファイルが見つかりません。")
        return pd.DataFrame()

    logger.info(f"株価ファイル数: {len(csv_files)}個")

    all_dataframes = []
    usecols = ["Date", "Ticker", "AdjustmentClose"]

    for csv_file in tqdm(csv_files, desc="株価ファイル読み込み中"):
        file_path = os.path.join(ohlcv_dir, csv_file)
        try:
            df = pd.read_csv(
                file_path,
                usecols=usecols,
                parse_dates=["Date"],
                dtype={"Ticker": "Int64", "AdjustmentClose": "float32"},
            )
            df = df.drop_duplicates(subset=["Ticker", "Date"], keep="first")
            all_dataframes.append(df)
        except Exception as e:
            logger.warning(f"ファイル読み込みエラー ({csv_file}): {e}")
            continue

    if not all_dataframes:
        return pd.DataFrame()

    logger.info("全ファイルを結合中...")
    df_all = pd.concat(all_dataframes, ignore_index=True)
    logger.info(f"結合完了: {len(df_all):,}件")

    df_all = df_all.rename(columns={"Ticker": "Code", "AdjustmentClose": "Close"})
    df_all["Code"] = df_all["Code"].astype("str").str.replace("<NA>", "0").str.zfill(4)
    df_all = df_all.dropna(subset=["Close"])

    logger.info("重複除去 & ソート中...")
    df_all = df_all.sort_values(["Code", "Date"])
    df_all = df_all.drop_duplicates(subset=["Code", "Date"], keep="first")
    logger.info(f"重複除去後: {len(df_all):,}件")

    return df_all


def build_prices_by_code(prices_df: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    d = {}
    tmp = prices_df[["Code", "Date", "Close"]].copy()
    tmp["Code"] = tmp["Code"].astype(str).map(normalize_code)
    tmp = tmp[tmp["Code"] != ""].copy()
    tmp = tmp.sort_values(["Code", "Date"])

    for code, g in tmp.groupby("Code", sort=False):
        gg = g[["Date", "Close"]].drop_duplicates(subset=["Date"], keep="last").sort_values("Date").copy()
        gg = gg.set_index("Date", drop=False)
        d[code] = gg

    logger.info(f"prices_by_code built: {len(d):,} codes")
    return d


def get_near_price_fast(prices_by_code: Dict[str, pd.DataFrame],
                        code: str,
                        ref_date: pd.Timestamp,
                        kind: str = "last") -> Optional[float]:
    code = normalize_code(code)
    if not code or code not in prices_by_code:
        return None
    g = prices_by_code[code]
    lo = ref_date - pd.Timedelta(days=5)
    hi = ref_date + pd.Timedelta(days=5)
    w = g.loc[(g["Date"] >= lo) & (g["Date"] <= hi)]
    if w.empty:
        return None
    return float(w.iloc[0]["Close"]) if kind == "first" else float(w.iloc[-1]["Close"])


def safe_code_to_int(code_series: pd.Series) -> pd.Series:
    cleaned = code_series.astype(str).str.replace(r"\D", "", regex=True)
    cleaned = cleaned.replace("", "0")
    return pd.to_numeric(cleaned, errors="coerce").fillna(0).astype("int64")


def calculate_market_metrics_fast_chunked(statements_df: pd.DataFrame,
                                          prices_df: pd.DataFrame,
                                          chunk_size: int = 200) -> pd.DataFrame:
    logger.info("時価総額・PBR・ROE計算中（チャンク処理版）...")

    if prices_df.empty:
        logger.error("株価データが空です。")
        return pd.DataFrame()

    req_cols = {"Code", "Date", "Close"}
    if not req_cols.issubset(set(prices_df.columns)):
        logger.error(f"株価データに必要な列が存在しません。存在する列: {prices_df.columns.tolist()}")
        return pd.DataFrame()

    statements_df = statements_df.copy()
    statements_df["Profit"] = pd.to_numeric(statements_df["Profit"], errors="coerce").fillna(0)
    statements_df["Equity"] = pd.to_numeric(statements_df["Equity"], errors="coerce").fillna(0)
    statements_df["IssuedShareTotal"] = pd.to_numeric(statements_df["IssuedShareTotal"], errors="coerce").fillna(1)

    statements_df = statements_df[(statements_df["Equity"] > 0) & (statements_df["IssuedShareTotal"] > 0)]
    logger.info(f"有効な財務データ: {len(statements_df):,}件")

    statements_df["Code_int"] = safe_code_to_int(statements_df["Code"])
    prices_df = prices_df.copy()
    prices_df["Code_int"] = safe_code_to_int(prices_df["Code"])

    statements_df = statements_df[statements_df["Code_int"] > 0]
    prices_df = prices_df[prices_df["Code_int"] > 0]

    statements_df = statements_df.sort_values(["Code_int", "DisclosedDate"]).reset_index(drop=True)
    prices_df = prices_df.sort_values(["Code_int", "Date"]).reset_index(drop=True)

    statements_df = statements_df.drop_duplicates(subset=["Code_int", "DisclosedDate"], keep="first")
    prices_df = prices_df.drop_duplicates(subset=["Code_int", "Date"], keep="first")

    logger.info(f"ソート・重複除去後: 財務 {len(statements_df):,}件, 株価 {len(prices_df):,}件")

    statements_groups = list(statements_df.groupby("Code_int"))
    prices_dict = {code: group for code, group in prices_df.groupby("Code_int")}

    merged_list = []
    num_chunks = (len(statements_groups) + chunk_size - 1) // chunk_size

    for chunk_idx in tqdm(range(num_chunks), desc="マージ処理"):
        start_idx = chunk_idx * chunk_size
        end_idx = min((chunk_idx + 1) * chunk_size, len(statements_groups))
        chunk_groups = statements_groups[start_idx:end_idx]

        for code, stmt_code in chunk_groups:
            if code not in prices_dict:
                continue
            price_code = prices_dict[code]
            if len(price_code) == 0:
                continue

            stmt_code = stmt_code.sort_values("DisclosedDate").reset_index(drop=True)
            price_code = price_code.sort_values("Date").reset_index(drop=True)

            try:
                merged = pd.merge_asof(
                    stmt_code,
                    price_code[["Date", "Close"]],
                    left_on="DisclosedDate",
                    right_on="Date",
                    direction="backward",
                    tolerance=pd.Timedelta(days=10),
                )
                if not merged.empty:
                    merged_list.append(merged)
            except Exception:
                continue

    if not merged_list:
        logger.error("マージ結果が空です")
        return pd.DataFrame()

    df_merged = pd.concat(merged_list, ignore_index=True)
    df_merged = df_merged.dropna(subset=["Close"])
    logger.info(f"マージ完了: {len(df_merged):,}件")

    df_merged["MarketCap"] = df_merged["Close"] * df_merged["IssuedShareTotal"]
    df_merged["PBR"] = df_merged["MarketCap"] / df_merged["Equity"]
    df_merged["ROE"] = (df_merged["Profit"] / df_merged["Equity"]) * 100

    result_cols = ["Code", "DisclosedDate", "Close", "MarketCap", "PBR", "ROE", "Date"]
    if "CompanyName" in df_merged.columns:
        result_cols.insert(1, "CompanyName")

    result_df = df_merged[result_cols].copy()
    result_df = result_df.rename(columns={"Close": "StockPrice", "Date": "PriceDate"})

    mask = (
        (result_df["PBR"] > 0) &
        (result_df["PBR"] < 50) &
        (result_df["ROE"] > -100) &
        (result_df["ROE"] < 100) &
        (result_df["MarketCap"] > 1_000_000_000)
    )
    result_df = result_df[mask].copy()

    logger.info(f"計算完了: {len(result_df):,}件")
    return result_df


def build_unit_share_portfolio(stock_candidates: pd.DataFrame,
                               target_positions: int = 20,
                               initial_capital: float = 10_000_000) -> dict:
    if len(stock_candidates) == 0:
        return {"stocks": [], "shares": [], "prices": [], "amounts": []}

    selected = stock_candidates.head(target_positions).copy()
    capital_per_stock = initial_capital / len(selected)

    stocks, shares_list, prices_list, amounts_list = [], [], [], []

    for _, row in selected.iterrows():
        code = row["Code"]
        price = row["StockPrice"]
        required_amount = price * UNIT_SHARES

        if required_amount <= capital_per_stock:
            shares = int(capital_per_stock // required_amount) * UNIT_SHARES
            if shares > 0:
                stocks.append(code)
                shares_list.append(shares)
                prices_list.append(price)
                amounts_list.append(shares * price)

    return {"stocks": stocks, "shares": shares_list, "prices": prices_list, "amounts": amounts_list}


def make_month_ends(start: pd.Timestamp, end: pd.Timestamp) -> List[pd.Timestamp]:
    m0 = pd.Timestamp(start.year, start.month, 1) + pd.offsets.MonthEnd(0)
    m1 = pd.Timestamp(end.year, end.month, 1) + pd.offsets.MonthEnd(0)
    months = pd.date_range(m0, m1, freq="M")
    return [pd.Timestamp(x).normalize() for x in months]


def build_daily_equity_curve_for_period(
    portfolio: dict,
    start_date: pd.Timestamp,
    end_date: pd.Timestamp,
    prices_by_code: Dict[str, pd.DataFrame],
    invest_ratio_by_monthend: Optional[Dict[pd.Timestamp, float]] = None,
) -> pd.DataFrame:
    stocks = portfolio.get("stocks", [])
    shares = portfolio.get("shares", [])
    if not stocks:
        return pd.DataFrame()

    start_date = pd.to_datetime(start_date).normalize()
    end_date = pd.to_datetime(end_date).normalize()

    date_sets = []
    for code in stocks:
        code = normalize_code(code)
        if code in prices_by_code:
            g = prices_by_code[code]
            d = g[(g["Date"] >= start_date - pd.Timedelta(days=10)) & (g["Date"] <= end_date + pd.Timedelta(days=10))]["Date"]
            if len(d):
                date_sets.append(d)

    if not date_sets:
        return pd.DataFrame()

    dates = pd.Index(sorted(pd.unique(pd.concat(date_sets)))).astype("datetime64[ns]")
    dates = dates[(dates >= start_date) & (dates <= end_date)]
    if len(dates) == 0:
        return pd.DataFrame()

    values = []
    for dt in dates:
        v = 0.0
        ok = False
        for i, code in enumerate(stocks):
            code = normalize_code(code)
            if code not in prices_by_code:
                continue
            p = get_near_price_fast(prices_by_code, code, pd.Timestamp(dt), kind="last")
            if p is None:
                continue
            v += float(shares[i]) * float(p)
            ok = True
        values.append(v if ok else np.nan)

    df = pd.DataFrame({"Date": pd.to_datetime(dates), "equity_stock": values})
    df = df.dropna(subset=["equity_stock"]).copy()
    if df.empty:
        return df

    df = df.sort_values("Date").reset_index(drop=True)
    df["ret_stock"] = df["equity_stock"].pct_change().fillna(0.0)
    df["MonthEnd"] = (df["Date"] + pd.offsets.MonthEnd(0)).dt.normalize()

    if invest_ratio_by_monthend is None:
        df["invest_ratio"] = 1.0
        df["ret_total"] = df["ret_stock"]
    else:
        df["invest_ratio"] = df["MonthEnd"].map(invest_ratio_by_monthend).fillna(1.0).astype(float)
        df["ret_total"] = df["invest_ratio"] * df["ret_stock"]

    df["equity_total"] = (1.0 + df["ret_total"]).cumprod()
    peak = df["equity_total"].cummax()
    df["dd_total"] = df["equity_total"] / peak - 1.0

    return df[["Date", "MonthEnd", "equity_stock", "ret_stock", "invest_ratio", "ret_total", "equity_total", "dd_total"]]


def load_ff5mom_factors_monthly() -> pd.DataFrame:
    fac = pd.read_parquet(FF5MOM_FACTOR_PATH).copy()
    fac["MonthEnd"] = pd.to_datetime(fac["MonthEnd"], errors="coerce").dt.normalize()
    need = ["MonthEnd"] + FACTOR_COLS
    missing = [c for c in need if c not in fac.columns]
    if missing:
        raise KeyError(f"FF5MOM factors missing columns: {missing} in {FF5MOM_FACTOR_PATH}")
    fac = fac[need].sort_values("MonthEnd").reset_index(drop=True)
    return fac


def compute_regime_off(fac: pd.DataFrame, month_end: pd.Timestamp) -> Tuple[bool, Dict]:
    fac2 = fac.set_index("MonthEnd")
    idx = fac2.index[fac2.index < month_end]
    if len(idx) < REGIME_LOOKBACK_M:
        return False, {"regime_ready": False}

    win = idx[-REGIME_LOOKBACK_M:]
    s_mkt = float(pd.to_numeric(fac2.loc[win, "MKT"], errors="coerce").sum())
    s_wml = float(pd.to_numeric(fac2.loc[win, "WML"], errors="coerce").sum())

    # Option3: OR
    off = (s_mkt < REGIME_OFF_IF_SUM_MKT_LT) or (s_wml < REGIME_OFF_IF_SUM_WML_LT)
    return bool(off), {"regime_ready": True, "sum_MKT_3m": s_mkt, "sum_WML_3m": s_wml}


def estimate_port_beta(monthly_port_rets: pd.DataFrame, fac: pd.DataFrame, month_end: pd.Timestamp) -> Tuple[bool, Dict]:
    fac2 = fac.set_index("MonthEnd")
    idx = fac2.index[fac2.index < month_end]
    if len(idx) < BETA_EST_WINDOW_M:
        return False, {"beta_ready": False}

    win = idx[-BETA_EST_WINDOW_M:]

    m = monthly_port_rets.copy()
    m["MonthEnd"] = pd.to_datetime(m["MonthEnd"], errors="coerce").dt.normalize()
    m = m.dropna(subset=["MonthEnd", "port_ret"]).copy()

    m = (
        m.groupby("MonthEnd", as_index=False)["port_ret"]
         .apply(lambda s: (1.0 + s.astype(float)).prod() - 1.0)
    )

    df = m[m["MonthEnd"].isin(win)].merge(
        fac, on="MonthEnd", how="left", validate="one_to_one"
    ).dropna(subset=["port_ret"] + FACTOR_COLS)

    if len(df) < BETA_EST_MIN_OBS:
        return False, {"beta_ready": False, "n_obs": int(len(df))}

    y = df["port_ret"].to_numpy(dtype=float)
    X = df[FACTOR_COLS].to_numpy(dtype=float)
    alpha, betas, r2 = ols_alpha_beta(y, X)

    out = {"beta_ready": True, "n_obs": int(len(df)), "r2": float(r2), "alpha": float(alpha)}
    for c, b in zip(FACTOR_COLS, betas):
        out[f"beta_{c}"] = float(b)
    return True, out


def decide_invest_ratio(month_end: pd.Timestamp,
                        fac: pd.DataFrame,
                        port_hist: pd.DataFrame) -> Tuple[float, Dict]:
    invest_ratio = 1.0
    diag = {
        "MonthEnd": month_end,
        "invest_ratio": 1.0,
        "regime_off": False,
        "beta_off": False,
        "regime_ready": False,
        "beta_ready": False,
        "sum_MKT_3m": np.nan,
        "sum_WML_3m": np.nan,
        "beta_MKT": np.nan,
        "beta_CMA": np.nan,
        "beta_n_obs": np.nan,
        "beta_r2": np.nan,
    }

    off_reg, info = compute_regime_off(fac, month_end)
    diag["regime_off"] = bool(off_reg)
    diag["regime_ready"] = bool(info.get("regime_ready", False))
    diag["sum_MKT_3m"] = info.get("sum_MKT_3m", np.nan)
    diag["sum_WML_3m"] = info.get("sum_WML_3m", np.nan)
    if off_reg:
        invest_ratio = min(invest_ratio, RISKOFF_RATIO)

    ok_beta, binfo = estimate_port_beta(port_hist, fac, month_end)
    diag["beta_ready"] = bool(binfo.get("beta_ready", False))
    if binfo.get("beta_ready", False):
        b_mkt = float(binfo.get("beta_MKT", np.nan))
        b_cma = float(binfo.get("beta_CMA", np.nan))
        diag["beta_MKT"] = b_mkt
        diag["beta_CMA"] = b_cma
        diag["beta_n_obs"] = float(binfo.get("n_obs", np.nan))
        diag["beta_r2"] = float(binfo.get("r2", np.nan))

        off_beta = (b_cma < BETA_CAP_CMA_LT) or (abs(b_mkt) > BETA_CAP_ABS_MKT_GT)
        diag["beta_off"] = bool(off_beta)
        if off_beta:
            invest_ratio = min(invest_ratio, RISKOFF_RATIO)

    diag["invest_ratio"] = float(invest_ratio)
    return float(invest_ratio), diag


def compute_monthly_portfolio_returns_with_riskcontrol_fast(
    portfolio: dict,
    start_date: pd.Timestamp,
    end_date: pd.Timestamp,
    prices_by_code: Dict[str, pd.DataFrame],
    fac: pd.DataFrame
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    stocks = portfolio["stocks"]
    shares = portfolio["shares"]
    if not stocks:
        return pd.DataFrame(), pd.DataFrame()

    month_ends = make_month_ends(start_date, end_date)
    boundaries = [start_date] + [me for me in month_ends if (me > start_date) and (me < end_date)] + [end_date]

    rows = []
    ir_rows = []
    port_hist = pd.DataFrame(columns=["MonthEnd", "port_ret"])

    for j in range(len(boundaries) - 1):
        d0 = boundaries[j]
        d1 = boundaries[j + 1]

        me = pd.Timestamp(d0.year, d0.month, 1) + pd.offsets.MonthEnd(0)
        me = pd.Timestamp(me).normalize()

        start_vals = []
        end_vals = []
        for i, code in enumerate(stocks):
            p0 = get_near_price_fast(prices_by_code, code, d0, kind="first")
            p1 = get_near_price_fast(prices_by_code, code, d1, kind="last")
            if p0 is None or p1 is None:
                continue
            sh = shares[i]
            start_vals.append(sh * p0)
            end_vals.append(sh * p1)

        if len(start_vals) == 0:
            continue

        start_v = float(np.sum(start_vals))
        end_v = float(np.sum(end_vals))
        stock_ret = (end_v / start_v) - 1.0

        port_hist = pd.concat([port_hist, pd.DataFrame([{"MonthEnd": me, "port_ret": stock_ret}])], ignore_index=True)

        invest_ratio, diag = decide_invest_ratio(me, fac, port_hist)
        ir_rows.append(diag)

        total_ret = invest_ratio * stock_ret

        rows.append({
            "MonthEnd": me,
            "period_start": d0,
            "period_end": d1,
            "port_ret_stock": stock_ret,
            "invest_ratio": invest_ratio,
            "port_ret_total": total_ret,
        })

    monthly_df = pd.DataFrame(rows)
    ir_diag_df = pd.DataFrame(ir_rows)
    return monthly_df, ir_diag_df


def build_long_candidates(enhanced_financial_data: pd.DataFrame, rebalance_date: pd.Timestamp) -> pd.DataFrame:
    current_data = enhanced_financial_data[enhanced_financial_data["DisclosedDate"] <= rebalance_date].copy()
    current_data = current_data.sort_values("DisclosedDate").groupby("Code").tail(1)

    if len(current_data) < 100:
        return pd.DataFrame()

    current_data["PBR_Rank"] = current_data["PBR"].rank(method="first", ascending=True)
    current_data["ROE_Rank"] = current_data["ROE"].rank(method="first", ascending=False)

    current_data["PBR_Quartile"] = pd.qcut(current_data["PBR_Rank"], q=4, labels=[1, 2, 3, 4])
    current_data["ROE_Quartile"] = pd.qcut(current_data["ROE_Rank"], q=4, labels=[1, 2, 3, 4])

    long_candidates = current_data[
        (current_data["PBR_Quartile"] == 1) &
        (current_data["ROE_Quartile"] == 4)
    ].nsmallest(50, "PBR")

    return long_candidates


def annual_return_from_monthly(monthly_rets: pd.Series) -> float:
    if monthly_rets.empty:
        return 0.0
    return float((1.0 + monthly_rets).prod() - 1.0)


def apply_tax_annual(gross_return: float, initial_capital: float) -> Tuple[float, float]:
    profit = gross_return * initial_capital
    taxable = max(profit, 0.0)
    tax = taxable * TAX_RATE
    net_profit = profit - tax
    net_return = net_profit / initial_capital
    tax_rate_total = tax / initial_capital
    return float(net_return), float(tax_rate_total)


def run_annual_backtest_with_and_without_riskcontrol_fast(
    enhanced_financial_data: pd.DataFrame,
    prices_df: pd.DataFrame,
    prices_by_code: Dict[str, pd.DataFrame],
    fac: pd.DataFrame,
    initial_capital: float = INITIAL_CAPITAL
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    rebalance_dates = [pd.Timestamp(f"{year}-10-01") for year in range(2016, 2026)]

    results_raw = []
    results_rc = []
    diag_ir_all = []

    daily_curves_raw = []
    daily_curves_rc  = []

    for i, rebalance_date in enumerate(tqdm(rebalance_dates[:-1], desc="10月1日リバランス（統合・高速）")):
        next_rebalance = rebalance_dates[i + 1]

        long_candidates = build_long_candidates(enhanced_financial_data, rebalance_date)
        if long_candidates.empty:
            logger.warning(f"{rebalance_date}: データ不足でスキップ")
            continue

        long_portfolio = build_unit_share_portfolio(
            long_candidates,
            target_positions=20,
            initial_capital=initial_capital
        )

        n_long = len(long_portfolio["stocks"])
        inv_amount = float(np.sum(long_portfolio["amounts"])) if n_long > 0 else 0.0
        inv_ratio_raw = inv_amount / initial_capital if initial_capital > 0 else 0.0

        logger.info(f"{rebalance_date.strftime('%Y-%m')}: long={n_long} invest={inv_amount:,.0f} ratio={inv_ratio_raw:.3f}")

        # --- RAW（年次 start->end の価格で計算）
        total_profit = 0.0
        for j, code in enumerate(long_portfolio["stocks"]):
            sh = long_portfolio["shares"][j]
            p0 = long_portfolio["prices"][j]
            p1 = get_near_price_fast(prices_by_code, code, next_rebalance, kind="last")
            if p1 is None:
                continue
            start_value = sh * p0
            end_value = sh * p1
            total_profit += (end_value - start_value)

        gross_return_raw = (total_profit / initial_capital) if initial_capital > 0 else 0.0
        net_return_raw, tax_rate_raw = apply_tax_annual(gross_return_raw, initial_capital)

        results_raw.append({
            "date": next_rebalance,
            "strategy_return_gross": gross_return_raw,
            "strategy_return_net": net_return_raw,
            "tax": tax_rate_raw,
            "long_count": n_long,
            "investment_ratio": inv_ratio_raw,
        })

        # --- RiskControl（月次で縮尺）
        monthly_df, ir_df = compute_monthly_portfolio_returns_with_riskcontrol_fast(
            long_portfolio, rebalance_date, next_rebalance, prices_by_code, fac
        )

        if monthly_df.empty:
            gross_return_rc = 0.0
            inv_ratio_rc_avg = 0.0
        else:
            gross_return_rc = annual_return_from_monthly(monthly_df["port_ret_total"])
            inv_ratio_rc_avg = float(monthly_df["invest_ratio"].mean())

        net_return_rc, tax_rate_rc = apply_tax_annual(gross_return_rc, initial_capital)

        results_rc.append({
            "date": next_rebalance,
            "strategy_return_gross": gross_return_rc,
            "strategy_return_net": net_return_rc,
            "tax": tax_rate_rc,
            "long_count": n_long,
            "investment_ratio": inv_ratio_rc_avg,
        })

        if not ir_df.empty:
            ir_df = ir_df.copy()
            ir_df["rebalance_start"] = rebalance_date
            ir_df["rebalance_end"] = next_rebalance
            diag_ir_all.append(ir_df)

        invest_ratio_map = None
        if not ir_df.empty:
            tmp_ir = ir_df.copy()
            tmp_ir["MonthEnd"] = pd.to_datetime(tmp_ir["MonthEnd"], errors="coerce").dt.normalize()
            tmp_ir = tmp_ir.dropna(subset=["MonthEnd"])
            tmp_ir = tmp_ir.sort_values("MonthEnd").drop_duplicates(subset=["MonthEnd"], keep="last")
            invest_ratio_map = dict(zip(tmp_ir["MonthEnd"], tmp_ir["invest_ratio"].astype(float)))

        daily_raw = build_daily_equity_curve_for_period(
            long_portfolio, rebalance_date, next_rebalance, prices_by_code, invest_ratio_by_monthend=None
        )
        if not daily_raw.empty:
            daily_raw["rebalance_start"] = rebalance_date
            daily_raw["rebalance_end"] = next_rebalance
            daily_curves_raw.append(daily_raw)

        daily_rc = build_daily_equity_curve_for_period(
            long_portfolio, rebalance_date, next_rebalance, prices_by_code, invest_ratio_by_monthend=invest_ratio_map
        )
        if not daily_rc.empty:
            daily_rc["rebalance_start"] = rebalance_date
            daily_rc["rebalance_end"] = next_rebalance
            daily_curves_rc.append(daily_rc)

    df_raw = pd.DataFrame(results_raw)
    df_rc = pd.DataFrame(results_rc)
    diag_ir = pd.concat(diag_ir_all, ignore_index=True) if len(diag_ir_all) else pd.DataFrame()

    daily_raw_all = pd.concat(daily_curves_raw, ignore_index=True) if len(daily_curves_raw) else pd.DataFrame()
    daily_rc_all  = pd.concat(daily_curves_rc,  ignore_index=True) if len(daily_curves_rc)  else pd.DataFrame()

    # 既存互換
    run_annual_backtest_with_and_without_riskcontrol_fast._daily_raw_all = daily_raw_all
    run_annual_backtest_with_and_without_riskcontrol_fast._daily_rc_all = daily_rc_all

    return df_raw, df_rc, diag_ir


def build_annual_factor_from_monthly(fac: pd.DataFrame, start_date: pd.Timestamp, end_date: pd.Timestamp) -> Dict:
    fac2 = fac.copy()
    fac2 = fac2[(fac2["MonthEnd"] >= (start_date + pd.offsets.MonthEnd(0))) &
                (fac2["MonthEnd"] <= (end_date + pd.offsets.MonthEnd(-1)))].copy()
    out = {}
    for c in FACTOR_COLS:
        s = pd.to_numeric(fac2[c], errors="coerce").dropna()
        out[c] = float((1.0 + s).prod() - 1.0) if len(s) else np.nan
    return out


def run_factor_regression(results_df: pd.DataFrame, fac: pd.DataFrame) -> pd.DataFrame:
    if results_df.empty:
        return pd.DataFrame()

    rows = []
    for _, row in results_df.iterrows():
        end_date = pd.Timestamp(row["date"])
        start_date = end_date - pd.DateOffset(years=1)
        ann_fac = build_annual_factor_from_monthly(fac, start_date, end_date)
        rec = {"date": end_date}
        rec.update(ann_fac)
        rec["y"] = float(row["strategy_return_net"])
        rows.append(rec)

    df = pd.DataFrame(rows).dropna(subset=["y"] + FACTOR_COLS).copy()
    if len(df) < 3:
        return pd.DataFrame([{
            "n_years": int(len(df)),
            "R2": np.nan,
            "alpha": np.nan,
            **{f"beta_{c}": np.nan for c in FACTOR_COLS}
        }])

    y = df["y"].to_numpy(dtype=float)
    X = df[FACTOR_COLS].to_numpy(dtype=float)
    alpha, betas, r2 = ols_alpha_beta(y, X)

    out = {"n_years": int(len(df)), "R2": float(r2), "alpha": float(alpha)}
    for c, b in zip(FACTOR_COLS, betas):
        out[f"beta_{c}"] = float(b)
    return pd.DataFrame([out])


def save_annual_bundle(df: pd.DataFrame, out_annual_csv: Path, out_curve_csv: Path, out_perf_csv: Path):
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").reset_index(drop=True)

    df.to_csv(out_annual_csv, index=False, encoding="utf-8-sig")

    r = df["strategy_return_net"].astype(float)
    cum = (1.0 + r).cumprod()
    dd = compute_drawdown(cum)
    curve = pd.DataFrame({"date": df["date"], "ret": r, "cum": cum, "dd": dd})
    curve.to_csv(out_curve_csv, index=False, encoding="utf-8-sig")

    stats = perf_stats(r)
    perf = pd.DataFrame([stats])
    perf.to_csv(out_perf_csv, index=False, encoding="utf-8-sig")


def run_one_case(
    mkt_thr: float,
    wml_thr: float,
    enhanced_financial_data: pd.DataFrame,
    prices_df: pd.DataFrame,
    prices_by_code: Dict[str, pd.DataFrame],
    fac: pd.DataFrame
) -> Dict:
    """
    1ケース実行して、比較指標を辞書で返す（加えてCSVも保存する）
    """
    global REGIME_OFF_IF_SUM_MKT_LT, REGIME_OFF_IF_SUM_WML_LT

    REGIME_OFF_IF_SUM_MKT_LT = float(mkt_thr)
    REGIME_OFF_IF_SUM_WML_LT = float(wml_thr)

    suffix = f"mkt_{fmt_thr(mkt_thr)}__wml_{fmt_thr(wml_thr)}"
    set_output_dir(BASE_OUT_DIR, suffix)

    logger.info("=" * 110)
    logger.info(f"[GRID] START case: RISKOFF_RATIO={RISKOFF_RATIO} | MKT_THR={REGIME_OFF_IF_SUM_MKT_LT} | WML_THR={REGIME_OFF_IF_SUM_WML_LT}")
    logger.info(f"[GRID] OUT_DIR={OUT_DIR}")
    logger.info("=" * 110)

    df_raw, df_rc, diag_ir = run_annual_backtest_with_and_without_riskcontrol_fast(
        enhanced_financial_data, prices_df, prices_by_code, fac, initial_capital=INITIAL_CAPITAL
    )

    if df_raw.empty or df_rc.empty:
        logger.error("年次バックテスト結果が空です")
        return {
            "mkt_thr": REGIME_OFF_IF_SUM_MKT_LT,
            "wml_thr": REGIME_OFF_IF_SUM_WML_LT,
            "daily_mdd": np.nan,
            "cagr": np.nan,
            "days_to_recovery": np.nan,
            "out_dir": str(OUT_DIR),
            "note": "empty_result"
        }

    # 年次出力
    save_annual_bundle(df_raw, OUT_ANNUAL_RAW, OUT_CURVE_RAW, OUT_PERF_RAW)
    save_annual_bundle(df_rc,  OUT_ANNUAL_RC,  OUT_CURVE_RC,  OUT_PERF_RC)

    # 回帰
    reg_raw = run_factor_regression(df_raw, fac)
    reg_rc  = run_factor_regression(df_rc, fac)
    reg_raw.to_csv(OUT_REG_RAW, index=False, encoding="utf-8-sig")
    reg_rc.to_csv(OUT_REG_RC, index=False, encoding="utf-8-sig")

    # 診断（月次）
    if not diag_ir.empty:
        diag_ir = diag_ir.copy()
        diag_ir["MonthEnd"] = pd.to_datetime(diag_ir["MonthEnd"])
        diag_ir.sort_values(["rebalance_start", "MonthEnd"], inplace=True)
        diag_ir.to_csv(OUT_IR_DIAG, index=False, encoding="utf-8-sig")

    # 日次出力
    daily_raw_all = getattr(run_annual_backtest_with_and_without_riskcontrol_fast, "_daily_raw_all", pd.DataFrame())
    daily_rc_all  = getattr(run_annual_backtest_with_and_without_riskcontrol_fast, "_daily_rc_all", pd.DataFrame())

    if not daily_raw_all.empty:
        daily_raw_all.to_csv(OUT_DAILY_CURVE_RAW, index=False, encoding="utf-8-sig")
    if not daily_rc_all.empty:
        daily_rc_all.to_csv(OUT_DAILY_CURVE_RC, index=False, encoding="utf-8-sig")

    # 日次MDD + DDイベント
    mdd_rc_d = daily_mdd(daily_rc_all)

    ev_rc = extract_max_drawdown_event_from_daily_curve(daily_rc_all, label="RISKCONTROL")
    ev_raw = extract_max_drawdown_event_from_daily_curve(daily_raw_all, label="RAW")

    ev_raw.to_csv(OUT_DD_EVENTS_RAW, index=False, encoding="utf-8-sig")
    ev_rc.to_csv(OUT_DD_EVENTS_RC, index=False, encoding="utf-8-sig")
    pd.concat([ev_raw, ev_rc], ignore_index=True).to_csv(OUT_DD_EVENTS_MAX, index=False, encoding="utf-8-sig")

    pd.DataFrame([{
        "daily_maxDD_raw": daily_mdd(daily_raw_all),
        "daily_maxDD_riskcontrol": mdd_rc_d,
        "n_days_raw": int(len(daily_raw_all)) if not daily_raw_all.empty else 0,
        "n_days_riskcontrol": int(len(daily_rc_all)) if not daily_rc_all.empty else 0,
        "RISKOFF_RATIO": RISKOFF_RATIO,
        "REGIME_OFF_IF_SUM_MKT_LT": REGIME_OFF_IF_SUM_MKT_LT,
        "REGIME_OFF_IF_SUM_WML_LT": REGIME_OFF_IF_SUM_WML_LT,
    }]).to_csv(OUT_DAILY_MDD_SUMMARY, index=False, encoding="utf-8-sig")

    # CAGR（RC側）
    rc_perf = pd.read_csv(OUT_PERF_RC).iloc[0].to_dict() if OUT_PERF_RC.exists() else {}
    cagr_rc = float(rc_perf.get("CAGR", np.nan))

    # 回復日数（RC側）
    days_to_recovery = float(ev_rc.iloc[0].get("days_to_recovery", np.nan)) if not ev_rc.empty else np.nan

    logger.info(f"[GRID] DONE case: MKT_THR={REGIME_OFF_IF_SUM_MKT_LT} | WML_THR={REGIME_OFF_IF_SUM_WML_LT} | dailyMDD={mdd_rc_d:.6f} | CAGR={cagr_rc:.6f} | days_to_recovery={days_to_recovery}")

    return {
        "riskoff_ratio": float(RISKOFF_RATIO),
        "mkt_thr": float(REGIME_OFF_IF_SUM_MKT_LT),
        "wml_thr": float(REGIME_OFF_IF_SUM_WML_LT),
        "daily_mdd": float(mdd_rc_d) if np.isfinite(mdd_rc_d) else np.nan,
        "cagr": float(cagr_rc) if np.isfinite(cagr_rc) else np.nan,
        "days_to_recovery": float(days_to_recovery) if np.isfinite(days_to_recovery) else np.nan,
        "out_dir": str(OUT_DIR),
        "note": ""
    }


def main():
    logger.info("=" * 110)
    logger.info("10/1 年次「割安高質」バックテスト + FF5+MOM RiskControl(C) [REGIME THRESHOLD GRID]")
    logger.info("=" * 110)
    logger.info(f"FF5MOM_FACTOR_PATH: {FF5MOM_FACTOR_PATH}")
    logger.info(f"BASE_OUT_DIR: {BASE_OUT_DIR}")
    logger.info(f"Tax mode: annual simple tax only (TAX_RATE={TAX_RATE}) | daily tax ignored")
    logger.info(f"Fixed: RISKOFF_RATIO={RISKOFF_RATIO}")
    logger.info("Grid: REGIME_OFF_IF_SUM_MKT_LT x REGIME_OFF_IF_SUM_WML_LT in (0.00, -0.02, -0.05)")
    logger.info("Regime: off if (sum_MKT_3m < MKT_THR) OR (sum_WML_3m < WML_THR)")
    logger.info("=" * 110)

    # ---- 入力データ読み込み（ここは1回だけ）
    statements_df = load_financial_data()
    if statements_df.empty:
        logger.error("財務データ読み込み失敗")
        return

    prices_df = load_existing_price_data()
    if prices_df.empty:
        logger.error("株価データ読み込み失敗")
        return

    fac = load_ff5mom_factors_monthly()
    prices_by_code = build_prices_by_code(prices_df)

    enhanced_financial_data = calculate_market_metrics_fast_chunked(statements_df, prices_df, chunk_size=200)
    if enhanced_financial_data.empty:
        logger.error("財務指標計算失敗")
        return

    # ---- グリッド実験（9ケース）
    grid_vals = [0.00, -0.02, -0.05]
    rows = []
    for mkt_thr in grid_vals:
        for wml_thr in grid_vals:
            try:
                rec = run_one_case(
                    mkt_thr=mkt_thr,
                    wml_thr=wml_thr,
                    enhanced_financial_data=enhanced_financial_data,
                    prices_df=prices_df,
                    prices_by_code=prices_by_code,
                    fac=fac,
                )
                rows.append(rec)
            except Exception as e:
                logger.error(f"[GRID] case failed: MKT_THR={mkt_thr} | WML_THR={wml_thr} | err={e}", exc_info=True)
                rows.append({
                    "riskoff_ratio": float(RISKOFF_RATIO),
                    "mkt_thr": float(mkt_thr),
                    "wml_thr": float(wml_thr),
                    "daily_mdd": np.nan,
                    "cagr": np.nan,
                    "days_to_recovery": np.nan,
                    "out_dir": "",
                    "note": f"error: {e}"
                })

    summary = pd.DataFrame(rows)
    summary = summary.sort_values(["mkt_thr", "wml_thr"]).reset_index(drop=True)
    summary.to_csv(OUT_GRID_SUMMARY, index=False, encoding="utf-8-sig")

    logger.info("-" * 110)
    logger.info("✅ GRID SUMMARY SAVED")
    logger.info(f"  - {OUT_GRID_SUMMARY}")
    logger.info("-" * 110)
    logger.info("[GRID TABLE]")
    logger.info(summary.to_string(index=False))


if __name__ == "__main__":
    main()


In [2]:
# -*- coding: utf-8 -*-
"""
10/1 年次「割安高質」バックテスト（100株単位・税引後）に
FF5+MOMのリスク制御（C：レジーム減速＋β制約）を統合。

【追加：日次DDストップ + DD回復復帰（方式1） グリッド実験】
- 月次RiskControl（regime/beta）で出る invest_ratio（月次）をベースに日次カーブを作成
- 日次 dd_total を監視し、閾値dd2で 0.0 に退避
- dd_total が回復したら段階的に復帰（ヒステリシス）
    dd <= dd2  : invest_ratio = 0.0
    dd >= rec1 : invest_ratio = 0.3
    dd >= rec2 : invest_ratio = base（月次のriskcontrol比率に戻す）
- 指標: 日次MDD / CAGR（年次税引後）/ days_to_recovery（最大DDイベントの回復日数）
- 出力: ケース別サブフォルダ + grid_summary.csv

税:
- 年次のみ簡易課税
- 日次は税無視

依存:
  pip install pandas numpy pyarrow tqdm
"""

import warnings
warnings.filterwarnings("ignore")

import os
import logging
from pathlib import Path
from typing import Dict, Tuple, List, Optional

import numpy as np
import pandas as pd
from tqdm import tqdm


# ===================================
# ロギング
# ===================================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler("backtest_october_unit_with_ff5mom_ddstop_recover_grid.log", encoding="utf-8"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)


# ===================================
# パス（あなたの環境）
# ===================================
FACTORS_DIR = Path(r"C:\Users\yongr\Project\merged_data_all_stocks\factors")
FF5MOM_FACTOR_PATH = FACTORS_DIR / "ff5_mom_factors_monthly.parquet"

CACHE_FILE = "topix_quarterly_statements.csv"
OHLCV_DIR = "./OHLCV_Adjusted"


# ===================================
# 出力（グリッド用ベース）
# ===================================
BASE_OUT_DIR = FACTORS_DIR / "bt_october_unit_with_ff5mom_ddstop_recover_grid"
BASE_OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_GRID_SUMMARY = BASE_OUT_DIR / "grid_summary.csv"


# ===================================
# 税・単位株
# ===================================
TAX_RATE = 0.20315
UNIT_SHARES = 100
INITIAL_CAPITAL = 10_000_000


# ===================================
# リスク制御パラメータ（固定）
# ===================================
RISKOFF_RATIO = 0.7

REGIME_LOOKBACK_M = 3
REGIME_OFF_IF_SUM_MKT_LT = 0.0
REGIME_OFF_IF_SUM_WML_LT = 0.0

BETA_CAP_CMA_LT = -0.8
BETA_CAP_ABS_MKT_GT = 0.9

BETA_EST_WINDOW_M = 12
BETA_EST_MIN_OBS = 10

FACTOR_COLS = ["MKT", "SMB", "HML", "RMW", "CMA", "WML"]
BAD_CODE_STRINGS = {"None", "nan", "", "NaN", "NULL", "null"}


# ===================================
# 出力パス（ケースごとに切替）
# ===================================
OUT_DIR = BASE_OUT_DIR / "case_tmp"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_ANNUAL_RAW = OUT_DIR / "annual_returns_raw.csv"
OUT_ANNUAL_RC  = OUT_DIR / "annual_returns_riskcontrol.csv"
OUT_ANNUAL_DDREC = OUT_DIR / "annual_returns_ddstop_recover.csv"

OUT_CURVE_RAW  = OUT_DIR / "cumulative_curve_raw.csv"
OUT_CURVE_RC   = OUT_DIR / "cumulative_curve_riskcontrol.csv"
OUT_CURVE_DDREC = OUT_DIR / "cumulative_curve_ddstop_recover.csv"

OUT_PERF_RAW   = OUT_DIR / "performance_summary_raw.csv"
OUT_PERF_RC    = OUT_DIR / "performance_summary_riskcontrol.csv"
OUT_PERF_DDREC = OUT_DIR / "performance_summary_ddstop_recover.csv"

OUT_REG_RAW    = OUT_DIR / "regression_ff5mom_raw.csv"
OUT_REG_RC     = OUT_DIR / "regression_ff5mom_riskcontrol.csv"
OUT_REG_DDREC  = OUT_DIR / "regression_ff5mom_ddstop_recover.csv"

OUT_IR_DIAG    = OUT_DIR / "diag_invest_ratio_monthly.csv"

OUT_DAILY_CURVE_RAW = OUT_DIR / "daily_equity_curve_raw.csv"
OUT_DAILY_CURVE_RC  = OUT_DIR / "daily_equity_curve_riskcontrol.csv"
OUT_DAILY_CURVE_DDREC = OUT_DIR / "daily_equity_curve_ddstop_recover.csv"

OUT_DAILY_MDD_SUMMARY = OUT_DIR / "daily_mdd_summary.csv"

OUT_DD_EVENTS_RAW = OUT_DIR / "daily_drawdown_events_raw.csv"
OUT_DD_EVENTS_RC  = OUT_DIR / "daily_drawdown_events_riskcontrol.csv"
OUT_DD_EVENTS_DDREC = OUT_DIR / "daily_drawdown_events_ddstop_recover.csv"
OUT_DD_EVENTS_MAX = OUT_DIR / "daily_drawdown_events_max_summary.csv"


def set_output_dir(base_dir: Path, suffix: str):
    global OUT_DIR
    global OUT_ANNUAL_RAW, OUT_ANNUAL_RC, OUT_ANNUAL_DDREC
    global OUT_CURVE_RAW, OUT_CURVE_RC, OUT_CURVE_DDREC
    global OUT_PERF_RAW, OUT_PERF_RC, OUT_PERF_DDREC
    global OUT_REG_RAW, OUT_REG_RC, OUT_REG_DDREC
    global OUT_IR_DIAG
    global OUT_DAILY_CURVE_RAW, OUT_DAILY_CURVE_RC, OUT_DAILY_CURVE_DDREC
    global OUT_DAILY_MDD_SUMMARY
    global OUT_DD_EVENTS_RAW, OUT_DD_EVENTS_RC, OUT_DD_EVENTS_DDREC, OUT_DD_EVENTS_MAX

    OUT_DIR = base_dir / suffix
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    OUT_ANNUAL_RAW = OUT_DIR / "annual_returns_raw.csv"
    OUT_ANNUAL_RC  = OUT_DIR / "annual_returns_riskcontrol.csv"
    OUT_ANNUAL_DDREC = OUT_DIR / "annual_returns_ddstop_recover.csv"

    OUT_CURVE_RAW  = OUT_DIR / "cumulative_curve_raw.csv"
    OUT_CURVE_RC   = OUT_DIR / "cumulative_curve_riskcontrol.csv"
    OUT_CURVE_DDREC = OUT_DIR / "cumulative_curve_ddstop_recover.csv"

    OUT_PERF_RAW   = OUT_DIR / "performance_summary_raw.csv"
    OUT_PERF_RC    = OUT_DIR / "performance_summary_riskcontrol.csv"
    OUT_PERF_DDREC = OUT_DIR / "performance_summary_ddstop_recover.csv"

    OUT_REG_RAW    = OUT_DIR / "regression_ff5mom_raw.csv"
    OUT_REG_RC     = OUT_DIR / "regression_ff5mom_riskcontrol.csv"
    OUT_REG_DDREC  = OUT_DIR / "regression_ff5mom_ddstop_recover.csv"

    OUT_IR_DIAG    = OUT_DIR / "diag_invest_ratio_monthly.csv"

    OUT_DAILY_CURVE_RAW = OUT_DIR / "daily_equity_curve_raw.csv"
    OUT_DAILY_CURVE_RC  = OUT_DIR / "daily_equity_curve_riskcontrol.csv"
    OUT_DAILY_CURVE_DDREC = OUT_DIR / "daily_equity_curve_ddstop_recover.csv"

    OUT_DAILY_MDD_SUMMARY = OUT_DIR / "daily_mdd_summary.csv"

    OUT_DD_EVENTS_RAW = OUT_DIR / "daily_drawdown_events_raw.csv"
    OUT_DD_EVENTS_RC  = OUT_DIR / "daily_drawdown_events_riskcontrol.csv"
    OUT_DD_EVENTS_DDREC = OUT_DIR / "daily_drawdown_events_ddstop_recover.csv"
    OUT_DD_EVENTS_MAX = OUT_DIR / "daily_drawdown_events_max_summary.csv"


def fmt_cut(x: float) -> str:
    x = float(x)
    s = f"{abs(x):.2f}".replace(".", "p")
    return f"m{s}" if x < 0 else s


# ===================================
# ユーティリティ
# ===================================
def normalize_code(code) -> str:
    if code is None:
        return ""
    s = str(code).strip()
    if s in BAD_CODE_STRINGS:
        return ""
    return s


def ols_alpha_beta(y: np.ndarray, X: np.ndarray) -> Tuple[float, np.ndarray, float]:
    n = len(y)
    if n < 3:
        return np.nan, np.full(X.shape[1], np.nan), np.nan

    X1 = np.column_stack([np.ones(n), X])
    XtX = X1.T @ X1
    try:
        inv = np.linalg.inv(XtX)
    except np.linalg.LinAlgError:
        inv = np.linalg.pinv(XtX)
    b = inv @ (X1.T @ y)

    yhat = X1 @ b
    resid = y - yhat
    sse = float(np.sum(resid**2))
    sst = float(np.sum((y - y.mean())**2))
    r2 = np.nan if sst <= 0 else (1.0 - sse / sst)

    alpha = float(b[0])
    betas = b[1:].astype(float)
    return alpha, betas, float(r2)


def compute_drawdown(cum: pd.Series) -> pd.Series:
    peak = cum.cummax()
    return cum / peak - 1.0


def perf_stats(annual_ret: pd.Series) -> Dict:
    r = annual_ret.dropna().astype(float)
    if r.empty:
        return {"n_years": 0, "CAGR": np.nan, "ann_mean": np.nan, "ann_vol": np.nan, "sharpe0": np.nan, "maxDD": np.nan, "cum_end": np.nan}

    n = len(r)
    cum = (1.0 + r).cumprod()
    years = n
    cagr = float(cum.iloc[-1] ** (1/years) - 1.0) if years > 0 else np.nan
    ann_mean = float(r.mean())
    ann_vol = float(r.std(ddof=1)) if n >= 2 else np.nan
    sharpe0 = float(ann_mean / ann_vol) if ann_vol and ann_vol > 0 else np.nan
    dd = compute_drawdown(cum)
    maxdd = float(dd.min())

    return {
        "n_years": int(n),
        "CAGR": cagr,
        "ann_mean": ann_mean,
        "ann_vol": ann_vol,
        "sharpe0": sharpe0,
        "maxDD": maxdd,
        "cum_end": float(cum.iloc[-1]),
    }


def extract_max_drawdown_event_from_daily_curve(df_daily: pd.DataFrame, label: str = "") -> pd.DataFrame:
    if df_daily is None or df_daily.empty:
        return pd.DataFrame([{
            "label": label,
            "peak_date": pd.NaT,
            "trough_date": pd.NaT,
            "recovery_date": pd.NaT,
            "dd_min": np.nan,
            "peak_equity": np.nan,
            "trough_equity": np.nan,
            "recovery_equity": np.nan,
            "days_to_trough": np.nan,
            "days_to_recovery": np.nan,
            "dd_duration_days": np.nan,
        }])

    df = df_daily.copy()
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df = df.dropna(subset=["Date"]).sort_values("Date").reset_index(drop=True)

    for col in ["equity_total", "dd_total"]:
        if col not in df.columns:
            raise KeyError(f"daily curve missing required column: {col}")

    df["equity_total"] = pd.to_numeric(df["equity_total"], errors="coerce")
    df["dd_total"] = pd.to_numeric(df["dd_total"], errors="coerce")
    df = df.dropna(subset=["equity_total", "dd_total"]).copy()
    if df.empty:
        return pd.DataFrame([{
            "label": label,
            "peak_date": pd.NaT,
            "trough_date": pd.NaT,
            "recovery_date": pd.NaT,
            "dd_min": np.nan,
            "peak_equity": np.nan,
            "trough_equity": np.nan,
            "recovery_equity": np.nan,
            "days_to_trough": np.nan,
            "days_to_recovery": np.nan,
            "dd_duration_days": np.nan,
        }])

    trough_idx = int(df["dd_total"].idxmin())
    dd_min = float(df.loc[trough_idx, "dd_total"])
    trough_date = pd.Timestamp(df.loc[trough_idx, "Date"])
    trough_equity = float(df.loc[trough_idx, "equity_total"])

    df_pre = df.loc[:trough_idx].copy()
    peak_equity = float(df_pre["equity_total"].max())
    peak_idx = int(df_pre["equity_total"].idxmax())
    peak_date = pd.Timestamp(df.loc[peak_idx, "Date"])

    df_post = df.loc[trough_idx:].copy()
    rec = df_post[df_post["equity_total"] >= peak_equity]
    if len(rec) == 0:
        recovery_date = pd.NaT
        recovery_equity = np.nan
        days_to_recovery = np.nan
        dd_duration_days = np.nan
    else:
        rec_idx = int(rec.index[0])
        recovery_date = pd.Timestamp(df.loc[rec_idx, "Date"])
        recovery_equity = float(df.loc[rec_idx, "equity_total"])
        days_to_recovery = int((recovery_date - peak_date).days)
        dd_duration_days = int((recovery_date - peak_date).days)

    days_to_trough = int((trough_date - peak_date).days)

    return pd.DataFrame([{
        "label": label,
        "peak_date": peak_date,
        "trough_date": trough_date,
        "recovery_date": recovery_date,
        "dd_min": dd_min,
        "peak_equity": peak_equity,
        "trough_equity": trough_equity,
        "recovery_equity": recovery_equity,
        "days_to_trough": days_to_trough,
        "days_to_recovery": days_to_recovery,
        "dd_duration_days": dd_duration_days,
    }])


def daily_mdd(df_daily: pd.DataFrame) -> float:
    if df_daily is None or df_daily.empty or "dd_total" not in df_daily.columns:
        return np.nan
    s = pd.to_numeric(df_daily["dd_total"], errors="coerce").dropna()
    return float(s.min()) if len(s) else np.nan


# ===================================
# A. 財務データ読み込み（列名自動判定）
# ===================================
def load_financial_data(cache_filename: str = CACHE_FILE) -> pd.DataFrame:
    if not os.path.exists(cache_filename):
        logger.error(f"財務データファイル '{cache_filename}' が見つかりません")
        return pd.DataFrame()

    try:
        df = pd.read_csv(cache_filename, encoding="utf-8-sig", parse_dates=["DisclosedDate"], low_memory=False)
        logger.info(f"財務データ読み込み成功: {len(df):,}件")

        column_mapping = {
            "IssuedShareTotal": ["IssuedShareTotal", "NumberOfIssuedAndOutstandingSharesAtTheEndOfFiscalYearIncludingTreasuryStock"],
            "Equity": ["Equity", "NetAssets", "TotalEquity"],
            "Profit": ["Profit", "NetIncome", "ProfitAttributableToOwnersOfParent"],
        }

        available_cols = df.columns.tolist()
        required_columns = {}
        for target_col, possible_names in column_mapping.items():
            found = False
            for possible_name in possible_names:
                if possible_name in available_cols:
                    required_columns[target_col] = possible_name
                    found = True
                    break
            if not found:
                required_columns[target_col] = None

        rename_dict = {v: k for k, v in required_columns.items() if v is not None}
        df = df.rename(columns=rename_dict)

        for col in ["IssuedShareTotal", "Equity", "Profit"]:
            if col not in df.columns:
                df[col] = 0

        base_cols = ["Code", "DisclosedDate"]
        if "CompanyName" in df.columns:
            base_cols.append("CompanyName")
        final_cols = base_cols + ["Profit", "Equity", "IssuedShareTotal"]
        df = df[final_cols].copy()

        logger.info(f"使用列: {df.columns.tolist()}")
        return df

    except Exception as e:
        logger.error(f"財務データ読み込みエラー: {e}", exc_info=True)
        return pd.DataFrame()


# ===================================
# B. 株価データ読み込み
# ===================================
def load_existing_price_data(ohlcv_dir: str = OHLCV_DIR) -> pd.DataFrame:
    if not os.path.exists(ohlcv_dir):
        logger.error(f"株価データディレクトリ '{ohlcv_dir}' が見つかりません。")
        return pd.DataFrame()

    csv_files = sorted([f for f in os.listdir(ohlcv_dir)
                        if f.startswith("OHLCV_Adjusted_") and f.endswith(".csv") and f != "OHLCV_Adjusted_TOPIX.csv"])

    if not csv_files:
        logger.error(f"ディレクトリ '{ohlcv_dir}' 内にCSVファイルが見つかりません。")
        return pd.DataFrame()

    logger.info(f"株価ファイル数: {len(csv_files)}個")

    all_dataframes = []
    usecols = ["Date", "Ticker", "AdjustmentClose"]

    for csv_file in tqdm(csv_files, desc="株価ファイル読み込み中"):
        file_path = os.path.join(ohlcv_dir, csv_file)
        try:
            df = pd.read_csv(
                file_path,
                usecols=usecols,
                parse_dates=["Date"],
                dtype={"Ticker": "Int64", "AdjustmentClose": "float32"},
            )
            df = df.drop_duplicates(subset=["Ticker", "Date"], keep="first")
            all_dataframes.append(df)
        except Exception as e:
            logger.warning(f"ファイル読み込みエラー ({csv_file}): {e}")
            continue

    if not all_dataframes:
        return pd.DataFrame()

    logger.info("全ファイルを結合中...")
    df_all = pd.concat(all_dataframes, ignore_index=True)
    logger.info(f"結合完了: {len(df_all):,}件")

    df_all = df_all.rename(columns={"Ticker": "Code", "AdjustmentClose": "Close"})
    df_all["Code"] = df_all["Code"].astype("str").str.replace("<NA>", "0").str.zfill(4)
    df_all = df_all.dropna(subset=["Close"])

    logger.info("重複除去 & ソート中...")
    df_all = df_all.sort_values(["Code", "Date"])
    df_all = df_all.drop_duplicates(subset=["Code", "Date"], keep="first")
    logger.info(f"重複除去後: {len(df_all):,}件")

    return df_all


def build_prices_by_code(prices_df: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    d = {}
    tmp = prices_df[["Code", "Date", "Close"]].copy()
    tmp["Code"] = tmp["Code"].astype(str).map(normalize_code)
    tmp = tmp[tmp["Code"] != ""].copy()
    tmp = tmp.sort_values(["Code", "Date"])

    for code, g in tmp.groupby("Code", sort=False):
        gg = g[["Date", "Close"]].drop_duplicates(subset=["Date"], keep="last").sort_values("Date").copy()
        gg = gg.set_index("Date", drop=False)
        d[code] = gg

    logger.info(f"prices_by_code built: {len(d):,} codes")
    return d


def get_near_price_fast(prices_by_code: Dict[str, pd.DataFrame],
                        code: str,
                        ref_date: pd.Timestamp,
                        kind: str = "last") -> Optional[float]:
    code = normalize_code(code)
    if not code or code not in prices_by_code:
        return None
    g = prices_by_code[code]
    lo = ref_date - pd.Timedelta(days=5)
    hi = ref_date + pd.Timedelta(days=5)
    w = g.loc[(g["Date"] >= lo) & (g["Date"] <= hi)]
    if w.empty:
        return None
    return float(w.iloc[0]["Close"]) if kind == "first" else float(w.iloc[-1]["Close"])


def safe_code_to_int(code_series: pd.Series) -> pd.Series:
    cleaned = code_series.astype(str).str.replace(r"\D", "", regex=True)
    cleaned = cleaned.replace("", "0")
    return pd.to_numeric(cleaned, errors="coerce").fillna(0).astype("int64")


def calculate_market_metrics_fast_chunked(statements_df: pd.DataFrame,
                                          prices_df: pd.DataFrame,
                                          chunk_size: int = 200) -> pd.DataFrame:
    logger.info("時価総額・PBR・ROE計算中（チャンク処理版）...")

    if prices_df.empty:
        logger.error("株価データが空です。")
        return pd.DataFrame()

    req_cols = {"Code", "Date", "Close"}
    if not req_cols.issubset(set(prices_df.columns)):
        logger.error(f"株価データに必要な列が存在しません。存在する列: {prices_df.columns.tolist()}")
        return pd.DataFrame()

    statements_df = statements_df.copy()
    statements_df["Profit"] = pd.to_numeric(statements_df["Profit"], errors="coerce").fillna(0)
    statements_df["Equity"] = pd.to_numeric(statements_df["Equity"], errors="coerce").fillna(0)
    statements_df["IssuedShareTotal"] = pd.to_numeric(statements_df["IssuedShareTotal"], errors="coerce").fillna(1)

    statements_df = statements_df[(statements_df["Equity"] > 0) & (statements_df["IssuedShareTotal"] > 0)]
    logger.info(f"有効な財務データ: {len(statements_df):,}件")

    statements_df["Code_int"] = safe_code_to_int(statements_df["Code"])
    prices_df = prices_df.copy()
    prices_df["Code_int"] = safe_code_to_int(prices_df["Code"])

    statements_df = statements_df[statements_df["Code_int"] > 0]
    prices_df = prices_df[prices_df["Code_int"] > 0]

    statements_df = statements_df.sort_values(["Code_int", "DisclosedDate"]).reset_index(drop=True)
    prices_df = prices_df.sort_values(["Code_int", "Date"]).reset_index(drop=True)

    statements_df = statements_df.drop_duplicates(subset=["Code_int", "DisclosedDate"], keep="first")
    prices_df = prices_df.drop_duplicates(subset=["Code_int", "Date"], keep="first")

    logger.info(f"ソート・重複除去後: 財務 {len(statements_df):,}件, 株価 {len(prices_df):,}件")

    statements_groups = list(statements_df.groupby("Code_int"))
    prices_dict = {code: group for code, group in prices_df.groupby("Code_int")}

    merged_list = []
    num_chunks = (len(statements_groups) + chunk_size - 1) // chunk_size

    for chunk_idx in tqdm(range(num_chunks), desc="マージ処理"):
        start_idx = chunk_idx * chunk_size
        end_idx = min((chunk_idx + 1) * chunk_size, len(statements_groups))
        chunk_groups = statements_groups[start_idx:end_idx]

        for code, stmt_code in chunk_groups:
            if code not in prices_dict:
                continue
            price_code = prices_dict[code]
            if len(price_code) == 0:
                continue

            stmt_code = stmt_code.sort_values("DisclosedDate").reset_index(drop=True)
            price_code = price_code.sort_values("Date").reset_index(drop=True)

            try:
                merged = pd.merge_asof(
                    stmt_code,
                    price_code[["Date", "Close"]],
                    left_on="DisclosedDate",
                    right_on="Date",
                    direction="backward",
                    tolerance=pd.Timedelta(days=10),
                )
                if not merged.empty:
                    merged_list.append(merged)
            except Exception:
                continue

    if not merged_list:
        logger.error("マージ結果が空です")
        return pd.DataFrame()

    df_merged = pd.concat(merged_list, ignore_index=True)
    df_merged = df_merged.dropna(subset=["Close"])
    logger.info(f"マージ完了: {len(df_merged):,}件")

    df_merged["MarketCap"] = df_merged["Close"] * df_merged["IssuedShareTotal"]
    df_merged["PBR"] = df_merged["MarketCap"] / df_merged["Equity"]
    df_merged["ROE"] = (df_merged["Profit"] / df_merged["Equity"]) * 100

    result_cols = ["Code", "DisclosedDate", "Close", "MarketCap", "PBR", "ROE", "Date"]
    if "CompanyName" in df_merged.columns:
        result_cols.insert(1, "CompanyName")

    result_df = df_merged[result_cols].copy()
    result_df = result_df.rename(columns={"Close": "StockPrice", "Date": "PriceDate"})

    mask = (
        (result_df["PBR"] > 0) &
        (result_df["PBR"] < 50) &
        (result_df["ROE"] > -100) &
        (result_df["ROE"] < 100) &
        (result_df["MarketCap"] > 1_000_000_000)
    )
    result_df = result_df[mask].copy()

    logger.info(f"計算完了: {len(result_df):,}件")
    return result_df


def build_unit_share_portfolio(stock_candidates: pd.DataFrame,
                               target_positions: int = 20,
                               initial_capital: float = 10_000_000) -> dict:
    if len(stock_candidates) == 0:
        return {"stocks": [], "shares": [], "prices": [], "amounts": []}

    selected = stock_candidates.head(target_positions).copy()
    capital_per_stock = initial_capital / len(selected)

    stocks, shares_list, prices_list, amounts_list = [], [], [], []

    for _, row in selected.iterrows():
        code = row["Code"]
        price = row["StockPrice"]
        required_amount = price * UNIT_SHARES

        if required_amount <= capital_per_stock:
            shares = int(capital_per_stock // required_amount) * UNIT_SHARES
            if shares > 0:
                stocks.append(code)
                shares_list.append(shares)
                prices_list.append(price)
                amounts_list.append(shares * price)

    return {"stocks": stocks, "shares": shares_list, "prices": prices_list, "amounts": amounts_list}


def make_month_ends(start: pd.Timestamp, end: pd.Timestamp) -> List[pd.Timestamp]:
    m0 = pd.Timestamp(start.year, start.month, 1) + pd.offsets.MonthEnd(0)
    m1 = pd.Timestamp(end.year, end.month, 1) + pd.offsets.MonthEnd(0)
    months = pd.date_range(m0, m1, freq="M")
    return [pd.Timestamp(x).normalize() for x in months]


def build_daily_equity_curve_for_period(
    portfolio: dict,
    start_date: pd.Timestamp,
    end_date: pd.Timestamp,
    prices_by_code: Dict[str, pd.DataFrame],
    invest_ratio_by_monthend: Optional[Dict[pd.Timestamp, float]] = None,
) -> pd.DataFrame:
    stocks = portfolio.get("stocks", [])
    shares = portfolio.get("shares", [])
    if not stocks:
        return pd.DataFrame()

    start_date = pd.to_datetime(start_date).normalize()
    end_date = pd.to_datetime(end_date).normalize()

    date_sets = []
    for code in stocks:
        code = normalize_code(code)
        if code in prices_by_code:
            g = prices_by_code[code]
            d = g[(g["Date"] >= start_date - pd.Timedelta(days=10)) & (g["Date"] <= end_date + pd.Timedelta(days=10))]["Date"]
            if len(d):
                date_sets.append(d)

    if not date_sets:
        return pd.DataFrame()

    dates = pd.Index(sorted(pd.unique(pd.concat(date_sets)))).astype("datetime64[ns]")
    dates = dates[(dates >= start_date) & (dates <= end_date)]
    if len(dates) == 0:
        return pd.DataFrame()

    values = []
    for dt in dates:
        v = 0.0
        ok = False
        for i, code in enumerate(stocks):
            code = normalize_code(code)
            if code not in prices_by_code:
                continue
            p = get_near_price_fast(prices_by_code, code, pd.Timestamp(dt), kind="last")
            if p is None:
                continue
            v += float(shares[i]) * float(p)
            ok = True
        values.append(v if ok else np.nan)

    df = pd.DataFrame({"Date": pd.to_datetime(dates), "equity_stock": values})
    df = df.dropna(subset=["equity_stock"]).copy()
    if df.empty:
        return df

    df = df.sort_values("Date").reset_index(drop=True)
    df["ret_stock"] = df["equity_stock"].pct_change().fillna(0.0)
    df["MonthEnd"] = (df["Date"] + pd.offsets.MonthEnd(0)).dt.normalize()

    if invest_ratio_by_monthend is None:
        df["invest_ratio"] = 1.0
        df["ret_total"] = df["ret_stock"]
    else:
        df["invest_ratio"] = df["MonthEnd"].map(invest_ratio_by_monthend).fillna(1.0).astype(float)
        df["ret_total"] = df["invest_ratio"] * df["ret_stock"]

    df["equity_total"] = (1.0 + df["ret_total"]).cumprod()
    peak = df["equity_total"].cummax()
    df["dd_total"] = df["equity_total"] / peak - 1.0

    return df[["Date", "MonthEnd", "equity_stock", "ret_stock", "invest_ratio", "ret_total", "equity_total", "dd_total"]]


def load_ff5mom_factors_monthly() -> pd.DataFrame:
    fac = pd.read_parquet(FF5MOM_FACTOR_PATH).copy()
    fac["MonthEnd"] = pd.to_datetime(fac["MonthEnd"], errors="coerce").dt.normalize()
    need = ["MonthEnd"] + FACTOR_COLS
    missing = [c for c in need if c not in fac.columns]
    if missing:
        raise KeyError(f"FF5MOM factors missing columns: {missing} in {FF5MOM_FACTOR_PATH}")
    fac = fac[need].sort_values("MonthEnd").reset_index(drop=True)
    return fac


def compute_regime_off(fac: pd.DataFrame, month_end: pd.Timestamp) -> Tuple[bool, Dict]:
    fac2 = fac.set_index("MonthEnd")
    idx = fac2.index[fac2.index < month_end]
    if len(idx) < REGIME_LOOKBACK_M:
        return False, {"regime_ready": False}

    win = idx[-REGIME_LOOKBACK_M:]
    s_mkt = float(pd.to_numeric(fac2.loc[win, "MKT"], errors="coerce").sum())
    s_wml = float(pd.to_numeric(fac2.loc[win, "WML"], errors="coerce").sum())

    off = (s_mkt < REGIME_OFF_IF_SUM_MKT_LT) or (s_wml < REGIME_OFF_IF_SUM_WML_LT)
    return bool(off), {"regime_ready": True, "sum_MKT_3m": s_mkt, "sum_WML_3m": s_wml}


def estimate_port_beta(monthly_port_rets: pd.DataFrame, fac: pd.DataFrame, month_end: pd.Timestamp) -> Tuple[bool, Dict]:
    fac2 = fac.set_index("MonthEnd")
    idx = fac2.index[fac2.index < month_end]
    if len(idx) < BETA_EST_WINDOW_M:
        return False, {"beta_ready": False}

    win = idx[-BETA_EST_WINDOW_M:]

    m = monthly_port_rets.copy()
    m["MonthEnd"] = pd.to_datetime(m["MonthEnd"], errors="coerce").dt.normalize()
    m = m.dropna(subset=["MonthEnd", "port_ret"]).copy()

    m = (
        m.groupby("MonthEnd", as_index=False)["port_ret"]
         .apply(lambda s: (1.0 + s.astype(float)).prod() - 1.0)
    )

    df = m[m["MonthEnd"].isin(win)].merge(
        fac, on="MonthEnd", how="left", validate="one_to_one"
    ).dropna(subset=["port_ret"] + FACTOR_COLS)

    if len(df) < BETA_EST_MIN_OBS:
        return False, {"beta_ready": False, "n_obs": int(len(df))}

    y = df["port_ret"].to_numpy(dtype=float)
    X = df[FACTOR_COLS].to_numpy(dtype=float)
    alpha, betas, r2 = ols_alpha_beta(y, X)

    out = {"beta_ready": True, "n_obs": int(len(df)), "r2": float(r2), "alpha": float(alpha)}
    for c, b in zip(FACTOR_COLS, betas):
        out[f"beta_{c}"] = float(b)
    return True, out


def decide_invest_ratio(month_end: pd.Timestamp,
                        fac: pd.DataFrame,
                        port_hist: pd.DataFrame) -> Tuple[float, Dict]:
    invest_ratio = 1.0
    diag = {
        "MonthEnd": month_end,
        "invest_ratio": 1.0,
        "regime_off": False,
        "beta_off": False,
        "regime_ready": False,
        "beta_ready": False,
        "sum_MKT_3m": np.nan,
        "sum_WML_3m": np.nan,
        "beta_MKT": np.nan,
        "beta_CMA": np.nan,
        "beta_n_obs": np.nan,
        "beta_r2": np.nan,
    }

    off_reg, info = compute_regime_off(fac, month_end)
    diag["regime_off"] = bool(off_reg)
    diag["regime_ready"] = bool(info.get("regime_ready", False))
    diag["sum_MKT_3m"] = info.get("sum_MKT_3m", np.nan)
    diag["sum_WML_3m"] = info.get("sum_WML_3m", np.nan)
    if off_reg:
        invest_ratio = min(invest_ratio, RISKOFF_RATIO)

    ok_beta, binfo = estimate_port_beta(port_hist, fac, month_end)
    diag["beta_ready"] = bool(binfo.get("beta_ready", False))
    if binfo.get("beta_ready", False):
        b_mkt = float(binfo.get("beta_MKT", np.nan))
        b_cma = float(binfo.get("beta_CMA", np.nan))
        diag["beta_MKT"] = b_mkt
        diag["beta_CMA"] = b_cma
        diag["beta_n_obs"] = float(binfo.get("n_obs", np.nan))
        diag["beta_r2"] = float(binfo.get("r2", np.nan))

        off_beta = (b_cma < BETA_CAP_CMA_LT) or (abs(b_mkt) > BETA_CAP_ABS_MKT_GT)
        diag["beta_off"] = bool(off_beta)
        if off_beta:
            invest_ratio = min(invest_ratio, RISKOFF_RATIO)

    diag["invest_ratio"] = float(invest_ratio)
    return float(invest_ratio), diag


def compute_monthly_portfolio_returns_with_riskcontrol_fast(
    portfolio: dict,
    start_date: pd.Timestamp,
    end_date: pd.Timestamp,
    prices_by_code: Dict[str, pd.DataFrame],
    fac: pd.DataFrame
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    stocks = portfolio["stocks"]
    shares = portfolio["shares"]
    if not stocks:
        return pd.DataFrame(), pd.DataFrame()

    month_ends = make_month_ends(start_date, end_date)
    boundaries = [start_date] + [me for me in month_ends if (me > start_date) and (me < end_date)] + [end_date]

    rows = []
    ir_rows = []
    port_hist = pd.DataFrame(columns=["MonthEnd", "port_ret"])

    for j in range(len(boundaries) - 1):
        d0 = boundaries[j]
        d1 = boundaries[j + 1]

        me = pd.Timestamp(d0.year, d0.month, 1) + pd.offsets.MonthEnd(0)
        me = pd.Timestamp(me).normalize()

        start_vals = []
        end_vals = []
        for i, code in enumerate(stocks):
            p0 = get_near_price_fast(prices_by_code, code, d0, kind="first")
            p1 = get_near_price_fast(prices_by_code, code, d1, kind="last")
            if p0 is None or p1 is None:
                continue
            sh = shares[i]
            start_vals.append(sh * p0)
            end_vals.append(sh * p1)

        if len(start_vals) == 0:
            continue

        start_v = float(np.sum(start_vals))
        end_v = float(np.sum(end_vals))
        stock_ret = (end_v / start_v) - 1.0

        port_hist = pd.concat([port_hist, pd.DataFrame([{"MonthEnd": me, "port_ret": stock_ret}])], ignore_index=True)

        invest_ratio, diag = decide_invest_ratio(me, fac, port_hist)
        ir_rows.append(diag)

        total_ret = invest_ratio * stock_ret

        rows.append({
            "MonthEnd": me,
            "period_start": d0,
            "period_end": d1,
            "port_ret_stock": stock_ret,
            "invest_ratio": invest_ratio,
            "port_ret_total": total_ret,
        })

    monthly_df = pd.DataFrame(rows)
    ir_diag_df = pd.DataFrame(ir_rows)
    return monthly_df, ir_diag_df


def annual_return_from_monthly(monthly_rets: pd.Series) -> float:
    if monthly_rets.empty:
        return 0.0
    return float((1.0 + monthly_rets).prod() - 1.0)


def apply_tax_annual(gross_return: float, initial_capital: float) -> Tuple[float, float]:
    profit = gross_return * initial_capital
    taxable = max(profit, 0.0)
    tax = taxable * TAX_RATE
    net_profit = profit - tax
    net_return = net_profit / initial_capital
    tax_rate_total = tax / initial_capital
    return float(net_return), float(tax_rate_total)


def save_annual_bundle(df: pd.DataFrame, out_annual_csv: Path, out_curve_csv: Path, out_perf_csv: Path):
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").reset_index(drop=True)

    df.to_csv(out_annual_csv, index=False, encoding="utf-8-sig")

    r = df["strategy_return_net"].astype(float)
    cum = (1.0 + r).cumprod()
    dd = compute_drawdown(cum)
    curve = pd.DataFrame({"date": df["date"], "ret": r, "cum": cum, "dd": dd})
    curve.to_csv(out_curve_csv, index=False, encoding="utf-8-sig")

    stats = perf_stats(r)
    perf = pd.DataFrame([stats])
    perf.to_csv(out_perf_csv, index=False, encoding="utf-8-sig")


def build_long_candidates(enhanced_financial_data: pd.DataFrame, rebalance_date: pd.Timestamp) -> pd.DataFrame:
    current_data = enhanced_financial_data[enhanced_financial_data["DisclosedDate"] <= rebalance_date].copy()
    current_data = current_data.sort_values("DisclosedDate").groupby("Code").tail(1)
    if len(current_data) < 100:
        return pd.DataFrame()

    current_data["PBR_Rank"] = current_data["PBR"].rank(method="first", ascending=True)
    current_data["ROE_Rank"] = current_data["ROE"].rank(method="first", ascending=False)

    current_data["PBR_Quartile"] = pd.qcut(current_data["PBR_Rank"], q=4, labels=[1, 2, 3, 4])
    current_data["ROE_Quartile"] = pd.qcut(current_data["ROE_Rank"], q=4, labels=[1, 2, 3, 4])

    long_candidates = current_data[
        (current_data["PBR_Quartile"] == 1) &
        (current_data["ROE_Quartile"] == 4)
    ].nsmallest(50, "PBR")

    return long_candidates


def run_factor_regression(results_df: pd.DataFrame, fac: pd.DataFrame) -> pd.DataFrame:
    if results_df.empty:
        return pd.DataFrame()

    rows = []
    for _, row in results_df.iterrows():
        end_date = pd.Timestamp(row["date"])
        start_date = end_date - pd.DateOffset(years=1)

        fac2 = fac.copy()
        fac2 = fac2[(fac2["MonthEnd"] >= (start_date + pd.offsets.MonthEnd(0))) &
                    (fac2["MonthEnd"] <= (end_date + pd.offsets.MonthEnd(-1)))].copy()

        ann_fac = {}
        for c in FACTOR_COLS:
            s = pd.to_numeric(fac2[c], errors="coerce").dropna()
            ann_fac[c] = float((1.0 + s).prod() - 1.0) if len(s) else np.nan

        rec = {"date": end_date}
        rec.update(ann_fac)
        rec["y"] = float(row["strategy_return_net"])
        rows.append(rec)

    df = pd.DataFrame(rows).dropna(subset=["y"] + FACTOR_COLS).copy()
    if len(df) < 3:
        return pd.DataFrame([{
            "n_years": int(len(df)),
            "R2": np.nan,
            "alpha": np.nan,
            **{f"beta_{c}": np.nan for c in FACTOR_COLS}
        }])

    y = df["y"].to_numpy(dtype=float)
    X = df[FACTOR_COLS].to_numpy(dtype=float)
    alpha, betas, r2 = ols_alpha_beta(y, X)

    out = {"n_years": int(len(df)), "R2": float(r2), "alpha": float(alpha)}
    for c, b in zip(FACTOR_COLS, betas):
        out[f"beta_{c}"] = float(b)
    return pd.DataFrame([out])


# ===================================
# ★ 日次：DDストップ＋DD回復復帰（方式1）
# ===================================
def apply_dd_stop_recover_to_daily_curve(
    df_daily_rc: pd.DataFrame,
    dd2: float,
    rec1: float,
    rec2: float,
    ratio_rec1: float = 0.3,
) -> pd.DataFrame:
    """
    ベース投資比率: df_daily_rc['invest_ratio']（月次RiskControl）
    0退避: dd <= dd2
    0.3復帰: dd >= rec1
    ベースへ復帰: dd >= rec2

    状態機械:
      state = "BASE" | "CASH" | "PARTIAL"
      BASE:    ir = base
              if dd <= dd2 -> CASH
      CASH:    ir = 0
              if dd >= rec1 -> PARTIAL
      PARTIAL: ir = min(base, ratio_rec1)
              if dd <= dd2 -> CASH
              if dd >= rec2 -> BASE
    """
    if df_daily_rc is None or df_daily_rc.empty:
        return pd.DataFrame()

    if not (dd2 < rec1 < rec2):
        # 例：dd2=-0.18, rec1=-0.10, rec2=-0.06 のように「浅くなる順」
        return pd.DataFrame()

    df = df_daily_rc.copy()
    df = df.sort_values("Date").reset_index(drop=True)

    df["ret_stock"] = pd.to_numeric(df["ret_stock"], errors="coerce").fillna(0.0).astype(float)
    base_ir = pd.to_numeric(df["invest_ratio"], errors="coerce").fillna(1.0).astype(float).to_numpy()

    ir = np.empty(len(df), dtype=float)
    eq = np.empty(len(df), dtype=float)
    peak = np.empty(len(df), dtype=float)
    dd = np.empty(len(df), dtype=float)

    state = "BASE"
    eq_val = 1.0
    peak_val = 1.0

    for i in range(len(df)):
        cur_dd = dd[i-1] if i > 0 else 0.0

        # 状態遷移（前日までのddで判定）
        if state == "BASE":
            if cur_dd <= dd2:
                state = "CASH"
        elif state == "CASH":
            if cur_dd >= rec1:
                state = "PARTIAL"
        elif state == "PARTIAL":
            if cur_dd <= dd2:
                state = "CASH"
            elif cur_dd >= rec2:
                state = "BASE"

        # 状態に応じた投資比率
        if state == "BASE":
            ir_i = base_ir[i]
        elif state == "CASH":
            ir_i = 0.0
        else:  # PARTIAL
            ir_i = min(base_ir[i], float(ratio_rec1))

        # 当日リターン反映
        r_total = ir_i * float(df.loc[i, "ret_stock"])
        eq_val *= (1.0 + r_total)

        peak_val = max(peak_val, eq_val)
        dd_val = (eq_val / peak_val) - 1.0

        ir[i] = ir_i
        eq[i] = eq_val
        peak[i] = peak_val
        dd[i] = dd_val

    out = df.copy()
    out["invest_ratio_ddrec"] = ir
    out["ret_total_ddrec"] = out["ret_stock"] * out["invest_ratio_ddrec"]
    out["equity_total_ddrec"] = eq
    out["dd_total_ddrec"] = dd

    # 互換ビュー列
    out_view = out.copy()
    out_view["invest_ratio"] = out_view["invest_ratio_ddrec"]
    out_view["ret_total"] = out_view["ret_total_ddrec"]
    out_view["equity_total"] = out_view["equity_total_ddrec"]
    out_view["dd_total"] = out_view["dd_total_ddrec"]

    return out_view


def build_annual_from_daily_equity(df_daily: pd.DataFrame, equity_col: str) -> pd.DataFrame:
    if df_daily is None or df_daily.empty:
        return pd.DataFrame()

    df = df_daily.copy()
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df = df.dropna(subset=["Date"]).sort_values("Date").reset_index(drop=True)

    rebalance_dates = [pd.Timestamp(f"{year}-10-01") for year in range(2016, 2026)]
    rows = []
    for i in range(len(rebalance_dates) - 1):
        start = rebalance_dates[i]
        end = rebalance_dates[i + 1]

        w = df[(df["Date"] >= start) & (df["Date"] <= end)].copy()
        if w.empty:
            continue

        eq0 = float(w.iloc[0][equity_col])
        eq1 = float(w.iloc[-1][equity_col])
        if eq0 <= 0:
            continue

        gross = (eq1 / eq0) - 1.0
        net, tax = apply_tax_annual(gross, INITIAL_CAPITAL)

        rows.append({
            "date": end,
            "strategy_return_gross": float(gross),
            "strategy_return_net": float(net),
            "tax": float(tax),
            "long_count": np.nan,
            "investment_ratio": np.nan,
        })

    return pd.DataFrame(rows)


def run_annual_backtest_with_and_without_riskcontrol_fast(
    enhanced_financial_data: pd.DataFrame,
    prices_df: pd.DataFrame,
    prices_by_code: Dict[str, pd.DataFrame],
    fac: pd.DataFrame,
    initial_capital: float = INITIAL_CAPITAL
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:

    rebalance_dates = [pd.Timestamp(f"{year}-10-01") for year in range(2016, 2026)]

    results_raw = []
    results_rc = []
    diag_ir_all = []

    daily_curves_raw = []
    daily_curves_rc  = []

    for i, rebalance_date in enumerate(tqdm(rebalance_dates[:-1], desc="10月1日リバランス（統合・高速）")):
        next_rebalance = rebalance_dates[i + 1]

        long_candidates = build_long_candidates(enhanced_financial_data, rebalance_date)
        if long_candidates.empty:
            logger.warning(f"{rebalance_date}: データ不足でスキップ")
            continue

        long_portfolio = build_unit_share_portfolio(
            long_candidates,
            target_positions=20,
            initial_capital=initial_capital
        )

        n_long = len(long_portfolio["stocks"])
        inv_amount = float(np.sum(long_portfolio["amounts"])) if n_long > 0 else 0.0
        inv_ratio_raw = inv_amount / initial_capital if initial_capital > 0 else 0.0

        logger.info(f"{rebalance_date.strftime('%Y-%m')}: long={n_long} invest={inv_amount:,.0f} ratio={inv_ratio_raw:.3f}")

        # RAW（年次）
        total_profit = 0.0
        for j, code in enumerate(long_portfolio["stocks"]):
            sh = long_portfolio["shares"][j]
            p0 = long_portfolio["prices"][j]
            p1 = get_near_price_fast(prices_by_code, code, next_rebalance, kind="last")
            if p1 is None:
                continue
            total_profit += (sh * p1 - sh * p0)

        gross_return_raw = (total_profit / initial_capital) if initial_capital > 0 else 0.0
        net_return_raw, tax_rate_raw = apply_tax_annual(gross_return_raw, initial_capital)

        results_raw.append({
            "date": next_rebalance,
            "strategy_return_gross": gross_return_raw,
            "strategy_return_net": net_return_raw,
            "tax": tax_rate_raw,
            "long_count": n_long,
            "investment_ratio": inv_ratio_raw,
        })

        # RiskControl（月次縮尺）
        monthly_df, ir_df = compute_monthly_portfolio_returns_with_riskcontrol_fast(
            long_portfolio, rebalance_date, next_rebalance, prices_by_code, fac
        )

        if monthly_df.empty:
            gross_return_rc = 0.0
            inv_ratio_rc_avg = 0.0
        else:
            gross_return_rc = annual_return_from_monthly(monthly_df["port_ret_total"])
            inv_ratio_rc_avg = float(monthly_df["invest_ratio"].mean())

        net_return_rc, tax_rate_rc = apply_tax_annual(gross_return_rc, initial_capital)

        results_rc.append({
            "date": next_rebalance,
            "strategy_return_gross": gross_return_rc,
            "strategy_return_net": net_return_rc,
            "tax": tax_rate_rc,
            "long_count": n_long,
            "investment_ratio": inv_ratio_rc_avg,
        })

        if not ir_df.empty:
            ir_df = ir_df.copy()
            ir_df["rebalance_start"] = rebalance_date
            ir_df["rebalance_end"] = next_rebalance
            diag_ir_all.append(ir_df)

        invest_ratio_map = None
        if not ir_df.empty:
            tmp_ir = ir_df.copy()
            tmp_ir["MonthEnd"] = pd.to_datetime(tmp_ir["MonthEnd"], errors="coerce").dt.normalize()
            tmp_ir = tmp_ir.dropna(subset=["MonthEnd"])
            tmp_ir = tmp_ir.sort_values("MonthEnd").drop_duplicates(subset=["MonthEnd"], keep="last")
            invest_ratio_map = dict(zip(tmp_ir["MonthEnd"], tmp_ir["invest_ratio"].astype(float)))

        daily_raw = build_daily_equity_curve_for_period(
            long_portfolio, rebalance_date, next_rebalance, prices_by_code, invest_ratio_by_monthend=None
        )
        if not daily_raw.empty:
            daily_raw["rebalance_start"] = rebalance_date
            daily_raw["rebalance_end"] = next_rebalance
            daily_curves_raw.append(daily_raw)

        daily_rc = build_daily_equity_curve_for_period(
            long_portfolio, rebalance_date, next_rebalance, prices_by_code, invest_ratio_by_monthend=invest_ratio_map
        )
        if not daily_rc.empty:
            daily_rc["rebalance_start"] = rebalance_date
            daily_rc["rebalance_end"] = next_rebalance
            daily_curves_rc.append(daily_rc)

    df_raw = pd.DataFrame(results_raw)
    df_rc = pd.DataFrame(results_rc)
    diag_ir = pd.concat(diag_ir_all, ignore_index=True) if len(diag_ir_all) else pd.DataFrame()

    daily_raw_all = pd.concat(daily_curves_raw, ignore_index=True) if len(daily_curves_raw) else pd.DataFrame()
    daily_rc_all  = pd.concat(daily_curves_rc,  ignore_index=True) if len(daily_curves_rc)  else pd.DataFrame()

    run_annual_backtest_with_and_without_riskcontrol_fast._daily_raw_all = daily_raw_all
    run_annual_backtest_with_and_without_riskcontrol_fast._daily_rc_all = daily_rc_all

    return df_raw, df_rc, diag_ir


# ===================================
# 1ケース実行（DD stop + recover）
# ===================================
def run_one_case_ddrec(
    dd2: float,
    rec1: float,
    rec2: float,
    enhanced_financial_data: pd.DataFrame,
    prices_df: pd.DataFrame,
    prices_by_code: Dict[str, pd.DataFrame],
    fac: pd.DataFrame,
) -> Dict:

    suffix = f"dd2_{fmt_cut(dd2)}__rec1_{fmt_cut(rec1)}__rec2_{fmt_cut(rec2)}"
    set_output_dir(BASE_OUT_DIR, suffix)

    logger.info("=" * 110)
    logger.info(f"[GRID] START DDREC case | dd2={dd2} ->0.0 | rec1={rec1} ->0.3 | rec2={rec2} ->base")
    logger.info(f"[GRID] OUT_DIR={OUT_DIR}")
    logger.info("=" * 110)

    df_raw, df_rc, diag_ir = run_annual_backtest_with_and_without_riskcontrol_fast(
        enhanced_financial_data, prices_df, prices_by_code, fac, initial_capital=INITIAL_CAPITAL
    )
    if df_raw.empty or df_rc.empty:
        return {
            "dd2": dd2, "rec1": rec1, "rec2": rec2,
            "daily_mdd": np.nan, "cagr": np.nan, "days_to_recovery": np.nan,
            "out_dir": str(OUT_DIR), "note": "empty_result"
        }

    # 保存（RAW/RC）
    save_annual_bundle(df_raw, OUT_ANNUAL_RAW, OUT_CURVE_RAW, OUT_PERF_RAW)
    save_annual_bundle(df_rc,  OUT_ANNUAL_RC,  OUT_CURVE_RC,  OUT_PERF_RC)

    reg_raw = run_factor_regression(df_raw, fac)
    reg_rc  = run_factor_regression(df_rc, fac)
    reg_raw.to_csv(OUT_REG_RAW, index=False, encoding="utf-8-sig")
    reg_rc.to_csv(OUT_REG_RC,  index=False, encoding="utf-8-sig")

    if not diag_ir.empty:
        diag_ir = diag_ir.copy()
        diag_ir["MonthEnd"] = pd.to_datetime(diag_ir["MonthEnd"])
        diag_ir.sort_values(["rebalance_start", "MonthEnd"], inplace=True)
        diag_ir.to_csv(OUT_IR_DIAG, index=False, encoding="utf-8-sig")

    daily_raw_all = getattr(run_annual_backtest_with_and_without_riskcontrol_fast, "_daily_raw_all", pd.DataFrame())
    daily_rc_all  = getattr(run_annual_backtest_with_and_without_riskcontrol_fast, "_daily_rc_all",  pd.DataFrame())

    if not daily_raw_all.empty:
        daily_raw_all.to_csv(OUT_DAILY_CURVE_RAW, index=False, encoding="utf-8-sig")
    if not daily_rc_all.empty:
        daily_rc_all.to_csv(OUT_DAILY_CURVE_RC, index=False, encoding="utf-8-sig")

    # DD stop + recover を適用
    ddrec_daily = apply_dd_stop_recover_to_daily_curve(
        daily_rc_all, dd2=dd2, rec1=rec1, rec2=rec2, ratio_rec1=0.3
    )
    if ddrec_daily.empty:
        return {
            "dd2": dd2, "rec1": rec1, "rec2": rec2,
            "daily_mdd": np.nan, "cagr": np.nan, "days_to_recovery": np.nan,
            "out_dir": str(OUT_DIR),
            "note": "invalid_threshold_order (need dd2 < rec1 < rec2)"
        }

    ddrec_daily.to_csv(OUT_DAILY_CURVE_DDREC, index=False, encoding="utf-8-sig")

    # 指標（日次）
    mdd_ddrec = daily_mdd(ddrec_daily)
    ev_raw = extract_max_drawdown_event_from_daily_curve(daily_raw_all, label="RAW")
    ev_rc  = extract_max_drawdown_event_from_daily_curve(daily_rc_all,  label="RISKCONTROL")
    ev_dd  = extract_max_drawdown_event_from_daily_curve(ddrec_daily,    label="DDSTOP_RECOVER")

    ev_raw.to_csv(OUT_DD_EVENTS_RAW, index=False, encoding="utf-8-sig")
    ev_rc.to_csv(OUT_DD_EVENTS_RC, index=False, encoding="utf-8-sig")
    ev_dd.to_csv(OUT_DD_EVENTS_DDREC, index=False, encoding="utf-8-sig")
    pd.concat([ev_raw, ev_rc, ev_dd], ignore_index=True).to_csv(OUT_DD_EVENTS_MAX, index=False, encoding="utf-8-sig")

    # 年次（ddrec日次equityから再構築）
    df_ddrec = build_annual_from_daily_equity(ddrec_daily, equity_col="equity_total")
    if df_ddrec.empty:
        cagr_ddrec = np.nan
    else:
        save_annual_bundle(df_ddrec, OUT_ANNUAL_DDREC, OUT_CURVE_DDREC, OUT_PERF_DDREC)
        reg_dd = run_factor_regression(df_ddrec, fac)
        reg_dd.to_csv(OUT_REG_DDREC, index=False, encoding="utf-8-sig")
        perf_dd = pd.read_csv(OUT_PERF_DDREC).iloc[0].to_dict()
        cagr_ddrec = float(perf_dd.get("CAGR", np.nan))

    days_to_recovery = float(ev_dd.iloc[0].get("days_to_recovery", np.nan)) if not ev_dd.empty else np.nan

    pd.DataFrame([{
        "daily_maxDD_raw": daily_mdd(daily_raw_all),
        "daily_maxDD_riskcontrol": daily_mdd(daily_rc_all),
        "daily_maxDD_ddstop_recover": mdd_ddrec,
        "dd2": dd2, "rec1": rec1, "rec2": rec2,
    }]).to_csv(OUT_DAILY_MDD_SUMMARY, index=False, encoding="utf-8-sig")

    logger.info(f"[GRID] DONE DDREC | dailyMDD={mdd_ddrec:.6f} | CAGR={cagr_ddrec} | days_to_recovery={days_to_recovery}")

    return {
        "riskoff_ratio": float(RISKOFF_RATIO),
        "dd2": float(dd2),
        "rec1": float(rec1),
        "rec2": float(rec2),
        "daily_mdd": float(mdd_ddrec) if np.isfinite(mdd_ddrec) else np.nan,
        "cagr": float(cagr_ddrec) if np.isfinite(cagr_ddrec) else np.nan,
        "days_to_recovery": float(days_to_recovery) if np.isfinite(days_to_recovery) else np.nan,
        "out_dir": str(OUT_DIR),
        "note": ""
    }


def main():
    logger.info("=" * 110)
    logger.info("10/1 年次「割安高質」バックテスト + FF5+MOM RiskControl(C) [DD STOP + DD RECOVER GRID]")
    logger.info("=" * 110)
    logger.info(f"FF5MOM_FACTOR_PATH: {FF5MOM_FACTOR_PATH}")
    logger.info(f"BASE_OUT_DIR: {BASE_OUT_DIR}")
    logger.info(f"Tax: annual only (TAX_RATE={TAX_RATE}) | daily tax ignored")
    logger.info(f"Fixed: RISKOFF_RATIO={RISKOFF_RATIO}")
    logger.info("=" * 110)

    # 入力データ（1回だけ）
    statements_df = load_financial_data()
    if statements_df.empty:
        logger.error("財務データ読み込み失敗")
        return

    prices_df = load_existing_price_data()
    if prices_df.empty:
        logger.error("株価データ読み込み失敗")
        return

    fac = load_ff5mom_factors_monthly()
    prices_by_code = build_prices_by_code(prices_df)

    enhanced_financial_data = calculate_market_metrics_fast_chunked(statements_df, prices_df, chunk_size=200)
    if enhanced_financial_data.empty:
        logger.error("財務指標計算失敗")
        return

    # ===== グリッド（必要に応じて調整）
    dd2_list  = [-0.15, -0.18]
    rec1_list = [-0.10, -0.08, -0.06]
    rec2_list = [-0.06, -0.04, -0.03]

    rows = []
    for dd2 in dd2_list:
        for rec1 in rec1_list:
            for rec2 in rec2_list:
                try:
                    rec = run_one_case_ddrec(
                        dd2=dd2,
                        rec1=rec1,
                        rec2=rec2,
                        enhanced_financial_data=enhanced_financial_data,
                        prices_df=prices_df,
                        prices_by_code=prices_by_code,
                        fac=fac,
                    )
                    rows.append(rec)
                except Exception as e:
                    logger.error(f"[GRID] failed dd2={dd2} rec1={rec1} rec2={rec2} | err={e}", exc_info=True)
                    rows.append({
                        "riskoff_ratio": float(RISKOFF_RATIO),
                        "dd2": float(dd2), "rec1": float(rec1), "rec2": float(rec2),
                        "daily_mdd": np.nan, "cagr": np.nan, "days_to_recovery": np.nan,
                        "out_dir": "",
                        "note": f"error: {e}"
                    })

    summary = pd.DataFrame(rows)
    summary = summary.sort_values(["dd2", "rec1", "rec2"]).reset_index(drop=True)
    summary.to_csv(OUT_GRID_SUMMARY, index=False, encoding="utf-8-sig")

    logger.info("-" * 110)
    logger.info("✅ GRID SUMMARY SAVED")
    logger.info(f"  - {OUT_GRID_SUMMARY}")
    logger.info("-" * 110)
    logger.info("[GRID TABLE]")
    logger.info(summary.to_string(index=False))


if __name__ == "__main__":
    main()


2026-01-27 04:44:00,064 - INFO - ==============================================================================================================
2026-01-27 04:44:00,066 - INFO - 10/1 年次「割安高質」バックテスト + FF5+MOM RiskControl(C) [DD STOP + DD RECOVER GRID]
2026-01-27 04:44:00,066 - INFO - ==============================================================================================================
2026-01-27 04:44:00,067 - INFO - FF5MOM_FACTOR_PATH: C:\Users\yongr\Project\merged_data_all_stocks\factors\ff5_mom_factors_monthly.parquet
2026-01-27 04:44:00,068 - INFO - BASE_OUT_DIR: C:\Users\yongr\Project\merged_data_all_stocks\factors\bt_october_unit_with_ff5mom_ddstop_recover_grid
2026-01-27 04:44:00,068 - INFO - Tax: annual only (TAX_RATE=0.20315) | daily tax ignored
2026-01-27 04:44:00,069 - INFO - Fixed: RISKOFF_RATIO=0.7
2026-01-27 04:44:00,069 - INFO - ==============================================================================================================
2026-01-27 04:44:00,855 - I

In [3]:
# -*- coding: utf-8 -*-
"""
Build FF5+MOM monthly factors from LOCAL J-Quants data and show "monthly important factors".

Inputs (LOCAL):
- daily bars parquet dir (date=YYYY-MM-DD.parquet): contains Code, Date, AdjustedClose (or similar)
- normalized fins parquet dir (date=YYYY-MM-DD.parquet): contains Code, Eq, TA, NP, shares columns (ShOutFY/TrShFY/AvgSh)

Outputs:
- price_month_end.parquet
- month_end_snapshot.parquet
- ff5_mom_factors_monthly.parquet
- diag: factor correlation, factor importance (ΔR²), factor stats

Run:
  python build_ff5mom_and_show_monthly_importance.py
"""

from __future__ import annotations

import re
from pathlib import Path
import numpy as np
import pandas as pd


# ============================================================
# PATHS (edit if needed)
# ============================================================
BARS_DIR = Path(r"C:\Users\yongr\Project\jquants_daily_bars_10y_parquet\daily_parquet")
FINS_DIR = Path(r"C:\Users\yongr\Project\jquants_fins_summary_10y_parquet\daily_parquet_norm")

FACTORS_DIR = Path(r"C:\Users\yongr\Project\merged_data_all_stocks\factors")
FACTORS_DIR.mkdir(parents=True, exist_ok=True)

PRICE_MONTH_END_PATH = FACTORS_DIR / "price_month_end.parquet"
SNAPSHOT_PATH = FACTORS_DIR / "month_end_snapshot.parquet"
FF5MOM_OUT = FACTORS_DIR / "ff5_mom_factors_monthly.parquet"

DIAG_DIR = FACTORS_DIR / "diag_ff5mom_monthly_only"
DIAG_DIR.mkdir(parents=True, exist_ok=True)

DIAG_FACTOR_CORR = DIAG_DIR / "diag_factor_corr.csv"
DIAG_FACTOR_STATS = DIAG_DIR / "diag_factor_stats.csv"
DIAG_IMPORTANCE = DIAG_DIR / "diag_monthly_factor_importance_deltaR2.csv"
DIAG_IMPORTANCE_DROPONE = DIAG_DIR / "diag_monthly_factor_importance_dropone.csv"
DIAG_WARN_MONTHS = DIAG_DIR / "diag_factor_months_with_nan_or_low_coverage.csv"

# ============================================================
# SETTINGS
# ============================================================
BAD_CODE_STRINGS = {"None", "nan", "", "NaN", "NULL", "null"}

# month window (adjust)
START_DATE = pd.Timestamp("2014-01-01")   # build enough history for MOM 12-1 and INV lag12
END_DATE   = pd.Timestamp("2025-09-30")

# factor build constraints
P30, P70 = 0.30, 0.70
MIN_STOCKS_PER_MONTH = 500
WINSOR_Q = (0.01, 0.99)

FACTOR_COLS = ["MKT", "SMB", "HML", "RMW", "CMA", "WML"]

# filename pattern
date_pat = re.compile(r"date=(\d{4}-\d{2}-\d{2})\.parquet$", re.I)


# ============================================================
# UTIL
# ============================================================
def force_dt64ns(x):
    dt = pd.to_datetime(x, errors="coerce")
    if hasattr(dt, "astype"):
        return dt.astype("datetime64[ns]")
    if pd.isna(dt):
        return np.datetime64("NaT", "ns")
    return pd.Timestamp(dt).to_datetime64()

def month_end(ts: pd.Series) -> pd.Series:
    return ts.dt.to_period("M").dt.to_timestamp("M")

def safe_num(s):
    return pd.to_numeric(s, errors="coerce")

def winsorize_series(x: pd.Series, q=(0.01, 0.99)) -> pd.Series:
    x = pd.to_numeric(x, errors="coerce")
    if x.notna().any():
        lo = x.quantile(q[0])
        hi = x.quantile(q[1])
        return x.clip(lower=lo, upper=hi)
    return x

def vwap_return(ret: pd.Series, w: pd.Series) -> float:
    ret = pd.to_numeric(ret, errors="coerce")
    w = pd.to_numeric(w, errors="coerce")
    m = ret.notna() & w.notna() & (w > 0)
    if m.sum() == 0:
        return np.nan
    return float((ret[m] * w[m]).sum() / w[m].sum())

def list_local_dates(folder: Path) -> list[pd.Timestamp]:
    dates = []
    for p in folder.glob("date=*.parquet"):
        m = date_pat.search(p.name)
        if not m:
            continue
        d = pd.Timestamp(m.group(1))
        dates.append(d)
    return sorted(dates)

def _pick_adjclose_column(df: pd.DataFrame) -> str | None:
    for c in ["AdjustedClose", "AdjustmentClose", "AdjC", "AdjClose", "AdjCl"]:
        if c in df.columns:
            return c
    return None


# ============================================================
# STEP 1: Build price_month_end from daily bars (LOCAL)
# ============================================================
def build_price_month_end_from_daily_bars(
    bars_dir: Path,
    start_date: pd.Timestamp,
    end_date: pd.Timestamp,
) -> pd.DataFrame:
    dates = list_local_dates(bars_dir)
    dates = [d for d in dates if (d >= start_date) and (d <= end_date)]
    if not dates:
        raise RuntimeError("No bar files in requested range.")

    parts = []
    for d in dates:
        fp = bars_dir / f"date={d.strftime('%Y-%m-%d')}.parquet"
        df = pd.read_parquet(fp)
        if df.empty or ("Code" not in df.columns):
            continue
        adj_col = _pick_adjclose_column(df)
        if adj_col is None:
            continue
        if "Date" not in df.columns:
            # sometimes Date absent; use file date
            df = df.copy()
            df["Date"] = pd.Timestamp(d)

        x = df[["Date", "Code", adj_col]].copy()
        x = x.rename(columns={adj_col: "AdjustedClose"})
        x["Date"] = pd.to_datetime(x["Date"], errors="coerce")
        x["Code"] = x["Code"].astype(str).str.strip()
        x = x[x["Code"].notna() & ~x["Code"].isin(BAD_CODE_STRINGS)].copy()
        x = x[x["Date"].notna()].copy()
        x["AdjustedClose"] = safe_num(x["AdjustedClose"])
        x = x[x["AdjustedClose"].notna() & (x["AdjustedClose"] > 0)].copy()
        x["MonthEnd"] = month_end(x["Date"])

        parts.append(x)

    if not parts:
        raise RuntimeError("No valid daily rows read for price build.")

    bars = pd.concat(parts, ignore_index=True)
    bars = bars.sort_values(["Code", "MonthEnd", "Date"], kind="mergesort")
    pm = bars.groupby(["Code", "MonthEnd"], as_index=False).tail(1)

    # unify columns
    pm["MarketCap"] = np.nan
    pm = pm[["Code", "MonthEnd", "Date", "AdjustedClose", "MarketCap"]].copy()

    if pm.duplicated(["Code", "MonthEnd"]).any():
        dup = pm[pm.duplicated(["Code","MonthEnd"], keep=False)].sort_values(["Code","MonthEnd","Date"])
        dup.head(5000).to_csv(DIAG_DIR / "diag_price_month_end_duplicates.csv", index=False, encoding="utf-8-sig")
        raise RuntimeError("price_month_end has duplicates. see diag_price_month_end_duplicates.csv")

    return pm


# ============================================================
# STEP 2: Build month_end_snapshot using fins norm (asof)
# ============================================================
def load_fins_norm_range(fins_dir: Path, start_need: pd.Timestamp, end_need: pd.Timestamp) -> pd.DataFrame:
    fin_dates = list_local_dates(fins_dir)
    fin_dates = [d for d in fin_dates if (d >= start_need) and (d <= end_need)]
    parts = []

    for d in fin_dates:
        fp = fins_dir / f"date={d.strftime('%Y-%m-%d')}.parquet"
        if not fp.exists():
            continue
        fin = pd.read_parquet(fp)
        if fin.empty or ("Code" not in fin.columns):
            continue
        fin = fin.copy()
        fin["Code"] = fin["Code"].astype(str).str.strip()
        fin = fin[fin["Code"].notna() & ~fin["Code"].isin(BAD_CODE_STRINGS)].copy()

        # snapshot_date is file date
        fin["snapshot_date"] = pd.Timestamp(d).to_datetime64()
        fin["snapshot_date"] = force_dt64ns(fin["snapshot_date"])

        for c in ["Eq", "TA", "NP", "ShOutFY", "TrShFY", "AvgSh"]:
            if c in fin.columns:
                fin[c] = safe_num(fin[c])
            else:
                fin[c] = np.nan

        # shares priority (same as your pipeline)
        fin["SharesOut_raw"] = fin["TrShFY"]
        fin.loc[fin["SharesOut_raw"].isna(), "SharesOut_raw"] = fin["ShOutFY"]
        fin.loc[fin["SharesOut_raw"].isna(), "SharesOut_raw"] = fin["AvgSh"]

        keep = ["Code", "snapshot_date", "Eq", "TA", "NP", "SharesOut_raw"]
        fin = fin[keep].copy()

        parts.append(fin)

    if not parts:
        return pd.DataFrame(columns=["Code","snapshot_date","Eq","TA","NP","SharesOut_raw"])

    right = pd.concat(parts, ignore_index=True)
    right = right[right["snapshot_date"].notna()].copy()
    right = right.sort_values(["snapshot_date","Code"], kind="mergesort")
    right = right.drop_duplicates(["Code","snapshot_date"], keep="last").reset_index(drop=True)
    return right


def build_month_end_snapshot(price_me: pd.DataFrame, fins_dir: Path) -> pd.DataFrame:
    # fins read range: need enough past for asof (and for TA lag12 feature)
    rebuild_start = pd.Timestamp(price_me["MonthEnd"].min())
    rebuild_end = pd.Timestamp(price_me["MonthEnd"].max())

    start_need = rebuild_start - pd.Timedelta(days=800)  # ~2y buffer for safety
    end_need = rebuild_end

    right = load_fins_norm_range(fins_dir, start_need, end_need)

    left = price_me[["Code","MonthEnd","Date","AdjustedClose"]].copy()
    left["Date"] = force_dt64ns(left["Date"])
    left["MonthEnd"] = force_dt64ns(left["MonthEnd"])
    right["snapshot_date"] = force_dt64ns(right["snapshot_date"])

    left = left.sort_values(["Date","Code"], kind="mergesort").reset_index(drop=True)
    right = right.sort_values(["snapshot_date","Code"], kind="mergesort").reset_index(drop=True)

    merged = pd.merge_asof(
        left,
        right,
        left_on="Date",
        right_on="snapshot_date",
        by="Code",
        direction="backward",
        allow_exact_matches=True
    )

    # features
    merged["MarketCap"] = safe_num(merged["AdjustedClose"]) * safe_num(merged["SharesOut_raw"])
    merged.loc[merged["MarketCap"] <= 0, "MarketCap"] = np.nan

    merged["BM_Ratio"] = safe_num(merged["Eq"]) / safe_num(merged["MarketCap"])
    merged.loc[(merged["BM_Ratio"] <= 0) | (merged["BM_Ratio"] > 10), "BM_Ratio"] = np.nan

    merged["ROE"] = safe_num(merged["NP"]) / safe_num(merged["Eq"])
    merged.loc[safe_num(merged["Eq"]) <= 0, "ROE"] = np.nan

    merged["TA"] = safe_num(merged["TA"])
    merged = merged.sort_values(["Code","MonthEnd"], kind="mergesort").reset_index(drop=True)
    merged["TA_lag12"] = merged.groupby("Code")["TA"].shift(12)
    merged["INV_Growth"] = (merged["TA"] - merged["TA_lag12"]) / merged["TA_lag12"]
    merged.loc[merged["TA_lag12"] <= 0, "INV_Growth"] = np.nan

    for c in ["BM_Ratio","ROE","INV_Growth"]:
        merged[c] = winsorize_series(merged[c], WINSOR_Q)

    keep_cols = ["Code","MonthEnd","Date","AdjustedClose","Eq","TA","NP","SharesOut_raw",
                 "MarketCap","BM_Ratio","ROE","INV_Growth"]
    out = merged[[c for c in keep_cols if c in merged.columns]].copy()

    if out.duplicated(["Code","MonthEnd"]).any():
        dup = out[out.duplicated(["Code","MonthEnd"], keep=False)].sort_values(["Code","MonthEnd","Date"])
        dup.head(5000).to_csv(DIAG_DIR / "diag_snapshot_duplicates.csv", index=False, encoding="utf-8-sig")
        raise RuntimeError("snapshot has duplicates. see diag_snapshot_duplicates.csv")

    return out


# ============================================================
# STEP 3: Build returns + MOM and compute monthly FF5+MOM factors
# ============================================================
def build_monthly_returns_and_mom(price_me: pd.DataFrame) -> pd.DataFrame:
    p = price_me.copy()
    p["Code"] = p["Code"].astype(str).str.strip()
    p["MonthEnd"] = force_dt64ns(p["MonthEnd"])
    p["AdjustedClose"] = safe_num(p["AdjustedClose"])
    p = p.sort_values(["Code","MonthEnd"], kind="mergesort").reset_index(drop=True)

    p["Adj_lag1"] = p.groupby("Code")["AdjustedClose"].shift(1)
    p["Adj_lag12"] = p.groupby("Code")["AdjustedClose"].shift(12)

    # realized monthly return: P_t / P_{t-1} - 1
    p["ret_m"] = (p["AdjustedClose"] / p["Adj_lag1"]) - 1.0
    p.loc[(p["AdjustedClose"] <= 0) | (p["Adj_lag1"] <= 0), "ret_m"] = np.nan

    # MOM 12-1 at month t: P_{t-1} / P_{t-12} - 1
    p["MOM_12_1"] = (p["Adj_lag1"] / p["Adj_lag12"]) - 1.0
    p.loc[(p["Adj_lag1"] <= 0) | (p["Adj_lag12"] <= 0), "MOM_12_1"] = np.nan

    return p[["Code","MonthEnd","ret_m","MOM_12_1"]].copy()


def compute_ff5_mom_factors(snapshot: pd.DataFrame, price_me: pd.DataFrame) -> pd.DataFrame:
    pm = build_monthly_returns_and_mom(price_me)
    df = snapshot.merge(pm, on=["Code","MonthEnd"], how="left")

    for c in ["MarketCap","BM_Ratio","ROE","INV_Growth","ret_m","MOM_12_1"]:
        df[c] = safe_num(df[c])

    for c in ["BM_Ratio","ROE","INV_Growth","ret_m","MOM_12_1"]:
        df[c] = winsorize_series(df[c], WINSOR_Q)

    months = sorted(df["MonthEnd"].dropna().unique())
    rows = []
    warn_rows = []

    for me in months:
        d = df[df["MonthEnd"] == me].copy()
        n_total = int(d["Code"].nunique())

        # market: value-weighted return using MarketCap
        mkt = vwap_return(d["ret_m"], d["MarketCap"])

        mc = d["MarketCap"]
        if mc.notna().sum() < MIN_STOCKS_PER_MONTH:
            warn_rows.append({"MonthEnd": me, "reason": "low_mcap_coverage", "n_total": n_total})
            continue

        size_cut = mc.median()
        d["SB"] = np.where(d["MarketCap"] <= size_cut, "S", "B")

        def q30_70(x: pd.Series) -> tuple[float, float]:
            x = pd.to_numeric(x, errors="coerce").dropna()
            if len(x) < MIN_STOCKS_PER_MONTH:
                return (np.nan, np.nan)
            return (x.quantile(P30), x.quantile(P70))

        q30, q70 = q30_70(d["BM_Ratio"])
        d["VAL"] = np.where(d["BM_Ratio"] <= q30, "L", np.where(d["BM_Ratio"] >= q70, "H", "M"))

        q30p, q70p = q30_70(d["ROE"])
        d["PROF"] = np.where(d["ROE"] <= q30p, "W", np.where(d["ROE"] >= q70p, "R", "M"))

        q30i, q70i = q30_70(d["INV_Growth"])
        d["INV"] = np.where(d["INV_Growth"] <= q30i, "C", np.where(d["INV_Growth"] >= q70i, "A", "M"))

        q30m, q70m = q30_70(d["MOM_12_1"])
        d["MOM"] = np.where(d["MOM_12_1"] <= q30m, "L", np.where(d["MOM_12_1"] >= q70m, "W", "M"))

        def bucket_ret(mask) -> float:
            dd = d[mask].copy()
            return vwap_return(dd["ret_m"], dd["MarketCap"])

        # HML
        r_SH = bucket_ret((d["SB"]=="S") & (d["VAL"]=="H"))
        r_BH = bucket_ret((d["SB"]=="B") & (d["VAL"]=="H"))
        r_SL = bucket_ret((d["SB"]=="S") & (d["VAL"]=="L"))
        r_BL = bucket_ret((d["SB"]=="B") & (d["VAL"]=="L"))
        hml = np.nanmean([r_SH, r_BH]) - np.nanmean([r_SL, r_BL])

        # RMW
        r_SR = bucket_ret((d["SB"]=="S") & (d["PROF"]=="R"))
        r_BR = bucket_ret((d["SB"]=="B") & (d["PROF"]=="R"))
        r_SW = bucket_ret((d["SB"]=="S") & (d["PROF"]=="W"))
        r_BW = bucket_ret((d["SB"]=="B") & (d["PROF"]=="W"))
        rmw = np.nanmean([r_SR, r_BR]) - np.nanmean([r_SW, r_BW])

        # CMA
        r_SC = bucket_ret((d["SB"]=="S") & (d["INV"]=="C"))
        r_BC = bucket_ret((d["SB"]=="B") & (d["INV"]=="C"))
        r_SA = bucket_ret((d["SB"]=="S") & (d["INV"]=="A"))
        r_BA = bucket_ret((d["SB"]=="B") & (d["INV"]=="A"))
        cma = np.nanmean([r_SC, r_BC]) - np.nanmean([r_SA, r_BA])

        # WML
        r_SWm = bucket_ret((d["SB"]=="S") & (d["MOM"]=="W"))
        r_BWm = bucket_ret((d["SB"]=="B") & (d["MOM"]=="W"))
        r_SLm = bucket_ret((d["SB"]=="S") & (d["MOM"]=="L"))
        r_BLm = bucket_ret((d["SB"]=="B") & (d["MOM"]=="L"))
        wml = np.nanmean([r_SWm, r_BWm]) - np.nanmean([r_SLm, r_BLm])

        # SMB (average of 4 SMB legs: value/prof/inv/mom)
        s_val = np.nanmean([bucket_ret((d["SB"]=="S") & (d["VAL"]==x)) for x in ["H","M","L"]])
        b_val = np.nanmean([bucket_ret((d["SB"]=="B") & (d["VAL"]==x)) for x in ["H","M","L"]])
        smb_val = s_val - b_val

        s_prof = np.nanmean([bucket_ret((d["SB"]=="S") & (d["PROF"]==x)) for x in ["R","M","W"]])
        b_prof = np.nanmean([bucket_ret((d["SB"]=="B") & (d["PROF"]==x)) for x in ["R","M","W"]])
        smb_prof = s_prof - b_prof

        s_inv = np.nanmean([bucket_ret((d["SB"]=="S") & (d["INV"]==x)) for x in ["C","M","A"]])
        b_inv = np.nanmean([bucket_ret((d["SB"]=="B") & (d["INV"]==x)) for x in ["C","M","A"]])
        smb_inv = s_inv - b_inv

        s_mom = np.nanmean([bucket_ret((d["SB"]=="S") & (d["MOM"]==x)) for x in ["W","M","L"]])
        b_mom = np.nanmean([bucket_ret((d["SB"]=="B") & (d["MOM"]==x)) for x in ["W","M","L"]])
        smb_mom = s_mom - b_mom

        smb = np.nanmean([smb_val, smb_prof, smb_inv, smb_mom])

        used_val = int(d["BM_Ratio"].notna().sum())
        used_prof = int(d["ROE"].notna().sum())
        used_inv = int(d["INV_Growth"].notna().sum())
        used_mom = int(d["MOM_12_1"].notna().sum())

        rows.append({
            "MonthEnd": me,
            "MKT": mkt,
            "SMB": smb,
            "HML": hml,
            "RMW": rmw,
            "CMA": cma,
            "WML": wml,
            "n_total": n_total,
            "used_val": used_val,
            "used_prof": used_prof,
            "used_inv": used_inv,
            "used_mom": used_mom,
        })

    fac = pd.DataFrame(rows).sort_values("MonthEnd").reset_index(drop=True)
    warn = pd.DataFrame(warn_rows)
    warn.to_csv(DIAG_WARN_MONTHS, index=False, encoding="utf-8-sig")

    if fac.empty:
        raise RuntimeError("No factor months computed. Check coverage and diagnostics.")

    return fac


# ============================================================
# STEP 4: "Monthly important factors" (no strategy returns)
# ============================================================
def run_ols_with_intercept(y: np.ndarray, X: np.ndarray) -> tuple[np.ndarray, float]:
    y = y.reshape(-1, 1)
    X = np.asarray(X)
    X1 = np.concatenate([np.ones((X.shape[0], 1)), X], axis=1)
    beta = np.linalg.pinv(X1) @ y
    y_hat = X1 @ beta
    resid = y - y_hat
    ss_res = float((resid ** 2).sum())
    ss_tot = float(((y - y.mean()) ** 2).sum()) if y.shape[0] > 1 else np.nan
    r2 = 1.0 - ss_res / ss_tot if (ss_tot is not np.nan and ss_tot > 0) else np.nan
    return beta.flatten(), r2

def monthly_factor_importance_delta_r2(fac: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Define "importance" without strategy:
    - Target: MKT
    - Predictors: other factors
    - Importance: drop-one ΔR² (how much R² drops when the factor is removed)

    Returns:
      - importance_summary (sorted)
      - dropone_table (each factor's R2_full, R2_drop, deltaR2)
    """
    d = fac.copy()
    d = d.dropna(subset=FACTOR_COLS).copy()

    if len(d) < 24:
        raise RuntimeError(f"Too few monthly rows for regression: {len(d)}")

    y = d["MKT"].to_numpy(dtype=float)

    other = [c for c in FACTOR_COLS if c != "MKT"]
    X_full = d[other].to_numpy(dtype=float)
    _, r2_full = run_ols_with_intercept(y, X_full)

    rows = []
    for c in other:
        cols_drop = [x for x in other if x != c]
        X_drop = d[cols_drop].to_numpy(dtype=float)
        _, r2_drop = run_ols_with_intercept(y, X_drop)
        delta = (r2_full - r2_drop) if np.isfinite(r2_full) and np.isfinite(r2_drop) else np.nan
        rows.append({
            "target": "MKT",
            "factor_dropped": c,
            "R2_full": float(r2_full) if np.isfinite(r2_full) else np.nan,
            "R2_drop": float(r2_drop) if np.isfinite(r2_drop) else np.nan,
            "deltaR2": float(delta) if np.isfinite(delta) else np.nan,
            "n_months": int(len(d)),
        })

    dropone = pd.DataFrame(rows).sort_values("deltaR2", ascending=False).reset_index(drop=True)
    summary = dropone[["factor_dropped","deltaR2","n_months","R2_full"]].rename(columns={
        "factor_dropped":"factor",
        "deltaR2":"importance_deltaR2"
    }).copy()
    return summary, dropone


def main():
    print("=== Build FF5+MOM monthly factors (LOCAL) and show monthly importance ===")

    print(f"[1] Build price_month_end from daily bars: {BARS_DIR}")
    price_me = build_price_month_end_from_daily_bars(BARS_DIR, START_DATE, END_DATE)
    price_me.to_parquet(PRICE_MONTH_END_PATH, engine="pyarrow", index=False)
    print(f"  saved: {PRICE_MONTH_END_PATH} rows={len(price_me):,} months={price_me['MonthEnd'].nunique():,}")

    print(f"[2] Build month_end_snapshot using fins norm: {FINS_DIR}")
    snapshot = build_month_end_snapshot(price_me, FINS_DIR)
    snapshot.to_parquet(SNAPSHOT_PATH, engine="pyarrow", index=False)
    print(f"  saved: {SNAPSHOT_PATH} rows={len(snapshot):,} months={snapshot['MonthEnd'].nunique():,}")

    print("[3] Compute FF5+MOM monthly factors")
    fac = compute_ff5_mom_factors(snapshot, price_me)
    fac.to_parquet(FF5MOM_OUT, engine="pyarrow", index=False)
    print(f"  saved: {FF5MOM_OUT} months={len(fac):,} {fac['MonthEnd'].min()} .. {fac['MonthEnd'].max()}")

    # diagnostics
    corr = fac[FACTOR_COLS].corr()
    corr.to_csv(DIAG_FACTOR_CORR, encoding="utf-8-sig")

    stats = fac[FACTOR_COLS].agg(["count","mean","std","min","max"]).T.reset_index().rename(columns={"index":"factor"})
    stats.to_csv(DIAG_FACTOR_STATS, index=False, encoding="utf-8-sig")

    # "importance"
    summary, dropone = monthly_factor_importance_delta_r2(fac)
    summary.to_csv(DIAG_IMPORTANCE, index=False, encoding="utf-8-sig")
    dropone.to_csv(DIAG_IMPORTANCE_DROPONE, index=False, encoding="utf-8-sig")

    print("\n=== MONTHLY FACTOR STATS (mean/std) ===")
    print(stats[["factor","mean","std","min","max","count"]].to_string(index=False))

    print("\n=== MONTHLY IMPORTANT FACTORS (ΔR²; target=MKT, drop-one) ===")
    print(summary.to_string(index=False))

    if len(summary) and pd.notna(summary.iloc[0]["importance_deltaR2"]):
        top = summary.iloc[0]
        print(f"\n[TOP] 月次で一番“効いている”（MKT説明への寄与が最大）: {top['factor']}  ΔR²={top['importance_deltaR2']:.6f}")
        print(f"[SAVE] {DIAG_IMPORTANCE}")
        print(f"[SAVE] {DIAG_FACTOR_CORR}")
        print(f"[SAVE] {DIAG_FACTOR_STATS}")

    print("\nDONE.")


if __name__ == "__main__":
    main()


=== Build FF5+MOM monthly factors (LOCAL) and show monthly importance ===
[1] Build price_month_end from daily bars: C:\Users\yongr\Project\jquants_daily_bars_10y_parquet\daily_parquet
  saved: C:\Users\yongr\Project\merged_data_all_stocks\factors\price_month_end.parquet rows=474,003 months=117
[2] Build month_end_snapshot using fins norm: C:\Users\yongr\Project\jquants_fins_summary_10y_parquet\daily_parquet_norm
  saved: C:\Users\yongr\Project\merged_data_all_stocks\factors\month_end_snapshot.parquet rows=474,003 months=117
[3] Compute FF5+MOM monthly factors
  saved: C:\Users\yongr\Project\merged_data_all_stocks\factors\ff5_mom_factors_monthly.parquet months=117 2016-01-31 00:00:00 .. 2025-09-30 00:00:00

=== MONTHLY FACTOR STATS (mean/std) ===
factor      mean      std       min      max  count
   MKT  0.008752 0.035632 -0.090198 0.095802  116.0
   SMB -0.002378 0.018443 -0.060136 0.040359  116.0
   HML  0.001815 0.044278 -0.166050 0.098367  114.0
   RMW  0.008221 0.017842 -0.043032

In [4]:
# -*- coding: utf-8 -*-
"""
Show which FF5+MOM monthly factors are statistically significant (t-stat of mean).

Input:
  - ff5_mom_factors_monthly.parquet

Output (under diag dir):
  - diag_monthly_factor_significance.csv

Run:
  python show_monthly_factor_significance.py
"""

from pathlib import Path
import numpy as np
import pandas as pd
import math

FACTORS_DIR = Path(r"C:\Users\yongr\Project\merged_data_all_stocks\factors")
FF5MOM_PATH = FACTORS_DIR / "ff5_mom_factors_monthly.parquet"

DIAG_DIR = FACTORS_DIR / "diag_ff5mom_monthly_only"
DIAG_DIR.mkdir(parents=True, exist_ok=True)
OUT_SIG = DIAG_DIR / "diag_monthly_factor_significance.csv"

FACTOR_COLS = ["MKT", "SMB", "HML", "RMW", "CMA", "WML"]


def norm_cdf(x: float) -> float:
    # Standard normal CDF via erf
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))


def two_sided_pvalue_norm_approx(t: float) -> float:
    # two-sided p-value using normal approx
    if not np.isfinite(t):
        return np.nan
    z = abs(float(t))
    return 2.0 * (1.0 - norm_cdf(z))


def main():
    fac = pd.read_parquet(FF5MOM_PATH)

    rows = []
    for c in FACTOR_COLS:
        x = pd.to_numeric(fac[c], errors="coerce").dropna()
        n = int(len(x))
        if n < 3:
            rows.append({
                "factor": c, "n": n,
                "mean": np.nan, "std": np.nan, "t_stat": np.nan, "p_value_norm": np.nan,
                "sharpe_mean_over_std": np.nan,
            })
            continue

        mu = float(x.mean())
        sd = float(x.std(ddof=1))
        se = sd / math.sqrt(n) if sd > 0 else np.nan
        t = (mu / se) if (se is not None and np.isfinite(se) and se > 0) else np.nan
        p = two_sided_pvalue_norm_approx(t)
        sharpe = (mu / sd) if (sd > 0) else np.nan

        rows.append({
            "factor": c,
            "n": n,
            "mean": mu,
            "std": sd,
            "t_stat": float(t) if np.isfinite(t) else np.nan,
            "p_value_norm": float(p) if np.isfinite(p) else np.nan,
            "sharpe_mean_over_std": float(sharpe) if np.isfinite(sharpe) else np.nan,
        })

    out = pd.DataFrame(rows)

    # ranking: by |t|
    out["abs_t"] = out["t_stat"].abs()
    out = out.sort_values(["abs_t", "n"], ascending=[False, False]).reset_index(drop=True)

    out.to_csv(OUT_SIG, index=False, encoding="utf-8-sig")

    print("=== MONTHLY FACTOR SIGNIFICANCE (t-stat of mean; p-value normal approx) ===")
    show_cols = ["factor", "n", "mean", "std", "t_stat", "p_value_norm", "sharpe_mean_over_std"]
    print(out[show_cols].to_string(index=False))

    if len(out) and pd.notna(out.loc[0, "t_stat"]):
        top = out.loc[0]
        print(f"\n[TOP] 月次で統計的に一番“有意”（|t|最大）: {top['factor']}  "
              f"t={top['t_stat']:.3f}  p≈{top['p_value_norm']:.4g}  n={int(top['n'])}")
        print(f"[SAVE] {OUT_SIG}")


if __name__ == "__main__":
    main()


=== MONTHLY FACTOR SIGNIFICANCE (t-stat of mean; p-value normal approx) ===
factor   n      mean      std    t_stat  p_value_norm  sharpe_mean_over_std
   RMW 116  0.008221 0.017842  4.962766  6.949638e-07              0.460781
   CMA 105 -0.006401 0.021506 -3.049936  2.288900e-03             -0.297643
   MKT 116  0.008752 0.035632  2.645374  8.160060e-03              0.245617
   SMB 116 -0.002378 0.018443 -1.388436  1.650044e-01             -0.128913
   HML 114  0.001815 0.044278  0.437739  6.615757e-01              0.040998
   WML 105 -0.001151 0.027645 -0.426526  6.697249e-01             -0.041625

[TOP] 月次で統計的に一番“有意”（|t|最大）: RMW  t=4.963  p≈6.95e-07  n=116
[SAVE] C:\Users\yongr\Project\merged_data_all_stocks\factors\diag_ff5mom_monthly_only\diag_monthly_factor_significance.csv


In [5]:
# -*- coding: utf-8 -*-
"""
Backtest: Top20 ROE portfolio, monthly rebalance, equal-weight, simple tax model.

Inputs (LOCAL):
- price_month_end.parquet  (Code, MonthEnd, AdjustedClose ...)
- month_end_snapshot.parquet (Code, MonthEnd, ROE, MarketCap, ...)

Outputs:
- backtest_top20_roe_monthly_results.csv
- backtest_top20_roe_monthly_holdings.csv
- backtest_top20_roe_monthly_summary.txt

Run:
  python backtest_top20_roe_monthly_rebalance_tax_simple.py
"""

from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd
import math

# ===================== PATHS =====================
FACTORS_DIR = Path(r"C:\Users\yongr\Project\merged_data_all_stocks\factors")

PRICE_PATH = FACTORS_DIR / "price_month_end.parquet"
SNAP_PATH  = FACTORS_DIR / "month_end_snapshot.parquet"

OUT_DIR = FACTORS_DIR / "backtest_top20_roe_monthly"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_MONTHLY = OUT_DIR / "backtest_top20_roe_monthly_results.csv"
OUT_HOLD    = OUT_DIR / "backtest_top20_roe_monthly_holdings.csv"
OUT_SUMMARY = OUT_DIR / "backtest_top20_roe_monthly_summary.txt"

# ===================== SETTINGS =====================
TOP_N = 20
TAX_RATE = 0.20315  # 20.315%
MIN_NAMES = 20      # 最低保有銘柄数（Top20を満たせない月はスキップ）

BAD_CODE_STRINGS = {"None", "nan", "", "NaN", "NULL", "null"}


# ===================== UTIL =====================
def safe_num(s):
    return pd.to_numeric(s, errors="coerce")

def to_dt(x):
    return pd.to_datetime(x, errors="coerce").astype("datetime64[ns]")

def compound_returns(r: pd.Series) -> float:
    r = pd.to_numeric(r, errors="coerce").dropna()
    if len(r) == 0:
        return np.nan
    return float((1.0 + r).prod() - 1.0)

def annualized_return(r: pd.Series) -> float:
    r = pd.to_numeric(r, errors="coerce").dropna()
    if len(r) == 0:
        return np.nan
    total = float((1.0 + r).prod())
    years = len(r) / 12.0
    if years <= 0:
        return np.nan
    return total ** (1.0 / years) - 1.0

def annualized_vol(r: pd.Series) -> float:
    r = pd.to_numeric(r, errors="coerce").dropna()
    if len(r) < 2:
        return np.nan
    return float(r.std(ddof=1) * math.sqrt(12.0))

def sharpe_ann(r: pd.Series, rf=0.0) -> float:
    r = pd.to_numeric(r, errors="coerce").dropna()
    if len(r) < 2:
        return np.nan
    mu = float(r.mean() * 12.0)
    vol = annualized_vol(r)
    if not np.isfinite(vol) or vol <= 0:
        return np.nan
    return (mu - rf) / vol

def max_drawdown_from_monthly(r: pd.Series) -> float:
    r = pd.to_numeric(r, errors="coerce").dropna()
    if len(r) == 0:
        return np.nan
    eq = (1.0 + r).cumprod()
    peak = eq.cummax()
    dd = (eq / peak) - 1.0
    return float(dd.min())

def simple_tax_net_return(r_gross: float, tax_rate: float) -> float:
    # 月次リターンがプラスのときのみ税金を控除（簡易モデル）
    if not np.isfinite(r_gross):
        return np.nan
    tax = tax_rate * max(r_gross, 0.0)
    return float(r_gross - tax)


# ===================== CORE =====================
def build_monthly_returns_from_price(price: pd.DataFrame) -> pd.DataFrame:
    """
    realized monthly return: P_t / P_{t-1} - 1
    """
    p = price.copy()
    p["Code"] = p["Code"].astype(str).str.strip()
    p = p[~p["Code"].isin(BAD_CODE_STRINGS)].copy()

    p["MonthEnd"] = to_dt(p["MonthEnd"]).dt.normalize()
    p["AdjustedClose"] = safe_num(p["AdjustedClose"])
    p = p[p["MonthEnd"].notna() & p["AdjustedClose"].notna() & (p["AdjustedClose"] > 0)].copy()

    p = p.sort_values(["Code","MonthEnd"], kind="mergesort").reset_index(drop=True)
    p["Adj_lag1"] = p.groupby("Code")["AdjustedClose"].shift(1)
    p["ret_m"] = (p["AdjustedClose"] / p["Adj_lag1"]) - 1.0
    p.loc[(p["Adj_lag1"] <= 0) | (p["AdjustedClose"] <= 0), "ret_m"] = np.nan

    return p[["Code","MonthEnd","ret_m","AdjustedClose"]].copy()


def backtest_topn_by_roe_monthly(
    snap: pd.DataFrame,
    rets: pd.DataFrame,
    top_n: int = 20,
    min_names: int = 20,
    tax_rate: float = 0.20315
) -> tuple[pd.DataFrame, pd.DataFrame, str]:
    """
    At MonthEnd t:
      - rank by ROE (high to low) using snapshot at t
      - hold next month return (MonthEnd t+1 realized ret) -> we align using ret at t+1
    Implementation:
      - pick holdings at t
      - portfolio return uses rets at t+1 for those Codes
    """
    s = snap.copy()
    s["Code"] = s["Code"].astype(str).str.strip()
    s = s[~s["Code"].isin(BAD_CODE_STRINGS)].copy()
    s["MonthEnd"] = to_dt(s["MonthEnd"]).dt.normalize()
    s["ROE"] = safe_num(s["ROE"])
    s["MarketCap"] = safe_num(s["MarketCap"]) if "MarketCap" in s.columns else np.nan

    r = rets.copy()
    r["MonthEnd"] = to_dt(r["MonthEnd"]).dt.normalize()
    r["ret_m"] = safe_num(r["ret_m"])

    # merge snapshot with next-month returns:
    # holdings decided at t -> use return at t+1
    months = sorted(s["MonthEnd"].dropna().unique())
    months = [pd.Timestamp(m) for m in months]

    monthly_rows = []
    holding_rows = []

    # to get t+1 return, we need ret table keyed by MonthEnd
    ret_map = r.set_index(["Code","MonthEnd"])["ret_m"]

    for i in range(len(months)-1):
        t = months[i]
        t1 = months[i+1]

        ss = s[s["MonthEnd"] == t].copy()

        # universe: need ROE and next month return available
        ss = ss[ss["ROE"].notna()].copy()
        if ss.empty:
            continue

        # rank by ROE desc
        ss = ss.sort_values("ROE", ascending=False, kind="mergesort")
        ss = ss.head(top_n).copy()

        # require enough names
        if len(ss) < min_names:
            continue

        codes = ss["Code"].tolist()

        # fetch next month returns
        rr = []
        for c in codes:
            v = ret_map.get((c, t1), np.nan)
            rr.append(v)

        rr = pd.Series(rr, index=codes, dtype="float64").dropna()

        # if too many missing (e.g. delisted), skip this month
        if len(rr) < min_names:
            continue

        # equal-weight portfolio gross return
        r_gross = float(rr.mean())

        r_net = simple_tax_net_return(r_gross, tax_rate)

        # turnover vs previous holdings (for info)
        if holding_rows:
            prev_month = monthly_rows[-1]["formation_monthend"]
            prev_codes = [x["Code"] for x in holding_rows if x["formation_monthend"] == prev_month]
            prev_set = set(prev_codes)
            cur_set = set(rr.index)
            turnover = 1.0 - (len(prev_set & cur_set) / len(prev_set)) if len(prev_set) else np.nan
        else:
            turnover = np.nan

        monthly_rows.append({
            "formation_monthend": t,
            "holding_monthend": t1,
            "n_holdings": int(len(rr)),
            "ret_gross": r_gross,
            "ret_net_tax_simple": r_net,
            "tax_rate": tax_rate,
            "turnover": turnover,
            "avg_roe": float(ss.loc[ss["Code"].isin(rr.index), "ROE"].mean()),
            "median_roe": float(ss.loc[ss["Code"].isin(rr.index), "ROE"].median()),
        })

        w = 1.0 / len(rr)
        for c in rr.index:
            holding_rows.append({
                "formation_monthend": t,
                "holding_monthend": t1,
                "Code": c,
                "weight": w,
                "roe_at_formation": float(ss.loc[ss["Code"] == c, "ROE"].iloc[0]) if (ss["Code"] == c).any() else np.nan,
                "next_month_return": float(rr.loc[c]),
            })

    monthly = pd.DataFrame(monthly_rows)
    holdings = pd.DataFrame(holding_rows)

    if monthly.empty:
        raise RuntimeError("No monthly rows generated. Check ROE coverage and price continuity.")

    # summary text
    r_g = monthly["ret_gross"]
    r_n = monthly["ret_net_tax_simple"]

    summary_lines = []
    summary_lines.append("=== Top20 ROE monthly rebalance (equal-weight) ===")
    summary_lines.append(f"months: {len(monthly)}  from {monthly['holding_monthend'].min().date()} to {monthly['holding_monthend'].max().date()}")
    summary_lines.append(f"tax model: simple monthly tax on positive returns, tax_rate={tax_rate:.5f}")
    summary_lines.append("")
    summary_lines.append("[GROSS]")
    summary_lines.append(f" total_return: {compound_returns(r_g):.4f}")
    summary_lines.append(f" CAGR       : {annualized_return(r_g):.4f}")
    summary_lines.append(f" vol_ann    : {annualized_vol(r_g):.4f}")
    summary_lines.append(f" sharpe_ann : {sharpe_ann(r_g):.3f}")
    summary_lines.append(f" maxDD      : {max_drawdown_from_monthly(r_g):.4f}")
    summary_lines.append("")
    summary_lines.append("[NET (tax simple)]")
    summary_lines.append(f" total_return: {compound_returns(r_n):.4f}")
    summary_lines.append(f" CAGR       : {annualized_return(r_n):.4f}")
    summary_lines.append(f" vol_ann    : {annualized_vol(r_n):.4f}")
    summary_lines.append(f" sharpe_ann : {sharpe_ann(r_n):.3f}")
    summary_lines.append(f" maxDD      : {max_drawdown_from_monthly(r_n):.4f}")

    return monthly, holdings, "\n".join(summary_lines)


def main():
    print("=== Backtest Top20 by ROE (monthly rebalance, equal-weight, simple tax) ===")

    if not PRICE_PATH.exists():
        raise RuntimeError(f"Missing: {PRICE_PATH}")
    if not SNAP_PATH.exists():
        raise RuntimeError(f"Missing: {SNAP_PATH}")

    price = pd.read_parquet(PRICE_PATH)
    snap = pd.read_parquet(SNAP_PATH)

    # build realized monthly returns from price_month_end
    rets = build_monthly_returns_from_price(price)

    monthly, holdings, summary = backtest_topn_by_roe_monthly(
        snap=snap,
        rets=rets,
        top_n=TOP_N,
        min_names=MIN_NAMES,
        tax_rate=TAX_RATE,
    )

    monthly.to_csv(OUT_MONTHLY, index=False, encoding="utf-8-sig")
    holdings.to_csv(OUT_HOLD, index=False, encoding="utf-8-sig")
    OUT_SUMMARY.write_text(summary, encoding="utf-8")

    print(summary)
    print(f"\n[SAVE] monthly:  {OUT_MONTHLY}")
    print(f"[SAVE] holdings: {OUT_HOLD}")
    print(f"[SAVE] summary:  {OUT_SUMMARY}")


if __name__ == "__main__":
    main()


=== Backtest Top20 by ROE (monthly rebalance, equal-weight, simple tax) ===
=== Top20 ROE monthly rebalance (equal-weight) ===
months: 114  from 2016-02-29 to 2025-09-30
tax model: simple monthly tax on positive returns, tax_rate=0.20315

[GROSS]
 total_return: 0.3686
 CAGR       : 0.0336
 vol_ann    : 0.2481
 sharpe_ann : 0.252
 maxDD      : -0.3743

[NET (tax simple)]
 total_return: -0.2502
 CAGR       : -0.0299
 vol_ann    : 0.2188
 sharpe_ann : -0.031
 maxDD      : -0.5537

[SAVE] monthly:  C:\Users\yongr\Project\merged_data_all_stocks\factors\backtest_top20_roe_monthly\backtest_top20_roe_monthly_results.csv
[SAVE] holdings: C:\Users\yongr\Project\merged_data_all_stocks\factors\backtest_top20_roe_monthly\backtest_top20_roe_monthly_holdings.csv
[SAVE] summary:  C:\Users\yongr\Project\merged_data_all_stocks\factors\backtest_top20_roe_monthly\backtest_top20_roe_monthly_summary.txt


In [2]:
import os
import pandas as pd
import numpy as np

TOPIX_CSV = r"C:\Users\yongr\Project\OHLCV_Adjusted\topix_index.csv"
OUT_DIR   = r"C:\Users\yongr\Project\OHLCV_Adjusted"
OUT_FILE  = "topix_month_end_with_filter.csv"

SMA_WINDOW = 10
MIN_PERIODS = 10

def main():
    if not os.path.exists(TOPIX_CSV):
        raise FileNotFoundError(f"TOPIX CSV not found: {TOPIX_CSV}")

    os.makedirs(OUT_DIR, exist_ok=True)
    out_path = os.path.join(OUT_DIR, OUT_FILE)

    df = pd.read_csv(TOPIX_CSV)
    need = {"Date", "Close", "Open"}
    missing = need - set(df.columns)
    if missing:
        raise ValueError(f"topix_index.csv missing columns: {missing}")

    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date").drop_duplicates("Date")
    df["ym"] = df["Date"].dt.to_period("M")

    # 月末（=最終取引日）
    idx = df.groupby("ym")["Date"].idxmax()
    me = df.loc[idx, ["Date", "Open", "Close"]].copy()
    me = me.rename(columns={"Date": "month_end_date",
                            "Open": "topix_open_me",
                            "Close": "topix_close_me"})
    me = me.sort_values("month_end_date").reset_index(drop=True)

    # SMA10（当月含む）
    me["topix_sma10"] = me["topix_close_me"].rolling(
        window=SMA_WINDOW, min_periods=MIN_PERIODS
    ).mean()

    # 当月末で判定
    me["risk_on"] = np.where(
        (me["topix_sma10"].notna()) & (me["topix_close_me"] > me["topix_sma10"]),
        1, 0
    ).astype(int)

    # 翌月適用（ラグ）
    me["risk_on_lag1"] = me["risk_on"].shift(1).fillna(0).astype(int)

    me.to_csv(out_path, index=False, encoding="utf-8-sig")

    print("Saved TOPIX month-end filter to:")
    print(out_path)
    print("Rows(months) =", len(me))
    print("First month_end_date =", me["month_end_date"].min())
    print("Last  month_end_date =", me["month_end_date"].max())

if __name__ == "__main__":
    main()


Saved TOPIX month-end filter to:
C:\Users\yongr\Project\OHLCV_Adjusted\topix_month_end_with_filter.csv
Rows(months) = 121
First month_end_date = 2015-12-30 00:00:00
Last  month_end_date = 2025-12-08 00:00:00


In [4]:
import os
import pandas as pd

RESULTS_CSV = r"C:\Users\yongr\Project\merged_data_all_stocks\factors\backtest_top20_roe_monthly\backtest_top20_roe_monthly_results.csv"
TOPIX_ME_CSV = r"C:\Users\yongr\Project\OHLCV_Adjusted\topix_month_end_with_filter.csv"
OUT_CSV = r"C:\Users\yongr\Project\merged_data_all_stocks\factors\backtest_top20_roe_monthly\backtest_top20_roe_monthly_results_with_topix_filter.csv"

def main():
    if not os.path.exists(RESULTS_CSV):
        raise FileNotFoundError(f"results csv not found: {RESULTS_CSV}")
    if not os.path.exists(TOPIX_ME_CSV):
        raise FileNotFoundError(f"topix month-end csv not found: {TOPIX_ME_CSV}")

    res = pd.read_csv(RESULTS_CSV)
    topix = pd.read_csv(TOPIX_ME_CSV)

    # 日付整備
    res["holding_monthend"] = pd.to_datetime(res["holding_monthend"])
    topix["month_end_date"] = pd.to_datetime(topix["month_end_date"])

    # 年月キー（YYYY-MM）を作る
    res["ym"] = res["holding_monthend"].dt.to_period("M").astype(str)
    topix["ym"] = topix["month_end_date"].dt.to_period("M").astype(str)

    # 年月でマージ
    m = res.merge(
        topix[["ym", "risk_on_lag1", "topix_close_me", "topix_sma10", "month_end_date"]],
        on="ym",
        how="left",
        validate="one_to_one"
    )

    # マージ失敗チェック
    if m["risk_on_lag1"].isna().any():
        missing_ym = m.loc[m["risk_on_lag1"].isna(), "ym"].head(30)
        raise ValueError(
            "TOPIX merge failed for some months (ym). Example missing ym:\n"
            + missing_ym.to_string(index=False)
        )

    # OFFは現金(0%) → リターン0
    m["ret_gross_topix_filter"] = m["ret_gross"] * m["risk_on_lag1"]
    m["ret_net_tax_simple_topix_filter"] = m["ret_net_tax_simple"] * m["risk_on_lag1"]

    m.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
    print("Saved:", OUT_CSV)
    on_ratio = m["risk_on_lag1"].mean()
    print(f"risk_on_lag1 ON ratio = {on_ratio:.2%}")

    # 参考：暦月末(holding_monthend)とTOPIXの最終取引日(month_end_date)のズレを数値化
    m["me_gap_days"] = (m["holding_monthend"] - m["month_end_date"]).dt.days
    print("month_end_date gap days (holding_monthend - topix month_end_date):")
    print(m["me_gap_days"].describe())

if __name__ == "__main__":
    main()


Saved: C:\Users\yongr\Project\merged_data_all_stocks\factors\backtest_top20_roe_monthly\backtest_top20_roe_monthly_results_with_topix_filter.csv
risk_on_lag1 ON ratio = 60.53%
month_end_date gap days (holding_monthend - topix month_end_date):
count    114.000000
mean       0.561404
std        0.882826
min        0.000000
25%        0.000000
50%        0.000000
75%        1.000000
max        4.000000
Name: me_gap_days, dtype: float64


In [5]:
import numpy as np
import pandas as pd
import os

IN_CSV = r"C:\Users\yongr\Project\merged_data_all_stocks\factors\backtest_top20_roe_monthly\backtest_top20_roe_monthly_results_with_topix_filter.csv"
OUT_SUMMARY_TXT = r"C:\Users\yongr\Project\merged_data_all_stocks\factors\backtest_top20_roe_monthly\topix_filter_effect_summary.txt"
OUT_EQUITY_CSV = r"C:\Users\yongr\Project\merged_data_all_stocks\factors\backtest_top20_roe_monthly\topix_filter_equity_curves.csv"

def perf_stats(r: pd.Series, periods_per_year=12):
    r = r.dropna().astype(float)
    n = len(r)
    if n == 0:
        return {}

    equity = (1.0 + r).cumprod()
    total_return = equity.iloc[-1] - 1.0
    cagr = equity.iloc[-1] ** (periods_per_year / n) - 1.0

    vol_ann = r.std(ddof=1) * np.sqrt(periods_per_year)
    mean_ann = r.mean() * periods_per_year
    sharpe = np.nan
    if vol_ann and vol_ann > 0:
        sharpe = mean_ann / vol_ann

    # max drawdown
    peak = equity.cummax()
    dd = equity / peak - 1.0
    max_dd = dd.min()

    calmar = np.nan
    if max_dd < 0:
        calmar = cagr / abs(max_dd)

    stats = {
        "n_months": n,
        "total_return": total_return,
        "CAGR": cagr,
        "vol_ann": vol_ann,
        "sharpe_ann": sharpe,
        "maxDD": max_dd,
        "calmar": calmar,
    }
    return stats, equity, dd

def fmt(stats):
    return "\n".join([
        f"n_months     : {stats['n_months']}",
        f"total_return : {stats['total_return']:.4f}",
        f"CAGR         : {stats['CAGR']:.4f}",
        f"vol_ann      : {stats['vol_ann']:.4f}",
        f"sharpe_ann   : {stats['sharpe_ann']:.4f}" if np.isfinite(stats['sharpe_ann']) else "sharpe_ann   : NaN",
        f"maxDD        : {stats['maxDD']:.4f}",
        f"calmar       : {stats['calmar']:.4f}" if np.isfinite(stats['calmar']) else "calmar       : NaN",
    ])

def main():
    if not os.path.exists(IN_CSV):
        raise FileNotFoundError(IN_CSV)

    df = pd.read_csv(IN_CSV)
    df["holding_monthend"] = pd.to_datetime(df["holding_monthend"])
    df = df.sort_values("holding_monthend").reset_index(drop=True)

    # 対象リターン列（税引前/後、フィルター無し/あり）
    col_map = {
        "GROSS (no filter)" : "ret_gross",
        "GROSS (TOPIX filter)" : "ret_gross_topix_filter",
        "NET tax_simple (no filter)" : "ret_net_tax_simple",
        "NET tax_simple (TOPIX filter)" : "ret_net_tax_simple_topix_filter",
    }

    lines = []
    equity_df = pd.DataFrame({"date": df["holding_monthend"]})
    dd_df = pd.DataFrame({"date": df["holding_monthend"]})

    for name, col in col_map.items():
        if col not in df.columns:
            lines.append(f"[SKIP] {name}: column not found: {col}")
            continue

        stats, equity, dd = perf_stats(df[col])
        lines.append("="*80)
        lines.append(name)
        lines.append("-"*80)
        lines.append(fmt(stats))

        # エクイティ/ドローダウン曲線も保存
        equity_df[name] = equity.values
        dd_df[name + " DD"] = dd.values

    # ON比率など補助情報
    lines.append("="*80)
    lines.append("Filter diagnostics")
    lines.append("-"*80)
    if "risk_on_lag1" in df.columns:
        lines.append(f"risk_on_lag1 ON ratio: {df['risk_on_lag1'].mean():.2%}")
    if "me_gap_days" in df.columns:
        lines.append(f"month_end gap days mean: {df['me_gap_days'].mean():.3f}, max: {df['me_gap_days'].max()}")

    # 保存
    with open(OUT_SUMMARY_TXT, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

    out_equity = equity_df.merge(dd_df, on="date", how="left")
    out_equity.to_csv(OUT_EQUITY_CSV, index=False, encoding="utf-8-sig")

    print("Saved summary:", OUT_SUMMARY_TXT)
    print("Saved curves :", OUT_EQUITY_CSV)

if __name__ == "__main__":
    main()


Saved summary: C:\Users\yongr\Project\merged_data_all_stocks\factors\backtest_top20_roe_monthly\topix_filter_effect_summary.txt
Saved curves : C:\Users\yongr\Project\merged_data_all_stocks\factors\backtest_top20_roe_monthly\topix_filter_equity_curves.csv


In [6]:
import pandas as pd
import numpy as np

CSV = r"C:\Users\yongr\Project\merged_data_all_stocks\factors\backtest_top20_roe_monthly\backtest_top20_roe_monthly_results_with_topix_filter.csv"

df = pd.read_csv(CSV)
df["holding_monthend"] = pd.to_datetime(df["holding_monthend"])
df = df.sort_values("holding_monthend").reset_index(drop=True)

# フィルター適用後（GROSS）のエクイティとDD
r = df["ret_gross_topix_filter"].astype(float)
eq = (1+r).cumprod()
peak = eq.cummax()
dd = eq/peak - 1.0
df["dd_filtered_gross"] = dd
df["eq_filtered_gross"] = eq

# DDが悪い月（=大きく落ちてる月）を確認
# 「DD水準が低い（よりマイナス）」順に上位を出す
cols = ["holding_monthend","ret_gross","ret_gross_topix_filter","risk_on_lag1","topix_close_me","topix_sma10","dd_filtered_gross"]
print(df.sort_values("dd_filtered_gross").head(20)[cols].to_string(index=False))


holding_monthend  ret_gross  ret_gross_topix_filter  risk_on_lag1  topix_close_me  topix_sma10  dd_filtered_gross
      2025-01-31  -0.023498               -0.023498             1         2788.66     2742.792          -0.526289
      2024-09-30  -0.013740               -0.013740             1         2645.94     2683.996          -0.514890
      2024-11-30   0.021470                0.000000             0         2680.71     2729.869          -0.514890
      2024-10-31  -0.051245               -0.000000             0         2695.51     2716.908          -0.514890
      2024-12-31   0.000429                0.000000             0         2784.92     2740.788          -0.514890
      2025-03-31  -0.018835               -0.000000             0         2658.73     2725.308          -0.513026
      2025-04-30   0.014530                0.000000             0         2667.29     2711.074          -0.513026
      2025-02-28   0.027999                0.027999             1         2682.09     27

In [7]:
import pandas as pd
import numpy as np

CSV = r"C:\Users\yongr\Project\merged_data_all_stocks\factors\backtest_top20_roe_monthly\backtest_top20_roe_monthly_results_with_topix_filter.csv"
df = pd.read_csv(CSV)
df["holding_monthend"] = pd.to_datetime(df["holding_monthend"])
df = df.sort_values("holding_monthend").reset_index(drop=True)

r = df["ret_gross"].astype(float)
on = df["risk_on_lag1"].astype(int)

def stats(x):
    eq = (1+x).cumprod()
    dd = eq/eq.cummax()-1
    n = len(x)
    cagr = eq.iloc[-1]**(12/n)-1
    return float(eq.iloc[-1]-1), float(cagr), float(dd.min())

for off_w in [0.0, 0.25, 0.5]:
    rf = on*r + (1-on)*(0.0) + (1-on)*off_w*r  # OFFのときoff_wだけ株を持つ
    tr, cagr, mdd = stats(rf)
    print(f"OFF weight={off_w:.2f}  total_return={tr:.4f}  CAGR={cagr:.4f}  MDD={mdd:.4f}")


OFF weight=0.00  total_return=-0.1793  CAGR=-0.0206  MDD=-0.5263
OFF weight=0.25  total_return=-0.0366  CAGR=-0.0039  MDD=-0.4789
OFF weight=0.50  total_return=0.1060  CAGR=0.0107  MDD=-0.4339


In [8]:
import pandas as pd
import numpy as np

CSV = r"C:\Users\yongr\Project\merged_data_all_stocks\factors\backtest_top20_roe_monthly\backtest_top20_roe_monthly_results_with_topix_filter.csv"
df = pd.read_csv(CSV)
df["holding_monthend"] = pd.to_datetime(df["holding_monthend"])
df = df.sort_values("holding_monthend").reset_index(drop=True)

r = df["ret_gross"].astype(float)

# risk_on（当月判定）と risk_on_lag1（翌月適用）が両方入っている前提ならrisk_onも使う
# 今のCSVにrisk_onが無ければ、topix_month_end_with_filter.csvからymでマージして足す必要あり
if "risk_on" not in df.columns:
    print("risk_on column not found. If you want lag0 test, merge risk_on from topix file.")
else:
    on0 = df["risk_on"].astype(int)
    on1 = df["risk_on_lag1"].astype(int)

    def stats(x):
        eq = (1+x).cumprod()
        dd = eq/eq.cummax()-1
        n = len(x)
        cagr = eq.iloc[-1]**(12/n)-1
        return float(eq.iloc[-1]-1), float(cagr), float(dd.min())

    for name, on in [("lag0", on0), ("lag1", on1)]:
        rf = on*r
        tr, cagr, mdd = stats(rf)
        print(f"{name}: total_return={tr:.4f}  CAGR={cagr:.4f}  MDD={mdd:.4f}")


risk_on column not found. If you want lag0 test, merge risk_on from topix file.


In [9]:
import pandas as pd
import numpy as np

RESULTS = r"C:\Users\yongr\Project\merged_data_all_stocks\factors\backtest_top20_roe_monthly\backtest_top20_roe_monthly_results_with_topix_filter.csv"
TOPIX   = r"C:\Users\yongr\Project\OHLCV_Adjusted\topix_month_end_with_filter.csv"
OUT     = r"C:\Users\yongr\Project\merged_data_all_stocks\factors\backtest_top20_roe_monthly\backtest_top20_roe_monthly_results_with_topix_filter_plus_risk_on.csv"

df = pd.read_csv(RESULTS)
df["holding_monthend"] = pd.to_datetime(df["holding_monthend"])
df["ym"] = df["holding_monthend"].dt.to_period("M").astype(str)

tx = pd.read_csv(TOPIX)
tx["month_end_date"] = pd.to_datetime(tx["month_end_date"])
tx["ym"] = tx["month_end_date"].dt.to_period("M").astype(str)

# risk_on（当月判定）を追加
m = df.merge(tx[["ym","risk_on"]], on="ym", how="left", validate="one_to_one")
if m["risk_on"].isna().any():
    miss = m.loc[m["risk_on"].isna(), "ym"].head(20)
    raise ValueError("missing risk_on for ym:\n" + miss.to_string(index=False))

m.to_csv(OUT, index=False, encoding="utf-8-sig")
print("Saved:", OUT)

# lag0/lag1 簡易比較（GROSS）
r = m["ret_gross"].astype(float)
on0 = m["risk_on"].astype(int)         # 当月判定で当月適用（診断用）
on1 = m["risk_on_lag1"].astype(int)    # 翌月適用（運用想定）

def stats(x):
    eq = (1+x).cumprod()
    dd = eq/eq.cummax()-1
    n = len(x)
    cagr = eq.iloc[-1]**(12/n)-1
    return float(eq.iloc[-1]-1), float(cagr), float(dd.min())

for name, on in [("lag0", on0), ("lag1", on1)]:
    tr, cagr, mdd = stats(on*r)
    print(f"{name}: total_return={tr:.4f}  CAGR={cagr:.4f}  MDD={mdd:.4f}")


Saved: C:\Users\yongr\Project\merged_data_all_stocks\factors\backtest_top20_roe_monthly\backtest_top20_roe_monthly_results_with_topix_filter_plus_risk_on.csv
lag0: total_return=1.0319  CAGR=0.0775  MDD=-0.2014
lag1: total_return=-0.1793  CAGR=-0.0206  MDD=-0.5263


In [11]:
import os
import re
import pandas as pd

# ====== ここは必要なら変更 ======
ROOT_DIR = r"C:\Users\yongr\Project\merged_data_all_stocks"   # 探索の起点
TARGET_KEYWORDS = [
    "month_end_snapshot", "snapshot",
    "price_month_end", "month_end",
    "holdings", "backtest_top20_roe_monthly"
]
MAX_FILES_TO_PRINT = 50
PREVIEW_ROWS = 5
# ==============================

def guess_cols(cols):
    cols_l = [c.lower() for c in cols]

    def pick(keys):
        res = []
        for c, cl in zip(cols, cols_l):
            if any(k in cl for k in keys):
                res.append(c)
        return res

    date_like = pick(["date", "month", "end", "formation", "holding", "time"])
    code_like = pick(["code", "ticker", "security", "sec", "localcode", "permno", "symbol", "銘柄"])
    roe_like  = pick(["roe", "returnonequity"])
    mom_like  = pick(["mom", "momentum", "12_1", "12-1", "12m"])
    ret_like  = pick(["ret", "return", "pnl", "profit"])

    return {
        "date_like": date_like,
        "code_like": code_like,
        "roe_like": roe_like,
        "mom_like": mom_like,
        "return_like": ret_like,
    }

def safe_read(path):
    ext = os.path.splitext(path)[1].lower()
    try:
        if ext == ".parquet":
            return pd.read_parquet(path)
        elif ext in [".csv", ".txt"]:
            return pd.read_csv(path)
        else:
            return None
    except Exception as e:
        return f"ERROR: {e}"

def is_interesting(filename):
    f = filename.lower()
    return any(k.lower() in f for k in TARGET_KEYWORDS)

def main():
    print("ROOT_DIR =", ROOT_DIR)
    if not os.path.exists(ROOT_DIR):
        raise FileNotFoundError(f"ROOT_DIR not found: {ROOT_DIR}")

    hits = []
    for root, dirs, files in os.walk(ROOT_DIR):
        for fn in files:
            if is_interesting(fn):
                hits.append(os.path.join(root, fn))

    hits = sorted(hits)
    print(f"Found {len(hits)} candidate files.")
    for i, p in enumerate(hits[:MAX_FILES_TO_PRINT], 1):
        print(f"[{i}] {p}")
    if len(hits) > MAX_FILES_TO_PRINT:
        print(f"... (showing first {MAX_FILES_TO_PRINT})")

    # 重要そうなファイルから順に中身を覗く
    # 優先順位：snapshot → price_month_end → holdings → others
    def priority(p):
        pl = p.lower()
        if "month_end_snapshot" in pl:
            return 0
        if "snapshot" in pl:
            return 1
        if "price_month_end" in pl:
            return 2
        if "holdings" in pl:
            return 3
        return 9

    hits2 = sorted(hits, key=priority)

    print("\n" + "="*100)
    print("INSPECTION (top 10 by priority)")
    print("="*100)

    for p in hits2[:10]:
        print("\n" + "-"*100)
        print("FILE:", p)

        obj = safe_read(p)
        if isinstance(obj, str) and obj.startswith("ERROR:"):
            print(obj)
            continue
        if obj is None:
            print("SKIP: unsupported file type")
            continue

        df = obj
        print("shape:", df.shape)

        cols = list(df.columns)
        print("columns:")
        for j, c in enumerate(cols):
            print(f"  {j:>3}: {c} (dtype={df[c].dtype})")

        g = guess_cols(cols)
        print("\n[guess cols]")
        for k, v in g.items():
            print(f"  {k}: {v if v else 'NONE'}")

        print("\nhead:")
        print(df.head(PREVIEW_ROWS).to_string(index=False))

if __name__ == "__main__":
    main()


ROOT_DIR = C:\Users\yongr\Project\merged_data_all_stocks
Found 19 candidate files.
[1] C:\Users\yongr\Project\merged_data_all_stocks\factors\backtest_top20_roe_monthly\backtest_top20_roe_monthly_holdings.csv
[2] C:\Users\yongr\Project\merged_data_all_stocks\factors\backtest_top20_roe_monthly\backtest_top20_roe_monthly_results.csv
[3] C:\Users\yongr\Project\merged_data_all_stocks\factors\backtest_top20_roe_monthly\backtest_top20_roe_monthly_results_with_topix_filter.csv
[4] C:\Users\yongr\Project\merged_data_all_stocks\factors\backtest_top20_roe_monthly\backtest_top20_roe_monthly_results_with_topix_filter_plus_risk_on.csv
[5] C:\Users\yongr\Project\merged_data_all_stocks\factors\backtest_top20_roe_monthly\backtest_top20_roe_monthly_summary.txt
[6] C:\Users\yongr\Project\merged_data_all_stocks\factors\diag_rebuild_eval_ff5mom\diag_snapshot_coverage_by_month_fullrebuild.csv
[7] C:\Users\yongr\Project\merged_data_all_stocks\factors\diag_update_norm\diag_snapshot_coverage_by_month.csv
[8] C

In [12]:
import pandas as pd

SNAPSHOT_PARQUET = r"C:\Users\yongr\Project\merged_data_all_stocks\factors\month_end_snapshot.parquet"
OUT_HOLDINGS = r"C:\Users\yongr\Project\merged_data_all_stocks\factors\backtest_top50_roe_buffer_monthly\holdings_top50_roe_buffer.csv"

N = 50
BUFFER = 30   # keep zone = top 80
MIN_ROE = None  # 例: 0.0 を入れるとROE<=0を除外。まずはNone推奨。

def select_with_buffer(df_month, prev_holdings, N=50, buffer=30):
    """
    df_month: 1ヶ月分のsnapshot（Code, MonthEnd, ROE, AdjustedClose などを含む）
    prev_holdings: 前月保有のlist（なければ空）
    """
    d = df_month.dropna(subset=["Code", "ROE", "AdjustedClose"]).copy()

    if MIN_ROE is not None:
        d = d[d["ROE"] > MIN_ROE]

    # ROE降順
    d = d.sort_values("ROE", ascending=False).reset_index(drop=True)
    d["rank"] = d.index + 1

    topN_codes = d.loc[d["rank"] <= N, "Code"].tolist()
    keep_zone = set(d.loc[d["rank"] <= (N + buffer), "Code"].tolist())

    prev_set = set(prev_holdings) if prev_holdings is not None else set()

    # 継続：keep_zoneに残っている前月保有を保持
    kept = [c for c in prev_set if c in keep_zone]

    # 不足分をTopNから補充
    final = kept[:]
    for c in topN_codes:
        if c not in final:
            final.append(c)
        if len(final) >= N:
            break

    # それでも不足ならROE上位から埋める
    if len(final) < N:
        for c in d["Code"].tolist():
            if c not in final:
                final.append(c)
            if len(final) >= N:
                break

    return final

def main():
    snap = pd.read_parquet(SNAPSHOT_PARQUET)
    snap["MonthEnd"] = pd.to_datetime(snap["MonthEnd"])
    snap = snap.sort_values(["MonthEnd", "Code"]).reset_index(drop=True)

    month_ends = sorted(snap["MonthEnd"].dropna().unique())
    rows = []
    prev = []

    for me in month_ends:
        dfm = snap[snap["MonthEnd"] == me]
        holdings = select_with_buffer(dfm, prev, N=N, buffer=BUFFER)
        for code in holdings:
            rows.append({"MonthEnd": me, "Code": code})
        prev = holdings

    out = pd.DataFrame(rows)
    # ディレクトリがなければ作る
    import os
    os.makedirs(os.path.dirname(OUT_HOLDINGS), exist_ok=True)
    out.to_csv(OUT_HOLDINGS, index=False, encoding="utf-8-sig")

    print("Saved holdings:", OUT_HOLDINGS)
    print("months:", out["MonthEnd"].nunique(), "rows:", len(out))
    print("avg holdings per month:", len(out)/out["MonthEnd"].nunique())

if __name__ == "__main__":
    main()


Saved holdings: C:\Users\yongr\Project\merged_data_all_stocks\factors\backtest_top50_roe_buffer_monthly\holdings_top50_roe_buffer.csv
months: 117 rows: 5850
avg holdings per month: 50.0


In [13]:
import pandas as pd

P = r"C:\Users\yongr\Project\merged_data_all_stocks\factors\price_month_end.parquet"
df = pd.read_parquet(P)
print("shape:", df.shape)
print("columns:")
for c in df.columns:
    print(" ", c, df[c].dtype)
print("\nhead:")
print(df.head(5).to_string(index=False))


shape: (474003, 5)
columns:
  Code object
  MonthEnd datetime64[ns]
  Date datetime64[ns]
  AdjustedClose float64
  MarketCap float64

head:
 Code   MonthEnd       Date  AdjustedClose  MarketCap
13010 2016-01-31 2016-01-29         2730.0        NaN
13010 2016-02-29 2016-02-29         2620.0        NaN
13010 2016-03-31 2016-03-31         2580.0        NaN
13010 2016-04-30 2016-04-28         2590.0        NaN
13010 2016-05-31 2016-05-31         2680.0        NaN


In [14]:
import os
import numpy as np
import pandas as pd

# ===== paths =====
HOLDINGS_CSV = r"C:\Users\yongr\Project\merged_data_all_stocks\factors\backtest_top50_roe_buffer_monthly\holdings_top50_roe_buffer.csv"
PRICE_PARQUET = r"C:\Users\yongr\Project\merged_data_all_stocks\factors\price_month_end.parquet"

OUT_DIR = r"C:\Users\yongr\Project\merged_data_all_stocks\factors\backtest_top50_roe_buffer_monthly"
OUT_RESULTS = os.path.join(OUT_DIR, "backtest_top50_roe_buffer_monthly_results.csv")
OUT_HOLDINGS = os.path.join(OUT_DIR, "backtest_top50_roe_buffer_monthly_holdings.csv")
OUT_SUMMARY = os.path.join(OUT_DIR, "backtest_top50_roe_buffer_monthly_summary.txt")

# ===== settings =====
TAX_RATE = 0.20315  # same as your tax_simple model
PERIODS_PER_YEAR = 12
DROP_FIRST_MONTH_WITHOUT_RETURN = True

def perf_stats(r: pd.Series, periods_per_year=12):
    r = r.dropna().astype(float)
    n = len(r)
    if n == 0:
        return None

    eq = (1.0 + r).cumprod()
    total_return = eq.iloc[-1] - 1.0
    cagr = eq.iloc[-1] ** (periods_per_year / n) - 1.0

    vol_ann = r.std(ddof=1) * np.sqrt(periods_per_year)
    mean_ann = r.mean() * periods_per_year
    sharpe = np.nan if vol_ann == 0 else mean_ann / vol_ann

    peak = eq.cummax()
    dd = eq / peak - 1.0
    max_dd = dd.min()

    calmar = np.nan
    if max_dd < 0:
        calmar = cagr / abs(max_dd)

    return {
        "n_months": n,
        "total_return": float(total_return),
        "CAGR": float(cagr),
        "vol_ann": float(vol_ann),
        "sharpe_ann": float(sharpe) if np.isfinite(sharpe) else np.nan,
        "maxDD": float(max_dd),
        "calmar": float(calmar) if np.isfinite(calmar) else np.nan,
    }

def main():
    os.makedirs(OUT_DIR, exist_ok=True)

    # --- load holdings ---
    h = pd.read_csv(HOLDINGS_CSV)
    h["MonthEnd"] = pd.to_datetime(h["MonthEnd"])
    h["Code"] = h["Code"].astype(str)

    # --- load prices ---
    p = pd.read_parquet(PRICE_PARQUET)
    p["MonthEnd"] = pd.to_datetime(p["MonthEnd"])
    p["Code"] = p["Code"].astype(str)

    # keep needed cols only
    p = p[["Code", "MonthEnd", "AdjustedClose"]].copy()
    p = p.dropna(subset=["AdjustedClose"])

    # --- compute per-stock monthly return from AdjustedClose ---
    p = p.sort_values(["Code", "MonthEnd"])
    p["ret_1m"] = p.groupby("Code")["AdjustedClose"].pct_change()

    # --- merge holdings with next-month return (holding month return) ---
    # holdings file has MonthEnd = holding_monthend (the month you hold)
    # We want the return over that month: ret_1m at that MonthEnd.
    hp = h.merge(
        p[["Code", "MonthEnd", "ret_1m"]],
        on=["Code", "MonthEnd"],
        how="left"
    )

    # If a holding has no return, it means:
    # - missing price at that MonthEnd OR
    # - first observation for that Code (pct_change NaN)
    # We'll leave it NaN and exclude from that month's average.
    # (Alternative: treat missing as 0; not recommended.)
    # --- portfolio monthly return (equal-weight across available returns) ---
    monthly = (
        hp.groupby("MonthEnd")
          .agg(
              n_holdings=("Code", "count"),
              n_ret_available=("ret_1m", lambda x: x.notna().sum()),
              ret_gross=("ret_1m", "mean"),
          )
          .reset_index()
          .rename(columns={"MonthEnd": "holding_monthend"})
          .sort_values("holding_monthend")
          .reset_index(drop=True)
    )

    # --- turnover based on holdings list ---
    # turnover = 0.5 * sum(|w_t - w_{t-1}|) with equal weights
    # With equal weights and varying membership, can compute via overlap:
    # turnover ≈ 1 - overlap_fraction (for equal weight, same N each month)
    # Here N is fixed at 50. We'll compute:
    # overlap = |A∩B| ; turnover = 1 - overlap/N
    h2 = h.sort_values(["MonthEnd", "Code"])
    month_list = sorted(h2["MonthEnd"].unique())
    prev_set = None
    turnover_list = []
    for me in month_list:
        cur = set(h2.loc[h2["MonthEnd"] == me, "Code"].tolist())
        if prev_set is None:
            turnover = np.nan
        else:
            overlap = len(cur.intersection(prev_set))
            turnover = 1.0 - overlap / float(len(cur)) if len(cur) > 0 else np.nan
        turnover_list.append((me, turnover))
        prev_set = cur

    turnover_df = pd.DataFrame(turnover_list, columns=["holding_monthend", "turnover"])
    turnover_df["holding_monthend"] = pd.to_datetime(turnover_df["holding_monthend"])
    monthly = monthly.merge(turnover_df, on="holding_monthend", how="left")

    # --- tax simple: tax on positive gross returns ---
    monthly["tax_rate"] = TAX_RATE
    monthly["tax_paid"] = np.where(monthly["ret_gross"] > 0, monthly["ret_gross"] * TAX_RATE, 0.0)
    monthly["ret_net_tax_simple"] = monthly["ret_gross"] - monthly["tax_paid"]

    # --- optional: drop first month (ret_1m is NaN for many codes) ---
    if DROP_FIRST_MONTH_WITHOUT_RETURN:
        # drop months where ret_gross is NaN (typically first month)
        monthly = monthly[monthly["ret_gross"].notna()].copy().reset_index(drop=True)

    # Save holdings (for audit)
    h_out = h.rename(columns={"MonthEnd": "holding_monthend"})
    h_out.to_csv(OUT_HOLDINGS, index=False, encoding="utf-8-sig")

    # Save results
    monthly.to_csv(OUT_RESULTS, index=False, encoding="utf-8-sig")

    # Summary
    gross_stats = perf_stats(monthly["ret_gross"], PERIODS_PER_YEAR)
    net_stats = perf_stats(monthly["ret_net_tax_simple"], PERIODS_PER_YEAR)

    lines = []
    lines.append("Backtest: Top50 ROE + Buffer (N=50, buffer=30), monthly rebalance, equal-weight")
    lines.append(f"Holdings file: {HOLDINGS_CSV}")
    lines.append(f"Price file   : {PRICE_PARQUET}")
    lines.append(f"Tax model    : simple tax on positive monthly return, tax_rate={TAX_RATE}")
    lines.append("")
    lines.append(f"Period: {monthly['holding_monthend'].min().date()} to {monthly['holding_monthend'].max().date()}")
    lines.append(f"Months: {len(monthly)}")
    lines.append("")
    lines.append("GROSS")
    for k, v in gross_stats.items():
        lines.append(f"  {k}: {v:.6f}" if isinstance(v, float) else f"  {k}: {v}")
    lines.append("")
    lines.append("NET (tax_simple)")
    for k, v in net_stats.items():
        lines.append(f"  {k}: {v:.6f}" if isinstance(v, float) else f"  {k}: {v}")

    with open(OUT_SUMMARY, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

    print("Saved results :", OUT_RESULTS)
    print("Saved holdings:", OUT_HOLDINGS)
    print("Saved summary :", OUT_SUMMARY)
    print("Months:", len(monthly))

if __name__ == "__main__":
    main()


Saved results : C:\Users\yongr\Project\merged_data_all_stocks\factors\backtest_top50_roe_buffer_monthly\backtest_top50_roe_buffer_monthly_results.csv
Saved holdings: C:\Users\yongr\Project\merged_data_all_stocks\factors\backtest_top50_roe_buffer_monthly\backtest_top50_roe_buffer_monthly_holdings.csv
Saved summary : C:\Users\yongr\Project\merged_data_all_stocks\factors\backtest_top50_roe_buffer_monthly\backtest_top50_roe_buffer_monthly_summary.txt
Months: 116


In [15]:
import numpy as np
import pandas as pd

IN_CSV = r"C:\Users\yongr\Project\merged_data_all_stocks\factors\backtest_top50_roe_buffer_monthly\backtest_top50_roe_buffer_monthly_results.csv"

PER_YEAR = 12

def perf(r):
    r = r.dropna().astype(float)
    n = len(r)
    eq = (1+r).cumprod()
    total_return = eq.iloc[-1] - 1
    cagr = eq.iloc[-1] ** (PER_YEAR/n) - 1

    vol = r.std(ddof=1) * np.sqrt(PER_YEAR)
    mean = r.mean() * PER_YEAR
    sharpe = np.nan if vol == 0 else mean/vol

    dd = eq/eq.cummax() - 1
    mdd = dd.min()
    calmar = np.nan if mdd >= 0 else cagr/abs(mdd)

    return {
        "n_months": n,
        "total_return": float(total_return),
        "CAGR": float(cagr),
        "vol_ann": float(vol),
        "sharpe_ann": float(sharpe) if np.isfinite(sharpe) else np.nan,
        "maxDD": float(mdd),
        "calmar": float(calmar) if np.isfinite(calmar) else np.nan,
    }

df = pd.read_csv(IN_CSV)
df["holding_monthend"] = pd.to_datetime(df["holding_monthend"])
df = df.sort_values("holding_monthend").reset_index(drop=True)

gross = perf(df["ret_gross"])
net = perf(df["ret_net_tax_simple"])

print("=== Top50 ROE + Buffer (GROSS) ===")
for k,v in gross.items():
    print(f"{k}: {v:.6f}" if isinstance(v,float) else f"{k}: {v}")

print("\n=== Top50 ROE + Buffer (NET tax_simple) ===")
for k,v in net.items():
    print(f"{k}: {v:.6f}" if isinstance(v,float) else f"{k}: {v}")

print("\nTurnover summary:")
print(df["turnover"].describe())


=== Top50 ROE + Buffer (GROSS) ===
n_months: 116
total_return: 3.828492
CAGR: 0.176899
vol_ann: 0.224220
sharpe_ann: 0.842195
maxDD: -0.333454
calmar: 0.530504

=== Top50 ROE + Buffer (NET tax_simple) ===
n_months: 116
total_return: 1.365412
CAGR: 0.093151
vol_ann: 0.198333
sharpe_ann: 0.550747
maxDD: -0.333454
calmar: 0.279350

Turnover summary:
count    116.000000
mean       0.247069
std        0.170304
min        0.040000
25%        0.120000
50%        0.200000
75%        0.320000
max        0.740000
Name: turnover, dtype: float64


In [16]:
import os
import numpy as np
import pandas as pd

# ===== Input files (確定) =====
SNAPSHOT_PARQUET = r"C:\Users\yongr\Project\merged_data_all_stocks\factors\month_end_snapshot.parquet"
PRICE_PARQUET    = r"C:\Users\yongr\Project\merged_data_all_stocks\factors\price_month_end.parquet"

# ===== Output dir =====
OUT_DIR = r"C:\Users\yongr\Project\merged_data_all_stocks\factors\param_sweep_topN_buffer"
os.makedirs(OUT_DIR, exist_ok=True)

# ===== Settings =====
TAX_RATE = 0.20315
PER_YEAR = 12

# 探索グリッド（必要なら増やしてOK）
N_LIST = [20, 50, 80, 100]
BUFFER_LIST = [0, 30, 60]   # 0=バンド無し（毎月TopNに寄せる）

# ROEフィルタ（まずは無し推奨）
MIN_ROE = None  # 例: 0.0 を入れるとROE<=0を除外

def perf_stats(r: pd.Series):
    r = r.dropna().astype(float)
    n = len(r)
    if n == 0:
        return None
    eq = (1+r).cumprod()
    total_return = float(eq.iloc[-1] - 1)
    cagr = float(eq.iloc[-1] ** (PER_YEAR/n) - 1)
    vol = float(r.std(ddof=1) * np.sqrt(PER_YEAR))
    mean = float(r.mean() * PER_YEAR)
    sharpe = np.nan if vol == 0 else float(mean/vol)
    dd = eq/eq.cummax() - 1
    mdd = float(dd.min())
    calmar = np.nan if mdd >= 0 else float(cagr/abs(mdd))
    return {
        "n_months": n,
        "total_return": total_return,
        "CAGR": cagr,
        "vol_ann": vol,
        "sharpe_ann": sharpe,
        "maxDD": mdd,
        "calmar": calmar
    }

def select_with_buffer(df_month, prev_holdings, N, buffer):
    """
    df_month: 1ヶ月分のsnapshot（Code, MonthEnd, ROE, AdjustedClose など）
    prev_holdings: list
    """
    d = df_month.dropna(subset=["Code", "ROE", "AdjustedClose"]).copy()
    if MIN_ROE is not None:
        d = d[d["ROE"] > MIN_ROE]

    d = d.sort_values("ROE", ascending=False).reset_index(drop=True)
    d["rank"] = d.index + 1

    topN = d.loc[d["rank"] <= N, "Code"].astype(str).tolist()

    if buffer <= 0:
        # バンド無し：毎月TopNをそのまま採用
        return topN

    keep_zone = set(d.loc[d["rank"] <= (N + buffer), "Code"].astype(str).tolist())
    prev_set = set([str(x) for x in prev_holdings]) if prev_holdings is not None else set()

    kept = [c for c in prev_set if c in keep_zone]

    final = kept[:]
    for c in topN:
        if c not in final:
            final.append(c)
        if len(final) >= N:
            break

    if len(final) < N:
        for c in d["Code"].astype(str).tolist():
            if c not in final:
                final.append(c)
            if len(final) >= N:
                break

    return final

def build_holdings(snap: pd.DataFrame, N, buffer):
    month_ends = sorted(snap["MonthEnd"].dropna().unique())
    rows = []
    prev = []
    for me in month_ends:
        dfm = snap[snap["MonthEnd"] == me]
        hold = select_with_buffer(dfm, prev, N=N, buffer=buffer)
        for code in hold:
            rows.append({"MonthEnd": me, "Code": str(code)})
        prev = hold
    return pd.DataFrame(rows)

def compute_strategy_results(holdings: pd.DataFrame, prices: pd.DataFrame):
    # prices: Code, MonthEnd, AdjustedClose
    p = prices.copy()
    p = p.dropna(subset=["AdjustedClose"])
    p = p.sort_values(["Code", "MonthEnd"])
    p["ret_1m"] = p.groupby("Code")["AdjustedClose"].pct_change()

    h = holdings.copy()
    h["Code"] = h["Code"].astype(str)
    hp = h.merge(p[["Code", "MonthEnd", "ret_1m"]], on=["Code", "MonthEnd"], how="left")

    monthly = (
        hp.groupby("MonthEnd")
          .agg(
              n_holdings=("Code", "count"),
              n_ret_available=("ret_1m", lambda x: x.notna().sum()),
              ret_gross=("ret_1m", "mean"),
          )
          .reset_index()
          .rename(columns={"MonthEnd": "holding_monthend"})
          .sort_values("holding_monthend")
          .reset_index(drop=True)
    )

    # turnover（等ウェイト前提、1 - overlap/N）
    h2 = h.sort_values(["MonthEnd", "Code"])
    month_list = sorted(h2["MonthEnd"].unique())
    prev_set = None
    trows = []
    for me in month_list:
        cur = set(h2.loc[h2["MonthEnd"] == me, "Code"].tolist())
        if prev_set is None:
            turnover = np.nan
        else:
            overlap = len(cur.intersection(prev_set))
            turnover = 1.0 - overlap / float(len(cur)) if len(cur) > 0 else np.nan
        trows.append((me, turnover))
        prev_set = cur
    turnover_df = pd.DataFrame(trows, columns=["holding_monthend", "turnover"])
    turnover_df["holding_monthend"] = pd.to_datetime(turnover_df["holding_monthend"])

    monthly = monthly.merge(turnover_df, on="holding_monthend", how="left")

    # tax_simple
    monthly["tax_rate"] = TAX_RATE
    monthly["tax_paid"] = np.where(monthly["ret_gross"] > 0, monthly["ret_gross"] * TAX_RATE, 0.0)
    monthly["ret_net_tax_simple"] = monthly["ret_gross"] - monthly["tax_paid"]

    # 先頭月（retが薄い/NaNになりやすい）を除外：Top50であなたが116か月になったのと合わせる
    monthly = monthly[monthly["ret_gross"].notna()].copy().reset_index(drop=True)

    return monthly

def main():
    # load snapshot & prices
    snap = pd.read_parquet(SNAPSHOT_PARQUET)
    snap["MonthEnd"] = pd.to_datetime(snap["MonthEnd"])
    snap["Code"] = snap["Code"].astype(str)

    prices = pd.read_parquet(PRICE_PARQUET)
    prices["MonthEnd"] = pd.to_datetime(prices["MonthEnd"])
    prices["Code"] = prices["Code"].astype(str)
    prices = prices[["Code", "MonthEnd", "AdjustedClose"]].copy()

    summary_rows = []

    for N in N_LIST:
        for B in BUFFER_LIST:
            tag = f"N{N}_B{B}"
            print("Running", tag)

            holdings = build_holdings(snap, N=N, buffer=B)
            results = compute_strategy_results(holdings, prices)

            gross = perf_stats(results["ret_gross"])
            net = perf_stats(results["ret_net_tax_simple"])
            avg_turn = float(results["turnover"].dropna().mean()) if results["turnover"].notna().any() else np.nan

            # save artifacts
            holdings_path = os.path.join(OUT_DIR, f"holdings_{tag}.csv")
            results_path  = os.path.join(OUT_DIR, f"results_{tag}.csv")
            holdings.to_csv(holdings_path, index=False, encoding="utf-8-sig")
            results.to_csv(results_path, index=False, encoding="utf-8-sig")

            row = {
                "tag": tag, "N": N, "buffer": B,
                "months": gross["n_months"],
                "avg_turnover": avg_turn,

                "gross_total_return": gross["total_return"],
                "gross_CAGR": gross["CAGR"],
                "gross_vol": gross["vol_ann"],
                "gross_sharpe": gross["sharpe_ann"],
                "gross_MDD": gross["maxDD"],
                "gross_calmar": gross["calmar"],

                "net_total_return": net["total_return"],
                "net_CAGR": net["CAGR"],
                "net_vol": net["vol_ann"],
                "net_sharpe": net["sharpe_ann"],
                "net_MDD": net["maxDD"],
                "net_calmar": net["calmar"],
            }
            summary_rows.append(row)

    summary = pd.DataFrame(summary_rows)
    out_csv = os.path.join(OUT_DIR, "param_sweep_summary.csv")
    summary.to_csv(out_csv, index=False, encoding="utf-8-sig")
    print("Saved summary table:", out_csv)

    # best-of quick report
    report_path = os.path.join(OUT_DIR, "best_by_metric.txt")
    lines = []
    lines.append("Best by metric (GROSS):")
    lines.append(str(summary.loc[summary["gross_calmar"].idxmax()][["tag","gross_CAGR","gross_MDD","gross_calmar","avg_turnover"]]))
    lines.append("")
    lines.append("Min MDD (GROSS):")
    lines.append(str(summary.loc[summary["gross_MDD"].idxmax()][["tag","gross_CAGR","gross_MDD","gross_calmar","avg_turnover"]]))  # maxDD is negative; closer to 0 is better so idxmax
    lines.append("")
    lines.append("Best by metric (NET):")
    lines.append(str(summary.loc[summary["net_calmar"].idxmax()][["tag","net_CAGR","net_MDD","net_calmar","avg_turnover"]]))
    lines.append("")
    lines.append("Min MDD (NET):")
    lines.append(str(summary.loc[summary["net_MDD"].idxmax()][["tag","net_CAGR","net_MDD","net_calmar","avg_turnover"]]))

    with open(report_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    print("Saved best report:", report_path)

if __name__ == "__main__":
    main()


Running N20_B0
Running N20_B30
Running N20_B60
Running N50_B0
Running N50_B30
Running N50_B60
Running N80_B0
Running N80_B30
Running N80_B60
Running N100_B0
Running N100_B30
Running N100_B60
Saved summary table: C:\Users\yongr\Project\merged_data_all_stocks\factors\param_sweep_topN_buffer\param_sweep_summary.csv
Saved best report: C:\Users\yongr\Project\merged_data_all_stocks\factors\param_sweep_topN_buffer\best_by_metric.txt


In [17]:
import pandas as pd

FILES = [
    r"C:\Users\yongr\Project\merged_data_all_stocks\merged_parts\merged-part-00001.parquet",
    r"C:\Users\yongr\Project\merged_data_all_stocks\merged_parts\merged-part-00002.parquet",
]

for f in FILES:
    print("\n" + "="*100)
    print("FILE:", f)
    df = pd.read_parquet(f)
    print("shape:", df.shape)
    print("columns:")
    for c in df.columns:
        print(f"  {c} ({df[c].dtype})")
    print("\nhead:")
    print(df.head(5).to_string(index=False))



FILE: C:\Users\yongr\Project\merged_data_all_stocks\merged_parts\merged-part-00001.parquet
shape: (8076101, 9)
columns:
  Date (datetime64[ns])
  Code (category)
  AdjustedClose (float64)
  Vo (float64)
  MarketCap (float32)
  BM_Ratio (float32)
  ROE (float32)
  Turnover (float32)
  INV_Growth (float32)

head:
      Date  Code  AdjustedClose       Vo  MarketCap  BM_Ratio  ROE     Turnover  INV_Growth
2016-01-15 13010         2650.0 114000.0        NaN       NaN  NaN  302100000.0         NaN
2016-01-18 13010         2630.0 173000.0        NaN       NaN  NaN  454990016.0         NaN
2016-01-19 13010         2600.0 140000.0        NaN       NaN  NaN  364000000.0         NaN
2016-01-20 13010         2550.0 400000.0        NaN       NaN  NaN 1020000000.0         NaN
2016-01-21 13010         2520.0 259000.0        NaN       NaN  NaN  652680000.0         NaN

FILE: C:\Users\yongr\Project\merged_data_all_stocks\merged_parts\merged-part-00002.parquet
shape: (1939961, 9)
columns:
  Date (datet

In [18]:
import os
import numpy as np
import pandas as pd

# ===== Paths =====
SNAPSHOT_PARQUET = r"C:\Users\yongr\Project\merged_data_all_stocks\factors\month_end_snapshot.parquet"

DAILY_PARTS = [
    r"C:\Users\yongr\Project\merged_data_all_stocks\merged_parts\merged-part-00001.parquet",
    r"C:\Users\yongr\Project\merged_data_all_stocks\merged_parts\merged-part-00002.parquet",
]

TOPIX_CSV = r"C:\Users\yongr\Project\OHLCV_Adjusted\topix_index.csv"

OUT_DIR = r"C:\Users\yongr\Project\merged_data_all_stocks\factors\backtest_top50_roe_daily_topix_intramonth_off"
os.makedirs(OUT_DIR, exist_ok=True)

OUT_DAILY = os.path.join(OUT_DIR, "daily_equity_curve.csv")
OUT_MONTHLY = os.path.join(OUT_DIR, "monthly_results.csv")
OUT_SUMMARY = os.path.join(OUT_DIR, "summary.txt")

# ===== Strategy params =====
N = 50          # Top50
PER_YEAR = 252  # daily
TOPIX_SMA_WINDOW = 200  # <- ここを100に変えればSMA100
USE_HYSTERESIS = False  # <- ここをTrueにすると 0.99*SMA をOFF閾値にできる
HYST_OFF_MULT = 0.99

def perf_stats_daily(r: pd.Series, periods_per_year=252):
    r = r.dropna().astype(float)
    n = len(r)
    eq = (1+r).cumprod()
    total_return = float(eq.iloc[-1] - 1)
    cagr = float(eq.iloc[-1] ** (periods_per_year/n) - 1)
    vol = float(r.std(ddof=1) * np.sqrt(periods_per_year))
    mean = float(r.mean() * periods_per_year)
    sharpe = np.nan if vol == 0 else float(mean/vol)
    dd = eq/eq.cummax() - 1
    mdd = float(dd.min())
    calmar = np.nan if mdd >= 0 else float(cagr/abs(mdd))
    return {"n_days": n, "total_return": total_return, "CAGR": cagr, "vol_ann": vol,
            "sharpe_ann": sharpe, "maxDD": mdd, "calmar": calmar}

def main():
    # --- TOPIX daily -> signal ---
    topix = pd.read_csv(TOPIX_CSV)
    topix["Date"] = pd.to_datetime(topix["Date"])
    topix = topix.sort_values("Date").drop_duplicates("Date")
    topix["topix_close"] = topix["Close"].astype(float)

    topix["topix_sma"] = topix["topix_close"].rolling(TOPIX_SMA_WINDOW, min_periods=TOPIX_SMA_WINDOW).mean()

    # ON/OFF
    if not USE_HYSTERESIS:
        topix["risk_on"] = np.where((topix["topix_sma"].notna()) & (topix["topix_close"] > topix["topix_sma"]), 1, 0).astype(int)
    else:
        # hysteresis: ON if close > sma, OFF if close < 0.99*sma, else keep prev state
        risk = []
        prev = 0
        for _, row in topix.iterrows():
            sma = row["topix_sma"]
            c = row["topix_close"]
            if pd.isna(sma):
                prev = 0
            else:
                if c > sma:
                    prev = 1
                elif c < HYST_OFF_MULT * sma:
                    prev = 0
            risk.append(prev)
        topix["risk_on"] = risk

    topix = topix[["Date", "risk_on", "topix_close", "topix_sma"]].copy()

    # --- Snapshot: monthly selection universe ---
    snap = pd.read_parquet(SNAPSHOT_PARQUET)
    snap["MonthEnd"] = pd.to_datetime(snap["MonthEnd"])
    snap["Code"] = snap["Code"].astype(str)
    snap = snap.dropna(subset=["ROE"])  # ROEが無いのは選定不可

    month_ends = sorted(snap["MonthEnd"].unique())

    # --- Load daily parts and concat (only needed cols) ---
    daily_list = []
    for fp in DAILY_PARTS:
        d = pd.read_parquet(fp)[["Date", "Code", "AdjustedClose"]].copy()
        d["Date"] = pd.to_datetime(d["Date"])
        d["Code"] = d["Code"].astype(str)  # category -> str
        d = d.dropna(subset=["AdjustedClose"])
        daily_list.append(d)

    px = pd.concat(daily_list, ignore_index=True)
    px = px.sort_values(["Code", "Date"]).reset_index(drop=True)

    # daily return per stock
    px["ret_d"] = px.groupby("Code")["AdjustedClose"].pct_change()

    # map daily date -> MonthEnd (calendar month end) for grouping
    # e.g. 2016-02-15 -> 2016-02-29
    px["MonthEnd"] = px["Date"] + pd.offsets.MonthEnd(0)

    # --- Build monthly holdings (Top50 ROE, B=0) ---
    holdings_rows = []
    for me in month_ends:
        dfm = snap[snap["MonthEnd"] == me].sort_values("ROE", ascending=False)
        top_codes = dfm["Code"].head(N).tolist()
        for c in top_codes:
            holdings_rows.append({"formation_monthend": me, "Code": c})

    holdings = pd.DataFrame(holdings_rows)

    # --- Apply holdings to NEXT month daily returns ---
    # formation_monthend=t で選ぶ → holding_monthend=t+1 の日次に適用
    holdings["holding_monthend"] = (holdings["formation_monthend"] + pd.offsets.MonthEnd(1)).dt.normalize()

    # join px with holdings by (Code, MonthEnd==holding_monthend)
    hp = holdings.merge(px, left_on=["Code", "holding_monthend"], right_on=["Code", "MonthEnd"], how="left")

    # portfolio daily return: equal weight across available stock returns each day
    port_daily = (
        hp.groupby("Date")
          .agg(
              n_holdings=("Code", "count"),
              n_ret_avail=("ret_d", lambda x: x.notna().sum()),
              port_ret=("ret_d", "mean"),
          )
          .reset_index()
          .sort_values("Date")
    )

    # merge TOPIX risk flag by Date
    port_daily = port_daily.merge(topix, on="Date", how="left")

    # 取引日一致しない場合は forward fill（TOPIXの休日等の差異に備える）
    port_daily["risk_on"] = port_daily["risk_on"].fillna(method="ffill").fillna(0).astype(int)

    # apply risk: OFF日は現金0%
    port_daily["port_ret_riskmanaged"] = port_daily["port_ret"] * port_daily["risk_on"]

    # equity curve
    port_daily["equity"] = (1 + port_daily["port_ret_riskmanaged"].fillna(0)).cumprod()
    peak = port_daily["equity"].cummax()
    port_daily["dd"] = port_daily["equity"]/peak - 1

    # monthly aggregation (for reporting)
    port_daily["holding_monthend"] = port_daily["Date"] + pd.offsets.MonthEnd(0)
    monthly = (
        port_daily.groupby("holding_monthend")
                  .agg(
                      gross_month_ret=("port_ret_riskmanaged", lambda x: (1+x.fillna(0)).prod() - 1),
                      avg_risk_on=("risk_on", "mean"),
                      n_days=("Date", "count")
                  )
                  .reset_index()
                  .sort_values("holding_monthend")
    )

    # outputs
    port_daily.to_csv(OUT_DAILY, index=False, encoding="utf-8-sig")
    monthly.to_csv(OUT_MONTHLY, index=False, encoding="utf-8-sig")

    stats = perf_stats_daily(port_daily["port_ret_riskmanaged"], periods_per_year=252)

    lines = []
    lines.append("Backtest: Top50 ROE (B=0, monthly selection) + TOPIX daily trend risk OFF intramonth")
    lines.append(f"Top50 selection snapshot: {SNAPSHOT_PARQUET}")
    lines.append(f"Daily parts: {DAILY_PARTS}")
    lines.append(f"TOPIX: {TOPIX_CSV}")
    lines.append(f"TOPIX_SMA_WINDOW={TOPIX_SMA_WINDOW}, USE_HYSTERESIS={USE_HYSTERESIS}, HYST_OFF_MULT={HYST_OFF_MULT}")
    lines.append("")
    lines.append(f"Period (daily): {port_daily['Date'].min().date()} to {port_daily['Date'].max().date()}")
    lines.append(f"Days: {int(stats['n_days'])}")
    lines.append("")
    for k in ["total_return","CAGR","vol_ann","sharpe_ann","maxDD","calmar"]:
        lines.append(f"{k}: {stats[k]:.6f}" if isinstance(stats[k], float) else f"{k}: {stats[k]}")

    with open(OUT_SUMMARY, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

    print("Saved daily  :", OUT_DAILY)
    print("Saved monthly:", OUT_MONTHLY)
    print("Saved summary:", OUT_SUMMARY)

if __name__ == "__main__":
    main()


Saved daily  : C:\Users\yongr\Project\merged_data_all_stocks\factors\backtest_top50_roe_daily_topix_intramonth_off\daily_equity_curve.csv
Saved monthly: C:\Users\yongr\Project\merged_data_all_stocks\factors\backtest_top50_roe_daily_topix_intramonth_off\monthly_results.csv
Saved summary: C:\Users\yongr\Project\merged_data_all_stocks\factors\backtest_top50_roe_daily_topix_intramonth_off\summary.txt


C:\Users\yongr\AppData\Local\Temp\ipykernel_32976\2979273067.py:135: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  port_daily["risk_on"] = port_daily["risk_on"].fillna(method="ffill").fillna(0).astype(int)


In [19]:
import os
import numpy as np
import pandas as pd

# ===== Paths =====
SNAPSHOT_PARQUET = r"C:\Users\yongr\Project\merged_data_all_stocks\factors\month_end_snapshot.parquet"

DAILY_PARTS = [
    r"C:\Users\yongr\Project\merged_data_all_stocks\merged_parts\merged-part-00001.parquet",
    r"C:\Users\yongr\Project\merged_data_all_stocks\merged_parts\merged-part-00002.parquet",
]

TOPIX_CSV = r"C:\Users\yongr\Project\OHLCV_Adjusted\topix_index.csv"

OUT_DIR = r"C:\Users\yongr\Project\merged_data_all_stocks\factors\backtest_top50_roexmom_daily_topix_intramonth_off_sma200"
os.makedirs(OUT_DIR, exist_ok=True)

OUT_DAILY = os.path.join(OUT_DIR, "daily_equity_curve.csv")
OUT_MONTHLY = os.path.join(OUT_DIR, "monthly_results.csv")
OUT_SUMMARY = os.path.join(OUT_DIR, "summary.txt")

# ===== Strategy params =====
N = 50
PER_YEAR = 252
TOPIX_SMA_WINDOW = 200

# MOM 12-1 (monthly)
MOM_LONG = 12
MOM_SKIP = 1

def zscore(s: pd.Series):
    x = s.astype(float)
    mu = x.mean()
    sd = x.std(ddof=0)
    if sd == 0 or np.isnan(sd):
        return pd.Series(np.nan, index=s.index)
    return (x - mu) / sd

def perf_stats_daily(r: pd.Series, periods_per_year=252):
    r = r.dropna().astype(float)
    n = len(r)
    eq = (1+r).cumprod()
    total_return = float(eq.iloc[-1] - 1)
    cagr = float(eq.iloc[-1] ** (periods_per_year/n) - 1)
    vol = float(r.std(ddof=1) * np.sqrt(periods_per_year))
    mean = float(r.mean() * periods_per_year)
    sharpe = np.nan if vol == 0 else float(mean/vol)
    dd = eq/eq.cummax() - 1
    mdd = float(dd.min())
    calmar = np.nan if mdd >= 0 else float(cagr/abs(mdd))
    return {"n_days": n, "total_return": total_return, "CAGR": cagr, "vol_ann": vol,
            "sharpe_ann": sharpe, "maxDD": mdd, "calmar": calmar}

def main():
    # --- TOPIX daily -> risk_on ---
    topix = pd.read_csv(TOPIX_CSV)
    topix["Date"] = pd.to_datetime(topix["Date"])
    topix = topix.sort_values("Date").drop_duplicates("Date")
    topix["topix_close"] = topix["Close"].astype(float)
    topix["topix_sma200"] = topix["topix_close"].rolling(
        TOPIX_SMA_WINDOW, min_periods=TOPIX_SMA_WINDOW
    ).mean()
    topix["risk_on"] = np.where(
        (topix["topix_sma200"].notna()) & (topix["topix_close"] > topix["topix_sma200"]),
        1, 0
    ).astype(int)
    topix = topix[["Date", "risk_on", "topix_close", "topix_sma200"]].copy()

    # --- Snapshot (ROE) ---
    snap = pd.read_parquet(SNAPSHOT_PARQUET)
    snap["MonthEnd"] = pd.to_datetime(snap["MonthEnd"])
    snap["Code"] = snap["Code"].astype(str)
    snap = snap.dropna(subset=["ROE"])
    month_ends = sorted(snap["MonthEnd"].unique())

    # --- Load daily parts (AdjustedClose) ---
    daily_list = []
    for fp in DAILY_PARTS:
        d = pd.read_parquet(fp)[["Date", "Code", "AdjustedClose"]].copy()
        d["Date"] = pd.to_datetime(d["Date"])
        d["Code"] = d["Code"].astype(str)
        d = d.dropna(subset=["AdjustedClose"])
        daily_list.append(d)
    px = pd.concat(daily_list, ignore_index=True)
    px = px.sort_values(["Code", "Date"]).reset_index(drop=True)

    # --- Build monthly price at calendar MonthEnd (use last trading day in that month) ---
    px["MonthEnd"] = (px["Date"] + pd.offsets.MonthEnd(0)).dt.normalize()
    # 月末バケット内で最後の取引日のAdjustedCloseを使う
    px_me = (
        px.sort_values(["Code", "Date"])
          .groupby(["Code", "MonthEnd"], as_index=False)
          .tail(1)[["Code", "MonthEnd", "AdjustedClose"]]
          .rename(columns={"AdjustedClose": "px_me"})
          .reset_index(drop=True)
    )

    # --- Compute MOM 12-1 using monthly px_me ---
    px_me = px_me.sort_values(["Code", "MonthEnd"])
    px_me["ret_1m"] = px_me.groupby("Code")["px_me"].pct_change()
    # MOM12_1 = (1+ret_{t-11}...t) - 1 から直に作るより、価格比で作るのが簡単：
    # mom12_1 = px_me(t-1) / px_me(t-12) - 1  （直近1か月を除外するためt-1を使う）
    px_me["px_lag1"] = px_me.groupby("Code")["px_me"].shift(MOM_SKIP)
    px_me["px_lag12"] = px_me.groupby("Code")["px_me"].shift(MOM_LONG)
    px_me["mom12_1"] = px_me["px_lag1"] / px_me["px_lag12"] - 1.0

    # --- Merge ROE snapshot with MOM at same MonthEnd (formation month) ---
    snap2 = snap.merge(px_me[["Code", "MonthEnd", "mom12_1"]], left_on=["Code", "MonthEnd"], right_on=["Code", "MonthEnd"], how="left")

    # --- Monthly selection by score = z(ROE) + z(MOM) ---
    holdings_rows = []
    for me in month_ends:
        dfm = snap2[snap2["MonthEnd"] == me].copy()
        dfm = dfm.dropna(subset=["ROE", "mom12_1"])  # MOMが作れない銘柄は除外（序盤は自然に減る）

        if len(dfm) == 0:
            continue

        dfm["roe_z"] = zscore(dfm["ROE"])
        dfm["mom_z"] = zscore(dfm["mom12_1"])
        dfm["score"] = dfm["roe_z"] + dfm["mom_z"]

        dfm = dfm.sort_values("score", ascending=False)
        top_codes = dfm["Code"].head(N).tolist()
        for c in top_codes:
            holdings_rows.append({"formation_monthend": me, "Code": c})

    holdings = pd.DataFrame(holdings_rows)
    holdings["formation_monthend"] = pd.to_datetime(holdings["formation_monthend"])
    holdings["holding_monthend"] = (holdings["formation_monthend"] + pd.offsets.MonthEnd(1)).dt.normalize()

    # --- Daily returns per stock ---
    px = px.sort_values(["Code", "Date"])
    px["ret_d"] = px.groupby("Code")["AdjustedClose"].pct_change()

    # --- Apply holdings to next month daily returns ---
    hp = holdings.merge(
        px,
        left_on=["Code", "holding_monthend"],
        right_on=["Code", "MonthEnd"],
        how="left"
    )

    port_daily = (
        hp.groupby("Date")
          .agg(
              n_holdings=("Code", "count"),
              n_ret_avail=("ret_d", lambda x: x.notna().sum()),
              port_ret=("ret_d", "mean"),
          )
          .reset_index()
          .sort_values("Date")
          .reset_index(drop=True)
    )

    # merge TOPIX risk flag
    port_daily = port_daily.merge(topix, on="Date", how="left")
    port_daily["risk_on"] = port_daily["risk_on"].ffill().fillna(0).astype(int)

    # OFF -> cash
    port_daily["port_ret_riskmanaged"] = port_daily["port_ret"].fillna(0) * port_daily["risk_on"]

    # equity / dd
    port_daily["equity"] = (1 + port_daily["port_ret_riskmanaged"]).cumprod()
    peak = port_daily["equity"].cummax()
    port_daily["dd"] = port_daily["equity"]/peak - 1

    # monthly aggregation
    port_daily["holding_monthend"] = (port_daily["Date"] + pd.offsets.MonthEnd(0)).dt.normalize()
    monthly = (
        port_daily.groupby("holding_monthend")
                  .agg(
                      gross_month_ret=("port_ret_riskmanaged", lambda x: (1+x).prod() - 1),
                      avg_risk_on=("risk_on", "mean"),
                      n_days=("Date", "count"),
                      avg_n_ret_avail=("n_ret_avail", "mean"),
                  )
                  .reset_index()
                  .sort_values("holding_monthend")
                  .reset_index(drop=True)
    )

    # save outputs
    port_daily.to_csv(OUT_DAILY, index=False, encoding="utf-8-sig")
    monthly.to_csv(OUT_MONTHLY, index=False, encoding="utf-8-sig")

    stats = perf_stats_daily(port_daily["port_ret_riskmanaged"], periods_per_year=252)

    lines = []
    lines.append("Backtest: Top50 score = z(ROE)+z(MOM12-1) (monthly selection) + TOPIX daily SMA200 intramonth OFF")
    lines.append(f"Snapshot: {SNAPSHOT_PARQUET}")
    lines.append(f"Daily parts: {DAILY_PARTS}")
    lines.append(f"TOPIX: {TOPIX_CSV}")
    lines.append(f"TOPIX_SMA_WINDOW={TOPIX_SMA_WINDOW}")
    lines.append(f"MOM: 12-1 using monthly last-trading-day px (lag1/lag12)")
    lines.append("")
    lines.append(f"Period (daily): {port_daily['Date'].min().date()} to {port_daily['Date'].max().date()}")
    lines.append(f"Days: {len(port_daily)}")
    lines.append("")
    for k in ["total_return","CAGR","vol_ann","sharpe_ann","maxDD","calmar"]:
        lines.append(f"{k}: {stats[k]:.6f}")

    with open(OUT_SUMMARY, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

    print("Saved daily  :", OUT_DAILY)
    print("Saved monthly:", OUT_MONTHLY)
    print("Saved summary:", OUT_SUMMARY)

if __name__ == "__main__":
    main()


Saved daily  : C:\Users\yongr\Project\merged_data_all_stocks\factors\backtest_top50_roexmom_daily_topix_intramonth_off_sma200\daily_equity_curve.csv
Saved monthly: C:\Users\yongr\Project\merged_data_all_stocks\factors\backtest_top50_roexmom_daily_topix_intramonth_off_sma200\monthly_results.csv
Saved summary: C:\Users\yongr\Project\merged_data_all_stocks\factors\backtest_top50_roexmom_daily_topix_intramonth_off_sma200\summary.txt


In [20]:
import os
import numpy as np
import pandas as pd

# ===== Paths =====
SNAPSHOT_PARQUET = r"C:\Users\yongr\Project\merged_data_all_stocks\factors\month_end_snapshot.parquet"
DAILY_PARTS = [
    r"C:\Users\yongr\Project\merged_data_all_stocks\merged_parts\merged-part-00001.parquet",
    r"C:\Users\yongr\Project\merged_data_all_stocks\merged_parts\merged-part-00002.parquet",
]
TOPIX_CSV = r"C:\Users\yongr\Project\OHLCV_Adjusted\topix_index.csv"

OUT_DIR = r"C:\Users\yongr\Project\merged_data_all_stocks\factors\w_sweep_roexmom_topix_sma200_intramonth_off"
os.makedirs(OUT_DIR, exist_ok=True)

OUT_SUMMARY_CSV = os.path.join(OUT_DIR, "w_sweep_summary.csv")
OUT_BEST_TXT = os.path.join(OUT_DIR, "w_sweep_best.txt")

# ===== Strategy params =====
N = 50
PER_YEAR = 252
TOPIX_SMA_WINDOW = 200

# MOM 12-1
MOM_LONG = 12
MOM_SKIP = 1

# sweep grid
W_LIST = [0.0, 0.1, 0.25, 0.35, 0.5, 0.75, 1.0]

def zscore(x: pd.Series):
    x = x.astype(float)
    mu = x.mean()
    sd = x.std(ddof=0)
    if sd == 0 or np.isnan(sd):
        return pd.Series(np.nan, index=x.index)
    return (x - mu) / sd

def perf_stats_daily(r: pd.Series, periods_per_year=252):
    r = r.dropna().astype(float)
    n = len(r)
    eq = (1+r).cumprod()
    total_return = float(eq.iloc[-1] - 1)
    cagr = float(eq.iloc[-1] ** (periods_per_year/n) - 1)
    vol = float(r.std(ddof=1) * np.sqrt(periods_per_year))
    mean = float(r.mean() * periods_per_year)
    sharpe = np.nan if vol == 0 else float(mean/vol)
    dd = eq/eq.cummax() - 1
    mdd = float(dd.min())
    calmar = np.nan if mdd >= 0 else float(cagr/abs(mdd))
    return {"n_days": int(n), "total_return": total_return, "CAGR": cagr, "vol_ann": vol,
            "sharpe_ann": sharpe, "maxDD": mdd, "calmar": calmar}

def prepare_common_data():
    # TOPIX risk_on
    topix = pd.read_csv(TOPIX_CSV)
    topix["Date"] = pd.to_datetime(topix["Date"])
    topix = topix.sort_values("Date").drop_duplicates("Date")
    topix["topix_close"] = topix["Close"].astype(float)
    topix["topix_sma200"] = topix["topix_close"].rolling(TOPIX_SMA_WINDOW, min_periods=TOPIX_SMA_WINDOW).mean()
    topix["risk_on"] = np.where(
        (topix["topix_sma200"].notna()) & (topix["topix_close"] > topix["topix_sma200"]),
        1, 0
    ).astype(int)
    topix = topix[["Date", "risk_on"]].copy()

    # Snapshot (ROE)
    snap = pd.read_parquet(SNAPSHOT_PARQUET)
    snap["MonthEnd"] = pd.to_datetime(snap["MonthEnd"])
    snap["Code"] = snap["Code"].astype(str)
    snap = snap.dropna(subset=["ROE"])
    month_ends = sorted(snap["MonthEnd"].unique())

    # Daily prices
    daily_list = []
    for fp in DAILY_PARTS:
        d = pd.read_parquet(fp)[["Date", "Code", "AdjustedClose"]].copy()
        d["Date"] = pd.to_datetime(d["Date"])
        d["Code"] = d["Code"].astype(str)
        d = d.dropna(subset=["AdjustedClose"])
        daily_list.append(d)

    px = pd.concat(daily_list, ignore_index=True)
    px = px.sort_values(["Code", "Date"]).reset_index(drop=True)

    # daily ret
    px["ret_d"] = px.groupby("Code")["AdjustedClose"].pct_change()

    # MonthEnd bucket (calendar)
    px["MonthEnd"] = (px["Date"] + pd.offsets.MonthEnd(0)).dt.normalize()

    # monthly last-trading-day price within each month
    px_me = (
        px.sort_values(["Code", "Date"])
          .groupby(["Code", "MonthEnd"], as_index=False)
          .tail(1)[["Code", "MonthEnd", "AdjustedClose"]]
          .rename(columns={"AdjustedClose": "px_me"})
          .reset_index(drop=True)
    )
    px_me = px_me.sort_values(["Code", "MonthEnd"])
    px_me["px_lag1"] = px_me.groupby("Code")["px_me"].shift(MOM_SKIP)
    px_me["px_lag12"] = px_me.groupby("Code")["px_me"].shift(MOM_LONG)
    px_me["mom12_1"] = px_me["px_lag1"] / px_me["px_lag12"] - 1.0

    # merge snapshot with mom
    snap2 = snap.merge(px_me[["Code", "MonthEnd", "mom12_1"]], on=["Code", "MonthEnd"], how="left")

    return topix, snap2, month_ends, px

def run_one_w(w, topix, snap2, month_ends, px):
    # build holdings by score
    rows = []
    for me in month_ends:
        dfm = snap2[snap2["MonthEnd"] == me].copy()
        dfm = dfm.dropna(subset=["ROE", "mom12_1"])
        if len(dfm) == 0:
            continue

        dfm["roe_z"] = zscore(dfm["ROE"])
        dfm["mom_z"] = zscore(dfm["mom12_1"])
        dfm["score"] = dfm["roe_z"] + w * dfm["mom_z"]

        top_codes = dfm.sort_values("score", ascending=False)["Code"].head(N).tolist()
        for c in top_codes:
            rows.append({"formation_monthend": me, "Code": c})

    holdings = pd.DataFrame(rows)
    if holdings.empty:
        return None

    holdings["formation_monthend"] = pd.to_datetime(holdings["formation_monthend"])
    holdings["holding_monthend"] = (holdings["formation_monthend"] + pd.offsets.MonthEnd(1)).dt.normalize()

    # apply holdings to daily returns by month bucket
    hp = holdings.merge(px, left_on=["Code", "holding_monthend"], right_on=["Code", "MonthEnd"], how="left")

    port_daily = (
        hp.groupby("Date")
          .agg(port_ret=("ret_d", "mean"))
          .reset_index()
          .sort_values("Date")
          .reset_index(drop=True)
    )

    port_daily = port_daily.merge(topix, on="Date", how="left")
    port_daily["risk_on"] = port_daily["risk_on"].ffill().fillna(0).astype(int)
    port_daily["port_ret_rm"] = port_daily["port_ret"].fillna(0) * port_daily["risk_on"]

    stats = perf_stats_daily(port_daily["port_ret_rm"], periods_per_year=PER_YEAR)
    stats["w"] = w
    stats["start"] = str(port_daily["Date"].min().date())
    stats["end"] = str(port_daily["Date"].max().date())
    return stats

def main():
    topix, snap2, month_ends, px = prepare_common_data()

    results = []
    for w in W_LIST:
        print("Running w =", w)
        s = run_one_w(w, topix, snap2, month_ends, px)
        if s is not None:
            results.append(s)

    df = pd.DataFrame(results).sort_values("w")
    df.to_csv(OUT_SUMMARY_CSV, index=False, encoding="utf-8-sig")
    print("Saved:", OUT_SUMMARY_CSV)

    # best candidates under DD constraint
    # target: maxDD >= -0.20 (closer to 0 is better)
    df_ok = df[df["maxDD"] >= -0.20].copy()
    lines = []
    lines.append("=== All results ===")
    lines.append(df.to_string(index=False))
    lines.append("\n=== Under DD constraint (maxDD >= -0.20) ===")
    if len(df_ok) == 0:
        lines.append("NONE")
    else:
        # maximize CAGR under constraint
        best_cagr = df_ok.sort_values("CAGR", ascending=False).head(1)
        # maximize calmar under constraint
        best_calmar = df_ok.sort_values("calmar", ascending=False).head(1)
        lines.append("Best CAGR under constraint:")
        lines.append(best_cagr.to_string(index=False))
        lines.append("\nBest Calmar under constraint:")
        lines.append(best_calmar.to_string(index=False))

    with open(OUT_BEST_TXT, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    print("Saved:", OUT_BEST_TXT)

if __name__ == "__main__":
    main()


Running w = 0.0
Running w = 0.1
Running w = 0.25
Running w = 0.35
Running w = 0.5
Running w = 0.75
Running w = 1.0
Saved: C:\Users\yongr\Project\merged_data_all_stocks\factors\w_sweep_roexmom_topix_sma200_intramonth_off\w_sweep_summary.csv
Saved: C:\Users\yongr\Project\merged_data_all_stocks\factors\w_sweep_roexmom_topix_sma200_intramonth_off\w_sweep_best.txt


In [21]:
import os
import numpy as np
import pandas as pd

# ===== Paths =====
SNAPSHOT_PARQUET = r"C:\Users\yongr\Project\merged_data_all_stocks\factors\month_end_snapshot.parquet"
DAILY_PARTS = [
    r"C:\Users\yongr\Project\merged_data_all_stocks\merged_parts\merged-part-00001.parquet",
    r"C:\Users\yongr\Project\merged_data_all_stocks\merged_parts\merged-part-00002.parquet",
]
TOPIX_CSV = r"C:\Users\yongr\Project\OHLCV_Adjusted\topix_index.csv"

OUT_DIR = r"C:\Users\yongr\Project\merged_data_all_stocks\factors\backtest_top50_roe_momfilter_daily_topix_intramonth_off_sma200"
os.makedirs(OUT_DIR, exist_ok=True)

OUT_SUMMARY = os.path.join(OUT_DIR, "summary.txt")
OUT_DAILY   = os.path.join(OUT_DIR, "daily_equity_curve.csv")
OUT_MONTHLY = os.path.join(OUT_DIR, "monthly_results.csv")

# ===== Strategy params =====
N_FINAL = 50
ROE_CANDIDATES = 200          # ROE候補を広めに取る（重要）
MOM_LONG = 12                 # 12-1 MOM
MOM_SKIP = 1
MOM_DROP_BOTTOM_Q = 0.30      # MOM下位30%を除外（まずは0.30推奨）

TOPIX_SMA_WINDOW = 200
PER_YEAR = 252

def perf_stats_daily(r: pd.Series, periods_per_year=252):
    r = r.dropna().astype(float)
    n = len(r)
    eq = (1+r).cumprod()
    total_return = float(eq.iloc[-1] - 1)
    cagr = float(eq.iloc[-1] ** (periods_per_year/n) - 1)
    vol = float(r.std(ddof=1) * np.sqrt(periods_per_year))
    mean = float(r.mean() * periods_per_year)
    sharpe = np.nan if vol == 0 else float(mean/vol)
    dd = eq/eq.cummax() - 1
    mdd = float(dd.min())
    calmar = np.nan if mdd >= 0 else float(cagr/abs(mdd))
    return {"n_days": int(n), "total_return": total_return, "CAGR": cagr, "vol_ann": vol,
            "sharpe_ann": sharpe, "maxDD": mdd, "calmar": calmar}

def main():
    # --- TOPIX risk_on (daily SMA200) ---
    topix = pd.read_csv(TOPIX_CSV)
    topix["Date"] = pd.to_datetime(topix["Date"])
    topix = topix.sort_values("Date").drop_duplicates("Date")
    topix["topix_close"] = topix["Close"].astype(float)
    topix["topix_sma200"] = topix["topix_close"].rolling(TOPIX_SMA_WINDOW, min_periods=TOPIX_SMA_WINDOW).mean()
    topix["risk_on"] = np.where(
        (topix["topix_sma200"].notna()) & (topix["topix_close"] > topix["topix_sma200"]),
        1, 0
    ).astype(int)
    topix = topix[["Date", "risk_on"]].copy()

    # --- Snapshot (ROE at MonthEnd) ---
    snap = pd.read_parquet(SNAPSHOT_PARQUET)
    snap["MonthEnd"] = pd.to_datetime(snap["MonthEnd"])
    snap["Code"] = snap["Code"].astype(str)
    snap = snap.dropna(subset=["ROE"])
    month_ends = sorted(snap["MonthEnd"].unique())

    # --- Daily prices (AdjustedClose) ---
    daily_list = []
    for fp in DAILY_PARTS:
        d = pd.read_parquet(fp)[["Date", "Code", "AdjustedClose"]].copy()
        d["Date"] = pd.to_datetime(d["Date"])
        d["Code"] = d["Code"].astype(str)
        d = d.dropna(subset=["AdjustedClose"])
        daily_list.append(d)

    px = pd.concat(daily_list, ignore_index=True)
    px = px.sort_values(["Code", "Date"]).reset_index(drop=True)

    # daily returns
    px["ret_d"] = px.groupby("Code")["AdjustedClose"].pct_change()

    # calendar MonthEnd bucket
    px["MonthEnd"] = (px["Date"] + pd.offsets.MonthEnd(0)).dt.normalize()

    # monthly last-trading-day price (per Code, MonthEnd)
    px_me = (
        px.sort_values(["Code", "Date"])
          .groupby(["Code", "MonthEnd"], as_index=False)
          .tail(1)[["Code", "MonthEnd", "AdjustedClose"]]
          .rename(columns={"AdjustedClose": "px_me"})
          .reset_index(drop=True)
    ).sort_values(["Code", "MonthEnd"])

    # MOM12-1 using px_me(t-1)/px_me(t-12)-1
    px_me["px_lag1"]  = px_me.groupby("Code")["px_me"].shift(MOM_SKIP)
    px_me["px_lag12"] = px_me.groupby("Code")["px_me"].shift(MOM_LONG)
    px_me["mom12_1"]  = px_me["px_lag1"] / px_me["px_lag12"] - 1.0

    snap2 = snap.merge(px_me[["Code", "MonthEnd", "mom12_1"]], on=["Code", "MonthEnd"], how="left")

    # --- Build holdings: ROE candidates -> drop bottom MOM quantile -> take top50 by ROE ---
    hold_rows = []
    for me in month_ends:
        dfm = snap2[snap2["MonthEnd"] == me].copy()
        dfm = dfm.dropna(subset=["ROE", "mom12_1"])
        if len(dfm) == 0:
            continue

        # ROE上位候補
        dfm = dfm.sort_values("ROE", ascending=False).head(ROE_CANDIDATES).copy()

        # MOM下位qを除外（地雷除去）
        q = dfm["mom12_1"].quantile(MOM_DROP_BOTTOM_Q)
        dfm_f = dfm[dfm["mom12_1"] > q].copy()

        # それでも足りなければ、除外を緩めて確実にN_FINAL確保
        if len(dfm_f) < N_FINAL:
            dfm_f = dfm.copy()

        top_codes = dfm_f.sort_values("ROE", ascending=False)["Code"].head(N_FINAL).tolist()
        for c in top_codes:
            hold_rows.append({"formation_monthend": me, "Code": c})

    holdings = pd.DataFrame(hold_rows)
    holdings["formation_monthend"] = pd.to_datetime(holdings["formation_monthend"])
    holdings["holding_monthend"] = (holdings["formation_monthend"] + pd.offsets.MonthEnd(1)).dt.normalize()

    # apply holdings to daily by MonthEnd bucket (next month)
    hp = holdings.merge(px, left_on=["Code", "holding_monthend"], right_on=["Code", "MonthEnd"], how="left")

    port_daily = (
        hp.groupby("Date")
          .agg(port_ret=("ret_d", "mean"))
          .reset_index()
          .sort_values("Date")
          .reset_index(drop=True)
    )

    # risk management
    port_daily = port_daily.merge(topix, on="Date", how="left")
    port_daily["risk_on"] = port_daily["risk_on"].ffill().fillna(0).astype(int)
    port_daily["port_ret_rm"] = port_daily["port_ret"].fillna(0) * port_daily["risk_on"]

    # equity / dd
    port_daily["equity"] = (1 + port_daily["port_ret_rm"]).cumprod()
    peak = port_daily["equity"].cummax()
    port_daily["dd"] = port_daily["equity"]/peak - 1

    # monthly aggregation
    port_daily["holding_monthend"] = (port_daily["Date"] + pd.offsets.MonthEnd(0)).dt.normalize()
    monthly = (
        port_daily.groupby("holding_monthend")
                  .agg(
                      gross_month_ret=("port_ret_rm", lambda x: (1+x).prod() - 1),
                      avg_risk_on=("risk_on", "mean"),
                      n_days=("Date", "count")
                  )
                  .reset_index()
                  .sort_values("holding_monthend")
                  .reset_index(drop=True)
    )

    # save
    port_daily.to_csv(OUT_DAILY, index=False, encoding="utf-8-sig")
    monthly.to_csv(OUT_MONTHLY, index=False, encoding="utf-8-sig")

    stats = perf_stats_daily(port_daily["port_ret_rm"], periods_per_year=PER_YEAR)

    lines = []
    lines.append("Backtest: Top50 ROE + MOM filter (drop bottom quantile) + TOPIX daily SMA200 intramonth OFF")
    lines.append(f"ROE_CANDIDATES={ROE_CANDIDATES}, N_FINAL={N_FINAL}, MOM_DROP_BOTTOM_Q={MOM_DROP_BOTTOM_Q}")
    lines.append(f"Period: {port_daily['Date'].min().date()} to {port_daily['Date'].max().date()}, Days={len(port_daily)}")
    for k in ["total_return","CAGR","vol_ann","sharpe_ann","maxDD","calmar"]:
        lines.append(f"{k}: {stats[k]:.6f}")
    with open(OUT_SUMMARY, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

    print("Saved summary:", OUT_SUMMARY)
    print("Saved daily  :", OUT_DAILY)
    print("Saved monthly:", OUT_MONTHLY)

if __name__ == "__main__":
    main()


Saved summary: C:\Users\yongr\Project\merged_data_all_stocks\factors\backtest_top50_roe_momfilter_daily_topix_intramonth_off_sma200\summary.txt
Saved daily  : C:\Users\yongr\Project\merged_data_all_stocks\factors\backtest_top50_roe_momfilter_daily_topix_intramonth_off_sma200\daily_equity_curve.csv
Saved monthly: C:\Users\yongr\Project\merged_data_all_stocks\factors\backtest_top50_roe_momfilter_daily_topix_intramonth_off_sma200\monthly_results.csv


In [22]:
import os
import numpy as np
import pandas as pd

# ===== Paths =====
SNAPSHOT_PARQUET = r"C:\Users\yongr\Project\merged_data_all_stocks\factors\month_end_snapshot.parquet"
DAILY_PARTS = [
    r"C:\Users\yongr\Project\merged_data_all_stocks\merged_parts\merged-part-00001.parquet",
    r"C:\Users\yongr\Project\merged_data_all_stocks\merged_parts\merged-part-00002.parquet",
]
TOPIX_CSV = r"C:\Users\yongr\Project\OHLCV_Adjusted\topix_index.csv"

OUT_DIR = r"C:\Users\yongr\Project\merged_data_all_stocks\factors\q_sweep_roe_momfilter_topix_sma200_intramonth_off"
os.makedirs(OUT_DIR, exist_ok=True)

OUT_SUMMARY_CSV = os.path.join(OUT_DIR, "q_sweep_summary.csv")
OUT_BEST_TXT = os.path.join(OUT_DIR, "q_sweep_best.txt")

# ===== Strategy params =====
N_FINAL = 50
ROE_CANDIDATES = 200
MOM_LONG = 12
MOM_SKIP = 1

Q_LIST = [0.20, 0.30, 0.40]  # <- 今回のスイープ

TOPIX_SMA_WINDOW = 200
PER_YEAR = 252

def perf_stats_daily(r: pd.Series, periods_per_year=252):
    r = r.dropna().astype(float)
    n = len(r)
    eq = (1+r).cumprod()
    total_return = float(eq.iloc[-1] - 1)
    cagr = float(eq.iloc[-1] ** (periods_per_year/n) - 1)
    vol = float(r.std(ddof=1) * np.sqrt(periods_per_year))
    mean = float(r.mean() * periods_per_year)
    sharpe = np.nan if vol == 0 else float(mean/vol)
    dd = eq/eq.cummax() - 1
    mdd = float(dd.min())
    calmar = np.nan if mdd >= 0 else float(cagr/abs(mdd))
    return {"n_days": int(n), "total_return": total_return, "CAGR": cagr, "vol_ann": vol,
            "sharpe_ann": sharpe, "maxDD": mdd, "calmar": calmar}

def prepare_common():
    # TOPIX risk_on
    topix = pd.read_csv(TOPIX_CSV)
    topix["Date"] = pd.to_datetime(topix["Date"])
    topix = topix.sort_values("Date").drop_duplicates("Date")
    topix["topix_close"] = topix["Close"].astype(float)
    topix["topix_sma200"] = topix["topix_close"].rolling(TOPIX_SMA_WINDOW, min_periods=TOPIX_SMA_WINDOW).mean()
    topix["risk_on"] = np.where(
        (topix["topix_sma200"].notna()) & (topix["topix_close"] > topix["topix_sma200"]),
        1, 0
    ).astype(int)
    topix = topix[["Date", "risk_on"]].copy()

    # snapshot
    snap = pd.read_parquet(SNAPSHOT_PARQUET)
    snap["MonthEnd"] = pd.to_datetime(snap["MonthEnd"])
    snap["Code"] = snap["Code"].astype(str)
    snap = snap.dropna(subset=["ROE"])
    month_ends = sorted(snap["MonthEnd"].unique())

    # daily px
    daily_list = []
    for fp in DAILY_PARTS:
        d = pd.read_parquet(fp)[["Date", "Code", "AdjustedClose"]].copy()
        d["Date"] = pd.to_datetime(d["Date"])
        d["Code"] = d["Code"].astype(str)
        d = d.dropna(subset=["AdjustedClose"])
        daily_list.append(d)
    px = pd.concat(daily_list, ignore_index=True).sort_values(["Code","Date"]).reset_index(drop=True)
    px["ret_d"] = px.groupby("Code")["AdjustedClose"].pct_change()
    px["MonthEnd"] = (px["Date"] + pd.offsets.MonthEnd(0)).dt.normalize()

    # monthly last trading day px
    px_me = (
        px.sort_values(["Code", "Date"])
          .groupby(["Code", "MonthEnd"], as_index=False)
          .tail(1)[["Code", "MonthEnd", "AdjustedClose"]]
          .rename(columns={"AdjustedClose":"px_me"})
          .reset_index(drop=True)
    ).sort_values(["Code","MonthEnd"])
    px_me["px_lag1"]  = px_me.groupby("Code")["px_me"].shift(MOM_SKIP)
    px_me["px_lag12"] = px_me.groupby("Code")["px_me"].shift(MOM_LONG)
    px_me["mom12_1"]  = px_me["px_lag1"] / px_me["px_lag12"] - 1.0

    snap2 = snap.merge(px_me[["Code","MonthEnd","mom12_1"]], on=["Code","MonthEnd"], how="left")
    return topix, snap2, month_ends, px

def run_one_q(q, topix, snap2, month_ends, px):
    rows = []
    for me in month_ends:
        dfm = snap2[snap2["MonthEnd"] == me].copy()
        dfm = dfm.dropna(subset=["ROE","mom12_1"])
        if len(dfm) == 0:
            continue

        dfm = dfm.sort_values("ROE", ascending=False).head(ROE_CANDIDATES).copy()
        thr = dfm["mom12_1"].quantile(q)
        dfm_f = dfm[dfm["mom12_1"] > thr].copy()
        if len(dfm_f) < N_FINAL:
            dfm_f = dfm.copy()

        top_codes = dfm_f.sort_values("ROE", ascending=False)["Code"].head(N_FINAL).tolist()
        for c in top_codes:
            rows.append({"formation_monthend": me, "Code": c})

    holdings = pd.DataFrame(rows)
    if holdings.empty:
        return None

    holdings["formation_monthend"] = pd.to_datetime(holdings["formation_monthend"])
    holdings["holding_monthend"] = (holdings["formation_monthend"] + pd.offsets.MonthEnd(1)).dt.normalize()

    hp = holdings.merge(px, left_on=["Code","holding_monthend"], right_on=["Code","MonthEnd"], how="left")
    port_daily = (
        hp.groupby("Date")
          .agg(port_ret=("ret_d","mean"))
          .reset_index()
          .sort_values("Date")
          .reset_index(drop=True)
    )
    port_daily = port_daily.merge(topix, on="Date", how="left")
    port_daily["risk_on"] = port_daily["risk_on"].ffill().fillna(0).astype(int)
    port_daily["port_ret_rm"] = port_daily["port_ret"].fillna(0) * port_daily["risk_on"]

    stats = perf_stats_daily(port_daily["port_ret_rm"], periods_per_year=PER_YEAR)
    stats["q"] = q
    stats["start"] = str(port_daily["Date"].min().date())
    stats["end"] = str(port_daily["Date"].max().date())
    return stats

def main():
    topix, snap2, month_ends, px = prepare_common()

    out = []
    for q in Q_LIST:
        print("Running q =", q)
        s = run_one_q(q, topix, snap2, month_ends, px)
        if s is not None:
            out.append(s)

    df = pd.DataFrame(out).sort_values("q")
    df.to_csv(OUT_SUMMARY_CSV, index=False, encoding="utf-8-sig")
    print("Saved:", OUT_SUMMARY_CSV)

    # best under DD constraint
    df_ok = df[df["maxDD"] >= -0.20].copy()
    lines = []
    lines.append("=== All results ===")
    lines.append(df.to_string(index=False))
    lines.append("\n=== Under DD constraint (maxDD >= -0.20) ===")
    if len(df_ok) == 0:
        lines.append("NONE")
    else:
        lines.append("Best CAGR under constraint:")
        lines.append(df_ok.sort_values("CAGR", ascending=False).head(1).to_string(index=False))
        lines.append("\nBest Calmar under constraint:")
        lines.append(df_ok.sort_values("calmar", ascending=False).head(1).to_string(index=False))

    with open(OUT_BEST_TXT, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    print("Saved:", OUT_BEST_TXT)

if __name__ == "__main__":
    main()


Running q = 0.2
Running q = 0.3
Running q = 0.4
Saved: C:\Users\yongr\Project\merged_data_all_stocks\factors\q_sweep_roe_momfilter_topix_sma200_intramonth_off\q_sweep_summary.csv
Saved: C:\Users\yongr\Project\merged_data_all_stocks\factors\q_sweep_roe_momfilter_topix_sma200_intramonth_off\q_sweep_best.txt


In [23]:
import os
import numpy as np
import pandas as pd

# ===== Paths =====
SNAPSHOT_PARQUET = r"C:\Users\yongr\Project\merged_data_all_stocks\factors\month_end_snapshot.parquet"
DAILY_PARTS = [
    r"C:\Users\yongr\Project\merged_data_all_stocks\merged_parts\merged-part-00001.parquet",
    r"C:\Users\yongr\Project\merged_data_all_stocks\merged_parts\merged-part-00002.parquet",
]
TOPIX_CSV = r"C:\Users\yongr\Project\OHLCV_Adjusted\topix_index.csv"

OUT_DIR = r"C:\Users\yongr\Project\merged_data_all_stocks\factors\candidates_sweep_roe_momfilter_topix_sma200_intramonth_off"
os.makedirs(OUT_DIR, exist_ok=True)

OUT_SUMMARY_CSV = os.path.join(OUT_DIR, "candidates_sweep_summary.csv")
OUT_BEST_TXT = os.path.join(OUT_DIR, "candidates_sweep_best.txt")

# ===== Fixed params (確定) =====
N_FINAL = 50
MOM_LONG = 12
MOM_SKIP = 1
MOM_DROP_BOTTOM_Q = 0.20      # <- 固定（あなたの最良q）
CANDIDATE_LIST = [200, 300, 500]

TOPIX_SMA_WINDOW = 200
PER_YEAR = 252

def perf_stats_daily(r: pd.Series, periods_per_year=252):
    r = r.dropna().astype(float)
    n = len(r)
    eq = (1+r).cumprod()
    total_return = float(eq.iloc[-1] - 1)
    cagr = float(eq.iloc[-1] ** (periods_per_year/n) - 1)
    vol = float(r.std(ddof=1) * np.sqrt(periods_per_year))
    mean = float(r.mean() * periods_per_year)
    sharpe = np.nan if vol == 0 else float(mean/vol)
    dd = eq/eq.cummax() - 1
    mdd = float(dd.min())
    calmar = np.nan if mdd >= 0 else float(cagr/abs(mdd))
    return {"n_days": int(n), "total_return": total_return, "CAGR": cagr, "vol_ann": vol,
            "sharpe_ann": sharpe, "maxDD": mdd, "calmar": calmar}

def prepare_common():
    # TOPIX risk_on
    topix = pd.read_csv(TOPIX_CSV)
    topix["Date"] = pd.to_datetime(topix["Date"])
    topix = topix.sort_values("Date").drop_duplicates("Date")
    topix["topix_close"] = topix["Close"].astype(float)
    topix["topix_sma200"] = topix["topix_close"].rolling(TOPIX_SMA_WINDOW, min_periods=TOPIX_SMA_WINDOW).mean()
    topix["risk_on"] = np.where(
        (topix["topix_sma200"].notna()) & (topix["topix_close"] > topix["topix_sma200"]),
        1, 0
    ).astype(int)
    topix = topix[["Date", "risk_on"]].copy()

    # snapshot
    snap = pd.read_parquet(SNAPSHOT_PARQUET)
    snap["MonthEnd"] = pd.to_datetime(snap["MonthEnd"])
    snap["Code"] = snap["Code"].astype(str)
    snap = snap.dropna(subset=["ROE"])
    month_ends = sorted(snap["MonthEnd"].unique())

    # daily px
    daily_list = []
    for fp in DAILY_PARTS:
        d = pd.read_parquet(fp)[["Date", "Code", "AdjustedClose"]].copy()
        d["Date"] = pd.to_datetime(d["Date"])
        d["Code"] = d["Code"].astype(str)
        d = d.dropna(subset=["AdjustedClose"])
        daily_list.append(d)
    px = pd.concat(daily_list, ignore_index=True).sort_values(["Code","Date"]).reset_index(drop=True)
    px["ret_d"] = px.groupby("Code")["AdjustedClose"].pct_change()
    px["MonthEnd"] = (px["Date"] + pd.offsets.MonthEnd(0)).dt.normalize()

    # monthly last-trading-day px
    px_me = (
        px.sort_values(["Code", "Date"])
          .groupby(["Code", "MonthEnd"], as_index=False)
          .tail(1)[["Code", "MonthEnd", "AdjustedClose"]]
          .rename(columns={"AdjustedClose":"px_me"})
          .reset_index(drop=True)
    ).sort_values(["Code","MonthEnd"])
    px_me["px_lag1"]  = px_me.groupby("Code")["px_me"].shift(MOM_SKIP)
    px_me["px_lag12"] = px_me.groupby("Code")["px_me"].shift(MOM_LONG)
    px_me["mom12_1"]  = px_me["px_lag1"] / px_me["px_lag12"] - 1.0

    snap2 = snap.merge(px_me[["Code","MonthEnd","mom12_1"]], on=["Code","MonthEnd"], how="left")
    return topix, snap2, month_ends, px

def run_one_candidates(roe_candidates, topix, snap2, month_ends, px):
    rows = []
    for me in month_ends:
        dfm = snap2[snap2["MonthEnd"] == me].copy()
        dfm = dfm.dropna(subset=["ROE","mom12_1"])
        if len(dfm) == 0:
            continue

        # ROE候補
        dfm = dfm.sort_values("ROE", ascending=False).head(roe_candidates).copy()

        # MOM下位q除外（地雷除去）
        thr = dfm["mom12_1"].quantile(MOM_DROP_BOTTOM_Q)
        dfm_f = dfm[dfm["mom12_1"] > thr].copy()

        # 念のため不足時はフォールバック
        if len(dfm_f) < N_FINAL:
            dfm_f = dfm.copy()

        top_codes = dfm_f.sort_values("ROE", ascending=False)["Code"].head(N_FINAL).tolist()
        for c in top_codes:
            rows.append({"formation_monthend": me, "Code": c})

    holdings = pd.DataFrame(rows)
    if holdings.empty:
        return None

    holdings["formation_monthend"] = pd.to_datetime(holdings["formation_monthend"])
    holdings["holding_monthend"] = (holdings["formation_monthend"] + pd.offsets.MonthEnd(1)).dt.normalize()

    hp = holdings.merge(px, left_on=["Code","holding_monthend"], right_on=["Code","MonthEnd"], how="left")
    port_daily = (
        hp.groupby("Date")
          .agg(port_ret=("ret_d","mean"))
          .reset_index()
          .sort_values("Date")
          .reset_index(drop=True)
    )
    port_daily = port_daily.merge(topix, on="Date", how="left")
    port_daily["risk_on"] = port_daily["risk_on"].ffill().fillna(0).astype(int)
    port_daily["port_ret_rm"] = port_daily["port_ret"].fillna(0) * port_daily["risk_on"]

    stats = perf_stats_daily(port_daily["port_ret_rm"], periods_per_year=PER_YEAR)
    stats["ROE_CANDIDATES"] = roe_candidates
    stats["q"] = MOM_DROP_BOTTOM_Q
    stats["start"] = str(port_daily["Date"].min().date())
    stats["end"] = str(port_daily["Date"].max().date())
    return stats

def main():
    topix, snap2, month_ends, px = prepare_common()

    out = []
    for c in CANDIDATE_LIST:
        print("Running ROE_CANDIDATES =", c)
        s = run_one_candidates(c, topix, snap2, month_ends, px)
        if s is not None:
            out.append(s)

    df = pd.DataFrame(out).sort_values("ROE_CANDIDATES")
    df.to_csv(OUT_SUMMARY_CSV, index=False, encoding="utf-8-sig")
    print("Saved:", OUT_SUMMARY_CSV)

    df_ok = df[df["maxDD"] >= -0.20].copy()

    lines = []
    lines.append("=== All results ===")
    lines.append(df.to_string(index=False))
    lines.append("\n=== Under DD constraint (maxDD >= -0.20) ===")
    if len(df_ok) == 0:
        lines.append("NONE")
    else:
        lines.append("Best CAGR under constraint:")
        lines.append(df_ok.sort_values("CAGR", ascending=False).head(1).to_string(index=False))
        lines.append("\nBest Calmar under constraint:")
        lines.append(df_ok.sort_values("calmar", ascending=False).head(1).to_string(index=False))

    with open(OUT_BEST_TXT, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    print("Saved:", OUT_BEST_TXT)

if __name__ == "__main__":
    main()


Running ROE_CANDIDATES = 200
Running ROE_CANDIDATES = 300
Running ROE_CANDIDATES = 500
Saved: C:\Users\yongr\Project\merged_data_all_stocks\factors\candidates_sweep_roe_momfilter_topix_sma200_intramonth_off\candidates_sweep_summary.csv
Saved: C:\Users\yongr\Project\merged_data_all_stocks\factors\candidates_sweep_roe_momfilter_topix_sma200_intramonth_off\candidates_sweep_best.txt


In [24]:
import os
import numpy as np
import pandas as pd

# =====================
# User fixed settings
# =====================
INITIAL_CAPITAL = 10_000_000  # 1000万 JPY
TAX_RATE = 0.20315
COST_BPS_ONE_WAY = 0.0030     # 30bp = 0.30% one-way
LOT_SIZE = 100               # 100株単位

TARGET_VOL_LIST = [0.18, 0.20]   # 年率目標ボラ（18%, 20%）
MAX_LEVERAGE = 1.50              # レバ上限（保守的）

# Strategy (fixed from your best)
N_FINAL = 50
ROE_CANDIDATES = 300
MOM_DROP_BOTTOM_Q = 0.20
TOPIX_SMA_WINDOW = 200

# Vol estimation window for targeting (daily)
VOL_LOOKBACK_DAYS = 60           # 例: 60営業日（必要なら20/63/126に変更可）
TRADING_DAYS_PER_YEAR = 252

# =====================
# Paths
# =====================
SNAPSHOT_PARQUET = r"C:\Users\yongr\Project\merged_data_all_stocks\factors\month_end_snapshot.parquet"
DAILY_PARTS = [
    r"C:\Users\yongr\Project\merged_data_all_stocks\merged_parts\merged-part-00001.parquet",
    r"C:\Users\yongr\Project\merged_data_all_stocks\merged_parts\merged-part-00002.parquet",
]
TOPIX_CSV = r"C:\Users\yongr\Project\OHLCV_Adjusted\topix_index.csv"

OUT_DIR = r"C:\Users\yongr\Project\merged_data_all_stocks\factors\bt_vol_target_cost_tax_lots"
os.makedirs(OUT_DIR, exist_ok=True)

def compute_topix_risk_on():
    topix = pd.read_csv(TOPIX_CSV)
    topix["Date"] = pd.to_datetime(topix["Date"])
    topix = topix.sort_values("Date").drop_duplicates("Date")
    topix["close"] = topix["Close"].astype(float)
    topix["sma200"] = topix["close"].rolling(TOPIX_SMA_WINDOW, min_periods=TOPIX_SMA_WINDOW).mean()
    topix["risk_on"] = ((topix["sma200"].notna()) & (topix["close"] > topix["sma200"])).astype(int)
    return topix[["Date", "risk_on", "close", "sma200"]].copy()

def build_monthly_mom12_1(px_daily):
    # month bucket
    px_daily["MonthEnd"] = (px_daily["Date"] + pd.offsets.MonthEnd(0)).dt.normalize()

    # monthly last trading day price
    px_me = (
        px_daily.sort_values(["Code","Date"])
                .groupby(["Code","MonthEnd"], as_index=False)
                .tail(1)[["Code","MonthEnd","AdjustedClose"]]
                .rename(columns={"AdjustedClose":"px_me"})
                .reset_index(drop=True)
                .sort_values(["Code","MonthEnd"])
    )
    # MOM 12-1: px(t-1)/px(t-12)-1
    px_me["px_lag1"]  = px_me.groupby("Code")["px_me"].shift(1)
    px_me["px_lag12"] = px_me.groupby("Code")["px_me"].shift(12)
    px_me["mom12_1"]  = px_me["px_lag1"] / px_me["px_lag12"] - 1.0
    return px_me[["Code","MonthEnd","mom12_1"]]

def build_holdings(snapshot, mom_monthly):
    snap = snapshot.merge(mom_monthly, left_on=["Code","MonthEnd"], right_on=["Code","MonthEnd"], how="left")
    month_ends = sorted(snap["MonthEnd"].unique())

    rows = []
    for me in month_ends:
        dfm = snap[snap["MonthEnd"] == me].copy()
        dfm = dfm.dropna(subset=["ROE","mom12_1"])
        if len(dfm) == 0:
            continue

        dfm = dfm.sort_values("ROE", ascending=False).head(ROE_CANDIDATES).copy()
        thr = dfm["mom12_1"].quantile(MOM_DROP_BOTTOM_Q)
        dfm_f = dfm[dfm["mom12_1"] > thr].copy()
        if len(dfm_f) < N_FINAL:
            dfm_f = dfm.copy()

        top_codes = dfm_f.sort_values("ROE", ascending=False)["Code"].head(N_FINAL).tolist()
        for c in top_codes:
            rows.append({"formation_monthend": me, "Code": c})

    h = pd.DataFrame(rows)
    h["formation_monthend"] = pd.to_datetime(h["formation_monthend"])
    h["holding_monthend"] = (h["formation_monthend"] + pd.offsets.MonthEnd(1)).dt.normalize()
    return h

def perf_stats_daily(r, periods_per_year=252):
    r = pd.Series(r).dropna().astype(float)
    n = len(r)
    eq = (1+r).cumprod()
    total_return = float(eq.iloc[-1] - 1)
    cagr = float(eq.iloc[-1] ** (periods_per_year/n) - 1)
    vol = float(r.std(ddof=1) * np.sqrt(periods_per_year))
    mean = float(r.mean() * periods_per_year)
    sharpe = np.nan if vol == 0 else float(mean/vol)
    dd = eq/eq.cummax() - 1
    mdd = float(dd.min())
    calmar = np.nan if mdd >= 0 else float(cagr/abs(mdd))
    return {"n_days": n, "total_return": total_return, "CAGR": cagr, "vol_ann": vol,
            "sharpe_ann": sharpe, "maxDD": mdd, "calmar": calmar}

def main():
    # --- Load snapshot (monthly ROE) ---
    snapshot = pd.read_parquet(SNAPSHOT_PARQUET)
    snapshot["MonthEnd"] = pd.to_datetime(snapshot["MonthEnd"])
    snapshot["Code"] = snapshot["Code"].astype(str)
    snapshot = snapshot.dropna(subset=["ROE"])

    # --- Load daily prices (needed cols only) ---
    daily_list = []
    for fp in DAILY_PARTS:
        d = pd.read_parquet(fp)[["Date","Code","AdjustedClose"]].copy()
        d["Date"] = pd.to_datetime(d["Date"])
        d["Code"] = d["Code"].astype(str)
        d = d.dropna(subset=["AdjustedClose"])
        daily_list.append(d)

    px = pd.concat(daily_list, ignore_index=True).sort_values(["Code","Date"]).reset_index(drop=True)

    # daily return by stock
    px["ret_d"] = px.groupby("Code")["AdjustedClose"].pct_change()

    # TOPIX risk flag
    topix = compute_topix_risk_on()

    # monthly MOM + holdings
    mom_m = build_monthly_mom12_1(px[["Date","Code","AdjustedClose"]].copy())
    holdings = build_holdings(snapshot[["Code","MonthEnd","ROE"]].copy(), mom_m)

    # Map each daily date to holding_monthend bucket
    px["holding_monthend"] = (px["Date"] + pd.offsets.MonthEnd(0)).dt.normalize()

    # For fast access: for each holding_monthend, the universe codes
    uni_by_month = holdings.groupby("holding_monthend")["Code"].apply(list).to_dict()

    # Create a panel-like daily table: we will compute portfolio from daily prices
    dates = sorted(px["Date"].unique())

    # Pre-index prices by (Date, Code) for quick lookup (memory heavy but simplest)
    # If memory is an issue, we can optimize later.
    px_key = px.set_index(["Date","Code"]).sort_index()

    # We also need yesterday price to compute PnL. We'll use ret_d directly when available.
    # Use ret_d at (Date, Code) = percent change from prev trading day.

    # Prepare TOPIX risk_on by date (ffill)
    topix2 = topix.set_index("Date").sort_index()
    # Align to trading dates via reindex+ffill
    risk_on_series = topix2["risk_on"].reindex(pd.Index(dates)).ffill().fillna(0).astype(int)
    topix_close_series = topix2["close"].reindex(pd.Index(dates)).ffill()

    # For vol targeting, we compute rolling vol of portfolio returns (pre-cost, pre-tax) on the fly.
    # We'll store realized daily returns for each target vol run.

    results_all = []

    for target_vol in TARGET_VOL_LIST:
        cash = INITIAL_CAPITAL
        positions = {}  # code -> shares (int)
        cost_paid_total = 0.0
        tax_paid_total = 0.0

        daily_rows = []
        port_rets_for_vol = []  # realized returns used for vol estimate (after applying risk_on but before cost/tax is acceptable; we will use after-cost return for conservatism)

        prev_nav = INITIAL_CAPITAL

        for i, dt in enumerate(dates):
            me = (pd.Timestamp(dt) + pd.offsets.MonthEnd(0)).normalize()
            risk_on = int(risk_on_series.loc[dt])

            # Universe for this month
            universe = uni_by_month.get(me, [])
            if len(universe) == 0:
                # no universe -> stay in cash
                nav = cash
                daily_rows.append({"Date": dt, "NAV": nav, "risk_on": risk_on, "leverage": 0.0,
                                   "cost_paid": 0.0, "tax_paid": 0.0})
                port_rets_for_vol.append(0.0)
                prev_nav = nav
                continue

            # Compute current NAV from positions using today's prices
            nav_stock = 0.0
            for code, sh in positions.items():
                if sh == 0:
                    continue
                try:
                    price = float(px_key.loc[(dt, code), "AdjustedClose"])
                except KeyError:
                    # missing price -> ignore (should be rare); treat as unchanged
                    continue
                nav_stock += sh * price
            nav = cash + nav_stock

            # Estimate realized vol from recent returns (after-cost). If insufficient history, use 1.0 to avoid crazy leverage.
            if len(port_rets_for_vol) >= VOL_LOOKBACK_DAYS:
                window = np.array(port_rets_for_vol[-VOL_LOOKBACK_DAYS:], dtype=float)
                est_vol = np.std(window, ddof=1) * np.sqrt(TRADING_DAYS_PER_YEAR)
                if not np.isfinite(est_vol) or est_vol <= 0:
                    est_vol = 1.0
            else:
                est_vol = 1.0

            # Target leverage (0 when risk_off)
            lev = 0.0 if risk_on == 0 else min(MAX_LEVERAGE, max(0.0, target_vol / est_vol))

            # Desired dollar exposure
            target_exposure = nav * lev

            # Desired per-stock dollar (equal weight)
            target_each = target_exposure / N_FINAL

            # Build target shares (100-share lots)
            target_shares = {}
            for code in universe:
                try:
                    price = float(px_key.loc[(dt, code), "AdjustedClose"])
                except KeyError:
                    continue
                if price <= 0:
                    continue
                raw_sh = target_each / price
                lot_sh = int(raw_sh // LOT_SIZE) * LOT_SIZE
                target_shares[code] = lot_sh

            # Execute trades to reach target shares (cost + realized tax on profitable sells)
            cost_paid_today = 0.0
            tax_paid_today = 0.0

            # Sell stocks not in target or reduced shares
            # Use a naive average-cost approximation: assume cost basis = yesterday price (simplification)
            # If you want full lot-level cost basis, we can extend later.
            for code, cur_sh in list(positions.items()):
                tgt_sh = target_shares.get(code, 0)
                if tgt_sh < cur_sh:
                    sell_sh = cur_sh - tgt_sh
                    try:
                        price = float(px_key.loc[(dt, code), "AdjustedClose"])
                    except KeyError:
                        continue
                    proceeds = sell_sh * price
                    cash += proceeds

                    # cost
                    cost = proceeds * COST_BPS_ONE_WAY
                    cash -= cost
                    cost_paid_today += cost

                    # tax (simplified): tax only if today's price > yesterday price and we sold
                    # To be more correct, we'd need cost basis tracking. This is a conservative/approx placeholder.
                    if i > 0:
                        prev_dt = dates[i-1]
                        try:
                            prev_price = float(px_key.loc[(prev_dt, code), "AdjustedClose"])
                        except KeyError:
                            prev_price = price
                        pnl = sell_sh * max(0.0, price - prev_price)
                        tax = pnl * TAX_RATE
                        cash -= tax
                        tax_paid_today += tax

                    positions[code] = tgt_sh

            # Buy / increase positions
            for code, tgt_sh in target_shares.items():
                cur_sh = positions.get(code, 0)
                if tgt_sh > cur_sh:
                    buy_sh = tgt_sh - cur_sh
                    try:
                        price = float(px_key.loc[(dt, code), "AdjustedClose"])
                    except KeyError:
                        continue
                    cost_gross = buy_sh * price
                    # cost
                    fee = cost_gross * COST_BPS_ONE_WAY
                    total = cost_gross + fee
                    if total <= cash:
                        cash -= total
                        cost_paid_today += fee
                        positions[code] = tgt_sh
                    else:
                        # insufficient cash: skip (or partial fill). For simplicity, skip.
                        pass

            cost_paid_total += cost_paid_today
            tax_paid_total += tax_paid_today

            # Recompute NAV after trades using today's close
            nav_stock = 0.0
            for code, sh in positions.items():
                if sh == 0:
                    continue
                try:
                    price = float(px_key.loc[(dt, code), "AdjustedClose"])
                except KeyError:
                    continue
                nav_stock += sh * price
            nav = cash + nav_stock

            # realized daily return (after cost/tax via NAV change)
            port_ret = 0.0 if prev_nav == 0 else (nav / prev_nav - 1.0)
            port_rets_for_vol.append(port_ret)
            prev_nav = nav

            daily_rows.append({
                "Date": dt,
                "NAV": nav,
                "risk_on": risk_on,
                "leverage": lev,
                "target_vol": target_vol,
                "est_vol": est_vol,
                "cost_paid": cost_paid_today,
                "tax_paid": tax_paid_today,
                "topix_close": float(topix_close_series.loc[dt]) if pd.notna(topix_close_series.loc[dt]) else np.nan
            })

        daily_df = pd.DataFrame(daily_rows)
        daily_df["ret"] = daily_df["NAV"].pct_change().fillna(0.0)

        stats = perf_stats_daily(daily_df["ret"], periods_per_year=TRADING_DAYS_PER_YEAR)
        stats.update({
            "target_vol": target_vol,
            "max_leverage": MAX_LEVERAGE,
            "cost_bps_one_way": COST_BPS_ONE_WAY,
            "tax_rate": TAX_RATE,
            "lot_size": LOT_SIZE,
            "total_cost_paid": float(daily_df["cost_paid"].sum()),
            "total_tax_paid": float(daily_df["tax_paid"].sum()),
            "start": str(daily_df["Date"].min().date()),
            "end": str(daily_df["Date"].max().date())
        })
        results_all.append(stats)

        # save per-run
        tag = f"tv{int(target_vol*100)}"
        daily_df.to_csv(os.path.join(OUT_DIR, f"daily_{tag}.csv"), index=False, encoding="utf-8-sig")

    summary = pd.DataFrame(results_all).sort_values("target_vol")
    summary.to_csv(os.path.join(OUT_DIR, "summary_table.csv"), index=False, encoding="utf-8-sig")

    # also write a readable summary
    lines = []
    lines.append("Vol targeting backtest (with cost, tax, 100-share lots) + TOPIX SMA200 intramonth OFF")
    lines.append(f"Initial capital: {INITIAL_CAPITAL}")
    lines.append(f"Cost: {COST_BPS_ONE_WAY*100:.2f}% one-way")
    lines.append(f"Tax: {TAX_RATE*100:.3f}% on realized gains (simplified)")
    lines.append(f"Lot size: {LOT_SIZE}")
    lines.append(f"ROE_CANDIDATES={ROE_CANDIDATES}, MOM q={MOM_DROP_BOTTOM_Q}, N={N_FINAL}")
    lines.append("")
    lines.append(summary.to_string(index=False))

    with open(os.path.join(OUT_DIR, "summary.txt"), "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

    print("Saved:", OUT_DIR)

if __name__ == "__main__":
    main()


Saved: C:\Users\yongr\Project\merged_data_all_stocks\factors\bt_vol_target_cost_tax_lots


In [25]:
import pandas as pd

p = r"C:\Users\yongr\Project\merged_data_all_stocks\factors\bt_vol_target_cost_tax_lots\summary_table.csv"
df = pd.read_csv(p)
print(df.to_string(index=False))


 n_days  total_return      CAGR  vol_ann  sharpe_ann     maxDD    calmar  target_vol  max_leverage  cost_bps_one_way  tax_rate  lot_size  total_cost_paid  total_tax_paid      start        end
   2440     -0.768266 -0.140162 0.249877   -0.394374 -0.827379 -0.169404        0.18           1.5             0.003   0.20315       100       3734929.89    1948624.9575 2016-01-15 2026-01-09
   2440     -0.837287 -0.170996 0.279743   -0.391162 -0.880657 -0.194168        0.20           1.5             0.003   0.20315       100       3834125.43    2056174.5990 2016-01-15 2026-01-09


In [26]:
import os
import numpy as np
import pandas as pd

# =====================
# Fixed user settings
# =====================
COST_BPS_ONE_WAY = 0.0030   # 30bp
TAX_RATE = 0.20315

TARGET_VOL_LIST = [0.18, 0.20]  # 18%, 20%
MAX_LEVERAGE = 1.50

VOL_LOOKBACK_DAYS = 60
TRADING_DAYS_PER_YEAR = 252

# Strategy fixed (your best)
N_FINAL = 50
ROE_CANDIDATES = 300
MOM_DROP_BOTTOM_Q = 0.20
TOPIX_SMA_WINDOW = 200

# =====================
# Paths
# =====================
SNAPSHOT_PARQUET = r"C:\Users\yongr\Project\merged_data_all_stocks\factors\month_end_snapshot.parquet"
DAILY_PARTS = [
    r"C:\Users\yongr\Project\merged_data_all_stocks\merged_parts\merged-part-00001.parquet",
    r"C:\Users\yongr\Project\merged_data_all_stocks\merged_parts\merged-part-00002.parquet",
]
TOPIX_CSV = r"C:\Users\yongr\Project\OHLCV_Adjusted\topix_index.csv"

OUT_DIR = r"C:\Users\yongr\Project\merged_data_all_stocks\factors\bt_vol_target_A_cost_tax"
os.makedirs(OUT_DIR, exist_ok=True)

def perf_stats_daily(r, periods_per_year=252):
    r = pd.Series(r).dropna().astype(float)
    n = len(r)
    eq = (1+r).cumprod()
    total_return = float(eq.iloc[-1] - 1)
    cagr = float(eq.iloc[-1] ** (periods_per_year/n) - 1)
    vol = float(r.std(ddof=1) * np.sqrt(periods_per_year))
    mean = float(r.mean() * periods_per_year)
    sharpe = np.nan if vol == 0 else float(mean/vol)
    dd = eq/eq.cummax() - 1
    mdd = float(dd.min())
    calmar = np.nan if mdd >= 0 else float(cagr/abs(mdd))
    return {"n_days": int(n), "total_return": total_return, "CAGR": cagr, "vol_ann": vol,
            "sharpe_ann": sharpe, "maxDD": mdd, "calmar": calmar}

def compute_topix_risk_on():
    topix = pd.read_csv(TOPIX_CSV)
    topix["Date"] = pd.to_datetime(topix["Date"])
    topix = topix.sort_values("Date").drop_duplicates("Date")
    topix["close"] = topix["Close"].astype(float)
    topix["sma200"] = topix["close"].rolling(TOPIX_SMA_WINDOW, min_periods=TOPIX_SMA_WINDOW).mean()
    topix["risk_on"] = ((topix["sma200"].notna()) & (topix["close"] > topix["sma200"])).astype(int)
    return topix[["Date","risk_on"]].copy()

def prepare_daily_prices():
    daily_list = []
    for fp in DAILY_PARTS:
        d = pd.read_parquet(fp)[["Date","Code","AdjustedClose"]].copy()
        d["Date"] = pd.to_datetime(d["Date"])
        d["Code"] = d["Code"].astype(str)
        d = d.dropna(subset=["AdjustedClose"])
        daily_list.append(d)
    px = pd.concat(daily_list, ignore_index=True).sort_values(["Code","Date"]).reset_index(drop=True)
    px["ret_d"] = px.groupby("Code")["AdjustedClose"].pct_change()
    px["MonthEnd"] = (px["Date"] + pd.offsets.MonthEnd(0)).dt.normalize()
    return px

def build_monthly_mom12_1(px):
    px_me = (
        px.sort_values(["Code","Date"])
          .groupby(["Code","MonthEnd"], as_index=False)
          .tail(1)[["Code","MonthEnd","AdjustedClose"]]
          .rename(columns={"AdjustedClose":"px_me"})
          .reset_index(drop=True)
          .sort_values(["Code","MonthEnd"])
    )
    px_me["px_lag1"]  = px_me.groupby("Code")["px_me"].shift(1)
    px_me["px_lag12"] = px_me.groupby("Code")["px_me"].shift(12)
    px_me["mom12_1"]  = px_me["px_lag1"] / px_me["px_lag12"] - 1.0
    return px_me[["Code","MonthEnd","mom12_1"]]

def build_universe_by_month(snapshot, mom_m):
    snap = snapshot.merge(mom_m, on=["Code","MonthEnd"], how="left")
    month_ends = sorted(snap["MonthEnd"].unique())
    uni = {}
    for me in month_ends:
        dfm = snap[snap["MonthEnd"] == me].copy()
        dfm = dfm.dropna(subset=["ROE","mom12_1"])
        if len(dfm) == 0:
            continue
        dfm = dfm.sort_values("ROE", ascending=False).head(ROE_CANDIDATES).copy()
        thr = dfm["mom12_1"].quantile(MOM_DROP_BOTTOM_Q)
        dfm_f = dfm[dfm["mom12_1"] > thr].copy()
        if len(dfm_f) < N_FINAL:
            dfm_f = dfm.copy()
        codes = dfm_f.sort_values("ROE", ascending=False)["Code"].head(N_FINAL).tolist()
        # formation month me -> apply next month
        hold_me = (pd.Timestamp(me) + pd.offsets.MonthEnd(1)).normalize()
        uni[hold_me] = codes
    return uni

def make_basket_daily_return(px, universe_by_holding_monthend):
    # Create per-date basket return = mean of member returns of that month
    # We'll do it by merging month bucket and filtering membership.
    # Build a DataFrame with (holding_monthend, Code) membership
    rows = []
    for hm, codes in universe_by_holding_monthend.items():
        for c in codes:
            rows.append({"holding_monthend": hm, "Code": c})
    mem = pd.DataFrame(rows)
    if mem.empty:
        return pd.DataFrame(columns=["Date","basket_ret"])

    # attach holding_monthend to px rows
    px2 = px[["Date","Code","ret_d","MonthEnd"]].rename(columns={"MonthEnd":"holding_monthend"}).copy()
    # join membership (keeps only members)
    hp = mem.merge(px2, on=["holding_monthend","Code"], how="left")

    basket = (hp.groupby("Date")["ret_d"].mean().reset_index().rename(columns={"ret_d":"basket_ret"}))
    basket = basket.sort_values("Date").reset_index(drop=True)
    return basket

def main():
    # Snapshot (ROE)
    snap = pd.read_parquet(SNAPSHOT_PARQUET)
    snap["MonthEnd"] = pd.to_datetime(snap["MonthEnd"])
    snap["Code"] = snap["Code"].astype(str)
    snap = snap.dropna(subset=["ROE"])[["Code","MonthEnd","ROE"]].copy()

    # Daily prices
    px = prepare_daily_prices()

    # Build universe by month
    mom_m = build_monthly_mom12_1(px[["Date","Code","AdjustedClose","MonthEnd"]].copy())
    uni = build_universe_by_month(snap, mom_m)

    # Basket daily return
    basket = make_basket_daily_return(px, uni)

    # TOPIX risk
    topix = compute_topix_risk_on()
    # align
    df = basket.merge(topix, on="Date", how="left").sort_values("Date").reset_index(drop=True)
    df["risk_on"] = df["risk_on"].ffill().fillna(0).astype(int)
    df["basket_ret"] = df["basket_ret"].fillna(0.0)

    # Run vol targeting for each target vol
    summary_rows = []
    for tv in TARGET_VOL_LIST:
        lev_prev = 0.0
        rets_net = []
        rets_after_cost = []
        lev_list = []
        cost_list = []
        tax_list = []

        for i in range(len(df)):
            r_b = float(df.loc[i, "basket_ret"])
            risk_on = int(df.loc[i, "risk_on"])

            # use AFTER-COST returns for vol estimate (conservative)
            if len(rets_after_cost) >= VOL_LOOKBACK_DAYS:
                window = np.array(rets_after_cost[-VOL_LOOKBACK_DAYS:], dtype=float)
                est_vol = np.std(window, ddof=1) * np.sqrt(TRADING_DAYS_PER_YEAR)
                if not np.isfinite(est_vol) or est_vol <= 0:
                    est_vol = 1.0
            else:
                est_vol = 1.0

            lev = 0.0 if risk_on == 0 else min(MAX_LEVERAGE, max(0.0, tv / est_vol))

            r_pre_cost = lev * r_b * risk_on

            # cost on leverage change (one-way bps)
            cost = COST_BPS_ONE_WAY * abs(lev - lev_prev)
            r_after_cost = r_pre_cost - cost

            # simple tax on positive daily return
            tax = TAX_RATE * max(r_after_cost, 0.0)
            r_net = r_after_cost - tax

            rets_after_cost.append(r_after_cost)
            rets_net.append(r_net)
            lev_list.append(lev)
            cost_list.append(cost)
            tax_list.append(tax)

            lev_prev = lev

        stats = perf_stats_daily(rets_net, periods_per_year=TRADING_DAYS_PER_YEAR)
        stats.update({
            "target_vol": tv,
            "max_leverage": MAX_LEVERAGE,
            "cost_bps_one_way": COST_BPS_ONE_WAY,
            "tax_rate": TAX_RATE,
            "lookback_days": VOL_LOOKBACK_DAYS,
            "avg_leverage": float(np.mean(lev_list)),
            "avg_cost_per_day": float(np.mean(cost_list)),
            "avg_tax_per_day": float(np.mean(tax_list)),
            "start": str(df["Date"].min().date()),
            "end": str(df["Date"].max().date())
        })
        summary_rows.append(stats)

        # save per-run daily diagnostics
        out = df[["Date","basket_ret","risk_on"]].copy()
        out["leverage"] = lev_list
        out["cost"] = cost_list
        out["tax"] = tax_list
        out["ret_net"] = rets_net
        out["equity"] = (1 + out["ret_net"]).cumprod()
        out.to_csv(os.path.join(OUT_DIR, f"daily_tv{int(tv*100)}.csv"), index=False, encoding="utf-8-sig")

    summary = pd.DataFrame(summary_rows).sort_values("target_vol")
    summary.to_csv(os.path.join(OUT_DIR, "summary_table.csv"), index=False, encoding="utf-8-sig")

    with open(os.path.join(OUT_DIR, "summary.txt"), "w", encoding="utf-8") as f:
        f.write(summary.to_string(index=False))

    print("Saved:", OUT_DIR)

if __name__ == "__main__":
    main()


Saved: C:\Users\yongr\Project\merged_data_all_stocks\factors\bt_vol_target_A_cost_tax


In [27]:
import pandas as pd

DIR = r"C:\Users\yongr\Project\merged_data_all_stocks\factors\bt_vol_target_A_cost_tax"
p = rf"{DIR}\summary_table.csv"

df = pd.read_csv(p)
print(df.to_string(index=False))


 n_days  total_return      CAGR  vol_ann  sharpe_ann     maxDD    calmar  target_vol  max_leverage  cost_bps_one_way  tax_rate  lookback_days  avg_leverage  avg_cost_per_day  avg_tax_per_day      start        end
   2139     -0.389558 -0.056490 0.139214   -0.347587 -0.571147 -0.098907        0.18           1.5             0.003   0.20315             60      0.756054          0.000180         0.000678 2017-02-01 2025-10-31
   2139     -0.412856 -0.060806 0.146002   -0.356105 -0.590500 -0.102974        0.20           1.5             0.003   0.20315             60      0.792808          0.000187         0.000711 2017-02-01 2025-10-31


In [28]:
import pandas as pd
import numpy as np

DIR = r"C:\Users\yongr\Project\merged_data_all_stocks\factors\bt_vol_target_A_cost_tax"
p = rf"{DIR}\daily_tv18.csv"  # tv20も同様に見てOK

df = pd.read_csv(p)
df["Date"] = pd.to_datetime(df["Date"])

print("shape:", df.shape)
print("columns:", list(df.columns))

# 基本統計
for c in ["basket_ret","risk_on","leverage","cost","tax","ret_net","equity"]:
    if c in df.columns:
        print(f"\n[{c}] describe:")
        print(df[c].describe())

# 極端な日（ret_netのワースト）
if "ret_net" in df.columns:
    cols = [c for c in ["Date","basket_ret","risk_on","leverage","cost","tax","ret_net","equity"] if c in df.columns]
    print("\nWorst 20 days by ret_net:")
    print(df.sort_values("ret_net").head(20)[cols].to_string(index=False))

    print("\nBest 20 days by ret_net:")
    print(df.sort_values("ret_net", ascending=False).head(20)[cols].to_string(index=False))


shape: (2139, 8)
columns: ['Date', 'basket_ret', 'risk_on', 'leverage', 'cost', 'tax', 'ret_net', 'equity']

[basket_ret] describe:
count    2139.000000
mean        0.000405
std         0.014554
min        -0.143638
25%        -0.006487
50%         0.001149
75%         0.008210
max         0.104755
Name: basket_ret, dtype: float64

[risk_on] describe:
count    2139.000000
mean        0.698925
std         0.458833
min         0.000000
25%         0.000000
50%         1.000000
75%         1.000000
max         1.000000
Name: risk_on, dtype: float64

[leverage] describe:
count    2139.000000
mean        0.756054
std         0.548712
min         0.000000
25%         0.000000
50%         0.978332
75%         1.144117
max         1.500000
Name: leverage, dtype: float64

[cost] describe:
count    2139.000000
mean        0.000180
std         0.000744
min         0.000000
25%         0.000000
50%         0.000003
75%         0.000034
max         0.004500
Name: cost, dtype: float64

[tax] describ

In [29]:
import os
import numpy as np
import pandas as pd

# =====================
# Fixed user settings
# =====================
COST_BPS_ONE_WAY = 0.0030   # 片道30bp = 0.30%
TAX_RATE = 0.20315         # 定義は残すが、USE_TAX=Falseで使わない
USE_TAX = False            # ★今回の修正ポイント：税を完全に無効化

TARGET_VOL_LIST = [0.18, 0.20]  # 18%, 20%
MAX_LEVERAGE = 1.50

VOL_LOOKBACK_DAYS = 60
TRADING_DAYS_PER_YEAR = 252

# Strategy fixed (your best)
N_FINAL = 50
ROE_CANDIDATES = 300
MOM_DROP_BOTTOM_Q = 0.20
TOPIX_SMA_WINDOW = 200

# =====================
# Paths
# =====================
SNAPSHOT_PARQUET = r"C:\Users\yongr\Project\merged_data_all_stocks\factors\month_end_snapshot.parquet"
DAILY_PARTS = [
    r"C:\Users\yongr\Project\merged_data_all_stocks\merged_parts\merged-part-00001.parquet",
    r"C:\Users\yongr\Project\merged_data_all_stocks\merged_parts\merged-part-00002.parquet",
]
TOPIX_CSV = r"C:\Users\yongr\Project\OHLCV_Adjusted\topix_index.csv"

# ★税なし版の出力ディレクトリ
OUT_DIR = r"C:\Users\yongr\Project\merged_data_all_stocks\factors\bt_vol_target_A_cost_NOTAX"
os.makedirs(OUT_DIR, exist_ok=True)

def perf_stats_daily(r, periods_per_year=252):
    r = pd.Series(r).dropna().astype(float)
    n = len(r)
    eq = (1 + r).cumprod()
    total_return = float(eq.iloc[-1] - 1)
    cagr = float(eq.iloc[-1] ** (periods_per_year / n) - 1)
    vol = float(r.std(ddof=1) * np.sqrt(periods_per_year))
    mean = float(r.mean() * periods_per_year)
    sharpe = np.nan if vol == 0 else float(mean / vol)
    dd = eq / eq.cummax() - 1
    mdd = float(dd.min())
    calmar = np.nan if mdd >= 0 else float(cagr / abs(mdd))
    return {
        "n_days": int(n),
        "total_return": total_return,
        "CAGR": cagr,
        "vol_ann": vol,
        "sharpe_ann": sharpe,
        "maxDD": mdd,
        "calmar": calmar
    }

def compute_topix_risk_on():
    topix = pd.read_csv(TOPIX_CSV)
    topix["Date"] = pd.to_datetime(topix["Date"])
    topix = topix.sort_values("Date").drop_duplicates("Date")
    topix["close"] = topix["Close"].astype(float)
    topix["sma200"] = topix["close"].rolling(TOPIX_SMA_WINDOW, min_periods=TOPIX_SMA_WINDOW).mean()
    topix["risk_on"] = ((topix["sma200"].notna()) & (topix["close"] > topix["sma200"])).astype(int)
    return topix[["Date", "risk_on"]].copy()

def prepare_daily_prices():
    daily_list = []
    for fp in DAILY_PARTS:
        d = pd.read_parquet(fp)[["Date", "Code", "AdjustedClose"]].copy()
        d["Date"] = pd.to_datetime(d["Date"])
        d["Code"] = d["Code"].astype(str)
        d = d.dropna(subset=["AdjustedClose"])
        daily_list.append(d)

    px = pd.concat(daily_list, ignore_index=True).sort_values(["Code", "Date"]).reset_index(drop=True)
    px["ret_d"] = px.groupby("Code")["AdjustedClose"].pct_change()
    px["MonthEnd"] = (px["Date"] + pd.offsets.MonthEnd(0)).dt.normalize()
    return px

def build_monthly_mom12_1(px):
    px_me = (
        px.sort_values(["Code", "Date"])
          .groupby(["Code", "MonthEnd"], as_index=False)
          .tail(1)[["Code", "MonthEnd", "AdjustedClose"]]
          .rename(columns={"AdjustedClose": "px_me"})
          .reset_index(drop=True)
          .sort_values(["Code", "MonthEnd"])
    )
    px_me["px_lag1"] = px_me.groupby("Code")["px_me"].shift(1)
    px_me["px_lag12"] = px_me.groupby("Code")["px_me"].shift(12)
    px_me["mom12_1"] = px_me["px_lag1"] / px_me["px_lag12"] - 1.0
    return px_me[["Code", "MonthEnd", "mom12_1"]]

def build_universe_by_month(snapshot, mom_m):
    snap = snapshot.merge(mom_m, on=["Code", "MonthEnd"], how="left")
    month_ends = sorted(snap["MonthEnd"].unique())
    uni = {}

    for me in month_ends:
        dfm = snap[snap["MonthEnd"] == me].copy()
        dfm = dfm.dropna(subset=["ROE", "mom12_1"])
        if len(dfm) == 0:
            continue

        # ROE候補を上位から
        dfm = dfm.sort_values("ROE", ascending=False).head(ROE_CANDIDATES).copy()

        # MOM下位q除外
        thr = dfm["mom12_1"].quantile(MOM_DROP_BOTTOM_Q)
        dfm_f = dfm[dfm["mom12_1"] > thr].copy()

        # 念のため不足ならフォールバック
        if len(dfm_f) < N_FINAL:
            dfm_f = dfm.copy()

        codes = dfm_f.sort_values("ROE", ascending=False)["Code"].head(N_FINAL).tolist()

        # formation me -> apply next month
        hold_me = (pd.Timestamp(me) + pd.offsets.MonthEnd(1)).normalize()
        uni[hold_me] = codes

    return uni

def make_basket_daily_return(px, universe_by_holding_monthend):
    rows = []
    for hm, codes in universe_by_holding_monthend.items():
        for c in codes:
            rows.append({"holding_monthend": hm, "Code": c})
    mem = pd.DataFrame(rows)
    if mem.empty:
        return pd.DataFrame(columns=["Date", "basket_ret"])

    px2 = px[["Date", "Code", "ret_d", "MonthEnd"]].rename(columns={"MonthEnd": "holding_monthend"}).copy()
    hp = mem.merge(px2, on=["holding_monthend", "Code"], how="left")

    basket = hp.groupby("Date")["ret_d"].mean().reset_index().rename(columns={"ret_d": "basket_ret"})
    basket = basket.sort_values("Date").reset_index(drop=True)
    return basket

def main():
    # Snapshot (ROE)
    snap = pd.read_parquet(SNAPSHOT_PARQUET)
    snap["MonthEnd"] = pd.to_datetime(snap["MonthEnd"])
    snap["Code"] = snap["Code"].astype(str)
    snap = snap.dropna(subset=["ROE"])[["Code", "MonthEnd", "ROE"]].copy()

    # Daily prices
    px = prepare_daily_prices()

    # Universe by month (ROE candidates + MOM filter)
    mom_m = build_monthly_mom12_1(px[["Date", "Code", "AdjustedClose", "MonthEnd"]].copy())
    uni = build_universe_by_month(snap, mom_m)

    # Basket daily return
    basket = make_basket_daily_return(px, uni)

    # TOPIX risk
    topix = compute_topix_risk_on()

    # Align
    df = basket.merge(topix, on="Date", how="left").sort_values("Date").reset_index(drop=True)
    df["risk_on"] = df["risk_on"].ffill().fillna(0).astype(int)
    df["basket_ret"] = df["basket_ret"].fillna(0.0)

    summary_rows = []

    for tv in TARGET_VOL_LIST:
        lev_prev = 0.0
        rets_net = []
        rets_after_cost = []
        lev_list = []
        cost_list = []
        tax_list = []

        for i in range(len(df)):
            r_b = float(df.loc[i, "basket_ret"])
            risk_on = int(df.loc[i, "risk_on"])

            # vol estimate (use after-cost series for conservatism)
            if len(rets_after_cost) >= VOL_LOOKBACK_DAYS:
                window = np.array(rets_after_cost[-VOL_LOOKBACK_DAYS:], dtype=float)
                est_vol = np.std(window, ddof=1) * np.sqrt(TRADING_DAYS_PER_YEAR)
                if not np.isfinite(est_vol) or est_vol <= 0:
                    est_vol = 1.0
            else:
                est_vol = 1.0

            lev = 0.0 if risk_on == 0 else min(MAX_LEVERAGE, max(0.0, tv / est_vol))

            # pre-cost return
            r_pre_cost = lev * r_b * risk_on

            # cost on leverage change (one-way bps)
            cost = COST_BPS_ONE_WAY * abs(lev - lev_prev)
            r_after_cost = r_pre_cost - cost

            # ★税なし
            tax = 0.0
            r_net = r_after_cost

            rets_after_cost.append(r_after_cost)
            rets_net.append(r_net)
            lev_list.append(lev)
            cost_list.append(cost)
            tax_list.append(tax)

            lev_prev = lev

        stats = perf_stats_daily(rets_net, periods_per_year=TRADING_DAYS_PER_YEAR)
        stats.update({
            "target_vol": tv,
            "max_leverage": MAX_LEVERAGE,
            "cost_bps_one_way": COST_BPS_ONE_WAY,
            "tax_rate": TAX_RATE,
            "use_tax": USE_TAX,
            "lookback_days": VOL_LOOKBACK_DAYS,
            "avg_leverage": float(np.mean(lev_list)),
            "avg_cost_per_day": float(np.mean(cost_list)),
            "avg_tax_per_day": float(np.mean(tax_list)),
            "start": str(df["Date"].min().date()),
            "end": str(df["Date"].max().date()),
        })
        summary_rows.append(stats)

        # save daily diagnostics
        out = df[["Date", "basket_ret", "risk_on"]].copy()
        out["leverage"] = lev_list
        out["cost"] = cost_list
        out["tax"] = tax_list
        out["ret_net"] = rets_net
        out["equity"] = (1 + out["ret_net"]).cumprod()
        out.to_csv(os.path.join(OUT_DIR, f"daily_tv{int(tv*100)}.csv"), index=False, encoding="utf-8-sig")

    summary = pd.DataFrame(summary_rows).sort_values("target_vol")
    summary.to_csv(os.path.join(OUT_DIR, "summary_table.csv"), index=False, encoding="utf-8-sig")

    with open(os.path.join(OUT_DIR, "summary.txt"), "w", encoding="utf-8") as f:
        f.write(summary.to_string(index=False))

    print("Saved:", OUT_DIR)

if __name__ == "__main__":
    main()


Saved: C:\Users\yongr\Project\merged_data_all_stocks\factors\bt_vol_target_A_cost_NOTAX


In [30]:
import pandas as pd
p = r"C:\Users\yongr\Project\merged_data_all_stocks\factors\bt_vol_target_A_cost_NOTAX\summary_table.csv"
df = pd.read_csv(p)
print(df[["target_vol","CAGR","maxDD","vol_ann","sharpe_ann","calmar","avg_leverage","avg_cost_per_day","start","end"]].to_string(index=False))


 target_vol     CAGR     maxDD  vol_ann  sharpe_ann   calmar  avg_leverage  avg_cost_per_day      start        end
       0.18 0.116945 -0.169433 0.153734    0.796645 0.690215      0.756054          0.000180 2017-02-01 2025-10-31
       0.20 0.120948 -0.175253 0.161146    0.789509 0.690134      0.792808          0.000187 2017-02-01 2025-10-31


In [31]:
import os
import numpy as np
import pandas as pd

# =====================
# Fixed user settings
# =====================
COST_BPS_ONE_WAY = 0.0030   # 片道30bp = 0.30%
TAX_RATE = 0.20315         # 定義は残す（今回 USE_TAX=False）
USE_TAX = False            # 税ゼロで評価（切り分け）

TARGET_VOL_LIST = [0.18, 0.20]  # 18%, 20%
MAX_LEVERAGE = 1.50

VOL_LOOKBACK_DAYS = 60
TRADING_DAYS_PER_YEAR = 252

# Strategy fixed (your best)
N_FINAL = 50
ROE_CANDIDATES = 300
MOM_DROP_BOTTOM_Q = 0.20
TOPIX_SMA_WINDOW = 200

# =====================
# Paths
# =====================
SNAPSHOT_PARQUET = r"C:\Users\yongr\Project\merged_data_all_stocks\factors\month_end_snapshot.parquet"
DAILY_PARTS = [
    r"C:\Users\yongr\Project\merged_data_all_stocks\merged_parts\merged-part-00001.parquet",
    r"C:\Users\yongr\Project\merged_data_all_stocks\merged_parts\merged-part-00002.parquet",
]
TOPIX_CSV = r"C:\Users\yongr\Project\OHLCV_Adjusted\topix_index.csv"

OUT_DIR = r"C:\Users\yongr\Project\merged_data_all_stocks\factors\bt_vol_target_Aprime_cost_NOTAX"
os.makedirs(OUT_DIR, exist_ok=True)

def perf_stats_daily(r, periods_per_year=252):
    r = pd.Series(r).dropna().astype(float)
    n = len(r)
    eq = (1 + r).cumprod()
    total_return = float(eq.iloc[-1] - 1)
    cagr = float(eq.iloc[-1] ** (periods_per_year / n) - 1)
    vol = float(r.std(ddof=1) * np.sqrt(periods_per_year))
    mean = float(r.mean() * periods_per_year)
    sharpe = np.nan if vol == 0 else float(mean / vol)
    dd = eq / eq.cummax() - 1
    mdd = float(dd.min())
    calmar = np.nan if mdd >= 0 else float(cagr / abs(mdd))
    return {
        "n_days": int(n),
        "total_return": total_return,
        "CAGR": cagr,
        "vol_ann": vol,
        "sharpe_ann": sharpe,
        "maxDD": mdd,
        "calmar": calmar
    }

def compute_topix_risk_on():
    topix = pd.read_csv(TOPIX_CSV)
    topix["Date"] = pd.to_datetime(topix["Date"])
    topix = topix.sort_values("Date").drop_duplicates("Date")
    topix["close"] = topix["Close"].astype(float)
    topix["sma200"] = topix["close"].rolling(TOPIX_SMA_WINDOW, min_periods=TOPIX_SMA_WINDOW).mean()
    topix["risk_on"] = ((topix["sma200"].notna()) & (topix["close"] > topix["sma200"])).astype(int)
    return topix[["Date", "risk_on"]].copy()

def prepare_daily_prices():
    daily_list = []
    for fp in DAILY_PARTS:
        d = pd.read_parquet(fp)[["Date", "Code", "AdjustedClose"]].copy()
        d["Date"] = pd.to_datetime(d["Date"])
        d["Code"] = d["Code"].astype(str)
        d = d.dropna(subset=["AdjustedClose"])
        daily_list.append(d)

    px = pd.concat(daily_list, ignore_index=True).sort_values(["Code", "Date"]).reset_index(drop=True)
    px["ret_d"] = px.groupby("Code")["AdjustedClose"].pct_change()
    px["MonthEnd"] = (px["Date"] + pd.offsets.MonthEnd(0)).dt.normalize()
    return px

def build_monthly_mom12_1(px):
    px_me = (
        px.sort_values(["Code", "Date"])
          .groupby(["Code", "MonthEnd"], as_index=False)
          .tail(1)[["Code", "MonthEnd", "AdjustedClose"]]
          .rename(columns={"AdjustedClose": "px_me"})
          .reset_index(drop=True)
          .sort_values(["Code", "MonthEnd"])
    )
    px_me["px_lag1"] = px_me.groupby("Code")["px_me"].shift(1)
    px_me["px_lag12"] = px_me.groupby("Code")["px_me"].shift(12)
    px_me["mom12_1"] = px_me["px_lag1"] / px_me["px_lag12"] - 1.0
    return px_me[["Code", "MonthEnd", "mom12_1"]]

def build_universe_by_month(snapshot, mom_m):
    snap = snapshot.merge(mom_m, on=["Code", "MonthEnd"], how="left")
    month_ends = sorted(snap["MonthEnd"].unique())
    uni = {}

    for me in month_ends:
        dfm = snap[snap["MonthEnd"] == me].copy()
        dfm = dfm.dropna(subset=["ROE", "mom12_1"])
        if len(dfm) == 0:
            continue

        dfm = dfm.sort_values("ROE", ascending=False).head(ROE_CANDIDATES).copy()
        thr = dfm["mom12_1"].quantile(MOM_DROP_BOTTOM_Q)
        dfm_f = dfm[dfm["mom12_1"] > thr].copy()
        if len(dfm_f) < N_FINAL:
            dfm_f = dfm.copy()

        codes = dfm_f.sort_values("ROE", ascending=False)["Code"].head(N_FINAL).tolist()

        hold_me = (pd.Timestamp(me) + pd.offsets.MonthEnd(1)).normalize()
        uni[hold_me] = codes

    return uni

def make_basket_daily_return(px, universe_by_holding_monthend):
    rows = []
    for hm, codes in universe_by_holding_monthend.items():
        for c in codes:
            rows.append({"holding_monthend": hm, "Code": c})
    mem = pd.DataFrame(rows)
    if mem.empty:
        return pd.DataFrame(columns=["Date", "basket_ret"])

    px2 = px[["Date", "Code", "ret_d", "MonthEnd"]].rename(columns={"MonthEnd": "holding_monthend"}).copy()
    hp = mem.merge(px2, on=["holding_monthend", "Code"], how="left")

    basket = hp.groupby("Date")["ret_d"].mean().reset_index().rename(columns={"ret_d": "basket_ret"})
    basket = basket.sort_values("Date").reset_index(drop=True)
    return basket

def main():
    # Snapshot (ROE)
    snap = pd.read_parquet(SNAPSHOT_PARQUET)
    snap["MonthEnd"] = pd.to_datetime(snap["MonthEnd"])
    snap["Code"] = snap["Code"].astype(str)
    snap = snap.dropna(subset=["ROE"])[["Code", "MonthEnd", "ROE"]].copy()

    # Daily prices
    px = prepare_daily_prices()

    # Universe by month (ROE candidates + MOM filter)
    mom_m = build_monthly_mom12_1(px[["Date", "Code", "AdjustedClose", "MonthEnd"]].copy())
    uni = build_universe_by_month(snap, mom_m)

    # Basket daily return
    basket = make_basket_daily_return(px, uni)

    # TOPIX risk
    topix = compute_topix_risk_on()

    # Align
    df = basket.merge(topix, on="Date", how="left").sort_values("Date").reset_index(drop=True)
    df["risk_on"] = df["risk_on"].ffill().fillna(0).astype(int)
    df["basket_ret"] = df["basket_ret"].fillna(0.0)

    # ★推定ボラ用：レバ無しの“リスク適用後”リターン
    # これで自己参照（レバ後で推定）を排除する
    df["ret_for_vol"] = df["basket_ret"] * df["risk_on"]

    summary_rows = []

    for tv in TARGET_VOL_LIST:
        lev_prev = 0.0
        rets_net = []
        lev_list = []
        cost_list = []
        tax_list = []
        est_vol_list = []

        for i in range(len(df)):
            r_b = float(df.loc[i, "basket_ret"])
            risk_on = int(df.loc[i, "risk_on"])

            # ★推定ボラは ret_for_vol（レバ無し）で計算
            if i >= VOL_LOOKBACK_DAYS:
                window = df.loc[i - VOL_LOOKBACK_DAYS:i - 1, "ret_for_vol"].to_numpy(dtype=float)
                # 0が多い（risk_off）場合もあるので、ddof=1を維持しつつ、0/NaN安全化
                if np.sum(np.isfinite(window)) >= 2:
                    est_vol = np.nanstd(window, ddof=1) * np.sqrt(TRADING_DAYS_PER_YEAR)
                else:
                    est_vol = np.nan
                if (not np.isfinite(est_vol)) or est_vol <= 0:
                    est_vol = 1.0
            else:
                est_vol = 1.0

            lev = 0.0 if risk_on == 0 else min(MAX_LEVERAGE, max(0.0, tv / est_vol))

            # pre-cost return
            r_pre_cost = lev * r_b * risk_on

            # cost on leverage change (one-way bps)
            cost = COST_BPS_ONE_WAY * abs(lev - lev_prev)
            r_after_cost = r_pre_cost - cost

            # tax disabled
            tax = 0.0
            r_net = r_after_cost

            rets_net.append(r_net)
            lev_list.append(lev)
            cost_list.append(cost)
            tax_list.append(tax)
            est_vol_list.append(est_vol)

            lev_prev = lev

        stats = perf_stats_daily(rets_net, periods_per_year=TRADING_DAYS_PER_YEAR)
        stats.update({
            "target_vol": tv,
            "max_leverage": MAX_LEVERAGE,
            "cost_bps_one_way": COST_BPS_ONE_WAY,
            "tax_rate": TAX_RATE,
            "use_tax": USE_TAX,
            "lookback_days": VOL_LOOKBACK_DAYS,
            "avg_leverage": float(np.mean(lev_list)),
            "avg_cost_per_day": float(np.mean(cost_list)),
            "avg_tax_per_day": float(np.mean(tax_list)),
            "avg_est_vol": float(np.mean(est_vol_list)),
            "start": str(df["Date"].min().date()),
            "end": str(df["Date"].max().date()),
        })
        summary_rows.append(stats)

        # save daily diagnostics
        out = df[["Date", "basket_ret", "risk_on", "ret_for_vol"]].copy()
        out["leverage"] = lev_list
        out["est_vol"] = est_vol_list
        out["cost"] = cost_list
        out["tax"] = tax_list
        out["ret_net"] = rets_net
        out["equity"] = (1 + out["ret_net"]).cumprod()
        out.to_csv(os.path.join(OUT_DIR, f"daily_tv{int(tv*100)}.csv"), index=False, encoding="utf-8-sig")

    summary = pd.DataFrame(summary_rows).sort_values("target_vol")
    summary.to_csv(os.path.join(OUT_DIR, "summary_table.csv"), index=False, encoding="utf-8-sig")

    with open(os.path.join(OUT_DIR, "summary.txt"), "w", encoding="utf-8") as f:
        f.write(summary.to_string(index=False))

    print("Saved:", OUT_DIR)

if __name__ == "__main__":
    main()


Saved: C:\Users\yongr\Project\merged_data_all_stocks\factors\bt_vol_target_Aprime_cost_NOTAX


In [32]:
import pandas as pd

p = r"C:\Users\yongr\Project\merged_data_all_stocks\factors\bt_vol_target_Aprime_cost_NOTAX\summary_table.csv"
df = pd.read_csv(p)

cols = ["target_vol","CAGR","maxDD","vol_ann","sharpe_ann","calmar","avg_leverage","avg_cost_per_day","avg_est_vol","start","end"]
cols = [c for c in cols if c in df.columns]
print(df[cols].to_string(index=False))


 target_vol     CAGR     maxDD  vol_ann  sharpe_ann   calmar  avg_leverage  avg_cost_per_day  avg_est_vol      start        end
       0.18 0.113606 -0.181638 0.162854    0.742607 0.625450      0.795490          0.000186     0.233775 2017-02-01 2025-10-31
       0.20 0.119215 -0.194543 0.175319    0.730619 0.612793      0.852381          0.000191     0.233775 2017-02-01 2025-10-31
